In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, hashlib, csv, re, collections, datetime
import pandas as pd
ROOT = Path('/content/drive/MyDrive')
OUT = ROOT / 'MDAS_Registration_Aware_Revision_2026' / 'Step01_Audit'
OUT.mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| audit output:', OUT)
print('Python:', __import__('sys').version.split()[0])

In [ ]:

# =============================================================================
# STEP 1 — MDAS GOOGLE DRIVE INVENTORY AND EXPERIMENT RECOVERY
# Project: Hyperspectral–Multispectral Image Fusion under Misregistration
#
# PURPOSE:
# 1. Mount Google Drive.
# 2. Locate MDAS experiment directories.
# 3. Audit SCSRNet and MSDCNN across 5 regimes and 3 seeds.
# 4. Search for checkpoints, CSV metrics, configurations and scripts.
# 5. Check archive files that may contain earlier experiment outputs.
# 6. Save an inventory without changing original research data.
# =============================================================================

from google.colab import drive
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import json
import os
import re
import hashlib
import zipfile
import tarfile

# -------------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# -------------------------------------------------------------------------

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")

PROJECT_DIR = DRIVE_ROOT / "MDAS_Registration_Aware_Revision_2026"
OUTPUT_DIR = PROJECT_DIR / "Step01_Audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 85)
print("MDAS — COMPLETE GOOGLE DRIVE INVENTORY")
print("=" * 85)
print("Drive root:", DRIVE_ROOT)
print("Output directory:", OUTPUT_DIR)
print("Audit time (UTC):", datetime.now(timezone.utc).isoformat())

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive mount is not accessible.")

# -------------------------------------------------------------------------
# 2. IDENTIFY RELEVANT FILE TYPES
# -------------------------------------------------------------------------

MODEL_EXT = {".pt", ".pth", ".ckpt", ".safetensors", ".h5", ".hdf5"}
DATA_EXT = {".tif", ".tiff", ".npy", ".npz", ".mat", ".h5", ".hdf5"}
RESULT_EXT = {".csv", ".json", ".xlsx", ".xls", ".parquet"}
CODE_EXT = {".py", ".ipynb", ".yaml", ".yml", ".toml", ".sh"}
ARCHIVE_EXT = {".zip", ".tar", ".tgz", ".gz", ".7z", ".rar"}

RELEVANT_TERMS = [
    "mdas", "scsrnet", "msdcnn", "hyperspectral",
    "multispectral", "misregistration",
    "registration", "shift", "fusion"
]

def is_relevant_path(path):
    s = str(path).lower()
    return any(term in s for term in RELEVANT_TERMS)

def classify_file(path):
    suffix = path.suffix.lower()
    name = path.name.lower()

    if suffix in MODEL_EXT:
        return "checkpoint"
    if suffix in ARCHIVE_EXT:
        return "archive"
    if suffix in DATA_EXT:
        return "image_or_array"
    if suffix in RESULT_EXT:
        return "results_or_metadata"
    if suffix in CODE_EXT:
        return "code_or_configuration"
    if suffix in {".txt", ".log"}:
        if any(x in name for x in ["log", "train", "metric", "result"]):
            return "log"
        return "text"
    if suffix in {".pdf", ".docx", ".png", ".jpg", ".jpeg"}:
        return "document_or_figure"

    return "other"

# -------------------------------------------------------------------------
# 3. SCAN DRIVE METADATA
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("STEP 1A — SCANNING GOOGLE DRIVE")
print("=" * 85)

all_records = []
folders = []
errors = []
scanned = 0

# os.walk avoids opening or loading large image/model files.
for root, dirs, files in os.walk(DRIVE_ROOT, followlinks=False):

    root_path = Path(root)
    dirs.sort()
    files.sort()

    # Avoid accidentally scanning generated outputs from earlier audit runs.
    if root_path == PROJECT_DIR:
        dirs[:] = [d for d in dirs if d != "Step01_Audit"]

    if is_relevant_path(root_path):
        folders.append(str(root_path))

    for filename in files:
        scanned += 1
        path = root_path / filename

        if not is_relevant_path(path):
            continue

        try:
            stat = path.stat()
            size = stat.st_size
            modified = datetime.fromtimestamp(
                stat.st_mtime, timezone.utc
            ).isoformat()
        except Exception as exc:
            size = None
            modified = None
            errors.append({
                "path": str(path),
                "error": str(exc)
            })

        all_records.append({
            "name": filename,
            "path": str(path),
            "parent": str(root_path),
            "extension": path.suffix.lower(),
            "category": classify_file(path),
            "size_bytes": size,
            "size_mb": round(size / 1024**2, 3)
                       if size is not None else None,
            "modified_utc": modified
        })

    if scanned and scanned % 1000 == 0:
        print(
            f"\rScanned {scanned:,} files | "
            f"relevant files: {len(all_records):,}",
            end=""
        )

inventory = pd.DataFrame(all_records)

columns = [
    "name", "path", "parent", "extension", "category",
    "size_bytes", "size_mb", "modified_utc"
]

if inventory.empty:
    inventory = pd.DataFrame(columns=columns)

inventory.to_csv(
    OUTPUT_DIR / "Step01_All_Relevant_Files.csv",
    index=False
)

print("\n")
print("Total files scanned:", scanned)
print("Relevant files identified:", len(inventory))
print("Relevant folder paths:", len(folders))
print("Metadata errors:", len(errors))

print("\nFILES BY CATEGORY")
if not inventory.empty:
    print(inventory["category"].value_counts().to_string())
else:
    print("No matching files found.")

# -------------------------------------------------------------------------
# 4. LOCATE THE ORIGINAL EXPERIMENT FOLDER
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("STEP 1B — LOCATING EXISTING EXPERIMENTS")
print("=" * 85)

experiment_roots = [
    Path(p) for p in folders
    if Path(p).name.lower() == "mdas_jstars_expanded"
]

print("MDAS_JSTARS_Expanded folders:", len(experiment_roots))

for path in experiment_roots:
    print("FOUND:", path)

EXPECTED_MODELS = ["SCSRNet", "MSDCNN"]
EXPECTED_REGIMES = [
    "p00_none",
    "p20_one",
    "p20_one_two",
    "p50_one",
    "p50_one_two"
]
EXPECTED_SEEDS = [7, 17, 37]

experiment_records = []

for model in EXPECTED_MODELS:
    for regime in EXPECTED_REGIMES:
        for seed in EXPECTED_SEEDS:

            expected_name = f"{model}_{regime}_s{seed}"
            matches = [
                Path(p) for p in folders
                if Path(p).name.lower() == expected_name.lower()
            ]

            if not matches:
                # Some empty directories may not have been included
                # in the relevance scan, so check the known roots.
                for parent in experiment_roots:
                    candidate = parent / expected_name
                    if candidate.is_dir():
                        matches.append(candidate)

            for candidate in matches:
                relative_files = inventory[
                    inventory["path"].str.startswith(
                        str(candidate) + os.sep,
                        na=False
                    )
                ]

                n_checkpoints = int(
                    (relative_files["category"] == "checkpoint").sum()
                )
                n_results = int(
                    (relative_files["category"] == "results_or_metadata").sum()
                )
                n_logs = int(
                    (relative_files["category"] == "log").sum()
                )

                experiment_records.append({
                    "experiment": expected_name,
                    "model": model,
                    "regime": regime,
                    "seed": seed,
                    "folder_found": True,
                    "folder_path": str(candidate),
                    "n_files": len(relative_files),
                    "n_checkpoints": n_checkpoints,
                    "n_results": n_results,
                    "n_logs": n_logs,
                    "status": (
                        "CHECKPOINT_AND_RESULTS_FOUND"
                        if n_checkpoints and n_results
                        else "CHECKPOINT_FOUND"
                        if n_checkpoints
                        else "RESULTS_FOUND"
                        if n_results
                        else "NO_FILES_DETECTED"
                    )
                })

            if not matches:
                experiment_records.append({
                    "experiment": expected_name,
                    "model": model,
                    "regime": regime,
                    "seed": seed,
                    "folder_found": False,
                    "folder_path": "",
                    "n_files": 0,
                    "n_checkpoints": 0,
                    "n_results": 0,
                    "n_logs": 0,
                    "status": "FOLDER_NOT_FOUND"
                })

experiments = pd.DataFrame(experiment_records)

experiments.to_csv(
    OUTPUT_DIR / "Step01_Experiment_Completeness.csv",
    index=False
)

print(experiments[
    [
        "experiment",
        "n_files",
        "n_checkpoints",
        "n_results",
        "status"
    ]
].to_string(index=False))

# -------------------------------------------------------------------------
# 5. IDENTIFY CHECKPOINTS AND RESULT FILES ANYWHERE IN DRIVE
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("STEP 1C — MODEL CHECKPOINTS")
print("=" * 85)

checkpoints = inventory[
    inventory["category"] == "checkpoint"
].copy()

checkpoints.to_csv(
    OUTPUT_DIR / "Step01_Checkpoint_Inventory.csv",
    index=False
)

if not checkpoints.empty:
    print(checkpoints[
        ["name", "size_mb", "path"]
    ].to_string(index=False, max_colwidth=110))
else:
    print("No relevant checkpoints detected.")

print("\n" + "=" * 85)
print("STEP 1D — RESULT TABLES AND TRAINING LOGS")
print("=" * 85)

results = inventory[
    inventory["category"].isin([
        "results_or_metadata",
        "log"
    ])
].copy()

results.to_csv(
    OUTPUT_DIR / "Step01_Results_Inventory.csv",
    index=False
)

if not results.empty:
    print(results[
        ["name", "category", "path"]
    ].to_string(index=False, max_colwidth=110))
else:
    print("No matching results or logs detected.")

# -------------------------------------------------------------------------
# 6. INSPECT ARCHIVES WITHOUT EXTRACTING THEM
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("STEP 1E — CHECKING COMPRESSED ARCHIVES")
print("=" * 85)

archive_rows = []
archives = inventory[inventory["category"] == "archive"]

MAX_ARCHIVES_TO_INSPECT = 30
MAX_ARCHIVE_MEMBERS = 10000

for _, record in archives.head(MAX_ARCHIVES_TO_INSPECT).iterrows():
    path = Path(record["path"])

    row = {
        "archive": str(path),
        "status": "",
        "member_count": None,
        "checkpoint_members": 0,
        "result_members": 0,
        "sample_members": ""
    }

    try:
        names = []

        if zipfile.is_zipfile(path):
            with zipfile.ZipFile(path) as zf:
                names = zf.namelist()[:MAX_ARCHIVE_MEMBERS]

        elif tarfile.is_tarfile(path):
            with tarfile.open(path, "r:*") as tf:
                names = [
                    m.name for m in tf
                    if m.isfile()
                ][:MAX_ARCHIVE_MEMBERS]

        else:
            row["status"] = "UNSUPPORTED_ARCHIVE_FORMAT"

        if names:
            row["status"] = "INSPECTED"
            row["member_count"] = len(names)
            row["checkpoint_members"] = sum(
                Path(n).suffix.lower() in MODEL_EXT for n in names
            )
            row["result_members"] = sum(
                Path(n).suffix.lower() in RESULT_EXT for n in names
            )
            row["sample_members"] = "; ".join(names[:8])

    except Exception as exc:
        row["status"] = "INSPECTION_ERROR"
        row["sample_members"] = str(exc)[:200]

    archive_rows.append(row)

archive_df = pd.DataFrame(archive_rows)

archive_df.to_csv(
    OUTPUT_DIR / "Step01_Archive_Inventory.csv",
    index=False
)

if not archive_df.empty:
    print(archive_df.to_string(index=False, max_colwidth=90))
else:
    print("No relevant archives found.")

# -------------------------------------------------------------------------
# 7. DETECT POSSIBLE EVALUATION TABLES
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("STEP 1F — IDENTIFYING NUMERICAL EVALUATION FILES")
print("=" * 85)

candidate_keywords = [
    "psnr", "sam", "ergas", "ssim", "rmse",
    "metric", "evaluation", "summary", "result",
    "shift", "benchmark", "robust"
]

candidates = results[
    results["name"].str.lower().apply(
        lambda name: any(k in name for k in candidate_keywords)
    )
].copy()

candidate_rows = []

for _, record in candidates.iterrows():
    path = Path(record["path"])

    row = {
        "path": str(path),
        "extension": path.suffix.lower(),
        "rows": None,
        "columns": None,
        "column_names": "",
        "read_status": ""
    }

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path, nrows=100)

            row["rows"] = len(df)
            row["columns"] = len(df.columns)
            row["column_names"] = ", ".join(map(str, df.columns))
            row["read_status"] = "SAMPLE_READ_OK"

        elif path.suffix.lower() == ".json":
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)

            row["read_status"] = "JSON_READ_OK"
            if isinstance(obj, dict):
                row["column_names"] = ", ".join(
                    list(map(str, obj.keys()))[:30]
                )

        else:
            row["read_status"] = "NOT_SAMPLED"

    except Exception as exc:
        row["read_status"] = f"READ_ERROR: {str(exc)[:120]}"

    candidate_rows.append(row)

candidate_df = pd.DataFrame(candidate_rows)

candidate_df.to_csv(
    OUTPUT_DIR / "Step01_Evaluation_Candidates.csv",
    index=False
)

if not candidate_df.empty:
    print(candidate_df.to_string(index=False, max_colwidth=100))
else:
    print("No matching evaluation tables identified.")

# -------------------------------------------------------------------------
# 8. CREATE MACHINE-READABLE AUDIT SUMMARY
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("STEP 1G — SAVING AUDIT REPORT")
print("=" * 85)

summary = {
    "project": "MDAS Registration-Aware Fusion Revision",
    "audit_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "drive_root": str(DRIVE_ROOT),
    "total_files_scanned": scanned,
    "relevant_files": int(len(inventory)),
    "experiment_folders_expected": 30,
    "experiment_folders_found": int(experiments["folder_found"].sum()),
    "experiments_with_checkpoints": int(
        (experiments["n_checkpoints"] > 0).sum()
    ),
    "experiments_with_results": int(
        (experiments["n_results"] > 0).sum()
    ),
    "checkpoint_files_detected": int(len(checkpoints)),
    "result_or_log_files_detected": int(len(results)),
    "archive_files_detected": int(len(archives)),
    "evaluation_candidates_detected": int(len(candidate_df)),
    "scan_errors": errors[:100]
}

with open(
    OUTPUT_DIR / "Step01_Audit_Summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

with open(
    OUTPUT_DIR / "Step01_Relevant_Folders.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(sorted(set(folders))))

# -------------------------------------------------------------------------
# 9. FINAL PRINTED REPORT
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("MDAS INVENTORY — FINAL SUMMARY")
print("=" * 85)

for key, value in summary.items():
    if key != "scan_errors":
        print(f"{key}: {value}")

print("\nSAVED OUTPUTS:")
for path in sorted(OUTPUT_DIR.glob("Step01_*")):
    if path.is_file():
        print(" ", path.name)

print("\n" + "=" * 85)
print("STEP 1 COMPLETED")
print("=" * 85)

if summary["experiments_with_checkpoints"] > 0:
    print("Checkpoints found. Existing trained models may be reusable.")
else:
    print("No experiment checkpoints located in the scanned Drive paths.")

if summary["experiments_with_results"] > 0:
    print("Experiment results found. Numerical recovery can proceed.")
else:
    print("No experiment result files located in the scanned paths.")

print("\nNEXT: Examine the inventory before choosing new training runs.")


In [ ]:

# =============================================================================
# STEP 2 — MDAS MULTI-SEED EXPERIMENT VERIFICATION
# SCSRNet + MSDCNN | 5 regimes | 3 seeds
#
# READ-ONLY AUDIT OF ORIGINAL EXPERIMENT FILES
# No retraining, checkpoint modification, or deletion.
# =============================================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# -------------------------------------------------------------------------
# 1. CONNECT GOOGLE DRIVE
# -------------------------------------------------------------------------

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
SOURCE = ROOT / "MDAS_JSTARS_Expanded"

OUT = (
    ROOT /
    "MDAS_Registration_Aware_Revision_2026" /
    "Step02_Verification"
)
OUT.mkdir(parents=True, exist_ok=True)

if not SOURCE.is_dir():
    raise FileNotFoundError(f"Source directory missing: {SOURCE}")

TABLES = SOURCE / "tables"

print("=" * 85)
print("STEP 2 — MDAS MULTI-SEED VERIFICATION")
print("=" * 85)
print("Source:", SOURCE)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 2. READ EXISTING CONSOLIDATED TABLES
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("2A — LOADING EXPERIMENTAL RESULTS")
print("=" * 85)

table_names = [
    "condition_summary.csv",
    "per_seed_summary.csv",
    "paired_to_aligned.csv"
]

tables = {}

for name in table_names:
    path = TABLES / name

    if path.exists():
        df = pd.read_csv(path)
        tables[name] = df

        print(f"\n{name}")
        print("Shape:", df.shape)
        print("Columns:", list(df.columns))
        display(df.head(10))
    else:
        print("MISSING:", path)

status_path = TABLES / "status.json"

if status_path.exists():
    print("\nEXPERIMENT STATUS:")
    with open(status_path, encoding="utf-8") as f:
        status = json.load(f)
    print(json.dumps(status, indent=2, default=str)[:12000])

# -------------------------------------------------------------------------
# 3. VERIFY CHECKPOINTS, MANIFESTS, LOGS, TEST FILES
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("2B — VERIFYING ALL 30 EXPERIMENT DIRECTORIES")
print("=" * 85)

MODELS = ["SCSRNet", "MSDCNN"]
REGIMES = [
    "p00_none",
    "p20_one",
    "p20_one_two",
    "p50_one",
    "p50_one_two"
]
SEEDS = [7, 17, 37]

records = []
manifest_records = []
all_test_frames = []
issues = []

for model in MODELS:
    for regime in REGIMES:
        for seed in SEEDS:

            experiment = f"{model}_{regime}_s{seed}"
            folder = SOURCE / experiment

            required = [
                "best.pt",
                "last.pt",
                "epoch_log.csv",
                "manifest.json",
                "test_rows.csv"
            ]

            present = {
                name: (folder / name).is_file()
                for name in required
            }

            row = {
                "experiment": experiment,
                "model": model,
                "regime": regime,
                "seed": seed,
                "folder_exists": folder.is_dir(),
                "all_files_exist": all(present.values()),
                **{name.replace(".", "_"): value
                   for name, value in present.items()}
            }

            if not all(present.values()):
                issues.append({
                    "experiment": experiment,
                    "issue": "Missing required file",
                    "details": str(
                        [k for k, v in present.items() if not v]
                    )
                })

            # Manifest metadata: inspect without assuming its structure.
            manifest_path = folder / "manifest.json"

            if manifest_path.exists():
                try:
                    with open(manifest_path, encoding="utf-8") as f:
                        manifest = json.load(f)

                    manifest_records.append({
                        "experiment": experiment,
                        "model": model,
                        "regime": regime,
                        "seed": seed,
                        "manifest": manifest
                    })

                except Exception as exc:
                    issues.append({
                        "experiment": experiment,
                        "issue": "Manifest read failure",
                        "details": str(exc)
                    })

            # Training history
            log_path = folder / "epoch_log.csv"

            if log_path.exists():
                try:
                    log = pd.read_csv(log_path)
                    row["log_rows"] = len(log)
                    row["log_columns"] = ", ".join(log.columns)
                except Exception as exc:
                    issues.append({
                        "experiment": experiment,
                        "issue": "Training log read failure",
                        "details": str(exc)
                    })

            # Patch-level test results
            test_path = folder / "test_rows.csv"

            if test_path.exists():
                try:
                    test = pd.read_csv(test_path)

                    row["test_rows"] = len(test)
                    row["test_columns"] = ", ".join(test.columns)

                    test.insert(0, "audit_experiment", experiment)
                    test.insert(1, "audit_model", model)
                    test.insert(2, "audit_regime", regime)
                    test.insert(3, "audit_seed", seed)

                    all_test_frames.append(test)

                except Exception as exc:
                    issues.append({
                        "experiment": experiment,
                        "issue": "Test table read failure",
                        "details": str(exc)
                    })

            records.append(row)

verification = pd.DataFrame(records)

verification.to_csv(
    OUT / "Step02_Experiment_Verification.csv",
    index=False
)

display(
    verification[
        [
            "experiment",
            "all_files_exist",
            "log_rows",
            "test_rows"
        ]
    ]
)

print("\nExperiments checked:", len(verification))
print("Complete file sets:", verification["all_files_exist"].sum())

# -------------------------------------------------------------------------
# 4. INSPECT MANIFESTS AND EXPERIMENT SETTINGS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("2C — TRAINING CONFIGURATION AUDIT")
print("=" * 85)

if manifest_records:

    print("\nEXAMPLE MANIFEST — FIRST EXPERIMENT")
    print(
        json.dumps(
            manifest_records[0]["manifest"],
            indent=2,
            default=str
        )[:12000]
    )

    flat_manifests = []

    for rec in manifest_records:
        manifest = rec["manifest"]

        flat = {
            "experiment": rec["experiment"],
            "model": rec["model"],
            "regime": rec["regime"],
            "seed": rec["seed"]
        }

        if isinstance(manifest, dict):
            flat.update(pd.json_normalize(manifest, sep=".").iloc[0].to_dict())

        flat_manifests.append(flat)

    manifest_df = pd.DataFrame(flat_manifests)

    manifest_df.to_csv(
        OUT / "Step02_Manifest_Comparison.csv",
        index=False
    )

    print("\nMANIFEST FIELDS:")
    print(list(manifest_df.columns))

    print("\nFIELDS WITH DIFFERENT VALUES ACROSS RUNS:")

    for col in manifest_df.columns:
        if col == "experiment":
            continue

        values = manifest_df[col].apply(str).unique()

        if len(values) > 1:
            print(f"{col}: {len(values)} distinct values")

# -------------------------------------------------------------------------
# 5. CHECK PER-SEED RESULTS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("2D — MULTI-SEED NUMERICAL SUMMARY")
print("=" * 85)

seed_df = tables.get("per_seed_summary.csv")

if seed_df is not None:

    expected_columns = {
        "model", "frequency", "magnitude", "seed",
        "aligned_psnr", "shift_1_psnr",
        "shift_2_psnr", "shift_4_psnr"
    }

    missing = expected_columns - set(seed_df.columns)

    if missing:
        issues.append({
            "experiment": "consolidated",
            "issue": "Missing per-seed columns",
            "details": str(sorted(missing))
        })
        print("Missing columns:", sorted(missing))

    else:
        metrics = [
            "aligned_psnr",
            "shift_1_psnr",
            "shift_2_psnr",
            "shift_4_psnr"
        ]

        seed_df = seed_df.copy()

        for metric in metrics:
            seed_df[metric] = pd.to_numeric(
                seed_df[metric], errors="coerce"
            )

        group_cols = ["model", "frequency", "magnitude"]

        # Check expected three seeds per experimental condition.
        seed_counts = (
            seed_df.groupby(group_cols, dropna=False)
            .agg(
                n_rows=("seed", "size"),
                n_seeds=("seed", "nunique")
            )
            .reset_index()
        )

        print("\nSEED COMPLETENESS")
        display(seed_counts)

        if not (
            (seed_counts["n_rows"] == 3) &
            (seed_counts["n_seeds"] == 3)
        ).all():
            issues.append({
                "experiment": "consolidated",
                "issue": "Unexpected seed counts",
                "details": "Inspect Step02_Seed_Counts.csv"
            })

        seed_counts.to_csv(
            OUT / "Step02_Seed_Counts.csv",
            index=False
        )

        # Mean and between-seed standard deviation.
        group_summary = (
            seed_df.groupby(group_cols, dropna=False)[metrics]
            .agg(["mean", "std", "count"])
        )

        group_summary.columns = [
            f"{metric}_{stat}"
            for metric, stat in group_summary.columns
        ]

        group_summary = group_summary.reset_index()

        group_summary.to_csv(
            OUT / "Step02_Recomputed_Multiseed_Summary.csv",
            index=False
        )

        print("\nRECOMPUTED RESULTS")
        display(group_summary)

        # Training regimes compared with their own model baseline.
        baseline = seed_df[
            pd.to_numeric(seed_df["frequency"], errors="coerce") == 0
        ].copy()

        paired_rows = []

        for model in MODELS:

            model_data = seed_df[seed_df["model"] == model]
            model_base = baseline[baseline["model"] == model]

            for _, run in model_data.iterrows():

                if float(run["frequency"]) == 0:
                    continue

                match = model_base[
                    model_base["seed"].astype(str) == str(run["seed"])
                ]

                if len(match) != 1:
                    issues.append({
                        "experiment": model,
                        "issue": "Baseline seed match missing or ambiguous",
                        "details": str(run["seed"])
                    })
                    continue

                base = match.iloc[0]

                paired_row = {
                    "model": model,
                    "frequency": run["frequency"],
                    "magnitude": run["magnitude"],
                    "seed": run["seed"]
                }

                for metric in metrics:
                    paired_row[f"delta_{metric}"] = (
                        run[metric] - base[metric]
                    )

                paired_rows.append(paired_row)

        paired_seed = pd.DataFrame(paired_rows)

        paired_seed.to_csv(
            OUT / "Step02_Seed_Paired_Effects.csv",
            index=False
        )

        print("\nPAIRED EFFECTS RELATIVE TO SAME-SEED BASELINE")
        display(paired_seed)

# -------------------------------------------------------------------------
# 6. VERIFY PATCH-LEVEL TEST TABLES
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("2E — PATCH-LEVEL TEST DATA AUDIT")
print("=" * 85)

if all_test_frames:

    test_all = pd.concat(
        all_test_frames,
        ignore_index=True,
        sort=False
    )

    test_all.to_csv(
        OUT / "Step02_Combined_Test_Rows.csv",
        index=False
    )

    print("Total combined rows:", len(test_all))
    print("Columns:", list(test_all.columns))

    print("\nFIRST FIVE TEST RECORDS:")
    display(test_all.head())

    print("\nROWS BY EXPERIMENT:")
    counts = (
        test_all.groupby("audit_experiment")
        .size()
        .reset_index(name="n_rows")
    )
    display(counts)

    counts.to_csv(
        OUT / "Step02_Test_Row_Counts.csv",
        index=False
    )

    # Do not assume test_rows.csv contains one row per unique patch:
    # it may include multiple displacements and directions.
    possible_patch_columns = [
        "patch", "patch_id", "patch_name",
        "filename", "file", "name", "image_id"
    ]

    patch_col = next(
        (c for c in possible_patch_columns if c in test_all.columns),
        None
    )

    if patch_col is not None:
        patch_counts = (
            test_all.groupby("audit_experiment")[patch_col]
            .nunique()
            .reset_index(name="unique_patches")
        )

        print("\nUNIQUE PATCH COUNTS:")
        display(patch_counts)

        patch_counts.to_csv(
            OUT / "Step02_Unique_Test_Patches.csv",
            index=False
        )

    print("\nMISSING DATA BY COLUMN:")
    missing_values = (
        test_all.isna().sum()
        .sort_values(ascending=False)
    )
    print(missing_values.to_string())

# -------------------------------------------------------------------------
# 7. FIGURE — MULTI-SEED PSNR SENSITIVITY
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("2F — EXPLORATORY MULTI-SEED PSNR FIGURE")
print("=" * 85)

if seed_df is not None and expected_columns.issubset(seed_df.columns):

    plot_columns = [
        "aligned_psnr",
        "shift_1_psnr",
        "shift_2_psnr",
        "shift_4_psnr"
    ]

    distances = np.array([0, 1, 2, 4])

    for model in MODELS:

        sub = seed_df[seed_df["model"] == model]

        fig, ax = plt.subplots(figsize=(8.5, 5.5))

        for (frequency, magnitude), group in sub.groupby(
            ["frequency", "magnitude"], dropna=False
        ):

            values = group[plot_columns].to_numpy(dtype=float)

            means = np.nanmean(values, axis=0)

            stds = (
                np.nanstd(values, axis=0, ddof=1)
                if len(values) > 1
                else np.full(len(distances), np.nan)
            )

            label = f"{frequency}% / {magnitude}"

            ax.errorbar(
                distances,
                means,
                yerr=stds,
                marker="o",
                capsize=3,
                linewidth=1.7,
                label=label
            )

        ax.set_xlabel("HR-MSI displacement (pixels)")
        ax.set_ylabel("PSNR (dB)")
        ax.set_xticks(distances)
        ax.grid(alpha=0.2)
        ax.legend(
            title="Training frequency / magnitude",
            fontsize=8,
            frameon=False
        )

        fig.tight_layout()

        fig.savefig(
            OUT / f"Step02_{model}_Multiseed_PSNR.png",
            dpi=900,
            bbox_inches="tight"
        )

        fig.savefig(
            OUT / f"Step02_{model}_Multiseed_PSNR.pdf",
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)

# -------------------------------------------------------------------------
# 8. FINAL VERIFICATION REPORT
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("2G — FINAL VERIFICATION SUMMARY")
print("=" * 85)

issue_df = pd.DataFrame(
    issues,
    columns=["experiment", "issue", "details"]
)

issue_df.to_csv(
    OUT / "Step02_Verification_Issues.csv",
    index=False
)

report = {
    "experiment_folders_checked": len(verification),
    "complete_file_sets": int(
        verification["all_files_exist"].sum()
    ),
    "manifest_files_read": len(manifest_records),
    "test_tables_read": len(all_test_frames),
    "numerical_tables_loaded": list(tables.keys()),
    "issues_detected": len(issues),
    "output_folder": str(OUT)
}

with open(
    OUT / "Step02_Verification_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

for key, value in report.items():
    print(f"{key}: {value}")

if issues:
    print("\nISSUES REQUIRING REVIEW:")
    display(issue_df)
else:
    print("\nNo structural issues detected by these checks.")

print("\n" + "=" * 85)
print("STEP 2 FINISHED")
print("=" * 85)
print("Original experiment files were not modified.")
print("Do not interpret exploratory plots as final manuscript results")
print("until evaluation definitions and test independence are verified.")


In [ ]:
# =============================================================================
# STEP 3 — MDAS NUMERICAL INTEGRITY AND ARCHITECTURE COMPARISON
# Read-only source data; outputs saved to new Step03 directory
# =============================================================================

from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
SOURCE = ROOT / "MDAS_JSTARS_Expanded"
OUT = ROOT / "MDAS_Registration_Aware_Revision_2026" / "Step03_Integrity"
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("STEP 3 — NUMERICAL INTEGRITY AND ARCHITECTURE COMPARISON")
print("=" * 80)

MODELS = ["SCSRNet", "MSDCNN"]
REGIMES = ["p00_none", "p20_one", "p20_one_two",
           "p50_one", "p50_one_two"]
SEEDS = [7, 17, 37]
METRICS = ["aligned", "shift_1", "shift_2", "shift_4"]

# -------------------------------------------------------------------------
# 1. RECONSTRUCT THE COMPLETE TEST DATASET
# -------------------------------------------------------------------------

frames = []

for model in MODELS:
    for regime in REGIMES:
        for seed in SEEDS:
            run = f"{model}_{regime}_s{seed}"
            path = SOURCE / run / "test_rows.csv"

            if not path.is_file():
                raise FileNotFoundError(path)

            df = pd.read_csv(path)

            required = {
                "image", "dy", "dx", "common_psnr",
                "model", "seed", "frequency", "magnitude"
            }

            missing = required - set(df.columns)
            if missing:
                raise ValueError(f"{run}: Missing columns {missing}")

            if len(df) != 520:
                raise ValueError(f"{run}: Expected 520 rows; got {len(df)}")

            df["run"] = run
            df["regime"] = regime
            frames.append(df)

data = pd.concat(frames, ignore_index=True)

for col in ["dy", "dx", "common_psnr"]:
    data[col] = pd.to_numeric(data[col], errors="raise")

data["distance"] = data["dy"].abs() + data["dx"].abs()

expected_offsets = {
    (0, 0),
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (2, 0), (-2, 0), (0, 2), (0, -2),
    (4, 0), (-4, 0), (0, 4), (0, -4)
}

assert len(data) == 15600
assert data["common_psnr"].notna().all()
assert np.isfinite(data["common_psnr"]).all()

print("\nTotal test observations:", len(data))
print("Experiment runs:", data["run"].nunique())
print("Unique image names:", data["image"].nunique())

# -------------------------------------------------------------------------
# 2. VERIFY EXACTLY 40 PATCHES AND 13 SHIFTS PER RUN
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3A — TEST PROTOCOL VERIFICATION")
print("=" * 80)

protocol_rows = []

reference_images = None

for run, group in data.groupby("run"):

    images = set(group["image"].astype(str).unique())

    if reference_images is None:
        reference_images = images

    same_images = images == reference_images

    patch_valid = True

    for image, patch in group.groupby("image"):
        offsets = set(
            zip(patch["dy"].astype(int),
                patch["dx"].astype(int))
        )

        if offsets != expected_offsets or len(patch) != 13:
            patch_valid = False
            break

    duplicate_count = int(group.duplicated(
        ["image", "dy", "dx"]
    ).sum())

    protocol_rows.append({
        "run": run,
        "rows": len(group),
        "patches": len(images),
        "same_patch_names": same_images,
        "exact_13_offsets": patch_valid,
        "duplicate_keys": duplicate_count
    })

protocol = pd.DataFrame(protocol_rows)
protocol.to_csv(OUT / "Protocol_Verification.csv", index=False)

print(protocol.to_string(index=False))

if not (
    (protocol["patches"] == 40)
    & protocol["same_patch_names"]
    & protocol["exact_13_offsets"]
    & (protocol["duplicate_keys"] == 0)
).all():
    raise AssertionError("Test protocol inconsistencies detected.")

print("\nPASS: All runs use the same 40 patch names and 13 offsets.")

# -------------------------------------------------------------------------
# 3. RECOMPUTE THE SEED-LEVEL PSNR METRICS
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3B — RAW-ROW TO SUMMARY RECONCILIATION")
print("=" * 80)

# Unit of spatial comparison = image/patch.
# Four directions are averaged within each patch at nonzero distances.
patch = (
    data.groupby(
        ["run", "model", "regime", "seed", "image", "distance"],
        as_index=False
    )["common_psnr"]
    .mean()
)

seed_results = (
    patch.groupby(
        ["run", "model", "regime", "seed", "distance"],
        as_index=False
    )["common_psnr"]
    .mean()
)

seed_wide = seed_results.pivot(
    index=["run", "model", "regime", "seed"],
    columns="distance",
    values="common_psnr"
).reset_index()

seed_wide = seed_wide.rename(columns={
    0: "aligned_psnr",
    1: "shift_1_psnr",
    2: "shift_2_psnr",
    4: "shift_4_psnr"
})

original = pd.read_csv(
    SOURCE / "tables" / "per_seed_summary.csv"
)

merged = seed_wide.merge(
    original,
    left_on="run",
    right_on="id",
    suffixes=("_recomputed", "_original"),
    validate="one_to_one"
)

reconciliation = []

for metric in [
    "aligned_psnr", "shift_1_psnr",
    "shift_2_psnr", "shift_4_psnr"
]:
    diff = (
        merged[f"{metric}_recomputed"]
        - merged[f"{metric}_original"]
    ).abs()

    reconciliation.append({
        "metric": metric,
        "max_absolute_difference": float(diff.max()),
        "mean_absolute_difference": float(diff.mean())
    })

reconciliation = pd.DataFrame(reconciliation)
reconciliation.to_csv(
    OUT / "Numerical_Reconciliation.csv", index=False
)

print(reconciliation.to_string(index=False))

if reconciliation["max_absolute_difference"].max() > 1e-5:
    raise AssertionError(
        "Raw-row reconstruction disagrees with saved summaries."
    )

print("\nPASS: Numerical summaries agree with the underlying test rows.")

# -------------------------------------------------------------------------
# 4. SAME-SEED, SAME-PATCH PAIRED AUGMENTATION EFFECTS
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3C — PAIRED AUGMENTATION EFFECTS")
print("=" * 80)

baseline = patch[
    patch["regime"] == "p00_none"
][["model", "seed", "image", "distance", "common_psnr"]]

baseline = baseline.rename(
    columns={"common_psnr": "baseline_psnr"}
)

paired = patch.merge(
    baseline,
    on=["model", "seed", "image", "distance"],
    how="left",
    validate="many_to_one"
)

assert paired["baseline_psnr"].notna().all()

paired["delta_psnr"] = (
    paired["common_psnr"] - paired["baseline_psnr"]
)

paired.to_csv(
    OUT / "Patch_Level_Paired_Effects.csv", index=False
)

# Average over patches first, retaining independent training seeds.
seed_effect = (
    paired.groupby(
        ["model", "regime", "seed", "distance"],
        as_index=False
    )["delta_psnr"]
    .mean()
)

effect_summary = (
    seed_effect.groupby(["model", "regime", "distance"])
    ["delta_psnr"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

effect_summary.to_csv(
    OUT / "Paired_Seed_Effect_Summary.csv", index=False
)

print(effect_summary.round(4).to_string(index=False))

# -------------------------------------------------------------------------
# 5. MATCHED ARCHITECTURE COMPARISON
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3D — MATCHED SCSRNET VS MSDCNN")
print("=" * 80)

arch = patch.pivot(
    index=["regime", "seed", "image", "distance"],
    columns="model",
    values="common_psnr"
).reset_index()

assert arch[MODELS].notna().all().all()

arch["scsrnet_minus_msdcnn"] = (
    arch["SCSRNet"] - arch["MSDCNN"]
)

arch.to_csv(
    OUT / "Matched_Architecture_Comparison.csv",
    index=False
)

arch_seed = (
    arch.groupby(["regime", "seed", "distance"])
    ["scsrnet_minus_msdcnn"]
    .mean()
    .reset_index()
)

arch_summary = (
    arch_seed.groupby(["regime", "distance"])
    ["scsrnet_minus_msdcnn"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

print(arch_summary.round(4).to_string(index=False))

arch_summary.to_csv(
    OUT / "Architecture_Comparison_Summary.csv",
    index=False
)

# -------------------------------------------------------------------------
# 6. CORRECTED FIGURES
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3E — GENERATING COMPARISON FIGURES")
print("=" * 80)

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "legend.fontsize": 9,
    "savefig.dpi": 900
})

labels = {
    "p00_none": "0% (aligned)",
    "p20_one": "20% (1 px)",
    "p20_one_two": "20% (1–2 px)",
    "p50_one": "50% (1 px)",
    "p50_one_two": "50% (1–2 px)"
}

for model in MODELS:

    subset = seed_wide[seed_wide["model"] == model]

    fig, ax = plt.subplots(figsize=(8.2, 5.2))

    for regime in REGIMES:

        runs = subset[subset["regime"] == regime]

        vals = runs[
            ["aligned_psnr", "shift_1_psnr",
             "shift_2_psnr", "shift_4_psnr"]
        ].to_numpy(float)

        means = vals.mean(axis=0)
        sds = vals.std(axis=0, ddof=1)

        ax.errorbar(
            [0, 1, 2, 4], means, yerr=sds,
            marker="o", linewidth=1.8,
            capsize=3, label=labels[regime]
        )

    ax.set_xlabel("HR-MSI displacement (pixels)")
    ax.set_ylabel("PSNR (dB)")
    ax.set_xticks([0, 1, 2, 4])
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, loc="best")

    fig.tight_layout()

    for ext in ["png", "pdf"]:
        fig.savefig(
            OUT / f"{model}_Verified_Sensitivity.{ext}",
            dpi=900, bbox_inches="tight"
        )

    plt.show()
    plt.close(fig)

# Architecture difference for the no-augmentation regime.
sub = arch_summary[
    arch_summary["regime"] == "p00_none"
].sort_values("distance")

fig, ax = plt.subplots(figsize=(7.5, 4.7))

ax.errorbar(
    sub["distance"],
    sub["mean"],
    yerr=sub["std"],
    marker="o",
    linewidth=2,
    capsize=4
)

ax.axhline(0, linewidth=1, linestyle="--", color="gray")
ax.set_xlabel("HR-MSI displacement (pixels)")
ax.set_ylabel("SCSRNet − MSDCNN PSNR (dB)")
ax.set_xticks([0, 1, 2, 4])
ax.grid(alpha=0.2)

fig.tight_layout()

for ext in ["png", "pdf"]:
    fig.savefig(
        OUT / f"Architecture_Difference.{ext}",
        dpi=900, bbox_inches="tight"
    )

plt.show()
plt.close(fig)

# -------------------------------------------------------------------------
# 7. SUMMARY
# -------------------------------------------------------------------------

report = {
    "observations": len(data),
    "runs": data["run"].nunique(),
    "unique_test_images": data["image"].nunique(),
    "protocol_valid": True,
    "maximum_summary_discrepancy": float(
        reconciliation["max_absolute_difference"].max()
    ),
    "seed_repetitions_per_condition": 3,
    "output_directory": str(OUT),
    "limitations": [
        "Three seeds do not establish broad training reproducibility.",
        "Patch independence requires geographic footprint verification.",
        "Synthetic shifts do not constitute real sensor misregistration.",
        "Architectural differences require implementation equivalence audit."
    ]
}

with open(OUT / "Step03_Report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 80)
print("STEP 3 — FINAL SUMMARY")
print("=" * 80)

print(json.dumps(report, indent=2))
print("\nResults saved successfully.")

In [ ]:
# =============================================================================
# STEP 4 — MDAS MODEL AND DATA READINESS AUDIT
# Purpose:
#   1. Inspect existing SCSRNet and MSDCNN checkpoints.
#   2. Verify training configurations and repository identifiers.
#   3. Locate MDAS image files and model source code.
#   4. Identify what is available for registration-aware experiments.
#
# Does not train models, alter checkpoints, or modify original files.
# =============================================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import json
import os
import re
from collections import Counter

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
SOURCE = ROOT / "MDAS_JSTARS_Expanded"
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
OUT = PROJECT / "Step04_Model_Readiness"

OUT.mkdir(parents=True, exist_ok=True)

print("=" * 85)
print("STEP 4 — REGISTRATION-AWARE FUSION READINESS AUDIT")
print("=" * 85)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Source:", SOURCE)

# -------------------------------------------------------------------------
# 1. CHECK ORIGINAL EXPERIMENTS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("4A — ORIGINAL EXPERIMENT CONFIGURATIONS")
print("=" * 85)

MODELS = ["SCSRNet", "MSDCNN"]
SEEDS = [7, 17, 37]
REGIMES = [
    "p00_none",
    "p20_one",
    "p20_one_two",
    "p50_one",
    "p50_one_two"
]

manifest_records = []
errors = []

for model in MODELS:
    for regime in REGIMES:
        for seed in SEEDS:

            run = f"{model}_{regime}_s{seed}"
            path = SOURCE / run / "manifest.json"

            if not path.exists():
                errors.append(f"Missing manifest: {run}")
                continue

            with open(path, encoding="utf-8") as f:
                manifest = json.load(f)

            spec = manifest.get("spec", {})

            manifest_records.append({
                "run": run,
                "model": model,
                "regime": regime,
                "seed": seed,
                "fingerprint": manifest.get("fingerprint"),
                "repo_head": manifest.get("repo_head"),
                "training_patches": manifest.get("train"),
                "validation_patches": manifest.get("val"),
                "test_patches": manifest.get("test"),
                "best_epoch": manifest.get("best_epoch"),
                "best_val_psnr": manifest.get("best_val_psnr"),
                "spec_model": spec.get("model"),
                "spec_seed": spec.get("seed"),
                "spec_frequency": spec.get("frequency"),
                "spec_magnitude": spec.get("magnitude")
            })

manifest_df = pd.DataFrame(manifest_records)

manifest_df.to_csv(
    OUT / "Step04_Manifest_Audit.csv",
    index=False
)

print("Manifests found:", len(manifest_df))
print("Repository commits:", manifest_df["repo_head"].nunique())
print("Unique fingerprints:", manifest_df["fingerprint"].nunique())

print("\nBEST CHECKPOINT EPOCHS")
print(
    manifest_df.groupby(["model", "regime"])
    ["best_epoch"]
    .agg(["min", "max", "mean"])
    .to_string()
)

# -------------------------------------------------------------------------
# 2. CHECK CHECKPOINT CONTENT
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("4B — MODEL CHECKPOINT INSPECTION")
print("=" * 85)

checkpoint_records = []
checkpoint_details = {}

# Inspect representative best checkpoints for both architectures
# under aligned and strongly augmented conditions.
selected_runs = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

for run in selected_runs:

    path = SOURCE / run / "best.pt"

    print("\n" + "-" * 75)
    print("CHECKPOINT:", run)

    if not path.exists():
        print("MISSING")
        errors.append(f"Missing checkpoint: {run}")
        continue

    print("Size:", round(path.stat().st_size / 1024**2, 3), "MiB")

    try:
        # Safe tensor-oriented loading; checkpoint is never executed.
        obj = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        if not isinstance(obj, dict):
            print("Unexpected checkpoint object:", type(obj))
            errors.append(f"Non-dictionary checkpoint: {run}")
            continue

        print("Top-level keys:", list(obj.keys())[:30])

        # Common checkpoint layouts.
        state = None
        state_source = None

        for key in [
            "model_state_dict",
            "state_dict",
            "model",
            "net",
            "network",
            "weights"
        ]:
            candidate = obj.get(key)

            if isinstance(candidate, dict):
                if any(
                    isinstance(v, torch.Tensor)
                    for v in candidate.values()
                ):
                    state = candidate
                    state_source = key
                    break

        if state is None and all(
            isinstance(v, torch.Tensor)
            for v in obj.values()
        ):
            state = obj
            state_source = "top_level"

        if state is None:
            print("No standard tensor state dictionary identified.")
            print("Inspect top-level checkpoint structure manually.")
            errors.append(f"State dictionary not identified: {run}")
            continue

        tensor_items = {
            k: v for k, v in state.items()
            if isinstance(v, torch.Tensor)
        }

        parameter_count = sum(
            v.numel() for v in tensor_items.values()
        )

        print("State source:", state_source)
        print("Tensor entries:", len(tensor_items))
        print("Stored tensor elements:", f"{parameter_count:,}")

        print("\nFIRST 20 TENSORS")

        tensor_preview = []

        for key, tensor in list(tensor_items.items())[:20]:
            shape = list(tensor.shape)
            print(f"{key}: {shape}")

            tensor_preview.append({
                "name": key,
                "shape": shape,
                "dtype": str(tensor.dtype)
            })

        checkpoint_details[run] = {
            "state_source": state_source,
            "tensor_count": len(tensor_items),
            "stored_tensor_elements": parameter_count,
            "first_tensors": tensor_preview
        }

        checkpoint_records.append({
            "run": run,
            "checkpoint_size_mb":
                round(path.stat().st_size / 1024**2, 3),
            "state_source": state_source,
            "tensor_entries": len(tensor_items),
            "stored_tensor_elements": parameter_count,
            "status": "READ_OK"
        })

        del obj, state

    except Exception as exc:
        print("Checkpoint inspection failed:", str(exc)[:500])
        errors.append(f"{run}: {exc}")

pd.DataFrame(checkpoint_records).to_csv(
    OUT / "Step04_Checkpoint_Summary.csv",
    index=False
)

with open(OUT / "Step04_Checkpoint_Details.json", "w") as f:
    json.dump(checkpoint_details, f, indent=2)

# -------------------------------------------------------------------------
# 3. LOCATE RELEVANT SOURCE CODE AND IMAGE DATA
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("4C — SEARCHING FOR DATASETS AND MODEL SOURCE CODE")
print("=" * 85)

# Search targeted project directories rather than rescanning
# all 119,081 files in Google Drive.
search_roots = []

for root in [ROOT, Path("/content")]:
    if not root.exists():
        continue

    try:
        children = list(root.iterdir())
    except Exception:
        continue

    for path in children:

        if root == Path("/content") and path.name == "drive":
            continue

        if not path.is_dir():
            continue

        name = path.name.lower()

        if any(term in name for term in [
            "mdas", "scsr", "msdcnn", "fusion",
            "hyperspectral", "registration"
        ]):
            search_roots.append(path)

# Always include the known experiment directory.
search_roots.append(SOURCE)

search_roots = list(dict.fromkeys(search_roots))

print("\nTARGETED SEARCH ROOTS")
for path in search_roots:
    print(" ", path)

code_ext = {".py", ".ipynb", ".yaml", ".yml"}
image_ext = {".tif", ".tiff", ".mat", ".npy", ".npz", ".h5"}

found_code = []
found_data = []
found_lists = []

MAX_DEPTH = 5

for base in search_roots:

    if not base.exists():
        continue

    for current, dirs, files in os.walk(base):

        current_path = Path(current)
        relative_depth = len(current_path.relative_to(base).parts)

        if relative_depth >= MAX_DEPTH:
            dirs[:] = []

        dirs[:] = [
            d for d in dirs
            if d not in {
                ".git", "__pycache__", ".ipynb_checkpoints",
                "node_modules"
            }
        ]

        for filename in files:

            path = current_path / filename
            ext = path.suffix.lower()
            name = filename.lower()

            if ext in code_ext:
                if any(x in str(path).lower() for x in [
                    "scsr", "msdcnn", "mdas", "fusion",
                    "model", "network", "dataset"
                ]):
                    found_code.append(str(path))

            if ext in image_ext:
                found_data.append(str(path))

            if ext in {".txt", ".csv", ".json"}:
                if any(x in name for x in [
                    "train", "test", "val", "split",
                    "patch", "config", "dataset"
                ]):
                    found_lists.append(str(path))

print("\nMODEL / DATASET SOURCE FILES:", len(found_code))
for p in found_code[:40]:
    print(" ", p)

print("\nIMAGE / ARRAY FILES:", len(found_data))
for p in found_data[:30]:
    print(" ", p)

print("\nSPLIT / CONFIGURATION FILES:", len(found_lists))
for p in found_lists[:30]:
    print(" ", p)

# -------------------------------------------------------------------------
# 4. CHECK IMAGE DATA METADATA WITHOUT LOADING FULL ARRAYS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("4D — IMAGE DATA READINESS")
print("=" * 85)

image_records = []

try:
    import tifffile
    tif_available = True
except ImportError:
    tif_available = False

for filename in found_data[:30]:

    path = Path(filename)

    row = {
        "path": filename,
        "extension": path.suffix.lower(),
        "size_mb": round(path.stat().st_size / 1024**2, 3),
        "shape": None,
        "dtype": None,
        "status": "NOT_INSPECTED"
    }

    try:
        if path.suffix.lower() in {".tif", ".tiff"} and tif_available:
            with tifffile.TiffFile(path) as tif:
                series = tif.series[0]
                row["shape"] = str(series.shape)
                row["dtype"] = str(series.dtype)
                row["status"] = "METADATA_OK"

        elif path.suffix.lower() == ".npy":
            arr = np.load(path, mmap_mode="r", allow_pickle=False)
            row["shape"] = str(arr.shape)
            row["dtype"] = str(arr.dtype)
            row["status"] = "METADATA_OK"
            del arr

        else:
            row["status"] = "FORMAT_NOT_INSPECTED"

    except Exception as exc:
        row["status"] = "READ_ERROR: " + str(exc)[:150]

    image_records.append(row)

image_df = pd.DataFrame(image_records)

image_df.to_csv(
    OUT / "Step04_Image_Metadata.csv",
    index=False
)

if not image_df.empty:
    print(image_df.to_string(index=False, max_colwidth=85))
else:
    print("No MDAS image files located in targeted directories.")

# -------------------------------------------------------------------------
# 5. FINAL READINESS REPORT
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("STEP 4 — FINAL READINESS REPORT")
print("=" * 85)

report = {
    "manifests_found": len(manifest_df),
    "repository_commits": int(manifest_df["repo_head"].nunique()),
    "checkpoints_inspected": len(checkpoint_records),
    "source_code_files_found": len(found_code),
    "image_files_found": len(found_data),
    "split_or_config_files_found": len(found_lists),
    "search_roots": [str(p) for p in search_roots],
    "errors": errors
}

with open(OUT / "Step04_Readiness_Report.json", "w") as f:
    json.dump(report, f, indent=2)

for key, value in report.items():
    print(f"{key}: {value}")

print("\nOutputs saved to:", OUT)
print("\nSTEP 4 COMPLETE")
print("Original experimental data remain unchanged.")

In [ ]:

# =============================================================================
# STEP 5 — SCSRNET SOURCE RECOVERY AND MDAS DATASET AUDIT
# Read-only checks against existing experiments.
# New repository copy saved separately in Google Drive.
# =============================================================================

from google.colab import drive
from pathlib import Path
import subprocess
import json
import os
import hashlib
import shutil
import torch
import pandas as pd
import tifffile
from collections import Counter

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
SOURCE = ROOT / "MDAS_JSTARS_Expanded"
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
OUT = PROJECT / "Step05_Source_Recovery"
REPO_DIR = OUT / "SCSRNet_Repository"

OUT.mkdir(parents=True, exist_ok=True)

print("=" * 85)
print("STEP 5 — SCSRNET SOURCE AND MDAS DATASET RECOVERY")
print("=" * 85)

# -------------------------------------------------------------------------
# 1. READ EXACT ORIGINAL REPOSITORY COMMIT
# -------------------------------------------------------------------------

manifest_path = (
    SOURCE / "SCSRNet_p00_none_s7" / "manifest.json"
)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)

RECORDED_COMMIT = manifest["repo_head"]
REPO_URL = "https://github.com/zsspo/SCSRNet.git"

print("\nRecorded commit:", RECORDED_COMMIT)
print("Repository:", REPO_URL)

def run_command(args, cwd=None, check=True):
    p = subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True
    )

    if p.stdout.strip():
        print(p.stdout[-5000:])

    if p.stderr.strip():
        print(p.stderr[-2000:])

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {args}\n"
            f"{p.stderr[-2000:]}"
        )

    return p

# -------------------------------------------------------------------------
# 2. CLONE REPOSITORY WITHOUT OVERWRITING AN EXISTING COPY
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("5A — RECOVERING PUBLIC MODEL REPOSITORY")
print("=" * 85)

if not REPO_DIR.exists():

    run_command([
        "git", "clone",
        REPO_URL,
        str(REPO_DIR)
    ])

else:
    print("Repository directory already exists:", REPO_DIR)

    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            "Existing directory is not a Git repository. "
            "No files were overwritten."
        )

# Fetch commit history without modifying the working tree.
run_command([
    "git", "-C", str(REPO_DIR),
    "fetch", "origin"
], check=False)

head = run_command([
    "git", "-C", str(REPO_DIR),
    "rev-parse", "HEAD"
]).stdout.strip()

print("\nCurrent repository HEAD:", head)

commit_check = subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "cat-file", "-e",
        f"{RECORDED_COMMIT}^{{commit}}"
    ],
    capture_output=True,
    text=True
)

commit_available = commit_check.returncode == 0

print("Original recorded commit available:", commit_available)

if commit_available:

    # Preserve the clone's default branch by creating a separate worktree.
    EXACT_DIR = OUT / "SCSRNet_Recorded_Commit"

    if not EXACT_DIR.exists():
        run_command([
            "git", "-C", str(REPO_DIR),
            "worktree", "add",
            "--detach",
            str(EXACT_DIR),
            RECORDED_COMMIT
        ])

    ACTIVE_REPO = EXACT_DIR

else:
    ACTIVE_REPO = REPO_DIR

    print("\nWARNING:")
    print("The original recorded commit is not available locally.")
    print("Current repository code may differ from training code.")
    print("No equivalence claim will be made.")

active_head = run_command([
    "git", "-C", str(ACTIVE_REPO),
    "rev-parse", "HEAD"
]).stdout.strip()

print("\nActive source commit:", active_head)

# -------------------------------------------------------------------------
# 3. IDENTIFY SOURCE FILES AND MDAS DATASETS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("5B — REPOSITORY CONTENTS")
print("=" * 85)

code_files = []
data_files = []
split_files = []

for root, dirs, files in os.walk(ACTIVE_REPO):

    dirs[:] = [
        d for d in dirs
        if d not in {".git", "__pycache__", ".ipynb_checkpoints"}
    ]

    for name in files:

        path = Path(root) / name
        ext = path.suffix.lower()

        if ext in {".py", ".ipynb", ".yaml", ".yml"}:
            code_files.append(path)

        if ext in {
            ".tif", ".tiff", ".mat",
            ".npy", ".npz", ".h5", ".hdf5"
        }:
            data_files.append(path)

        if ext in {".txt", ".json", ".csv"}:
            if any(x in name.lower() for x in [
                "train", "test", "val", "split",
                "config", "mdas"
            ]):
                split_files.append(path)

print("\nPython and configuration files:", len(code_files))
for path in code_files[:70]:
    print(" ", path.relative_to(ACTIVE_REPO))

print("\nImage/array files:", len(data_files))
for path in data_files[:35]:
    print(" ", path.relative_to(ACTIVE_REPO))

print("\nPotential data partitions/configurations:", len(split_files))
for path in split_files[:35]:
    print(" ", path.relative_to(ACTIVE_REPO))

# -------------------------------------------------------------------------
# 4. INSPECT MDAS DATA FILE METADATA
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("5C — MDAS IMAGE DIMENSION AUDIT")
print("=" * 85)

image_records = []

for path in data_files:

    row = {
        "path": str(path),
        "relative_path": str(path.relative_to(ACTIVE_REPO)),
        "extension": path.suffix.lower(),
        "size_mb": round(path.stat().st_size / 1024**2, 3),
        "shape": "",
        "dtype": "",
        "status": ""
    }

    try:

        if path.suffix.lower() in {".tif", ".tiff"}:

            with tifffile.TiffFile(path) as tif:
                series = tif.series[0]
                row["shape"] = str(series.shape)
                row["dtype"] = str(series.dtype)
                row["status"] = "OK"

        elif path.suffix.lower() == ".npy":

            arr = __import__("numpy").load(
                path,
                mmap_mode="r",
                allow_pickle=False
            )

            row["shape"] = str(arr.shape)
            row["dtype"] = str(arr.dtype)
            row["status"] = "OK"

        else:
            row["status"] = "FORMAT_NOT_INSPECTED"

    except Exception as exc:
        row["status"] = f"ERROR: {str(exc)[:150]}"

    image_records.append(row)

image_df = pd.DataFrame(image_records)

image_df.to_csv(
    OUT / "Step05_Repository_Data_Inventory.csv",
    index=False
)

print("\nIMAGE METADATA SAMPLE")

if not image_df.empty:
    print(
        image_df.head(40).to_string(
            index=False,
            max_colwidth=90
        )
    )
else:
    print("No image arrays detected in the repository.")

# -------------------------------------------------------------------------
# 5. EXTRACT MODEL SOURCE REFERENCES
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("5D — LOCATING NETWORK DEFINITIONS")
print("=" * 85)

keywords = [
    "SCSRNet",
    "MSDCNN",
    "DynamicFusionGate",
    "SpatialConditioned",
    "mdasn_dataset",
    "def forward",
    "class "
]

source_matches = []

for path in code_files:

    if path.suffix.lower() != ".py":
        continue

    try:
        lines = path.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()

        for number, line in enumerate(lines, 1):

            if any(k.lower() in line.lower() for k in keywords):

                source_matches.append({
                    "file": str(path.relative_to(ACTIVE_REPO)),
                    "line": number,
                    "text": line.strip()[:250]
                })

    except Exception:
        pass

matches_df = pd.DataFrame(source_matches)

matches_df.to_csv(
    OUT / "Step05_Model_Source_Locations.csv",
    index=False
)

print("\nMATCHED SOURCE LINES")

if not matches_df.empty:
    print(
        matches_df.head(100).to_string(
            index=False,
            max_colwidth=110
        )
    )

# -------------------------------------------------------------------------
# 6. CHECK WEIGHTS AGAINST SOURCE ARCHITECTURE INDICATORS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("5E — CHECKPOINT STRUCTURE COMPARISON")
print("=" * 85)

checkpoint_rows = []

for model in ["SCSRNet", "MSDCNN"]:

    for regime in ["p00_none", "p50_one_two"]:

        run = f"{model}_{regime}_s7"
        path = SOURCE / run / "best.pt"

        if not path.exists():
            continue

        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        state = checkpoint.get("model", checkpoint)

        if not isinstance(state, dict):
            raise TypeError(f"Invalid model state: {run}")

        tensors = {
            k: v for k, v in state.items()
            if isinstance(v, torch.Tensor)
        }

        # Buffers are included, so this is not strictly the
        # trainable parameter count.
        stored_elements = sum(
            tensor.numel() for tensor in tensors.values()
        )

        checkpoint_rows.append({
            "run": run,
            "model": model,
            "regime": regime,
            "state_entries": len(tensors),
            "stored_tensor_elements": stored_elements,
            "first_key": next(iter(tensors), ""),
            "checkpoint_epoch": checkpoint.get("epoch"),
            "best_val_psnr": checkpoint.get("best_val_psnr")
        })

        del checkpoint, state

checkpoint_df = pd.DataFrame(checkpoint_rows)

checkpoint_df.to_csv(
    OUT / "Step05_Checkpoint_Comparison.csv",
    index=False
)

print(checkpoint_df.to_string(index=False))

# -------------------------------------------------------------------------
# 7. READ REPOSITORY DATA PREPARATION INSTRUCTIONS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("5F — MDAS DATASET DOCUMENTATION")
print("=" * 85)

readme = ACTIVE_REPO / "README.md"

if readme.exists():

    lines = readme.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    indices = [
        i for i, line in enumerate(lines)
        if "mdas" in line.lower()
    ]

    for i in indices[:12]:

        start = max(0, i - 2)
        end = min(len(lines), i + 5)

        print("\n" + "-" * 55)
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")

# -------------------------------------------------------------------------
# 8. SAVE FINAL REPORT
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("STEP 5 — FINAL SOURCE RECOVERY REPORT")
print("=" * 85)

report = {
    "repository_url": REPO_URL,
    "recorded_training_commit": RECORDED_COMMIT,
    "downloaded_commit": head,
    "active_commit": active_head,
    "exact_commit_available": commit_available,
    "source_directory": str(ACTIVE_REPO),
    "code_files": len(code_files),
    "image_or_array_files": len(data_files),
    "potential_partition_files": len(split_files),
    "source_matches": len(source_matches),
    "checkpoint_records": len(checkpoint_rows),
    "important_note": (
        "Checkpoint architecture equivalence has not been "
        "verified by loading the trained weights into the model."
    )
}

with open(
    OUT / "Step05_Source_Recovery_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

print("\n" + "=" * 85)
print("STEP 5 COMPLETE")
print("=" * 85)
print("Source data and checkpoints were not modified.")
print("Outputs saved to:", OUT)


In [ ]:

# =============================================================================
# STEP 6 — MDAS SOURCE, CHECKPOINT AND DATASET COMPATIBILITY
# Original SCSRNet repository + saved expanded experiments
# READ-ONLY VERIFICATION — NO TRAINING
# =============================================================================

from google.colab import drive
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import torch
import tifffile
import inspect
import importlib
import sys
import json
import ast
import hashlib
import subprocess
import traceback
import re

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
EXPERIMENTS = ROOT / "MDAS_JSTARS_Expanded"
DATA = REPO / "datasets" / "mdas_new"
OUT = PROJECT / "Step06_Compatibility"

OUT.mkdir(parents=True, exist_ok=True)

print("=" * 88)
print("STEP 6 — MDAS CHECKPOINT AND DATASET COMPATIBILITY")
print("=" * 88)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Repository:", REPO)

assert REPO.is_dir(), "Recovered repository missing."
assert DATA.is_dir(), "MDAS dataset directory missing."
assert EXPERIMENTS.is_dir(), "Experiments directory missing."

# -------------------------------------------------------------------------
# 6A — VERIFY EXACT COMMIT
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6A — REPOSITORY REVISION")
print("=" * 88)

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

with open(
    EXPERIMENTS / "SCSRNet_p00_none_s7" / "manifest.json"
) as f:
    manifest = json.load(f)

recorded = manifest["repo_head"]

print("Recovered:", head)
print("Recorded: ", recorded)
print("Exact match:", head == recorded)

assert head == recorded, "Repository revision mismatch."

# -------------------------------------------------------------------------
# 6B — VERIFY DATASET PARTITIONS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6B — MDAS TRAIN / VALIDATION / TEST PARTITIONS")
print("=" * 88)

splits = {}

for split in ["train", "val", "test"]:

    file = DATA / f"{split}.txt"
    assert file.exists(), f"Missing split file: {file}"

    names = [
        line.strip().split()[0]
        for line in file.read_text().splitlines()
        if line.strip()
    ]

    splits[split] = names

    print(
        f"{split.upper():5s}: "
        f"{len(names)} entries | "
        f"{len(set(names))} unique"
    )

    if len(names) != len(set(names)):
        print("WARNING: Duplicate entries detected in", split)

expected_counts = {"train": 133, "val": 32, "test": 40}

for split, count in expected_counts.items():
    assert len(splits[split]) == count, (
        f"{split}: expected {count}, got {len(splits[split])}"
    )

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:

    overlap = set(splits[a]) & set(splits[b])

    print(f"{a} vs {b}: {len(overlap)} shared names")

    if overlap:
        print("OVERLAP:", sorted(overlap)[:10])

# Check actual test names against the previous evaluation.
evaluation = pd.read_csv(
    EXPERIMENTS / "SCSRNet_p00_none_s7" / "test_rows.csv"
)

evaluated_names = set(evaluation["image"].astype(str))
split_names = set(splits["test"])

print("\nTest-list names:", len(split_names))
print("Evaluated names:", len(evaluated_names))
print("Exact test-name agreement:", split_names == evaluated_names)

if split_names != evaluated_names:
    print("Only in test list:", sorted(split_names - evaluated_names)[:10])
    print("Only in evaluation:", sorted(evaluated_names - split_names)[:10])

# -------------------------------------------------------------------------
# 6C — VERIFY THE THREE DATA MODALITIES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6C — MDAS DATA MODALITIES")
print("=" * 88)

modalities = ["lrhsi", "hrmsi", "hrhsi"]

for modality in modalities:
    print(
        modality,
        "directory exists:",
        (DATA / modality).is_dir()
    )

# Inspect the actual test patch names in the available directories.
modality_records = []

for modality in modalities:

    folder = DATA / modality

    if not folder.is_dir():
        continue

    filenames = {
        p.name for p in folder.glob("*.tif")
    }

    print(
        f"\n{modality}: "
        f"{len(filenames)} TIFF files"
    )

    print(
        "Test patches present:",
        len(split_names & filenames),
        "/",
        len(split_names)
    )

    # Verify representative data without loading entire datasets.
    for name in splits["test"][:3]:

        path = folder / name

        if not path.exists():
            continue

        with tifffile.TiffFile(path) as tif:
            series = tif.series[0]
            shape = tuple(series.shape)
            dtype = str(series.dtype)

        modality_records.append({
            "modality": modality,
            "filename": name,
            "shape": str(shape),
            "dtype": dtype,
            "size_bytes": path.stat().st_size
        })

        print(name, shape, dtype)

pd.DataFrame(modality_records).to_csv(
    OUT / "Step06_Modality_Inventory.csv",
    index=False
)

# -------------------------------------------------------------------------
# 6D — INSPECT MODEL CONSTRUCTOR SIGNATURES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6D — ACTUAL MODEL DEFINITIONS")
print("=" * 88)

sys.path.insert(0, str(REPO))

model_information = {}
model_classes = {}

for model_name in ["SCSRNet", "MSDCNN"]:

    try:
        module = importlib.import_module(
            f"models.{model_name}"
        )

        cls = getattr(module, model_name)

        model_classes[model_name] = cls

        constructor = str(inspect.signature(cls))
        forward = str(inspect.signature(cls.forward))

        print("\nMODEL:", model_name)
        print("Constructor:", constructor)
        print("Forward:", forward)

        source = inspect.getsource(cls)

        model_information[model_name] = {
            "constructor": constructor,
            "forward": forward,
            "source_file": inspect.getfile(cls),
            "source_lines": len(source.splitlines())
        }

        # Save a separate text copy for methodological auditing.
        (OUT / f"Step06_{model_name}_Class_Source.txt").write_text(
            source,
            encoding="utf-8"
        )

        print("\nFIRST 45 SOURCE LINES")

        for line in source.splitlines()[:45]:
            print(line)

    except Exception as exc:
        model_information[model_name] = {
            "import_error": str(exc)
        }

        print("\nIMPORT FAILED:", model_name, repr(exc))
        traceback.print_exc(limit=2)

# -------------------------------------------------------------------------
# 6E — CHECK CHECKPOINT STATE STRUCTURE
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6E — CHECKPOINT STRUCTURE")
print("=" * 88)

checkpoint_records = []

runs = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

for run in runs:

    checkpoint_path = EXPERIMENTS / run / "best.pt"

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    state = ckpt["model"]

    tensor_count = sum(
        isinstance(v, torch.Tensor)
        for v in state.values()
    )

    print("\n", run)
    print("Checkpoint epoch:", ckpt.get("epoch"))
    print("State entries:", len(state))
    print("Tensor entries:", tensor_count)

    checkpoint_records.append({
        "run": run,
        "epoch": ckpt.get("epoch"),
        "state_entries": len(state),
        "tensor_entries": tensor_count,
        "checkpoint_fingerprint": ckpt.get("fingerprint")
    })

pd.DataFrame(checkpoint_records).to_csv(
    OUT / "Step06_Checkpoint_Inspection.csv",
    index=False
)

# -------------------------------------------------------------------------
# 6F — CONSTRUCTOR COMPATIBILITY TEST
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6F — STRICT CHECKPOINT LOADING")
print("=" * 88)

# SCSRNet is normally configured using a config dictionary.
# These values must be checked against the actual repository
# implementation before any scientific inference is reported.

candidate_config = {
    "spectral_bands": 242,
    "msi_bands": 4,
    "num_channels": 64,
    "hsi_channels": 242,
    "ms_channels": 4,
    "num_spectral": 242,
    "num_ms": 4,
    "n_feats": 64,
    "scale": 3
}

compatibility = []

for model_name in ["SCSRNet", "MSDCNN"]:

    cls = model_classes.get(model_name)

    if cls is None:
        continue

    run = f"{model_name}_p00_none_s7"

    state = torch.load(
        EXPERIMENTS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )["model"]

    status = "NOT_TESTED"
    error_message = ""
    model = None

    try:
        signature = inspect.signature(cls)
        params = signature.parameters

        if "config" in params:
            # Construct using the recovered source's explicit
            # config interface. A failure is reported, not bypassed.
            model = cls(config=candidate_config)

        elif all(
            p.default is not inspect.Parameter.empty
            for p in params.values()
        ):
            model = cls()

        else:
            status = "CONSTRUCTOR_ARGUMENTS_REQUIRE_VERIFICATION"
            error_message = str(signature)

        if model is not None:
            # STRICT: no missing or unexpected keys permitted.
            model.load_state_dict(state, strict=True)
            model.eval()

            status = "STRICT_LOAD_PASSED"

            trainable = sum(
                p.numel()
                for p in model.parameters()
                if p.requires_grad
            )

            print(
                f"{model_name}: STRICT LOAD PASSED | "
                f"Trainable parameters: {trainable:,}"
            )

    except Exception as exc:
        status = "LOAD_FAILED"
        error_message = str(exc)
        print(f"{model_name}: {status}")
        print("Reason:", error_message[:1000])

    compatibility.append({
        "model": model_name,
        "run": run,
        "status": status,
        "error": error_message
    })

pd.DataFrame(compatibility).to_csv(
    OUT / "Step06_Strict_Load_Results.csv",
    index=False
)

# -------------------------------------------------------------------------
# 6G — INSPECT DATA LOADER AND PSNR METRIC DEFINITION
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6G — DATA LOADER AND EVALUATION DEFINITIONS")
print("=" * 88)

targets = [
    REPO / "dataloaders" / "HSI_datasets.py",
    REPO / "utils" / "metrics.py",
    REPO / "models" / "SCSRNet.py",
    REPO / "models" / "MSDCNN.py"
]

patterns = [
    "max_value",
    "15000",
    "interpolate",
    "bilinear",
    "psnr",
    "forward",
    "def __getitem__",
    "class mdasn_dataset"
]

audit_lines = []

for path in targets:

    if not path.exists():
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for i, line in enumerate(lines, start=1):

        if any(p.lower() in line.lower() for p in patterns):

            audit_lines.append({
                "file": str(path.relative_to(REPO)),
                "line": i,
                "text": line.strip()[:300]
            })

audit_df = pd.DataFrame(audit_lines)

audit_df.to_csv(
    OUT / "Step06_Data_Metric_Definitions.csv",
    index=False
)

print("\nRELEVANT IMPLEMENTATION LINES")

for _, row in audit_df.head(100).iterrows():
    print(
        f"{row['file']}:{row['line']} "
        f"{row['text']}"
    )

# -------------------------------------------------------------------------
# 6H — SAVE FINAL AUDIT
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("STEP 6 — FINAL REPORT")
print("=" * 88)

report = {
    "exact_repository_commit": head == recorded,
    "training_patches": len(splits["train"]),
    "validation_patches": len(splits["val"]),
    "test_patches": len(splits["test"]),
    "test_names_match_evaluation": split_names == evaluated_names,
    "model_information": model_information,
    "checkpoint_compatibility": compatibility,
    "output_folder": str(OUT)
}

with open(
    OUT / "Step06_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

print("\nSTEP 6 COMPLETED.")
print("No original source files or checkpoints were modified.")


In [ ]:

# =============================================================================
# STEP 6B — RECOVER EXACT MDAS CONFIGURATION AND VERIFY INFERENCE
# No training. No changes to original checkpoints or repository.
# =============================================================================

from google.colab import drive
from pathlib import Path
import sys
import json
import copy
import inspect
import traceback
import numpy as np
import torch
import tifffile
import pandas as pd

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
DATA = REPO / "datasets" / "mdas_new"
OUT = PROJECT / "Step06B_Strict_Load"

OUT.mkdir(parents=True, exist_ok=True)

print("=" * 85)
print("STEP 6B — EXACT CONFIGURATION AND CHECKPOINT COMPATIBILITY")
print("=" * 85)

assert REPO.is_dir()
assert DATA.is_dir()
assert EXPS.is_dir()

sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

# -------------------------------------------------------------------------
# 1. DISCOVER REPOSITORY CONFIGURATION FILES
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("6B-A — DISCOVERING MDAS CONFIGURATION")
print("=" * 85)

config_candidates = []

for path in REPO.rglob("*.json"):

    if ".git" in path.parts:
        continue

    try:
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)

        if not isinstance(obj, dict):
            continue

        if "mdasn_dataset" not in obj:
            continue

        section = obj["mdasn_dataset"]

        if not isinstance(section, dict):
            continue

        config_candidates.append((path, obj))

    except (ValueError, OSError, UnicodeError):
        continue

print("MDAS configuration candidates:", len(config_candidates))

for path, obj in config_candidates:
    section = obj["mdasn_dataset"]

    print("\nFILE:", path.relative_to(REPO))
    print("Spectral bands:", section.get("spectral_bands"))
    print("MSI bands:", section.get("msi_bands"))
    print("Scale factor:", section.get("factor"))
    print("LR size:", section.get("LR_size"))
    print("HR size:", section.get("HR_size"))

# Select configurations consistent with the verified dataset.
matching = []

for path, obj in config_candidates:
    s = obj["mdasn_dataset"]

    try:
        valid = (
            int(s.get("spectral_bands", -1)) == 242
            and int(s.get("msi_bands", -1)) == 4
            and int(s.get("factor", -1)) == 3
            and int(s.get("LR_size", -1)) == 24
            and int(s.get("HR_size", -1)) == 72
        )
    except (ValueError, TypeError):
        valid = False

    if valid:
        matching.append((path, obj))

if not matching:
    raise RuntimeError(
        "No repository configuration matches the verified "
        "242-band, 4-band, 3x MDAS patches. "
        "Inspect the candidate configurations printed above."
    )

# Prefer a configuration explicitly named for MDAS.
matching.sort(
    key=lambda item: (
        "mdas" not in str(item[0]).lower(),
        str(item[0])
    )
)

CONFIG_PATH, source_config = matching[0]

CONFIG = copy.deepcopy(source_config)

# Required selector used by both model constructors.
CONFIG["train_dataset"] = "mdasn_dataset"

# Preserve repository-defined architecture choices and channel indices.
required_keys = [
    "spectral_bands",
    "msi_bands",
    "factor",
    "LR_size",
    "HR_size",
    "B",
    "G",
    "R"
]

missing_keys = [
    k for k in required_keys
    if k not in CONFIG["mdasn_dataset"]
]

if missing_keys:
    raise KeyError(
        "MDAS source configuration lacks required keys: "
        + ", ".join(missing_keys)
    )

print("\nSELECTED CONFIGURATION:", CONFIG_PATH)
print("\nMODEL-RELEVANT SETTINGS")

for key in required_keys:
    print(f"  {key}: {CONFIG['mdasn_dataset'][key]}")

print("\nNormalization value:",
      CONFIG["mdasn_dataset"].get("max_value"))

# Store the selected source path and relevant settings.
configuration_report = {
    "source_config_path": str(CONFIG_PATH),
    "train_dataset": CONFIG["train_dataset"],
    "mdas_settings": CONFIG["mdasn_dataset"]
}

with open(
    OUT / "Step06B_Selected_Config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(configuration_report, f, indent=2, default=str)

# -------------------------------------------------------------------------
# 2. LOAD REAL MDAS TEST INPUTS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("6B-B — READING REAL MDAS TEST PATCH")
print("=" * 85)

PATCH_NAME = "test_0_0.tif"

NORMALIZER = CONFIG["mdasn_dataset"].get("max_value")

if NORMALIZER is None:
    raise KeyError("No max_value found in MDAS configuration.")

NORMALIZER = float(NORMALIZER)

if NORMALIZER <= 0:
    raise ValueError("Invalid normalization value.")

def load_modality(folder):
    path = DATA / folder / PATCH_NAME

    if not path.is_file():
        raise FileNotFoundError(path)

    array = tifffile.imread(path).astype(np.float32)
    array = array / NORMALIZER

    if not np.isfinite(array).all():
        raise ValueError(f"Non-finite data in {path}")

    return array

lr = load_modality("lrhsi")
msi = load_modality("hrmsi")
ref = load_modality("hrhsi")

assert lr.shape == (24, 24, 242)
assert msi.shape == (72, 72, 4)
assert ref.shape == (72, 72, 242)

def to_tensor(array):
    return torch.from_numpy(
        np.ascontiguousarray(
            array.transpose(2, 0, 1)
        )
    ).unsqueeze(0).to(DEVICE)

LR = to_tensor(lr)
MSI = to_tensor(msi)
REF = to_tensor(ref)

print("LR-HSI:", tuple(LR.shape))
print("HR-MSI:", tuple(MSI.shape))
print("Reference:", tuple(REF.shape))
print("Normalization divisor:", NORMALIZER)

# -------------------------------------------------------------------------
# 3. STRICTLY LOAD SAVED MODEL CHECKPOINTS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("6B-C — STRICT CHECKPOINT LOADING")
print("=" * 85)

records = []
loaded_models = {}

MODEL_CLASSES = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

for run in RUNS:

    model_name = run.split("_")[0]
    checkpoint_path = EXPS / run / "best.pt"

    print("\n" + "-" * 70)
    print("RUN:", run)

    record = {
        "run": run,
        "strict_load": False,
        "forward_pass": False,
        "trainable_parameters": None,
        "output_shape": None,
        "output_finite": None,
        "error": ""
    }

    model = None

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        state = checkpoint["model"]

        # Instantiate architecture using recovered repository config.
        model = MODEL_CLASSES[model_name](
            config=copy.deepcopy(CONFIG)
        )

        # Enforce complete weight compatibility.
        model.load_state_dict(state, strict=True)

        record["strict_load"] = True

        trainable = sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )

        record["trainable_parameters"] = trainable

        print("STRICT LOAD: PASSED")
        print("Trainable parameters:", f"{trainable:,}")
        print("Checkpoint epoch:", checkpoint.get("epoch"))

        model = model.to(DEVICE)
        model.eval()

        # Real forward pass using the original input modalities.
        with torch.inference_mode():
            prediction = model(LR, MSI)

        if not isinstance(prediction, torch.Tensor):
            raise TypeError(
                "Model output is not a single tensor: "
                + str(type(prediction))
            )

        record["output_shape"] = str(tuple(prediction.shape))
        record["output_finite"] = bool(
            torch.isfinite(prediction).all().item()
        )

        expected_shape = (1, 242, 72, 72)

        if tuple(prediction.shape) != expected_shape:
            raise ValueError(
                f"Unexpected output: {tuple(prediction.shape)}"
            )

        if not record["output_finite"]:
            raise ValueError("Prediction contains NaN or infinity.")

        record["forward_pass"] = True

        print("FORWARD PASS: PASSED")
        print("Prediction shape:", tuple(prediction.shape))
        print("Prediction range:",
              float(prediction.min()),
              float(prediction.max()))

        loaded_models[run] = True

        del prediction, model, checkpoint, state

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    except Exception as exc:
        record["error"] = repr(exc)
        print("FAILED:", repr(exc))
        traceback.print_exc(limit=2)

        if model is not None:
            del model

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    records.append(record)

# -------------------------------------------------------------------------
# 4. SAVE COMPATIBILITY RESULTS
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("6B-D — FINAL COMPATIBILITY REPORT")
print("=" * 85)

results = pd.DataFrame(records)

results.to_csv(
    OUT / "Step06B_Model_Compatibility.csv",
    index=False
)

print(results.to_string(index=False))

summary = {
    "repository_config": str(CONFIG_PATH),
    "models_tested": len(results),
    "strict_load_passed": int(results["strict_load"].sum()),
    "forward_passed": int(results["forward_pass"].sum()),
    "normalization_divisor": NORMALIZER,
    "test_patch": PATCH_NAME,
    "ready_for_numerical_reproduction": bool(
        results["forward_pass"].all()
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step06B_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 85)
print("STEP 6B — FINAL SUMMARY")
print("=" * 85)

print(json.dumps(summary, indent=2))

if summary["ready_for_numerical_reproduction"]:
    print("\nPASS: Both architectures loaded and ran successfully.")
    print("NEXT: Reproduce saved patch-level PSNR before alignment tests.")
else:
    print("\nSome compatibility checks failed.")
    print("Inspect the error messages before proceeding.")

print("\nOriginal checkpoints and source files remain unchanged.")


In [ ]:

# =============================================================================
# STEP 6C — MODEL OUTPUT RECOVERY AND ARCHIVED PSNR REPRODUCTION
# SCSRNet + MSDCNN | Two checkpoints per architecture
# No training, no modification of source models or original checkpoints.
# =============================================================================

from google.colab import drive
from pathlib import Path
import sys
import json
import inspect
import traceback

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

REPO = (
    PROJECT / "Step05_Source_Recovery" /
    "SCSRNet_Recorded_Commit"
)

EXPS = ROOT / "MDAS_JSTARS_Expanded"
DATA = REPO / "datasets" / "mdas_new"
OUT = PROJECT / "Step06C_Numerical_Reproduction"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 88)
print("STEP 6C — MODEL OUTPUT AND PSNR REPRODUCTION")
print("=" * 88)
print("Device:", DEVICE)

# -------------------------------------------------------------------------
# 1. LOAD VERIFIED CONFIGURATION
# -------------------------------------------------------------------------

CONFIG_PATH = (
    REPO / "config_lk" / "config_rgb_lk_pavia.json"
)

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = json.load(f)

config["train_dataset"] = "mdasn_dataset"

NORMALIZER = float(
    config["mdasn_dataset"]["max_value"]
)

assert NORMALIZER == 15000.0

print("\nOriginal PSNR implementation:")
print(inspect.getsource(PSNR))

# -------------------------------------------------------------------------
# 2. LOAD ONE ORIGINAL TEST PATCH
# -------------------------------------------------------------------------

PATCH = "test_0_0.tif"

def load_patch(folder):
    path = DATA / folder / PATCH

    if not path.exists():
        raise FileNotFoundError(path)

    arr = tifffile.imread(path).astype(np.float32)
    arr = arr / NORMALIZER

    return torch.from_numpy(
        np.ascontiguousarray(arr.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)

LR = load_patch("lrhsi")
MSI = load_patch("hrmsi")
REF = load_patch("hrhsi")

assert LR.shape == (1, 242, 24, 24)
assert MSI.shape == (1, 4, 72, 72)
assert REF.shape == (1, 242, 72, 72)

print("\nInput shapes:")
print("LR-HSI:", tuple(LR.shape))
print("HR-MSI:", tuple(MSI.shape))
print("Reference:", tuple(REF.shape))

# -------------------------------------------------------------------------
# 3. INSPECT AND EXTRACT MODEL OUTPUT
# -------------------------------------------------------------------------

EXPECTED = tuple(REF.shape)

def extract_prediction(output):

    if isinstance(output, torch.Tensor):
        if tuple(output.shape) != EXPECTED:
            raise ValueError(
                f"Unexpected output shape: {tuple(output.shape)}"
            )
        return output, "tensor"

    if not isinstance(output, dict):
        raise TypeError(
            f"Unsupported output type: {type(output)}"
        )

    print("\nOutput dictionary keys:", list(output.keys()))

    candidates = []

    for key, value in output.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {key}: shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )

            if tuple(value.shape) == EXPECTED:
                candidates.append((key, value))

        else:
            print(f"  {key}: {type(value).__name__}")

    if len(candidates) != 1:
        raise ValueError(
            "Expected exactly one HR-HSI prediction tensor, "
            f"found {len(candidates)}. "
            "Inspect output keys before selecting a tensor."
        )

    key, tensor = candidates[0]

    return tensor, key

# -------------------------------------------------------------------------
# 4. CREATE REPLICATE-BOUNDARY MSI TRANSLATIONS
# -------------------------------------------------------------------------

def shift_msi(msi, dy, dx):

    if not isinstance(dy, int) or not isinstance(dx, int):
        raise TypeError("Integer shifts required.")

    _, _, h, w = msi.shape

    if abs(dy) >= h or abs(dx) >= w:
        raise ValueError("Shift exceeds image dimensions.")

    # Positive dy shifts image content downward.
    # Positive dx shifts image content rightward.
    padded = F.pad(
        msi,
        (max(dx, 0), max(-dx, 0),
         max(dy, 0), max(-dy, 0)),
        mode="replicate"
    )

    start_y = max(-dy, 0)
    start_x = max(-dx, 0)

    shifted = padded[
        :,
        :,
        start_y:start_y + h,
        start_x:start_x + w
    ]

    assert shifted.shape == msi.shape

    return shifted

# Confirm shift convention using a small synthetic array.
toy = torch.arange(9, dtype=torch.float32).reshape(1, 1, 3, 3)

assert shift_msi(toy, 1, 0)[0, 0, 1, 0] == toy[0, 0, 0, 0]
assert shift_msi(toy, 0, 1)[0, 0, 0, 1] == toy[0, 0, 0, 0]

# -------------------------------------------------------------------------
# 5. REPRODUCE SAVED PSNR RECORDS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6C-A — STRICT LOAD AND MODEL OUTPUT INSPECTION")
print("=" * 88)

MODELS = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

# Zero shift, plus four directions at one pixel and
# a representative four-pixel displacement.
OFFSETS = [
    (0, 0),
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (4, 0)
]

rows = []
failures = []

for run in RUNS:

    model_name = run.split("_")[0]

    print("\n" + "=" * 72)
    print("MODEL:", run)
    print("=" * 72)

    checkpoint_path = EXPS / run / "best.pt"
    saved_results_path = EXPS / run / "test_rows.csv"

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        model = MODELS[model_name](config=config)
        model.load_state_dict(
            checkpoint["model"],
            strict=True
        )

        model = model.to(DEVICE).eval()

        saved = pd.read_csv(saved_results_path)

        for dy, dx in OFFSETS:

            translated = shift_msi(MSI, dy, dx)

            with torch.inference_mode():
                raw_output = model(LR, translated)

                prediction, output_key = extract_prediction(
                    raw_output
                )

                assert torch.isfinite(prediction).all()

                # Fixed interior crop: 72 -> 64 pixels.
                pred_crop = prediction[:, :, 4:-4, 4:-4]
                ref_crop = REF[:, :, 4:-4, 4:-4]

                score = PSNR(pred_crop, ref_crop)

                if isinstance(score, torch.Tensor):
                    reproduced_psnr = float(
                        score.detach().cpu().item()
                    )
                else:
                    reproduced_psnr = float(score)

            archived = saved[
                (saved["image"].astype(str) == PATCH)
                & (saved["dy"].astype(int) == dy)
                & (saved["dx"].astype(int) == dx)
            ]

            if len(archived) != 1:
                raise ValueError(
                    f"Expected one archived record for "
                    f"{run}, dy={dy}, dx={dx}; "
                    f"found {len(archived)}"
                )

            archived_psnr = float(
                archived.iloc[0]["common_psnr"]
            )

            difference = reproduced_psnr - archived_psnr

            record = {
                "run": run,
                "model": model_name,
                "patch": PATCH,
                "dy": dy,
                "dx": dx,
                "output_key": output_key,
                "reproduced_psnr": reproduced_psnr,
                "archived_psnr": archived_psnr,
                "difference_db": difference,
                "absolute_difference_db": abs(difference)
            }

            rows.append(record)

            print(
                f"Shift ({dy:+d},{dx:+d}) | "
                f"new={reproduced_psnr:.6f} | "
                f"saved={archived_psnr:.6f} | "
                f"difference={difference:+.6f} dB"
            )

        del model, checkpoint

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    except Exception as exc:

        failures.append({
            "run": run,
            "error": repr(exc)
        })

        print("FAILED:", repr(exc))
        traceback.print_exc(limit=3)

# -------------------------------------------------------------------------
# 6. SAVE REPRODUCTION REPORT
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("6C-B — NUMERICAL REPRODUCTION SUMMARY")
print("=" * 88)

results = pd.DataFrame(rows)

results.to_csv(
    OUT / "Step06C_Patch_Reproduction.csv",
    index=False
)

if not results.empty:

    summary = (
        results.groupby("run")["absolute_difference_db"]
        .agg(["count", "mean", "max"])
        .reset_index()
    )

    print(summary.to_string(index=False))

    print("\nMaximum absolute difference:",
          results["absolute_difference_db"].max())

    print("\nChecks within 0.01 dB:",
          int((results["absolute_difference_db"] < 0.01).sum()),
          "/",
          len(results))

else:
    print("No numerical reproduction records generated.")

report = {
    "patch": PATCH,
    "models_attempted": len(RUNS),
    "offsets_per_model": len(OFFSETS),
    "reproduced_records": len(results),
    "failures": failures,
    "max_absolute_difference_db": (
        float(results["absolute_difference_db"].max())
        if not results.empty else None
    ),
    "all_records_within_0_01_db": (
        bool((results["absolute_difference_db"] < 0.01).all())
        if len(results) == len(RUNS) * len(OFFSETS)
        else False
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step06C_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 6C — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

if report["all_records_within_0_01_db"]:
    print("\nPASS: Selected archived predictions reproduced.")
    print("NEXT: Validate the full test set and registration baselines.")
else:
    print("\nNumerical reproduction not yet established.")
    print("Inspect prediction keys, PSNR implementation and shift logic.")

print("\nNo original checkpoint or dataset files were modified.")


In [ ]:

# =============================================================================
# STEP 6D — CORRECTED MDAS CHECKPOINT REPRODUCTION
# Four models × six displacements = 24 comparisons
# Fix: contiguous tensors before ORIGINAL repository PSNR
# =============================================================================

from google.colab import drive
from pathlib import Path
import sys, json, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
DATA = REPO / "datasets" / "mdas_new"
OUT = PROJECT / "Step06D_Corrected_Reproduction"

OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 85)
print("STEP 6D — CORRECTED MDAS NUMERICAL REPRODUCTION")
print("=" * 85)
print("Device:", DEVICE)

# -------------------------------------------------------------------------
# 1. LOAD VERIFIED CONFIGURATION
# -------------------------------------------------------------------------

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"
NORMALIZER = float(CONFIG["mdasn_dataset"]["max_value"])

assert NORMALIZER == 15000.0

# -------------------------------------------------------------------------
# 2. LOAD REAL MDAS TEST PATCH
# -------------------------------------------------------------------------

PATCH = "test_0_0.tif"

def load_patch(modality):
    path = DATA / modality / PATCH
    array = tifffile.imread(path).astype(np.float32)
    array /= NORMALIZER

    return torch.from_numpy(
        np.ascontiguousarray(array.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)

LR = load_patch("lrhsi")
MSI = load_patch("hrmsi")
REF = load_patch("hrhsi")

assert tuple(LR.shape) == (1, 242, 24, 24)
assert tuple(MSI.shape) == (1, 4, 72, 72)
assert tuple(REF.shape) == (1, 242, 72, 72)

# -------------------------------------------------------------------------
# 3. REPLICATE-BOUNDARY IMAGE TRANSLATION
# -------------------------------------------------------------------------

def shift_msi(image, dy, dx):
    """
    Positive dy moves content downward.
    Positive dx moves content rightward.
    Newly uncovered pixels replicate the nearest boundary.
    """
    dy, dx = int(dy), int(dx)

    height, width = image.shape[-2:]

    if abs(dy) >= height or abs(dx) >= width:
        raise ValueError("Invalid displacement.")

    padded = F.pad(
        image,
        (
            max(dx, 0),
            max(-dx, 0),
            max(dy, 0),
            max(-dy, 0)
        ),
        mode="replicate"
    )

    top = max(-dy, 0)
    left = max(-dx, 0)

    return padded[
        :, :, top:top + height, left:left + width
    ].contiguous()

# Verify direction convention independently.
test = torch.arange(16).reshape(1, 1, 4, 4).float()

assert torch.equal(
    shift_msi(test, 1, 0)[0, 0, 1, :],
    test[0, 0, 0, :]
)
assert torch.equal(
    shift_msi(test, 0, 1)[0, 0, :, 1],
    test[0, 0, :, 0]
)
assert torch.equal(
    shift_msi(test, -1, 0)[0, 0, 0, :],
    test[0, 0, 1, :]
)
assert torch.equal(
    shift_msi(test, 0, -1)[0, 0, :, 0],
    test[0, 0, :, 1]
)

print("Shift convention: VERIFIED")

# -------------------------------------------------------------------------
# 4. REPOSITORY PSNR WITH CONTIGUOUS CROPS
# -------------------------------------------------------------------------

def calculate_original_psnr(prediction, reference):

    pred_crop = prediction[:, :, 4:-4, 4:-4].contiguous()
    ref_crop = reference[:, :, 4:-4, 4:-4].contiguous()

    assert pred_crop.shape == (1, 242, 64, 64)
    assert ref_crop.shape == pred_crop.shape

    # This calls the ORIGINAL metric unchanged.
    result = PSNR(pred_crop, ref_crop)

    return float(result.detach().cpu().item())

# -------------------------------------------------------------------------
# 5. RUN THE REPRODUCTION EXPERIMENT
# -------------------------------------------------------------------------

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

OFFSETS = [
    (0, 0),
    (1, 0),
    (-1, 0),
    (0, 1),
    (0, -1),
    (4, 0)
]

CLASSES = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

rows = []
errors = []

for run in RUNS:

    print("\n" + "=" * 75)
    print("RUN:", run)
    print("=" * 75)

    model_name = run.split("_")[0]

    checkpoint_path = EXPS / run / "best.pt"
    results_path = EXPS / run / "test_rows.csv"

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        model = CLASSES[model_name](config=CONFIG)
        model.load_state_dict(
            checkpoint["model"],
            strict=True
        )

        model = model.to(DEVICE).eval()
        archived = pd.read_csv(results_path)

        print("Checkpoint: STRICT LOAD PASSED")

        for dy, dx in OFFSETS:

            shifted = shift_msi(MSI, dy, dx)

            with torch.inference_mode():
                output = model(LR, shifted)

                if not isinstance(output, dict):
                    raise TypeError("Expected model output dictionary.")

                prediction = output["pred"]

                assert prediction.shape == REF.shape
                assert torch.isfinite(prediction).all()

                reproduced = calculate_original_psnr(
                    prediction, REF
                )

            matching = archived[
                (archived["image"] == PATCH) &
                (archived["dy"] == dy) &
                (archived["dx"] == dx)
            ]

            if len(matching) != 1:
                raise ValueError(
                    f"Archived row count incorrect: {len(matching)}"
                )

            original = float(
                matching.iloc[0]["common_psnr"]
            )

            difference = reproduced - original

            rows.append({
                "run": run,
                "model": model_name,
                "image": PATCH,
                "dy": dy,
                "dx": dx,
                "reproduced_psnr": reproduced,
                "archived_psnr": original,
                "difference_db": difference,
                "absolute_difference_db": abs(difference)
            })

            print(
                f"dy={dy:+d}, dx={dx:+d} | "
                f"Reproduced={reproduced:.6f} | "
                f"Archived={original:.6f} | "
                f"Difference={difference:+.6f} dB"
            )

        del model, checkpoint

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    except Exception as exc:

        errors.append({
            "run": run,
            "error": repr(exc)
        })

        print("ERROR:", repr(exc))
        traceback.print_exc(limit=3)

# -------------------------------------------------------------------------
# 6. SAVE NUMERICAL RESULTS
# -------------------------------------------------------------------------

results = pd.DataFrame(rows)

results.to_csv(
    OUT / "Step06D_Numerical_Reproduction.csv",
    index=False
)

print("\n" + "=" * 85)
print("STEP 6D — REPRODUCTION SUMMARY")
print("=" * 85)

if not results.empty:

    summary = (
        results.groupby("run")
        ["absolute_difference_db"]
        .agg(["count", "mean", "max"])
        .reset_index()
    )

    print(summary.to_string(index=False))

    max_error = float(
        results["absolute_difference_db"].max()
    )

    n_passed = int(
        (results["absolute_difference_db"] <= 0.01).sum()
    )

else:
    max_error = None
    n_passed = 0

print("\nSuccessful numerical comparisons:", len(results), "/ 24")
print("Comparisons within 0.01 dB:", n_passed, "/ 24")
print("Maximum absolute difference:", max_error)

report = {
    "patch": PATCH,
    "expected_comparisons": 24,
    "completed_comparisons": len(results),
    "comparisons_within_0_01_db": n_passed,
    "maximum_absolute_difference_db": max_error,
    "full_reproduction_passed": (
        len(results) == 24 and n_passed == 24
    ),
    "errors": errors,
    "output_directory": str(OUT)
}

with open(
    OUT / "Step06D_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 85)
print("STEP 6D — FINAL REPORT")
print("=" * 85)
print(json.dumps(report, indent=2))

if report["full_reproduction_passed"]:
    print("\nPASS — Archived results reproduced within 0.01 dB.")
else:
    print("\nCHECK — Reproduction differs or is incomplete.")
    print("Review numerical differences before proceeding.")

print("\nOriginal research files remain unchanged.")



In [ ]:

# =============================================================================
# STEP 7 — FULL MDAS TEST-SET NUMERICAL REPRODUCTION
#
# 4 trained checkpoints × 40 patches × 13 offsets = 2,080 comparisons.
# Strictly reuse original model source, weights, image normalization,
# replicate-edge shifts, common 64×64 crop and repository PSNR.
#
# Features:
# - Live per-patch progress
# - Resume after Colab interruption
# - Atomic incremental CSV saving
# - Explicit PASS / FAIL summary
# - No retraining or modification of original checkpoints
# =============================================================================

from google.colab import drive
from pathlib import Path
import os
import sys
import json
import time
import tempfile
import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn.functional as F

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
DATA = REPO / "datasets" / "mdas_new"
OUT = PROJECT / "Step07_Full_Reproduction"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 88, flush=True)
print("STEP 7 — FULL MDAS TEST-SET REPRODUCTION", flush=True)
print("=" * 88, flush=True)
print("Device:", DEVICE, flush=True)
print("Outputs:", OUT, flush=True)

# -------------------------------------------------------------------------
# 7A — CONFIGURATION AND EXPERIMENT DEFINITIONS
# -------------------------------------------------------------------------

CONFIG_PATH = REPO / "config_lk" / "config_rgb_lk_pavia.json"

with open(CONFIG_PATH, encoding="utf-8") as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"
NORMALIZER = float(CONFIG["mdasn_dataset"]["max_value"])

assert NORMALIZER == 15000.0

MODELS = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

OFFSETS = [
    (0, 0),
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (2, 0), (-2, 0), (0, 2), (0, -2),
    (4, 0), (-4, 0), (0, 4), (0, -4)
]

TOLERANCE_DB = 0.01
SAVE_EVERY_PATCHES = 1
PROGRESS_EVERY_PATCHES = 5

# Read the authoritative ordered test list.
test_list = DATA / "test.txt"

PATCHES = [
    line.strip().split()[0]
    for line in test_list.read_text().splitlines()
    if line.strip()
]

assert len(PATCHES) == 40
assert len(set(PATCHES)) == 40

print("\nRuns:", len(RUNS), flush=True)
print("Test patches:", len(PATCHES), flush=True)
print("Offsets per patch:", len(OFFSETS), flush=True)

EXPECTED_TOTAL = len(RUNS) * len(PATCHES) * len(OFFSETS)

print("Total comparisons:", EXPECTED_TOTAL, flush=True)

# -------------------------------------------------------------------------
# 7B — EXACT INPUT PREPARATION
# -------------------------------------------------------------------------

def load_patch(modality, filename):
    path = DATA / modality / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    array = tifffile.imread(path).astype(np.float32)
    array = array / NORMALIZER

    if not np.isfinite(array).all():
        raise ValueError(f"Non-finite array: {path}")

    return torch.from_numpy(
        np.ascontiguousarray(array.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)


def shift_msi(image, dy, dx):
    """
    Replicate-edge integer translation.
    Positive dy = image content moves down.
    Positive dx = image content moves right.
    """
    dy, dx = int(dy), int(dx)

    h, w = image.shape[-2:]

    if abs(dy) >= h or abs(dx) >= w:
        raise ValueError("Invalid shift.")

    padded = F.pad(
        image,
        (
            max(dx, 0), max(-dx, 0),
            max(dy, 0), max(-dy, 0)
        ),
        mode="replicate"
    )

    start_y = max(-dy, 0)
    start_x = max(-dx, 0)

    return padded[
        :, :,
        start_y:start_y + h,
        start_x:start_x + w
    ].contiguous()


def get_prediction(output):
    if isinstance(output, dict):
        if "pred" not in output:
            raise KeyError(
                "Expected prediction key 'pred'. "
                f"Available keys: {list(output.keys())}"
            )
        output = output["pred"]

    if not isinstance(output, torch.Tensor):
        raise TypeError(
            f"Invalid prediction type: {type(output)}"
        )

    if tuple(output.shape) != (1, 242, 72, 72):
        raise ValueError(
            f"Invalid prediction shape: {tuple(output.shape)}"
        )

    if not torch.isfinite(output).all():
        raise ValueError("Prediction contains non-finite values.")

    return output


def original_psnr(pred, ref):
    # Common 64×64 crop, exactly as reproduced in Step 6D.
    a = pred[:, :, 4:-4, 4:-4].contiguous()
    b = ref[:, :, 4:-4, 4:-4].contiguous()

    score = PSNR(a, b)

    return float(score.detach().cpu().item())

# -------------------------------------------------------------------------
# 7C — CHECK SHIFT CONVENTION
# -------------------------------------------------------------------------

toy = torch.arange(25).float().reshape(1, 1, 5, 5)

for dy, dx in OFFSETS:
    shifted = shift_msi(toy, dy, dx)

    assert shifted.shape == toy.shape

assert torch.equal(shift_msi(toy, 0, 0), toy)

print("\nShift operation: VERIFIED", flush=True)

# -------------------------------------------------------------------------
# 7D — CHECKPOINT AND ARCHIVED RESULTS VALIDATION
# -------------------------------------------------------------------------

ARCHIVES = {}

for run in RUNS:
    path = EXPS / run / "test_rows.csv"

    if not path.is_file():
        raise FileNotFoundError(path)

    archive = pd.read_csv(path)

    required = {"image", "dy", "dx", "common_psnr"}
    missing = required - set(archive.columns)

    if missing:
        raise ValueError(f"{run}: Missing archive columns: {missing}")

    if len(archive) != 520:
        raise ValueError(
            f"{run}: Expected 520 archived rows, found {len(archive)}"
        )

    archive["image"] = archive["image"].astype(str)
    archive["dy"] = archive["dy"].astype(int)
    archive["dx"] = archive["dx"].astype(int)

    archive = archive.set_index(["image", "dy", "dx"])

    if not archive.index.is_unique:
        raise ValueError(f"{run}: Duplicate evaluation keys.")

    expected_keys = {
        (patch, dy, dx)
        for patch in PATCHES
        for dy, dx in OFFSETS
    }

    if set(archive.index.tolist()) != expected_keys:
        raise ValueError(
            f"{run}: Archived test keys differ from required protocol."
        )

    ARCHIVES[run] = archive

print("All archived result files: VERIFIED", flush=True)

# -------------------------------------------------------------------------
# 7E — RESUME EXISTING VALIDATED RECORDS
# -------------------------------------------------------------------------

OUTPUT_CSV = OUT / "Step07_Full_Reproduction.csv"

COLUMNS = [
    "run", "model", "image", "dy", "dx",
    "reproduced_psnr", "archived_psnr",
    "difference_db", "absolute_difference_db"
]

existing = pd.DataFrame(columns=COLUMNS)

if OUTPUT_CSV.exists():
    existing = pd.read_csv(OUTPUT_CSV)

    if not existing.empty:
        if not set(COLUMNS).issubset(existing.columns):
            raise ValueError(
                "Existing resume CSV has incompatible columns."
            )

        existing = existing[COLUMNS].copy()
        existing["dy"] = existing["dy"].astype(int)
        existing["dx"] = existing["dx"].astype(int)

        if existing.duplicated(
            ["run", "image", "dy", "dx"]
        ).any():
            raise ValueError(
                "Duplicate keys in the existing resume file."
            )

        # Verify that resumed archived values still match the source.
        for row in existing.itertuples(index=False):
            if row.run not in ARCHIVES:
                raise ValueError(
                    f"Unknown run in resume file: {row.run}"
                )

            expected = float(
                ARCHIVES[row.run].loc[
                    (row.image, row.dy, row.dx),
                    "common_psnr"
                ]
            )

            if abs(expected - row.archived_psnr) > 1e-7:
                raise ValueError(
                    "Resume data does not match its archived source: "
                    f"{row.run}, {row.image}, {row.dy}, {row.dx}"
                )

        print(
            f"Validated resumed records: {len(existing)}",
            flush=True
        )

records = existing.to_dict("records")

completed_keys = {
    (r["run"], r["image"], int(r["dy"]), int(r["dx"]))
    for r in records
}

def save_progress():
    frame = pd.DataFrame(records, columns=COLUMNS)
    frame = frame.sort_values(
        ["run", "image", "dy", "dx"]
    ).reset_index(drop=True)

    # Write a temporary complete CSV and replace the prior output
    # only after the temporary file is successfully closed.
    fd, temp_path = tempfile.mkstemp(
        prefix="step07_",
        suffix=".csv",
        dir=str(OUT)
    )

    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            frame.to_csv(f, index=False)

        os.replace(temp_path, OUTPUT_CSV)

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    return frame

# -------------------------------------------------------------------------
# 7F — FULL MODEL EVALUATION
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("7F — RUNNING FULL REPRODUCTION", flush=True)
print("=" * 88, flush=True)

overall_start = time.monotonic()

for run_number, run in enumerate(RUNS, start=1):

    model_name = run.split("_")[0]
    ckpt_path = EXPS / run / "best.pt"

    print(
        f"\nMODEL {run_number}/{len(RUNS)}: {run}",
        flush=True
    )

    pending = sum(
        (run, patch, dy, dx) not in completed_keys
        for patch in PATCHES
        for dy, dx in OFFSETS
    )

    print("Remaining comparisons:", pending, flush=True)

    if pending == 0:
        print("Already complete: skipping model.", flush=True)
        continue

    checkpoint = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=True
    )

    model = MODELS[model_name](config=CONFIG)
    model.load_state_dict(checkpoint["model"], strict=True)
    model = model.to(DEVICE).eval()

    print("Strict checkpoint load: PASSED", flush=True)

    run_start = time.monotonic()

    for patch_idx, patch_name in enumerate(PATCHES, start=1):

        patch_offsets = [
            (dy, dx)
            for dy, dx in OFFSETS
            if (run, patch_name, dy, dx) not in completed_keys
        ]

        if not patch_offsets:
            continue

        LR = load_patch("lrhsi", patch_name)
        MSI = load_patch("hrmsi", patch_name)
        REF = load_patch("hrhsi", patch_name)

        if tuple(LR.shape) != (1, 242, 24, 24):
            raise ValueError(f"Invalid LR-HSI: {patch_name}")

        if tuple(MSI.shape) != (1, 4, 72, 72):
            raise ValueError(f"Invalid HR-MSI: {patch_name}")

        if tuple(REF.shape) != (1, 242, 72, 72):
            raise ValueError(f"Invalid reference: {patch_name}")

        with torch.inference_mode():

            for dy, dx in patch_offsets:

                shifted = shift_msi(MSI, dy, dx)

                output = model(LR, shifted)
                prediction = get_prediction(output)

                new_score = original_psnr(prediction, REF)

                old_score = float(
                    ARCHIVES[run].loc[
                        (patch_name, dy, dx),
                        "common_psnr"
                    ]
                )

                difference = new_score - old_score

                record = {
                    "run": run,
                    "model": model_name,
                    "image": patch_name,
                    "dy": dy,
                    "dx": dx,
                    "reproduced_psnr": new_score,
                    "archived_psnr": old_score,
                    "difference_db": difference,
                    "absolute_difference_db": abs(difference)
                }

                records.append(record)
                completed_keys.add(
                    (run, patch_name, dy, dx)
                )

        if patch_idx % SAVE_EVERY_PATCHES == 0:
            save_progress()

        if (
            patch_idx % PROGRESS_EVERY_PATCHES == 0
            or patch_idx == len(PATCHES)
        ):
            run_records = [
                r for r in records if r["run"] == run
            ]

            largest_error = max(
                r["absolute_difference_db"]
                for r in run_records
            )

            elapsed = time.monotonic() - run_start

            print(
                f"  Patch {patch_idx:02d}/{len(PATCHES)} | "
                f"Records {len(run_records):03d}/520 | "
                f"Max error {largest_error:.8f} dB | "
                f"Elapsed {elapsed:.1f} s",
                flush=True
            )

        del LR, MSI, REF

    del model, checkpoint

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print(
        f"Completed model: {run}",
        flush=True
    )

# -------------------------------------------------------------------------
# 7G — FINAL INTEGRITY CHECK
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("7G — FINAL REPRODUCTION REPORT", flush=True)
print("=" * 88, flush=True)

result = save_progress()

if len(result) != EXPECTED_TOTAL:
    raise AssertionError(
        f"Expected {EXPECTED_TOTAL} records, found {len(result)}."
    )

if result.duplicated(["run", "image", "dy", "dx"]).any():
    raise AssertionError("Duplicate comparison keys detected.")

summary = (
    result.groupby("run")
    ["absolute_difference_db"]
    .agg(["count", "mean", "max"])
    .reset_index()
)

summary["within_tolerance"] = (
    summary["max"] <= TOLERANCE_DB
)

summary.to_csv(
    OUT / "Step07_Per_Model_Summary.csv",
    index=False
)

print(summary.to_string(index=False), flush=True)

passed = int(
    (result["absolute_difference_db"] <= TOLERANCE_DB).sum()
)

failed = int(len(result) - passed)

report = {
    "runs": len(RUNS),
    "test_patches": len(PATCHES),
    "offsets_per_patch": len(OFFSETS),
    "expected_comparisons": EXPECTED_TOTAL,
    "completed_comparisons": len(result),
    "passed_within_0_01_db": passed,
    "failed": failed,
    "maximum_absolute_difference_db": float(
        result["absolute_difference_db"].max()
    ),
    "full_reproduction_passed": (
        len(result) == EXPECTED_TOTAL and failed == 0
    ),
    "elapsed_seconds": round(
        time.monotonic() - overall_start, 2
    ),
    "repository_psnr_used": True,
    "crop_size": [64, 64],
    "normalization_divisor": NORMALIZER,
    "original_source_modified": False,
    "output_directory": str(OUT)
}

with open(
    OUT / "Step07_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + json.dumps(report, indent=2), flush=True)

print("\n" + "=" * 88, flush=True)

if report["full_reproduction_passed"]:
    print(
        "STEP 7 PASSED — FULL SELECTED-MODEL TEST SET REPRODUCED.",
        flush=True
    )
else:
    print(
        "STEP 7 REQUIRES REVIEW — INSPECT MODEL DIFFERENCES.",
        flush=True
    )

print("Results saved:", OUT, flush=True)

In [ ]:

# =============================================================================
# STEP 8 — ORACLE GEOMETRIC CORRECTION BASELINE
#
# Four verified checkpoints, 40 held-out patches, 13 shift conditions.
#
# Comparisons:
#   1. Uncorrected displaced HR-MSI (archived model PSNR)
#   2. Inverse-shifted HR-MSI (oracle displacement knowledge)
#   3. Original unshifted HR-MSI (clean aligned reference)
#
# No model training or source modifications.
# Checkpoint, normalization, crop and PSNR follow validated Step 7.
# =============================================================================

from google.colab import drive
from pathlib import Path
import os
import sys
import json
import time
import tempfile

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
DATA = REPO / "datasets" / "mdas_new"
OUT = PROJECT / "Step08_Oracle_Correction"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"
NORMALIZER = float(CONFIG["mdasn_dataset"]["max_value"])

assert NORMALIZER == 15000.0

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

CLASSES = {"SCSRNet": SCSRNet, "MSDCNN": MSDCNN}

OFFSETS = [
    (0, 0),
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (2, 0), (-2, 0), (0, 2), (0, -2),
    (4, 0), (-4, 0), (0, 4), (0, -4)
]

PATCHES = [
    line.split()[0]
    for line in (DATA / "test.txt").read_text().splitlines()
    if line.strip()
]

assert len(PATCHES) == 40
assert len(set(PATCHES)) == 40

EXPECTED_ROWS = len(RUNS) * len(PATCHES) * len(OFFSETS)
TOLERANCE = 0.01

print("=" * 88, flush=True)
print("STEP 8 — ORACLE GEOMETRIC CORRECTION", flush=True)
print("=" * 88, flush=True)
print("Device:", DEVICE)
print("Models:", len(RUNS))
print("Test patches:", len(PATCHES))
print("Displacements:", len(OFFSETS))
print("Expected records:", EXPECTED_ROWS)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 1. ORIGINAL DATA PREPARATION
# -------------------------------------------------------------------------

def load_patch(modality, filename):
    path = DATA / modality / filename
    arr = tifffile.imread(path).astype(np.float32) / NORMALIZER

    if not np.isfinite(arr).all():
        raise ValueError(f"Non-finite input: {path}")

    return torch.from_numpy(
        np.ascontiguousarray(arr.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)


def shift_image(image, dy, dx):
    """
    Positive dy shifts image content down.
    Positive dx shifts image content right.
    Boundary fill is nearest-pixel replication.
    """
    dy, dx = int(dy), int(dx)
    h, w = image.shape[-2:]

    assert abs(dy) < h and abs(dx) < w

    padded = F.pad(
        image,
        (
            max(dx, 0), max(-dx, 0),
            max(dy, 0), max(-dy, 0)
        ),
        mode="replicate"
    )

    top = max(-dy, 0)
    left = max(-dx, 0)

    return padded[
        :, :,
        top:top + h,
        left:left + w
    ].contiguous()


def predict(model, lr, msi):
    output = model(lr, msi)

    if isinstance(output, dict):
        output = output["pred"]

    if tuple(output.shape) != (1, 242, 72, 72):
        raise ValueError(f"Unexpected output shape: {output.shape}")

    if not torch.isfinite(output).all():
        raise ValueError("Non-finite prediction.")

    return output


def psnr_original(pred, ref):
    p = pred[:, :, 4:-4, 4:-4].contiguous()
    r = ref[:, :, 4:-4, 4:-4].contiguous()

    return float(PSNR(p, r).detach().cpu().item())


def core_msi_mse(a, b):
    # Geometric fidelity on the shared 64x64 MSI interior.
    a = a[:, :, 4:-4, 4:-4]
    b = b[:, :, 4:-4, 4:-4]

    return float(torch.mean((a - b) ** 2).item())


# -------------------------------------------------------------------------
# 2. VERIFY THE ORACLE-INVERSE SHIFT LOGIC
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("8A — TRANSLATION TEST")
print("=" * 88)

toy = torch.arange(100, dtype=torch.float32).reshape(1, 1, 10, 10)

for dy, dx in OFFSETS:
    distorted = shift_image(toy, dy, dx)
    corrected = shift_image(distorted, -dy, -dx)

    # All tested shifts are <= 4 pixels, so the 2-pixel interior
    # is NOT sufficient; use the 4-pixel interior.
    assert torch.equal(
        corrected[:, :, 4:-4, 4:-4],
        toy[:, :, 4:-4, 4:-4]
    )

print("PASS: Exact inverse offsets recover the interior of the toy image.")
print("NOTE: Boundary pixels may remain irrecoverable.")

# -------------------------------------------------------------------------
# 3. VALIDATE ARCHIVED RECORDS AND STEP 7
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("8B — BASELINE ARCHIVE VALIDATION")
print("=" * 88)

archives = {}

for run in RUNS:
    df = pd.read_csv(EXPS / run / "test_rows.csv")
    assert len(df) == 520

    df["dy"] = df["dy"].astype(int)
    df["dx"] = df["dx"].astype(int)

    df = df.set_index(["image", "dy", "dx"])
    assert df.index.is_unique

    expected = {
        (image, dy, dx)
        for image in PATCHES
        for dy, dx in OFFSETS
    }

    assert set(df.index.tolist()) == expected
    archives[run] = df

step7_path = PROJECT / "Step07_Full_Reproduction" / "Step07_Full_Reproduction.csv"
assert step7_path.is_file(), "Step 7 verified CSV is missing."

step7 = pd.read_csv(step7_path)
assert len(step7) == EXPECTED_ROWS
assert float(step7["absolute_difference_db"].max()) <= TOLERANCE

print("PASS: Step 7 and all four baseline archives verified.")

# -------------------------------------------------------------------------
# 4. SET UP RESUMABLE RECORD STORAGE
# -------------------------------------------------------------------------

CSV_PATH = OUT / "Step08_Oracle_Patch_Results.csv"

COLS = [
    "run", "model", "regime", "seed", "image",
    "dy", "dx", "distance",
    "uncorrected_psnr", "oracle_psnr", "clean_psnr",
    "oracle_gain_db", "oracle_vs_clean_db",
    "uncorrected_vs_clean_db",
    "msi_uncorrected_core_mse",
    "msi_corrected_core_mse"
]

if CSV_PATH.exists():
    prior = pd.read_csv(CSV_PATH)

    if not set(COLS).issubset(prior.columns):
        raise ValueError("Existing Step 8 CSV has incompatible columns.")

    prior = prior[COLS].copy()

    if prior.duplicated(["run", "image", "dy", "dx"]).any():
        raise ValueError("Duplicate records in resume file.")

    if not np.isfinite(
        prior[
            [
                "uncorrected_psnr", "oracle_psnr", "clean_psnr",
                "oracle_gain_db", "msi_corrected_core_mse"
            ]
        ].to_numpy(float)
    ).all():
        raise ValueError("Invalid numerical records in resume file.")

    if any(r not in RUNS for r in prior["run"].unique()):
        raise ValueError("Resume file contains an unexpected model.")

    # Recheck archived values so stale results aren't silently reused.
    for row in prior.itertuples(index=False):
        key = (row.image, int(row.dy), int(row.dx))
        if key not in archives[row.run].index:
            raise ValueError(f"Unexpected resume key: {row.run}, {key}")
        original = float(
            archives[row.run].loc[key, "common_psnr"]
        )
        if abs(original - row.uncorrected_psnr) > 1e-5:
            raise ValueError("Resume baseline differs from original archive.")

    records = prior.to_dict("records")
    print("Resuming existing records:", len(records))

else:
    records = []

completed = {
    (r["run"], r["image"], int(r["dy"]), int(r["dx"]))
    for r in records
}

def save_records():
    df = pd.DataFrame(records, columns=COLS)
    df = df.sort_values(
        ["run", "image", "dy", "dx"]
    ).reset_index(drop=True)

    fd, temp = tempfile.mkstemp(
        dir=str(OUT),
        prefix="step08_",
        suffix=".csv"
    )

    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            df.to_csv(f, index=False)
        os.replace(temp, CSV_PATH)
    finally:
        if os.path.exists(temp):
            os.remove(temp)

    return df

# -------------------------------------------------------------------------
# 5. ORACLE CORRECTION INFERENCE
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("8C — ORACLE CORRECTION EXPERIMENT", flush=True)
print("=" * 88, flush=True)

start = time.monotonic()

for run_index, run in enumerate(RUNS, 1):

    model_name = run.split("_")[0]
    regime = run.replace(model_name + "_", "").rsplit("_s", 1)[0]
    seed = int(run.rsplit("_s", 1)[1])

    pending = sum(
        (run, p, dy, dx) not in completed
        for p in PATCHES
        for dy, dx in OFFSETS
    )

    print(
        f"\nRUN {run_index}/{len(RUNS)}: {run} | "
        f"Remaining comparisons: {pending}",
        flush=True
    )

    if not pending:
        continue

    ckpt = torch.load(
        EXPS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    model = CLASSES[model_name](config=CONFIG)
    model.load_state_dict(ckpt["model"], strict=True)
    model = model.to(DEVICE).eval()

    print("STRICT LOAD PASSED", flush=True)

    run_start = time.monotonic()

    for patch_index, image in enumerate(PATCHES, 1):

        missing = [
            (dy, dx)
            for dy, dx in OFFSETS
            if (run, image, dy, dx) not in completed
        ]

        if not missing:
            continue

        lr = load_patch("lrhsi", image)
        msi = load_patch("hrmsi", image)
        ref = load_patch("hrhsi", image)

        with torch.inference_mode():
            clean_prediction = predict(model, lr, msi)
            clean_psnr = psnr_original(clean_prediction, ref)

            # Reconfirm the zero-displacement archived score.
            archived_clean = float(
                archives[run].loc[(image, 0, 0), "common_psnr"]
            )

            if abs(clean_psnr - archived_clean) > TOLERANCE:
                raise AssertionError(
                    f"Clean baseline disagreement: {run}, {image}"
                )

            for dy, dx in missing:

                distorted = shift_image(msi, dy, dx)

                # Oracle knowledge: use the true, known offset.
                corrected = shift_image(distorted, -dy, -dx)

                oracle_prediction = predict(model, lr, corrected)
                oracle_psnr = psnr_original(oracle_prediction, ref)

                uncorrected = float(
                    archives[run].loc[
                        (image, dy, dx), "common_psnr"
                    ]
                )

                rows = {
                    "run": run,
                    "model": model_name,
                    "regime": regime,
                    "seed": seed,
                    "image": image,
                    "dy": dy,
                    "dx": dx,
                    "distance": abs(dy) + abs(dx),
                    "uncorrected_psnr": uncorrected,
                    "oracle_psnr": oracle_psnr,
                    "clean_psnr": clean_psnr,
                    "oracle_gain_db": oracle_psnr - uncorrected,
                    "oracle_vs_clean_db": oracle_psnr - clean_psnr,
                    "uncorrected_vs_clean_db": uncorrected - clean_psnr,
                    "msi_uncorrected_core_mse": core_msi_mse(
                        distorted, msi
                    ),
                    "msi_corrected_core_mse": core_msi_mse(
                        corrected, msi
                    )
                }

                records.append(rows)
                completed.add((run, image, dy, dx))

        # Save one completed patch at a time.
        save_records()

        if patch_index % 5 == 0 or patch_index == len(PATCHES):
            run_rows = [r for r in records if r["run"] == run]

            print(
                f"  Patch {patch_index:02d}/40 | "
                f"Records {len(run_rows):03d}/520 | "
                f"Elapsed {time.monotonic()-run_start:.1f}s",
                flush=True
            )

        del lr, msi, ref, clean_prediction

    del model, ckpt
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. FINAL INTEGRITY AND SUMMARY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("8D — RESULTS AND QUALITY CONTROL")
print("=" * 88)

result = save_records()

assert len(result) == EXPECTED_ROWS
assert not result.duplicated(["run", "image", "dy", "dx"]).any()
assert np.isfinite(
    result[
        ["uncorrected_psnr", "oracle_psnr", "clean_psnr"]
    ].to_numpy()
).all()

# Corrected MSI interior should match original exactly under
# the integer-translation/replicate-boundary construction.
MAX_CORE_MSE = float(
    result["msi_corrected_core_mse"].max()
)

print("\nMaximum corrected-MSI interior MSE:", MAX_CORE_MSE)

if MAX_CORE_MSE > 1e-12:
    raise AssertionError(
        "Oracle geometric correction did not recover the MSI interior."
    )

# Important: the fusion network can still see altered boundary context.
# Thus oracle-fusion PSNR may differ from clean-fusion PSNR.
summary = (
    result.groupby(["model", "regime", "distance"])
    .agg(
        n_rows=("oracle_psnr", "size"),
        uncorrected_psnr=("uncorrected_psnr", "mean"),
        oracle_psnr=("oracle_psnr", "mean"),
        clean_psnr=("clean_psnr", "mean"),
        gain_db=("oracle_gain_db", "mean"),
        oracle_clean_gap_db=("oracle_vs_clean_db", "mean"),
        msi_corrected_mse=("msi_corrected_core_mse", "mean")
    )
    .reset_index()
)

summary.to_csv(
    OUT / "Step08_Oracle_Condition_Summary.csv",
    index=False
)

print("\nCONDITION-LEVEL RESULTS (MEAN OVER 40 PATCHES)")
print(summary.round(5).to_string(index=False))

# Per-image records remain available for paired spatial comparisons.
patch_summary = (
    result.groupby(
        ["model", "regime", "seed", "image", "distance"]
    )
    .agg(
        uncorrected_psnr=("uncorrected_psnr", "mean"),
        oracle_psnr=("oracle_psnr", "mean"),
        clean_psnr=("clean_psnr", "mean"),
        oracle_gain_db=("oracle_gain_db", "mean")
    )
    .reset_index()
)

patch_summary.to_csv(
    OUT / "Step08_Patch_Level_Condition_Summary.csv",
    index=False
)

report = {
    "models": len(RUNS),
    "held_out_test_patches": len(PATCHES),
    "offsets": len(OFFSETS),
    "expected_rows": EXPECTED_ROWS,
    "completed_rows": len(result),
    "maximum_corrected_msi_core_mse": MAX_CORE_MSE,
    "maximum_absolute_oracle_clean_psnr_difference_db": float(
        result["oracle_vs_clean_db"].abs().max()
    ),
    "oracle_fusion_is_not_assumed_equal_to_clean": True,
    "reference_hsi_used_for_alignment_estimation": False,
    "oracle_offsets_are_ground_truth": True,
    "experiment_type": "known-offset synthetic translation baseline",
    "training_seeds_evaluated": [7],
    "full_completion": len(result) == EXPECTED_ROWS,
    "elapsed_seconds": round(time.monotonic() - start, 2),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step08_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 8 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 8 COMPLETED.")
print("Original checkpoints and source code remain unchanged.")


### How we will interpret the results

#If the oracle-offset correction substantially improves PSNR, we have evidence that explicitly correcting displacement can recover some of the fusion accuracy lost to misregistration. This provides a benchmark for a practical alignment estimator.

#It will not yet demonstrate a new registration-aware fusion method. An oracle uses displacement information unavailable during ordinary deployment.

#**Send me the Step 8 condition-level results and final report.** The following stage will replace true offsets with image-estimated offsets and evaluate both registration error and fusion accuracy under the same controlled protocol.

In [ ]:

# =============================================================================
# STEP 9 — IMAGE-ESTIMATED CROSS-SENSOR REGISTRATION + FUSION
#
# Objective:
#   Estimate HR-MSI displacement from LR-HSI and HR-MSI alone.
#
# Protocol:
#   - Fit spectral projection on 133 TRAIN patches only.
#   - Use fixed cross-sensor registration scoring.
#   - Search integer offsets in [-4, +4] pixels.
#   - Evaluate 40 held-out test patches × 13 displacements.
#   - Compare estimated correction with archived uncorrected,
#     oracle-offset correction, and clean-input performance.
#   - Evaluate four verified checkpoints from seed 7.
#
# No HR-HSI reference is used to estimate registration.
# Ground-truth offsets are used ONLY for accuracy assessment.
# Original models/checkpoints/datasets are unchanged.
# =============================================================================

from google.colab import drive
from pathlib import Path
from scipy.ndimage import gaussian_filter, sobel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

import os
import sys
import json
import time
import tempfile
import hashlib

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile

# -------------------------------------------------------------------------
# 1. PATHS AND REPRODUCIBLE SETTINGS
# -------------------------------------------------------------------------

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
DATA = REPO / "datasets" / "mdas_new"
STEP8 = PROJECT / "Step08_Oracle_Correction"
OUT = PROJECT / "Step09_Estimated_Registration"

OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RANDOM_SEED = 2026
TRAIN_SAMPLES_PER_PATCH = 64
RIDGE_ALPHA = 100.0
BLUR_SIGMA = 1.2
SEARCH_RADIUS = 4
CROP_MARGIN = 8
INTENSITY_WEIGHT = 0.7
GRADIENT_WEIGHT = 0.3

MODELS = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

OFFSETS = [
    (0, 0),
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (2, 0), (-2, 0), (0, 2), (0, -2),
    (4, 0), (-4, 0), (0, 4), (0, -4)
]

print("=" * 88, flush=True)
print("STEP 9 — IMAGE-ESTIMATED REGISTRATION", flush=True)
print("=" * 88, flush=True)
print("Device:", DEVICE)
print("Search radius:", SEARCH_RADIUS)
print("Output directory:", OUT)

# -------------------------------------------------------------------------
# 2. RECOVER ORIGINAL CONFIGURATION
# -------------------------------------------------------------------------

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"

NORMALIZER = float(CONFIG["mdasn_dataset"]["max_value"])

assert NORMALIZER == 15000.0

def read_split(name):
    names = [
        line.strip().split()[0]
        for line in (DATA / f"{name}.txt").read_text().splitlines()
        if line.strip()
    ]
    assert len(names) == len(set(names))
    return names

TRAIN = read_split("train")
VAL = read_split("val")
TEST = read_split("test")

assert len(TRAIN) == 133
assert len(VAL) == 32
assert len(TEST) == 40

assert not (set(TRAIN) & set(VAL))
assert not (set(TRAIN) & set(TEST))
assert not (set(VAL) & set(TEST))

print("\nTrain:", len(TRAIN))
print("Validation:", len(VAL))
print("Test:", len(TEST))

# -------------------------------------------------------------------------
# 3. MODALITY LOADING
# -------------------------------------------------------------------------

def load_numpy(modality, filename):
    arr = tifffile.imread(
        DATA / modality / filename
    ).astype(np.float32)

    arr /= NORMALIZER

    if not np.isfinite(arr).all():
        raise ValueError(
            f"Non-finite input: {modality}/{filename}"
        )

    return arr

def to_tensor(arr):
    return torch.from_numpy(
        np.ascontiguousarray(arr.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)

def upsample_hsi(lrhsi):
    """
    Reconstruct a 72×72 feature map from the 24×24 LR-HSI.
    This interpolation is used only for registration features.
    """
    tensor = to_tensor(lrhsi)

    with torch.inference_mode():
        up = F.interpolate(
            tensor,
            size=(72, 72),
            mode="bicubic",
            align_corners=False
        )

    return (
        up.squeeze(0)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
        .astype(np.float32)
    )

# -------------------------------------------------------------------------
# 4. TRAIN-ONLY SPECTRAL PROJECTION
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("9A — FITTING TRAIN-ONLY HSI → MSI SPECTRAL MAPPING", flush=True)
print("=" * 88, flush=True)

rng = np.random.default_rng(RANDOM_SEED)

X_parts = []
Y_parts = []

for i, image in enumerate(TRAIN, 1):

    lr = load_numpy("lrhsi", image)
    msi = load_numpy("hrmsi", image)

    if lr.shape != (24, 24, 242):
        raise ValueError(f"Unexpected LR-HSI dimensions: {image}")

    if msi.shape != (72, 72, 4):
        raise ValueError(f"Unexpected MSI dimensions: {image}")

    up = upsample_hsi(lr)

    # Sample only the interior of aligned TRAIN data.
    grid_y, grid_x = np.mgrid[8:64, 8:64]

    coordinates = np.column_stack(
        [grid_y.ravel(), grid_x.ravel()]
    )

    choice = rng.choice(
        len(coordinates),
        size=TRAIN_SAMPLES_PER_PATCH,
        replace=False
    )

    positions = coordinates[choice]
    yy = positions[:, 0]
    xx = positions[:, 1]

    X_parts.append(up[yy, xx, :])
    Y_parts.append(msi[yy, xx, :])

    if i % 25 == 0 or i == len(TRAIN):
        print(
            f"  Training patch {i}/{len(TRAIN)} processed",
            flush=True
        )

X_train = np.concatenate(X_parts, axis=0)
Y_train = np.concatenate(Y_parts, axis=0)

print("\nTraining spectral pairs:", X_train.shape)
print("Target MSI observations:", Y_train.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

regressor = Ridge(
    alpha=RIDGE_ALPHA,
    fit_intercept=True
)

regressor.fit(X_scaled, Y_train)

fitted = regressor.predict(X_scaled)

print("\nTraining spectral fit:")
for b in range(4):
    print(
        f"  MSI band {b+1}: "
        f"R² = {r2_score(Y_train[:, b], fitted[:, b]):.4f}"
    )

# Save plain numerical arrays instead of serializing executable objects.
np.savez_compressed(
    OUT / "Step09_Spectral_Projection.npz",
    scaler_mean=scaler.mean_,
    scaler_scale=scaler.scale_,
    ridge_coefficients=regressor.coef_,
    ridge_intercept=regressor.intercept_
)

def project_hsi_to_msi(lrhsi):

    up = upsample_hsi(lrhsi)
    h, w, c = up.shape

    standardized = (
        up.reshape(-1, c) - scaler.mean_
    ) / scaler.scale_

    projected = regressor.predict(standardized)

    return projected.reshape(h, w, 4).astype(np.float32)

# -------------------------------------------------------------------------
# 5. VALIDATION-SET SPECTRAL DIAGNOSTIC
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("9B — SPECTRAL MAPPING VALIDATION", flush=True)
print("=" * 88, flush=True)

validation_records = []

for image in VAL:

    lr = load_numpy("lrhsi", image)
    observed = load_numpy("hrmsi", image)
    projected = project_hsi_to_msi(lr)

    # Validation R² is diagnostic only.
    # No hyperparameter tuning is performed here.
    for b in range(4):
        validation_records.append({
            "image": image,
            "band": b + 1,
            "r2": r2_score(
                observed[8:-8, 8:-8, b].ravel(),
                projected[8:-8, 8:-8, b].ravel()
            )
        })

val_df = pd.DataFrame(validation_records)
val_df.to_csv(
    OUT / "Step09_Spectral_Validation.csv",
    index=False
)

print(
    val_df.groupby("band")["r2"]
    .agg(["mean", "median", "min", "max"])
    .round(4)
    .to_string()
)

# -------------------------------------------------------------------------
# 6. CROSS-SENSOR REGISTRATION FEATURES
# -------------------------------------------------------------------------

def registration_features(msi_like):
    """
    Compute smoothed intensity and gradient representations.

    Input: H × W × 4
    Output:
       intensity: H × W × 4
       gradient:  H × W × 4
    """
    blurred = gaussian_filter(
        msi_like.astype(np.float32),
        sigma=(BLUR_SIGMA, BLUR_SIGMA, 0)
    )

    gx = sobel(blurred, axis=1, mode="nearest")
    gy = sobel(blurred, axis=0, mode="nearest")

    gradient = np.sqrt(gx * gx + gy * gy)

    return blurred, gradient

def translate_numpy(arr, dy, dx):
    """
    Exact replicate-edge translation matching previous PyTorch logic.
    """
    dy = int(dy)
    dx = int(dx)

    h, w = arr.shape[:2]

    pad_top = max(dy, 0)
    pad_bottom = max(-dy, 0)
    pad_left = max(dx, 0)
    pad_right = max(-dx, 0)

    padded = np.pad(
        arr,
        (
            (pad_top, pad_bottom),
            (pad_left, pad_right),
            (0, 0)
        ),
        mode="edge"
    )

    top = max(-dy, 0)
    left = max(-dx, 0)

    return np.ascontiguousarray(
        padded[top:top+h, left:left+w, :]
    )

def normalized_multiband_correlation(a, b):
    """
    Mean bandwise normalized cross-correlation.
    Does not use reference HR-HSI or ground-truth shifts.
    """
    a = a[CROP_MARGIN:-CROP_MARGIN,
          CROP_MARGIN:-CROP_MARGIN, :]

    b = b[CROP_MARGIN:-CROP_MARGIN,
          CROP_MARGIN:-CROP_MARGIN, :]

    a = a.reshape(-1, a.shape[-1]).astype(np.float64)
    b = b.reshape(-1, b.shape[-1]).astype(np.float64)

    a -= a.mean(axis=0, keepdims=True)
    b -= b.mean(axis=0, keepdims=True)

    numerator = np.sum(a * b, axis=0)

    denominator = np.sqrt(
        np.sum(a*a, axis=0)
        * np.sum(b*b, axis=0)
    )

    valid = denominator > 1e-12

    if not np.any(valid):
        return -1.0

    correlations = numerator[valid] / denominator[valid]

    return float(np.mean(correlations))

def registration_score(target_features, observed_features,
                       correction_dy, correction_dx):

    target_intensity, target_gradient = target_features
    observed_intensity, observed_gradient = observed_features

    shifted_intensity = translate_numpy(
        observed_intensity,
        correction_dy,
        correction_dx
    )

    shifted_gradient = translate_numpy(
        observed_gradient,
        correction_dy,
        correction_dx
    )

    intensity_score = normalized_multiband_correlation(
        target_intensity,
        shifted_intensity
    )

    gradient_score = normalized_multiband_correlation(
        target_gradient,
        shifted_gradient
    )

    combined = (
        INTENSITY_WEIGHT * intensity_score
        + GRADIENT_WEIGHT * gradient_score
    )

    return combined, intensity_score, gradient_score

def estimate_displacement(projected_msi, observed_msi):
    """
    Return estimated displacement of OBSERVED MSI relative
    to the HSI-derived target.

    A candidate correction is applied to observed MSI.
    Hence estimated displacement = negative correction.

    The known synthetic displacement is NEVER supplied here.
    """
    target_features = registration_features(projected_msi)
    observed_features = registration_features(observed_msi)

    candidates = []

    for correction_dy in range(
        -SEARCH_RADIUS, SEARCH_RADIUS + 1
    ):
        for correction_dx in range(
            -SEARCH_RADIUS, SEARCH_RADIUS + 1
        ):

            combined, intensity, gradient = registration_score(
                target_features,
                observed_features,
                correction_dy,
                correction_dx
            )

            candidates.append({
                "estimated_dy": -correction_dy,
                "estimated_dx": -correction_dx,
                "correction_dy": correction_dy,
                "correction_dx": correction_dx,
                "score": combined,
                "intensity_score": intensity,
                "gradient_score": gradient
            })

    candidates.sort(
        key=lambda r: r["score"],
        reverse=True
    )

    best = candidates[0]
    second = candidates[1]

    best["score_margin"] = (
        best["score"] - second["score"]
    )

    best["zero_shift_score"] = next(
        r["score"] for r in candidates
        if r["estimated_dy"] == 0
        and r["estimated_dx"] == 0
    )

    return best

# -------------------------------------------------------------------------
# 7. INDEPENDENT TRANSLATION UNIT TEST
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("9C — SHIFT-CONVENTION VERIFICATION")
print("=" * 88)

toy = np.arange(
    20 * 20 * 4, dtype=np.float32
).reshape(20, 20, 4)

for dy, dx in OFFSETS:

    moved_np = translate_numpy(toy, dy, dx)

    moved_torch = (
        F.pad(
            torch.from_numpy(toy).permute(2, 0, 1).unsqueeze(0),
            (
                max(dx, 0), max(-dx, 0),
                max(dy, 0), max(-dy, 0)
            ),
            mode="replicate"
        )
    )

    top = max(-dy, 0)
    left = max(-dx, 0)

    moved_torch = moved_torch[
        :, :, top:top+20, left:left+20
    ].squeeze().permute(1, 2, 0).numpy()

    assert np.array_equal(moved_np, moved_torch)

print("PASS: NumPy and PyTorch translations are identical.")

# -------------------------------------------------------------------------
# 8. ESTIMATE THE 520 TEST REGISTRATIONS
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("9D — ESTIMATING 520 UNKNOWN DISPLACEMENTS", flush=True)
print("=" * 88, flush=True)

REGISTRATION_CSV = OUT / "Step09_Registration_Results.csv"

registration_rows = []

start = time.monotonic()

for patch_number, image in enumerate(TEST, 1):

    lr = load_numpy("lrhsi", image)
    clean_msi = load_numpy("hrmsi", image)

    predicted_msi = project_hsi_to_msi(lr)

    for actual_dy, actual_dx in OFFSETS:

        shifted_msi = translate_numpy(
            clean_msi,
            actual_dy,
            actual_dx
        )

        estimate = estimate_displacement(
            predicted_msi,
            shifted_msi
        )

        estimated_dy = int(estimate["estimated_dy"])
        estimated_dx = int(estimate["estimated_dx"])

        error_dy = estimated_dy - actual_dy
        error_dx = estimated_dx - actual_dx

        registration_rows.append({
            "image": image,
            "actual_dy": actual_dy,
            "actual_dx": actual_dx,
            "distance": abs(actual_dy) + abs(actual_dx),
            "estimated_dy": estimated_dy,
            "estimated_dx": estimated_dx,
            "error_dy": error_dy,
            "error_dx": error_dx,
            "endpoint_error": float(
                np.hypot(error_dy, error_dx)
            ),
            "exact_match": int(
                error_dy == 0 and error_dx == 0
            ),
            "registration_score": estimate["score"],
            "score_margin": estimate["score_margin"],
            "zero_shift_score": estimate["zero_shift_score"],
            "intensity_score": estimate["intensity_score"],
            "gradient_score": estimate["gradient_score"]
        })

    # Save a complete snapshot after every test image.
    pd.DataFrame(registration_rows).to_csv(
        REGISTRATION_CSV,
        index=False
    )

    if patch_number % 5 == 0 or patch_number == len(TEST):
        current = pd.DataFrame(registration_rows)

        print(
            f"  Patch {patch_number:02d}/40 | "
            f"Registrations {len(current):03d}/520 | "
            f"Exact {current['exact_match'].mean()*100:.2f}% | "
            f"Mean EPE {current['endpoint_error'].mean():.3f} px | "
            f"Elapsed {time.monotonic()-start:.1f}s",
            flush=True
        )

registration_df = pd.DataFrame(registration_rows)

assert len(registration_df) == 520
assert not registration_df.duplicated(
    ["image", "actual_dy", "actual_dx"]
).any()

registration_summary = (
    registration_df.groupby("distance")
    .agg(
        n=("exact_match", "size"),
        exact_accuracy=("exact_match", "mean"),
        mean_epe=("endpoint_error", "mean"),
        median_epe=("endpoint_error", "median"),
        max_epe=("endpoint_error", "max"),
        mean_score_margin=("score_margin", "mean")
    )
    .reset_index()
)

registration_summary.to_csv(
    OUT / "Step09_Registration_Summary.csv",
    index=False
)

print("\nREGISTRATION ACCURACY BY DISPLACEMENT")
print(registration_summary.round(4).to_string(index=False))

# -------------------------------------------------------------------------
# 9. ARCHIVED AND ORACLE BASELINES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("9E — VERIFYING EXISTING FUSION BASELINES")
print("=" * 88)

oracle_path = STEP8 / "Step08_Oracle_Patch_Results.csv"

if not oracle_path.is_file():
    raise FileNotFoundError(
        "Step 8 oracle results are required: " + str(oracle_path)
    )

oracle = pd.read_csv(oracle_path)

assert len(oracle) == 2080

oracle["dy"] = oracle["dy"].astype(int)
oracle["dx"] = oracle["dx"].astype(int)

oracle = oracle.set_index(
    ["run", "image", "dy", "dx"]
)

assert oracle.index.is_unique

estimates = registration_df.set_index(
    ["image", "actual_dy", "actual_dx"]
)

assert estimates.index.is_unique

print("PASS: Oracle comparisons:", len(oracle))
print("PASS: Estimated shifts:", len(estimates))

# -------------------------------------------------------------------------
# 10. MODEL INFERENCE WITH ESTIMATED CORRECTION
# -------------------------------------------------------------------------

def shift_tensor(tensor, dy, dx):
    dy = int(dy)
    dx = int(dx)

    h, w = tensor.shape[-2:]

    padded = F.pad(
        tensor,
        (
            max(dx, 0), max(-dx, 0),
            max(dy, 0), max(-dy, 0)
        ),
        mode="replicate"
    )

    top = max(-dy, 0)
    left = max(-dx, 0)

    return padded[
        :, :, top:top+h, left:left+w
    ].contiguous()

def make_tensor(arr):
    return to_tensor(arr)

def predict(model, lr, msi):
    output = model(lr, msi)

    if isinstance(output, dict):
        output = output["pred"]

    if tuple(output.shape) != (1, 242, 72, 72):
        raise ValueError(
            f"Unexpected model output: {tuple(output.shape)}"
        )

    if not torch.isfinite(output).all():
        raise ValueError("Non-finite output")

    return output

def compute_psnr(pred, ref):
    a = pred[:, :, 4:-4, 4:-4].contiguous()
    b = ref[:, :, 4:-4, 4:-4].contiguous()
    return float(PSNR(a, b).detach().cpu().item())

FUSION_CSV = OUT / "Step09_Estimated_Fusion_Results.csv"

fusion_columns = [
    "run", "model", "regime", "seed",
    "image", "dy", "dx", "distance",
    "estimated_dy", "estimated_dx",
    "endpoint_error", "exact_match",
    "uncorrected_psnr", "estimated_psnr",
    "oracle_psnr", "clean_psnr",
    "estimated_gain_db",
    "estimated_minus_oracle_db",
    "estimated_minus_clean_db"
]

if FUSION_CSV.exists():
    existing = pd.read_csv(FUSION_CSV)

    if not set(fusion_columns).issubset(existing.columns):
        raise ValueError("Existing fusion CSV schema differs.")

    existing = existing[fusion_columns].copy()

    if existing.duplicated(["run", "image", "dy", "dx"]).any():
        raise ValueError("Duplicate resumed inference records.")

    # Check that stored estimates agree with the current
    # registration experiment before resuming.
    for r in existing.itertuples(index=False):
        e = estimates.loc[(r.image, int(r.dy), int(r.dx))]
        if (
            int(r.estimated_dy) != int(e["estimated_dy"])
            or int(r.estimated_dx) != int(e["estimated_dx"])
        ):
            raise ValueError(
                "Existing fusion results use different registration "
                "estimates. Preserve or relocate the old output first."
            )

    fusion_rows = existing.to_dict("records")
    print("Resumed fusion rows:", len(fusion_rows))

else:
    fusion_rows = []

completed = {
    (r["run"], r["image"], int(r["dy"]), int(r["dx"]))
    for r in fusion_rows
}

def save_fusion():

    df = pd.DataFrame(
        fusion_rows,
        columns=fusion_columns
    )

    df = df.sort_values(
        ["run", "image", "dy", "dx"]
    ).reset_index(drop=True)

    fd, temp = tempfile.mkstemp(
        dir=str(OUT),
        prefix="step09_",
        suffix=".csv"
    )

    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            df.to_csv(f, index=False)

        os.replace(temp, FUSION_CSV)

    finally:
        if os.path.exists(temp):
            os.remove(temp)

    return df

print("\n" + "=" * 88, flush=True)
print("9F — FUSION USING IMAGE-ESTIMATED DISPLACEMENTS", flush=True)
print("=" * 88, flush=True)

for run_number, run in enumerate(RUNS, 1):

    model_name = run.split("_")[0]
    regime = run.replace(model_name + "_", "").rsplit("_s", 1)[0]
    seed = int(run.rsplit("_s", 1)[1])

    pending = sum(
        (run, image, dy, dx) not in completed
        for image in TEST for dy, dx in OFFSETS
    )

    print(
        f"\nRUN {run_number}/4: {run} | Pending: {pending}",
        flush=True
    )

    if pending == 0:
        continue

    checkpoint = torch.load(
        EXPS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    model = MODELS[model_name](config=CONFIG)
    model.load_state_dict(checkpoint["model"], strict=True)
    model = model.to(DEVICE).eval()

    print("STRICT CHECKPOINT LOAD: PASSED", flush=True)

    for patch_index, image in enumerate(TEST, 1):

        missing = [
            (dy, dx) for dy, dx in OFFSETS
            if (run, image, dy, dx) not in completed
        ]

        if not missing:
            continue

        lr = make_tensor(load_numpy("lrhsi", image))
        msi = make_tensor(load_numpy("hrmsi", image))
        ref = make_tensor(load_numpy("hrhsi", image))

        with torch.inference_mode():

            for dy, dx in missing:

                estimate = estimates.loc[(image, dy, dx)]

                estimated_dy = int(estimate["estimated_dy"])
                estimated_dx = int(estimate["estimated_dx"])

                distorted = shift_tensor(msi, dy, dx)

                # Correction uses ONLY the image-estimated offset.
                corrected = shift_tensor(
                    distorted,
                    -estimated_dy,
                    -estimated_dx
                )

                reconstructed = predict(
                    model, lr, corrected
                )

                estimated_psnr = compute_psnr(
                    reconstructed, ref
                )

                reference_row = oracle.loc[
                    (run, image, dy, dx)
                ]

                uncorrected = float(
                    reference_row["uncorrected_psnr"]
                )

                oracle_psnr = float(
                    reference_row["oracle_psnr"]
                )

                clean_psnr = float(
                    reference_row["clean_psnr"]
                )

                fusion_rows.append({
                    "run": run,
                    "model": model_name,
                    "regime": regime,
                    "seed": seed,
                    "image": image,
                    "dy": dy,
                    "dx": dx,
                    "distance": abs(dy) + abs(dx),
                    "estimated_dy": estimated_dy,
                    "estimated_dx": estimated_dx,
                    "endpoint_error": float(
                        estimate["endpoint_error"]
                    ),
                    "exact_match": int(
                        estimate["exact_match"]
                    ),
                    "uncorrected_psnr": uncorrected,
                    "estimated_psnr": estimated_psnr,
                    "oracle_psnr": oracle_psnr,
                    "clean_psnr": clean_psnr,
                    "estimated_gain_db":
                        estimated_psnr - uncorrected,
                    "estimated_minus_oracle_db":
                        estimated_psnr - oracle_psnr,
                    "estimated_minus_clean_db":
                        estimated_psnr - clean_psnr
                })

                completed.add((run, image, dy, dx))

        save_fusion()

        if patch_index % 5 == 0 or patch_index == len(TEST):
            count = sum(
                r["run"] == run
                for r in fusion_rows
            )

            print(
                f"  Patch {patch_index:02d}/40 | "
                f"Model records {count:03d}/520",
                flush=True
            )

        del lr, msi, ref

    del model, checkpoint

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 11. SUMMARIZE THE ESTIMATED CORRECTION RESULTS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("9G — FUSION PERFORMANCE SUMMARY")
print("=" * 88)

fusion_df = save_fusion()

assert len(fusion_df) == 2080
assert not fusion_df.duplicated(
    ["run", "image", "dy", "dx"]
).any()

numeric = [
    "uncorrected_psnr", "estimated_psnr",
    "oracle_psnr", "clean_psnr",
    "estimated_gain_db"
]

assert np.isfinite(
    fusion_df[numeric].to_numpy(float)
).all()

fusion_summary = (
    fusion_df.groupby(["model", "regime", "distance"])
    .agg(
        n=("image", "size"),
        uncorrected_psnr=("uncorrected_psnr", "mean"),
        estimated_psnr=("estimated_psnr", "mean"),
        oracle_psnr=("oracle_psnr", "mean"),
        clean_psnr=("clean_psnr", "mean"),
        estimated_gain_db=("estimated_gain_db", "mean"),
        mean_epe=("endpoint_error", "mean"),
        exact_accuracy=("exact_match", "mean")
    )
    .reset_index()
)

fusion_summary.to_csv(
    OUT / "Step09_Fusion_Condition_Summary.csv",
    index=False
)

print(fusion_summary.round(4).to_string(index=False))

# Save per-image, per-shift-direction results for paired comparisons.
fusion_df.to_csv(
    OUT / "Step09_Fusion_Paired_Records.csv",
    index=False
)

# -------------------------------------------------------------------------
# 12. ADDITIONAL FAILURE ANALYSIS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("9H — REGISTRATION FAILURE DIAGNOSTICS")
print("=" * 88)

worst = (
    registration_df.sort_values(
        "endpoint_error",
        ascending=False
    )
    .head(20)
)

print("\n20 LARGEST GEOMETRIC ERRORS")
print(
    worst[
        [
            "image",
            "actual_dy", "actual_dx",
            "estimated_dy", "estimated_dx",
            "endpoint_error", "score_margin"
        ]
    ].to_string(index=False)
)

worst.to_csv(
    OUT / "Step09_Worst_Registration_Cases.csv",
    index=False
)

no_shift = registration_df[
    registration_df["distance"] == 0
]

print("\nALIGNED INPUT — FALSE CORRECTION RATE")
print(
    f"{100*(1-no_shift['exact_match'].mean()):.2f}%"
)

# -------------------------------------------------------------------------
# 13. FINAL REPORT
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("STEP 9 — FINAL REPORT")
print("=" * 88)

report = {
    "method": (
        "Training-calibrated HSI-to-MSI spectral projection "
        "with multiband NCC and gradient NCC integer search"
    ),
    "training_patches": len(TRAIN),
    "validation_patches": len(VAL),
    "test_patches": len(TEST),
    "registration_evaluations": len(registration_df),
    "fusion_evaluations": len(fusion_df),
    "search_radius_pixels": SEARCH_RADIUS,
    "registration_exact_accuracy": float(
        registration_df["exact_match"].mean()
    ),
    "registration_mean_epe_pixels": float(
        registration_df["endpoint_error"].mean()
    ),
    "aligned_false_correction_rate": float(
        1 - no_shift["exact_match"].mean()
    ),
    "reference_hrhsi_used_to_estimate_offsets": False,
    "ground_truth_shifts_used_to_estimate_offsets": False,
    "ground_truth_shifts_used_for_evaluation_only": True,
    "fusion_training_seeds": [7],
    "original_networks_modified": False,
    "output_directory": str(OUT)
}

with open(
    OUT / "Step09_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

print("\nSTEP 9 COMPLETE.")


#

In [ ]:

# =============================================================================
# STEP 10 — VALIDATION-CALIBRATED CROSS-SENSOR REGISTRATION
#
# Purpose:
# 1. Recover the Step 9 training-only spectral projection.
# 2. Estimate the zero-offset bias using 32 validation patches.
# 3. Freeze the bias correction before processing test results.
# 4. Evaluate validation shifts, then test shifts.
# 5. Measure corrected fusion PSNR using the 4 verified checkpoints.
#
# IMPORTANT:
# Test labels are used for evaluation, NEVER calibration.
# Since Step 9 test outcomes have already been examined,
# Step 10 is an exploratory method revision, not an independent
# final confirmatory experiment.
# =============================================================================

from google.colab import drive
from pathlib import Path
from scipy.ndimage import gaussian_filter, sobel

import os
import sys
import json
import time
import tempfile
import hashlib

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
DATA = REPO / "datasets" / "mdas_new"
EXPS = ROOT / "MDAS_JSTARS_Expanded"

STEP8 = PROJECT / "Step08_Oracle_Correction"
STEP9 = PROJECT / "Step09_Estimated_Registration"

OUT = PROJECT / "Step10_Validation_Calibration"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"
NORMALIZER = float(CONFIG["mdasn_dataset"]["max_value"])

assert NORMALIZER == 15000.0

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

MODEL_CLASSES = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

OFFSETS = [
    (0, 0),
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (2, 0), (-2, 0), (0, 2), (0, -2),
    (4, 0), (-4, 0), (0, 4), (0, -4)
]

SEARCH_RADIUS = 4
CROP_MARGIN = 8
BLUR_SIGMA = 1.2
INTENSITY_WEIGHT = 0.7
GRADIENT_WEIGHT = 0.3

print("=" * 88, flush=True)
print("STEP 10 — VALIDATION-CALIBRATED REGISTRATION", flush=True)
print("=" * 88, flush=True)
print("Device:", DEVICE, flush=True)
print("Output:", OUT, flush=True)

# -------------------------------------------------------------------------
# 10A — VERIFY DATA PARTITIONS
# -------------------------------------------------------------------------

def read_split(name):
    names = [
        line.strip().split()[0]
        for line in (DATA / f"{name}.txt").read_text().splitlines()
        if line.strip()
    ]
    assert len(names) == len(set(names))
    return names

TRAIN = read_split("train")
VAL = read_split("val")
TEST = read_split("test")

assert len(TRAIN) == 133
assert len(VAL) == 32
assert len(TEST) == 40

assert not (set(TRAIN) & set(VAL))
assert not (set(TRAIN) & set(TEST))
assert not (set(VAL) & set(TEST))

print("\n10A — DATA PARTITIONS")
print("Train:", len(TRAIN))
print("Validation:", len(VAL))
print("Test:", len(TEST))

# -------------------------------------------------------------------------
# 10B — RECOVER STEP 9 SPECTRAL PROJECTION
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("10B — RECOVERING FROZEN SPECTRAL PROJECTION")
print("=" * 88)

projection_path = STEP9 / "Step09_Spectral_Projection.npz"
assert projection_path.exists(), "Step 9 projection file missing."

with np.load(projection_path, allow_pickle=False) as z:
    scaler_mean = z["scaler_mean"].copy()
    scaler_scale = z["scaler_scale"].copy()
    ridge_coef = z["ridge_coefficients"].copy()
    ridge_intercept = z["ridge_intercept"].copy()

assert scaler_mean.shape == (242,)
assert scaler_scale.shape == (242,)
assert ridge_coef.shape == (4, 242)
assert ridge_intercept.shape == (4,)
assert np.all(scaler_scale > 0)

print("Spectral projection:", projection_path.name)
print("Input bands:", ridge_coef.shape[1])
print("Output bands:", ridge_coef.shape[0])

projection_sha = hashlib.sha256(
    projection_path.read_bytes()
).hexdigest()

def read_array(modality, name):
    arr = tifffile.imread(
        DATA / modality / name
    ).astype(np.float32) / NORMALIZER

    if not np.isfinite(arr).all():
        raise ValueError(f"Nonfinite input: {modality}/{name}")

    return arr

def to_tensor(arr):
    return torch.from_numpy(
        np.ascontiguousarray(arr.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)

def project_lrhsi(lrhsi):
    with torch.inference_mode():
        up = F.interpolate(
            to_tensor(lrhsi),
            size=(72, 72),
            mode="bicubic",
            align_corners=False
        )

    up = (
        up.squeeze(0)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )

    flat = up.reshape(-1, 242).astype(np.float64)
    standardized = (flat - scaler_mean) / scaler_scale

    prediction = standardized @ ridge_coef.T + ridge_intercept

    return prediction.reshape(72, 72, 4).astype(np.float32)

# -------------------------------------------------------------------------
# 10C — REPRODUCE ORIGINAL REGISTRATION SCORE
# -------------------------------------------------------------------------

def shift_np(arr, dy, dx):
    dy, dx = int(dy), int(dx)
    h, w = arr.shape[:2]

    pad = np.pad(
        arr,
        (
            (max(dy, 0), max(-dy, 0)),
            (max(dx, 0), max(-dx, 0)),
            (0, 0)
        ),
        mode="edge"
    )

    top = max(-dy, 0)
    left = max(-dx, 0)

    return np.ascontiguousarray(
        pad[top:top+h, left:left+w, :]
    )

def features(arr):
    blurred = gaussian_filter(
        arr.astype(np.float32),
        sigma=(BLUR_SIGMA, BLUR_SIGMA, 0)
    )

    gx = sobel(blurred, axis=1, mode="nearest")
    gy = sobel(blurred, axis=0, mode="nearest")

    gradient = np.sqrt(gx * gx + gy * gy)

    return blurred, gradient

def ncc(a, b):
    a = a[
        CROP_MARGIN:-CROP_MARGIN,
        CROP_MARGIN:-CROP_MARGIN
    ].reshape(-1, 4).astype(np.float64).copy()

    b = b[
        CROP_MARGIN:-CROP_MARGIN,
        CROP_MARGIN:-CROP_MARGIN
    ].reshape(-1, 4).astype(np.float64).copy()

    a -= a.mean(axis=0, keepdims=True)
    b -= b.mean(axis=0, keepdims=True)

    num = np.sum(a * b, axis=0)

    den = np.sqrt(
        np.sum(a*a, axis=0) *
        np.sum(b*b, axis=0)
    )

    valid = den > 1e-12

    if not np.any(valid):
        return -1.0

    return float(np.mean(num[valid] / den[valid]))

def estimate_shift(projected, observed):
    ti, tg = features(projected)
    oi, og = features(observed)

    candidates = []

    for cy in range(-SEARCH_RADIUS, SEARCH_RADIUS + 1):
        for cx in range(-SEARCH_RADIUS, SEARCH_RADIUS + 1):

            shifted_i = shift_np(oi, cy, cx)
            shifted_g = shift_np(og, cy, cx)

            score = (
                INTENSITY_WEIGHT * ncc(ti, shifted_i)
                + GRADIENT_WEIGHT * ncc(tg, shifted_g)
            )

            candidates.append((score, -cy, -cx))

    candidates.sort(key=lambda x: x[0], reverse=True)

    return {
        "raw_dy": int(candidates[0][1]),
        "raw_dx": int(candidates[0][2]),
        "score": float(candidates[0][0]),
        "score_margin": float(
            candidates[0][0] - candidates[1][0]
        )
    }

# -------------------------------------------------------------------------
# 10D — LEARN SPATIAL BIAS FROM ALIGNED VALIDATION PAIRS
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("10D — CALIBRATING BIAS ON 32 VALIDATION PATCHES", flush=True)
print("=" * 88, flush=True)

calibration_rows = []
validation_cache = {}

for idx, image in enumerate(VAL, 1):
    lr = read_array("lrhsi", image)
    msi = read_array("hrmsi", image)

    projected = project_lrhsi(lr)
    validation_cache[image] = (projected, msi)

    estimate = estimate_shift(projected, msi)

    calibration_rows.append({
        "image": image,
        "raw_dy": estimate["raw_dy"],
        "raw_dx": estimate["raw_dx"],
        "score_margin": estimate["score_margin"]
    })

    if idx % 8 == 0:
        print(
            f"  Aligned validation pairs: {idx}/32",
            flush=True
        )

calibration = pd.DataFrame(calibration_rows)
calibration.to_csv(
    OUT / "Step10_Validation_Origin_Estimates.csv",
    index=False
)

print("\nRAW ZERO-SHIFT ESTIMATE FREQUENCIES")
print(
    calibration.groupby(["raw_dy", "raw_dx"])
    .size()
    .sort_values(ascending=False)
    .to_string()
)

bias_dy = int(np.rint(calibration["raw_dy"].median()))
bias_dx = int(np.rint(calibration["raw_dx"].median()))

calibration_exact = float(
    (
        (calibration["raw_dy"] == bias_dy)
        & (calibration["raw_dx"] == bias_dx)
    ).mean()
)

print("\nValidation-derived origin bias:")
print("  dy:", bias_dy)
print("  dx:", bias_dx)
print("Validation agreement with modal/median bias:",
      f"{calibration_exact*100:.2f}%")

# Bias must be fixed before test labels are examined here.
frozen = {
    "bias_dy": bias_dy,
    "bias_dx": bias_dx,
    "calibration_n": len(VAL),
    "projection_sha256": projection_sha,
    "method": "median raw registration on aligned validation pairs",
    "uses_test_labels": False
}

with open(
    OUT / "Step10_Frozen_Calibration.json",
    "w"
) as f:
    json.dump(frozen, f, indent=2)

# -------------------------------------------------------------------------
# 10E — INDEPENDENT VALIDATION SHIFT DIAGNOSTICS
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("10E — VALIDATION SHIFT DIAGNOSTICS")
print("=" * 88, flush=True)

val_rows = []

for idx, image in enumerate(VAL, 1):
    projected, clean_msi = validation_cache[image]

    for dy, dx in OFFSETS:
        distorted = shift_np(clean_msi, dy, dx)
        raw = estimate_shift(projected, distorted)

        corrected_dy = raw["raw_dy"] - bias_dy
        corrected_dx = raw["raw_dx"] - bias_dx

        val_rows.append({
            "image": image,
            "true_dy": dy,
            "true_dx": dx,
            "distance": abs(dy) + abs(dx),
            "raw_dy": raw["raw_dy"],
            "raw_dx": raw["raw_dx"],
            "calibrated_dy": corrected_dy,
            "calibrated_dx": corrected_dx,
            "raw_epe": float(np.hypot(
                raw["raw_dy"]-dy,
                raw["raw_dx"]-dx
            )),
            "calibrated_epe": float(np.hypot(
                corrected_dy-dy,
                corrected_dx-dx
            )),
            "calibrated_exact": int(
                corrected_dy == dy and corrected_dx == dx
            )
        })

    if idx % 8 == 0:
        print(f"  Validation images {idx}/32", flush=True)

val_results = pd.DataFrame(val_rows)
val_results.to_csv(
    OUT / "Step10_Validation_Shifts.csv",
    index=False
)

print("\nVALIDATION ACCURACY")
print(
    val_results.groupby("distance")
    .agg(
        n=("image", "size"),
        raw_epe=("raw_epe", "mean"),
        calibrated_epe=("calibrated_epe", "mean"),
        calibrated_exact=("calibrated_exact", "mean")
    )
    .round(5)
    .to_string()
)

# -------------------------------------------------------------------------
# 10F — APPLY FROZEN BIAS TO EXISTING TEST ESTIMATES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("10F — TEST REGISTRATION WITH FROZEN CALIBRATION")
print("=" * 88)

test_path = STEP9 / "Step09_Registration_Results.csv"
test_raw = pd.read_csv(test_path)

assert len(test_raw) == 520
assert not test_raw.duplicated(
    ["image", "actual_dy", "actual_dx"]
).any()

assert set(test_raw["image"]) == set(TEST)

test_raw["calibrated_dy"] = (
    test_raw["estimated_dy"].astype(int) - bias_dy
)

test_raw["calibrated_dx"] = (
    test_raw["estimated_dx"].astype(int) - bias_dx
)

test_raw["calibrated_epe"] = np.hypot(
    test_raw["calibrated_dy"] - test_raw["actual_dy"],
    test_raw["calibrated_dx"] - test_raw["actual_dx"]
)

test_raw["calibrated_exact"] = (
    (test_raw["calibrated_dy"] == test_raw["actual_dy"])
    & (test_raw["calibrated_dx"] == test_raw["actual_dx"])
).astype(int)

test_raw.to_csv(
    OUT / "Step10_Calibrated_Test_Registration.csv",
    index=False
)

print("\nTEST REGISTRATION ACCURACY")
print(
    test_raw.groupby("distance")
    .agg(
        n=("image", "size"),
        exact=("calibrated_exact", "mean"),
        mean_epe=("calibrated_epe", "mean"),
        median_epe=("calibrated_epe", "median"),
        max_epe=("calibrated_epe", "max")
    )
    .round(5)
    .to_string()
)

aligned = test_raw[test_raw["distance"] == 0]
aligned_false_rate = float(
    1 - aligned["calibrated_exact"].mean()
)

print("\nAligned-input false correction rate:",
      f"{100*aligned_false_rate:.2f}%")

# -------------------------------------------------------------------------
# 10G — VERIFY SAVED ORACLE BASELINES
# -------------------------------------------------------------------------

oracle = pd.read_csv(
    STEP8 / "Step08_Oracle_Patch_Results.csv"
)

assert len(oracle) == 2080

oracle = oracle.set_index(
    ["run", "image", "dy", "dx"]
)

assert oracle.index.is_unique

estimated = test_raw.set_index(
    ["image", "actual_dy", "actual_dx"]
)

assert estimated.index.is_unique

# -------------------------------------------------------------------------
# 10H — FUSION INFERENCE USING CALIBRATED OFFSETS
# -------------------------------------------------------------------------

def shift_torch(x, dy, dx):
    dy, dx = int(dy), int(dx)
    h, w = x.shape[-2:]

    padded = F.pad(
        x,
        (
            max(dx, 0), max(-dx, 0),
            max(dy, 0), max(-dy, 0)
        ),
        mode="replicate"
    )

    top = max(-dy, 0)
    left = max(-dx, 0)

    return padded[
        :, :, top:top+h, left:left+w
    ].contiguous()

def fusion_psnr(model, lr, msi, ref):
    with torch.inference_mode():
        out = model(lr, msi)

        if isinstance(out, dict):
            out = out["pred"]

        assert tuple(out.shape) == (1, 242, 72, 72)

        result = PSNR(
            out[:, :, 4:-4, 4:-4].contiguous(),
            ref[:, :, 4:-4, 4:-4].contiguous()
        )

    return float(result.detach().cpu().item())

OUTPUT_CSV = OUT / "Step10_Calibrated_Fusion.csv"

COLUMNS = [
    "run", "model", "regime", "seed", "image",
    "dy", "dx", "distance",
    "calibrated_dy", "calibrated_dx",
    "calibrated_epe", "calibrated_exact",
    "uncorrected_psnr", "calibrated_psnr",
    "oracle_psnr", "clean_psnr",
    "calibrated_gain_db",
    "calibrated_oracle_gap_db"
]

if OUTPUT_CSV.exists():
    previous = pd.read_csv(OUTPUT_CSV)

    if not set(COLUMNS).issubset(previous.columns):
        raise ValueError("Incompatible Step 10 resume file.")

    previous = previous[COLUMNS].copy()

    if previous.duplicated(
        ["run", "image", "dy", "dx"]
    ).any():
        raise ValueError("Duplicate resumed records.")

    for r in previous.itertuples(index=False):
        estimate = estimated.loc[
            (r.image, int(r.dy), int(r.dx))
        ]

        if (
            int(r.calibrated_dy) != int(estimate["calibrated_dy"])
            or int(r.calibrated_dx) != int(estimate["calibrated_dx"])
        ):
            raise ValueError(
                "Saved results use another calibration. "
                "Preserve the existing file before changing the method."
            )

    records = previous.to_dict("records")
    print("\nResuming existing fusion records:", len(records))

else:
    records = []

completed = {
    (r["run"], r["image"], int(r["dy"]), int(r["dx"]))
    for r in records
}

def save_records():
    df = pd.DataFrame(records, columns=COLUMNS)
    df = df.sort_values(
        ["run", "image", "dy", "dx"]
    ).reset_index(drop=True)

    fd, temp = tempfile.mkstemp(
        dir=str(OUT),
        prefix="step10_",
        suffix=".csv"
    )

    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            df.to_csv(f, index=False)

        os.replace(temp, OUTPUT_CSV)

    finally:
        if os.path.exists(temp):
            os.remove(temp)

    return df

print("\n" + "=" * 88, flush=True)
print("10H — CALIBRATED FUSION EVALUATION", flush=True)
print("=" * 88, flush=True)

for run_index, run in enumerate(RUNS, 1):
    model_name = run.split("_")[0]
    regime = run.replace(model_name + "_", "").rsplit("_s", 1)[0]
    seed = int(run.rsplit("_s", 1)[1])

    remaining = sum(
        (run, image, dy, dx) not in completed
        for image in TEST
        for dy, dx in OFFSETS
    )

    print(
        f"\nRUN {run_index}/4: {run} | "
        f"Remaining: {remaining}",
        flush=True
    )

    if remaining == 0:
        continue

    checkpoint = torch.load(
        EXPS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    model = MODEL_CLASSES[model_name](config=CONFIG)
    model.load_state_dict(checkpoint["model"], strict=True)
    model = model.to(DEVICE).eval()

    print("Strict checkpoint loading: PASS", flush=True)

    for patch_index, image in enumerate(TEST, 1):
        missing = [
            (dy, dx) for dy, dx in OFFSETS
            if (run, image, dy, dx) not in completed
        ]

        if not missing:
            continue

        lr = to_tensor(read_array("lrhsi", image))
        msi = to_tensor(read_array("hrmsi", image))
        ref = to_tensor(read_array("hrhsi", image))

        for dy, dx in missing:
            shift_row = estimated.loc[(image, dy, dx)]

            pred_dy = int(shift_row["calibrated_dy"])
            pred_dx = int(shift_row["calibrated_dx"])

            distorted = shift_torch(msi, dy, dx)

            # No ground-truth shift is used to select correction.
            corrected = shift_torch(
                distorted,
                -pred_dy,
                -pred_dx
            )

            score = fusion_psnr(
                model, lr, corrected, ref
            )

            baseline = oracle.loc[
                (run, image, dy, dx)
            ]

            uncorrected = float(
                baseline["uncorrected_psnr"]
            )

            oracle_score = float(
                baseline["oracle_psnr"]
            )

            clean_score = float(
                baseline["clean_psnr"]
            )

            records.append({
                "run": run,
                "model": model_name,
                "regime": regime,
                "seed": seed,
                "image": image,
                "dy": dy,
                "dx": dx,
                "distance": abs(dy) + abs(dx),
                "calibrated_dy": pred_dy,
                "calibrated_dx": pred_dx,
                "calibrated_epe": float(
                    shift_row["calibrated_epe"]
                ),
                "calibrated_exact": int(
                    shift_row["calibrated_exact"]
                ),
                "uncorrected_psnr": uncorrected,
                "calibrated_psnr": score,
                "oracle_psnr": oracle_score,
                "clean_psnr": clean_score,
                "calibrated_gain_db": score - uncorrected,
                "calibrated_oracle_gap_db":
                    score - oracle_score
            })

            completed.add((run, image, dy, dx))

        save_records()

        if patch_index % 5 == 0 or patch_index == len(TEST):
            n = sum(r["run"] == run for r in records)

            print(
                f"  Patch {patch_index:02d}/40 | "
                f"Model records {n:03d}/520",
                flush=True
            )

    del model, checkpoint
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 10I — FINAL PERFORMANCE SUMMARY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("10I — FINAL FUSION COMPARISON")
print("=" * 88)

results = save_records()

assert len(results) == 2080
assert not results.duplicated(
    ["run", "image", "dy", "dx"]
).any()

summary = (
    results.groupby(["model", "regime", "distance"])
    .agg(
        n=("image", "size"),
        uncorrected=("uncorrected_psnr", "mean"),
        calibrated=("calibrated_psnr", "mean"),
        oracle=("oracle_psnr", "mean"),
        clean=("clean_psnr", "mean"),
        gain_db=("calibrated_gain_db", "mean"),
        mean_epe=("calibrated_epe", "mean"),
        exact_rate=("calibrated_exact", "mean")
    )
    .reset_index()
)

summary.to_csv(
    OUT / "Step10_Fusion_Condition_Summary.csv",
    index=False
)

print(summary.round(5).to_string(index=False))

report = {
    "method": "validation-median origin-bias correction",
    "validation_patches_used_for_calibration": len(VAL),
    "test_patches": len(TEST),
    "validation_origin_bias": [bias_dy, bias_dx],
    "validation_origin_agreement": calibration_exact,
    "validation_shift_evaluations": len(val_results),
    "test_registration_evaluations": len(test_raw),
    "test_registration_exact_accuracy": float(
        test_raw["calibrated_exact"].mean()
    ),
    "test_mean_epe": float(
        test_raw["calibrated_epe"].mean()
    ),
    "test_aligned_false_correction_rate":
        aligned_false_rate,
    "fusion_evaluations": len(results),
    "ground_truth_test_offsets_used_for_calibration": False,
    "reference_hrhsi_used_for_registration": False,
    "evaluation_status": (
        "exploratory_revision_after_inspection_of_step9_test_results"
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step10_Final_Report.json",
    "w"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 10 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 10 COMPLETED.")
print("Original checkpoints and MDAS imagery were not modified.")


In [ ]:
# =============================================================================
# STEP 11 — VALIDATION-SELECTED REGISTRATION SEARCH RADIUS
#
# 1. Reuse Step 9 train-only HSI → MSI spectral projection.
# 2. Evaluate candidate search radii {4, 5, 6} on 32 validation patches.
# 3. Calibrate each radius using aligned validation imagery.
# 4. Select radius by validation exact accuracy, then EPE.
# 5. Freeze radius and bias BEFORE evaluating 40 test patches.
# 6. Evaluate fusion on four verified seed-7 checkpoints.
# 7. Compare with Step 10 and oracle registration.
#
# Scientific status: exploratory development, because MDAS test
# results from earlier iterations have already been inspected.
# =============================================================================

from google.colab import drive
from pathlib import Path
from scipy.ndimage import gaussian_filter, sobel
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile
import json
import sys
import time
import os
import tempfile
import hashlib

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
DATA = REPO / "datasets" / "mdas_new"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
STEP8 = PROJECT / "Step08_Oracle_Correction"
STEP10 = PROJECT / "Step10_Validation_Calibration"
STEP9 = PROJECT / "Step09_Estimated_Registration"
OUT = PROJECT / "Step11_Search_Radius_Audit"

OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RADII = [4, 5, 6]
MAX_RADIUS = max(RADII)
CROP_MARGIN = 8
BLUR_SIGMA = 1.2
INTENSITY_WEIGHT = 0.7
GRADIENT_WEIGHT = 0.3

OFFSETS = [
    (0, 0),
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (2, 0), (-2, 0), (0, 2), (0, -2),
    (4, 0), (-4, 0), (0, 4), (0, -4)
]

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

MODEL_CLASSES = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"
NORMALIZER = float(CONFIG["mdasn_dataset"]["max_value"])

assert NORMALIZER == 15000

print("=" * 88)
print("STEP 11 — SEARCH-RADIUS BOUNDARY DIAGNOSTIC")
print("=" * 88)
print("Device:", DEVICE)
print("Search radii:", RADII)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 11A — DATA PARTITIONS AND FROZEN PROJECTION
# -------------------------------------------------------------------------

def read_split(name):
    result = [
        line.split()[0]
        for line in (DATA / f"{name}.txt").read_text().splitlines()
        if line.strip()
    ]
    assert len(result) == len(set(result))
    return result

TRAIN = read_split("train")
VAL = read_split("val")
TEST = read_split("test")

assert len(TRAIN) == 133
assert len(VAL) == 32
assert len(TEST) == 40
assert not set(TRAIN).intersection(VAL, TEST)
assert not (set(VAL) & set(TEST))

projection_path = STEP9 / "Step09_Spectral_Projection.npz"

with np.load(projection_path, allow_pickle=False) as z:
    scaler_mean = z["scaler_mean"].copy()
    scaler_scale = z["scaler_scale"].copy()
    ridge_coef = z["ridge_coefficients"].copy()
    ridge_intercept = z["ridge_intercept"].copy()

assert scaler_mean.shape == (242,)
assert scaler_scale.shape == (242,)
assert ridge_coef.shape == (4, 242)
assert ridge_intercept.shape == (4,)

projection_sha = hashlib.sha256(
    projection_path.read_bytes()
).hexdigest()

print("\nFrozen spectral mapping SHA256:", projection_sha)
print("Training:", len(TRAIN))
print("Validation:", len(VAL))
print("Test:", len(TEST))

def load_array(modality, image):
    a = tifffile.imread(
        DATA / modality / image
    ).astype(np.float32) / NORMALIZER
    if not np.isfinite(a).all():
        raise ValueError(f"Non-finite data: {modality}/{image}")
    return a

def to_tensor(a):
    return torch.from_numpy(
        np.ascontiguousarray(a.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)

def project_hsi(lrhsi):
    with torch.inference_mode():
        interpolated = F.interpolate(
            to_tensor(lrhsi),
            size=(72, 72),
            mode="bicubic",
            align_corners=False
        )

    a = (
        interpolated[0].permute(1, 2, 0)
        .cpu().numpy()
    )

    standardized = (
        a.reshape(-1, 242).astype(np.float64)
        - scaler_mean
    ) / scaler_scale

    projected = standardized @ ridge_coef.T + ridge_intercept

    return projected.reshape(72, 72, 4).astype(np.float32)

# -------------------------------------------------------------------------
# 11B — GEOMETRIC TRANSLATION AND REGISTRATION SCORING
# -------------------------------------------------------------------------

def shift_np(a, dy, dx):
    dy, dx = int(dy), int(dx)
    h, w = a.shape[:2]

    padded = np.pad(
        a,
        (
            (max(dy, 0), max(-dy, 0)),
            (max(dx, 0), max(-dx, 0)),
            (0, 0)
        ),
        mode="edge"
    )

    top, left = max(-dy, 0), max(-dx, 0)

    return np.ascontiguousarray(
        padded[top:top+h, left:left+w, :]
    )

def registration_features(a):
    blurred = gaussian_filter(
        a.astype(np.float32),
        sigma=(BLUR_SIGMA, BLUR_SIGMA, 0)
    )
    gx = sobel(blurred, axis=1, mode="nearest")
    gy = sobel(blurred, axis=0, mode="nearest")

    gradient = np.sqrt(gx * gx + gy * gy)

    return blurred, gradient

def ncc(a, b):
    a = a[
        CROP_MARGIN:-CROP_MARGIN,
        CROP_MARGIN:-CROP_MARGIN, :
    ].reshape(-1, 4).astype(np.float64).copy()

    b = b[
        CROP_MARGIN:-CROP_MARGIN,
        CROP_MARGIN:-CROP_MARGIN, :
    ].reshape(-1, 4).astype(np.float64).copy()

    a -= a.mean(axis=0, keepdims=True)
    b -= b.mean(axis=0, keepdims=True)

    numerator = np.sum(a*b, axis=0)
    denominator = np.sqrt(
        np.sum(a*a, axis=0) *
        np.sum(b*b, axis=0)
    )

    valid = denominator > 1e-12
    if not valid.any():
        return -1.0

    return float(np.mean(numerator[valid]/denominator[valid]))

def score_candidates(target_features, observed_features):
    ti, tg = target_features
    oi, og = observed_features

    records = []

    for correction_dy in range(-MAX_RADIUS, MAX_RADIUS+1):
        for correction_dx in range(-MAX_RADIUS, MAX_RADIUS+1):

            shifted_i = shift_np(oi, correction_dy, correction_dx)
            shifted_g = shift_np(og, correction_dy, correction_dx)

            score = (
                INTENSITY_WEIGHT * ncc(ti, shifted_i)
                + GRADIENT_WEIGHT * ncc(tg, shifted_g)
            )

            records.append((
                float(score),
                int(correction_dy),
                int(correction_dx)
            ))

    return records

def select_raw(scores, radius):
    eligible = [
        item for item in scores
        if abs(item[1]) <= radius
        and abs(item[2]) <= radius
    ]

    # Sort by the score only, matching the original Step 9
    # candidate-order convention for ties.
    eligible.sort(key=lambda x: x[0], reverse=True)

    score, cy, cx = eligible[0]

    return {
        "raw_dy": -cy,
        "raw_dx": -cx,
        "score": score,
        "margin": score-eligible[1][0],
        "at_boundary": int(
            abs(cy) == radius or abs(cx) == radius
        )
    }

# -------------------------------------------------------------------------
# 11C — COMPUTE VAL SCORE SURFACES ONCE
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("11C — VALIDATION SEARCH-SURFACE EVALUATION")
print("=" * 88)

val_rows = []
val_start = time.monotonic()

for index, image in enumerate(VAL, 1):
    lr = load_array("lrhsi", image)
    clean = load_array("hrmsi", image)

    projected = project_hsi(lr)
    target_f = registration_features(projected)

    for true_dy, true_dx in OFFSETS:
        observed = shift_np(clean, true_dy, true_dx)
        observed_f = registration_features(observed)

        scores = score_candidates(target_f, observed_f)

        for radius in RADII:
            estimate = select_raw(scores, radius)

            val_rows.append({
                "image": image,
                "radius": radius,
                "true_dy": true_dy,
                "true_dx": true_dx,
                "distance": abs(true_dy)+abs(true_dx),
                **estimate
            })

    if index % 8 == 0 or index == len(VAL):
        print(
            f"  Validation patches {index:02d}/{len(VAL)} | "
            f"Elapsed {time.monotonic()-val_start:.1f}s",
            flush=True
        )

val_df = pd.DataFrame(val_rows)
assert len(val_df) == 32*13*len(RADII)

val_df.to_csv(
    OUT / "Step11_Validation_Raw_Estimates.csv",
    index=False
)

# -------------------------------------------------------------------------
# 11D — LEARN A SEPARATE VAL-ONLY BIAS FOR EACH RADIUS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("11D — VALIDATION BIAS AND RADIUS SELECTION")
print("=" * 88)

selection_rows = []
calibrated_val = []

for radius in RADII:
    df = val_df[val_df["radius"] == radius].copy()
    aligned = df[df["distance"] == 0]

    bias_dy = int(np.rint(aligned["raw_dy"].median()))
    bias_dx = int(np.rint(aligned["raw_dx"].median()))

    df["estimated_dy"] = df["raw_dy"] - bias_dy
    df["estimated_dx"] = df["raw_dx"] - bias_dx

    df["epe"] = np.hypot(
        df["estimated_dy"]-df["true_dy"],
        df["estimated_dx"]-df["true_dx"]
    )

    df["exact"] = (
        (df["estimated_dy"] == df["true_dy"])
        & (df["estimated_dx"] == df["true_dx"])
    ).astype(int)

    aligned_cal = df[df["distance"] == 0]
    false_correction = 1.0 - float(
        aligned_cal["exact"].mean()
    )

    selection_rows.append({
        "radius": radius,
        "bias_dy": bias_dy,
        "bias_dx": bias_dx,
        "validation_exact_accuracy": float(df["exact"].mean()),
        "validation_mean_epe": float(df["epe"].mean()),
        "validation_aligned_false_rate": false_correction,
        "validation_4px_exact": float(
            df[df["distance"] == 4]["exact"].mean()
        ),
        "raw_boundary_rate": float(df["at_boundary"].mean())
    })

    calibrated_val.append(df)

selection = pd.DataFrame(selection_rows)

# Prefer zero false corrections, then maximum exact accuracy,
# then minimum mean EPE, then smallest radius.
selection = selection.sort_values(
    [
        "validation_aligned_false_rate",
        "validation_exact_accuracy",
        "validation_mean_epe",
        "radius"
    ],
    ascending=[True, False, True, True]
).reset_index(drop=True)

selection.to_csv(
    OUT / "Step11_Validation_Radius_Selection.csv",
    index=False
)

print(selection.round(6).to_string(index=False))

winner = selection.iloc[0]

SELECTED_RADIUS = int(winner["radius"])
BIAS_DY = int(winner["bias_dy"])
BIAS_DX = int(winner["bias_dx"])

val_cal = pd.concat(calibrated_val, ignore_index=True)
val_cal.to_csv(
    OUT / "Step11_Validation_Calibrated_Estimates.csv",
    index=False
)

frozen = {
    "selected_radius": SELECTED_RADIUS,
    "bias_dy": BIAS_DY,
    "bias_dx": BIAS_DX,
    "projection_sha256": projection_sha,
    "selection_set": "32 validation patches",
    "selection_method": "zero-shift safety, exact accuracy, EPE, radius",
    "test_shifts_used_in_selection": False,
    "note": "Exploratory development following previously observed MDAS test results"
}

with open(OUT / "Step11_Frozen_Method.json", "w") as f:
    json.dump(frozen, f, indent=2)

print("\nFROZEN METHOD")
print(json.dumps(frozen, indent=2))

print("\nSELECTED VALIDATION PERFORMANCE BY DISPLACEMENT")
print(
    val_cal[val_cal["radius"] == SELECTED_RADIUS]
    .groupby("distance")
    .agg(
        n=("exact", "size"),
        exact_rate=("exact", "mean"),
        mean_epe=("epe", "mean"),
        boundary_rate=("at_boundary", "mean")
    )
    .round(5).to_string()
)

# -------------------------------------------------------------------------
# 11E — TEST REGISTRATION, AFTER FREEZING THE METHOD
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("11E — TEST REGISTRATION WITH FROZEN PARAMETERS")
print("=" * 88)

test_rows = []
test_start = time.monotonic()

for index, image in enumerate(TEST, 1):
    lr = load_array("lrhsi", image)
    clean = load_array("hrmsi", image)

    projected = project_hsi(lr)
    target_f = registration_features(projected)

    for dy, dx in OFFSETS:
        observed = shift_np(clean, dy, dx)
        scores = score_candidates(
            target_f, registration_features(observed)
        )

        raw = select_raw(scores, SELECTED_RADIUS)

        estimated_dy = int(raw["raw_dy"] - BIAS_DY)
        estimated_dx = int(raw["raw_dx"] - BIAS_DX)

        error = float(np.hypot(
            estimated_dy-dy, estimated_dx-dx
        ))

        test_rows.append({
            "image": image,
            "dy": dy,
            "dx": dx,
            "distance": abs(dy)+abs(dx),
            "estimated_dy": estimated_dy,
            "estimated_dx": estimated_dx,
            "epe": error,
            "exact": int(error == 0),
            "score": raw["score"],
            "margin": raw["margin"],
            "at_boundary": raw["at_boundary"]
        })

    if index % 5 == 0 or index == len(TEST):
        current = pd.DataFrame(test_rows)
        print(
            f"  Test patches {index:02d}/{len(TEST)} | "
            f"Exact {current['exact'].mean()*100:.2f}% | "
            f"EPE {current['epe'].mean():.4f} px | "
            f"Elapsed {time.monotonic()-test_start:.1f}s",
            flush=True
        )

test_df = pd.DataFrame(test_rows)
assert len(test_df) == 520
assert not test_df.duplicated(["image", "dy", "dx"]).any()

test_df.to_csv(
    OUT / "Step11_Test_Registration.csv", index=False
)

registration_summary = (
    test_df.groupby("distance")
    .agg(
        n=("exact", "size"),
        exact_rate=("exact", "mean"),
        mean_epe=("epe", "mean"),
        max_epe=("epe", "max"),
        boundary_rate=("at_boundary", "mean")
    ).reset_index()
)

registration_summary.to_csv(
    OUT / "Step11_Test_Registration_Summary.csv",
    index=False
)

print("\nTEST REGISTRATION PERFORMANCE")
print(registration_summary.round(5).to_string(index=False))

# -------------------------------------------------------------------------
# 11F — ARCHIVED FUSION BASELINES AND PREVIOUS METHOD
# -------------------------------------------------------------------------

oracle_path = STEP8 / "Step08_Oracle_Patch_Results.csv"
old_path = STEP10 / "Step10_Calibrated_Fusion.csv"

assert oracle_path.is_file()
assert old_path.is_file()

oracle = pd.read_csv(oracle_path)
previous = pd.read_csv(old_path)

assert len(oracle) == 2080
assert len(previous) == 2080

oracle = oracle.set_index(
    ["run", "image", "dy", "dx"]
)
previous = previous.set_index(
    ["run", "image", "dy", "dx"]
)

estimates = test_df.set_index(["image", "dy", "dx"])

assert oracle.index.is_unique
assert previous.index.is_unique
assert estimates.index.is_unique

# -------------------------------------------------------------------------
# 11G — FUSION USING THE FROZEN ESTIMATES
# -------------------------------------------------------------------------

def shift_torch(x, dy, dx):
    dy, dx = int(dy), int(dx)
    h, w = x.shape[-2:]

    pad = F.pad(
        x,
        (
            max(dx, 0), max(-dx, 0),
            max(dy, 0), max(-dy, 0)
        ),
        mode="replicate"
    )

    top, left = max(-dy, 0), max(-dx, 0)

    return pad[:, :, top:top+h, left:left+w].contiguous()

def get_psnr(model, lr, msi, ref):
    with torch.inference_mode():
        output = model(lr, msi)

        if isinstance(output, dict):
            output = output["pred"]

        assert tuple(output.shape) == (1, 242, 72, 72)

        value = PSNR(
            output[:, :, 4:-4, 4:-4].contiguous(),
            ref[:, :, 4:-4, 4:-4].contiguous()
        )

        return float(value.detach().cpu().item())

CSV_PATH = OUT / "Step11_Fusion_Results.csv"

COLUMNS = [
    "run", "model", "regime", "seed", "image",
    "dy", "dx", "distance",
    "estimated_dy", "estimated_dx",
    "epe", "exact",
    "uncorrected_psnr", "step10_psnr",
    "step11_psnr", "oracle_psnr", "clean_psnr",
    "gain_vs_uncorrected", "gain_vs_step10",
    "gap_to_oracle"
]

# Reuse only results produced with the same frozen method.
if CSV_PATH.exists():
    previous_results = pd.read_csv(CSV_PATH)
    assert set(COLUMNS).issubset(previous_results.columns)
    previous_results = previous_results[COLUMNS]

    for r in previous_results.itertuples(index=False):
        est = estimates.loc[(r.image, int(r.dy), int(r.dx))]
        if (
            int(r.estimated_dy) != int(est["estimated_dy"])
            or int(r.estimated_dx) != int(est["estimated_dx"])
        ):
            raise ValueError(
                "Existing Step 11 inference used other offsets. "
                "Move the prior CSV before changing the method."
            )

    records = previous_results.to_dict("records")
    print("\nResuming", len(records), "existing fusion records.")
else:
    records = []

completed = {
    (r["run"], r["image"], int(r["dy"]), int(r["dx"]))
    for r in records
}

def save_fusion():
    frame = pd.DataFrame(records, columns=COLUMNS)
    frame = frame.sort_values(
        ["run", "image", "dy", "dx"]
    ).reset_index(drop=True)

    fd, temp_path = tempfile.mkstemp(
        dir=str(OUT), prefix="step11_", suffix=".csv"
    )

    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            frame.to_csv(f, index=False)
        os.replace(temp_path, CSV_PATH)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    return frame

print("\n" + "=" * 88)
print("11G — FUSION INFERENCE")
print("=" * 88)

for run_index, run in enumerate(RUNS, 1):
    model_name = run.split("_")[0]
    regime = run.replace(model_name + "_", "").rsplit("_s", 1)[0]
    seed = int(run.rsplit("_s", 1)[1])

    remaining = sum(
        (run, image, dy, dx) not in completed
        for image in TEST
        for dy, dx in OFFSETS
    )

    print(
        f"\nRUN {run_index}/{len(RUNS)}: {run} | "
        f"Pending {remaining}",
        flush=True
    )

    if remaining == 0:
        continue

    checkpoint = torch.load(
        EXPS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    model = MODEL_CLASSES[model_name](config=CONFIG)
    model.load_state_dict(checkpoint["model"], strict=True)
    model = model.to(DEVICE).eval()

    print("Strict checkpoint loading: PASS", flush=True)

    for index, image in enumerate(TEST, 1):
        missing = [
            (dy, dx) for dy, dx in OFFSETS
            if (run, image, dy, dx) not in completed
        ]

        if not missing:
            continue

        lr = to_tensor(load_array("lrhsi", image))
        msi = to_tensor(load_array("hrmsi", image))
        ref = to_tensor(load_array("hrhsi", image))

        for dy, dx in missing:
            est = estimates.loc[(image, dy, dx)]

            estimated_dy = int(est["estimated_dy"])
            estimated_dx = int(est["estimated_dx"])

            distorted = shift_torch(msi, dy, dx)

            corrected = shift_torch(
                distorted, -estimated_dy, -estimated_dx
            )

            new_psnr = get_psnr(model, lr, corrected, ref)

            key = (run, image, dy, dx)
            old_psnr = float(previous.loc[key, "calibrated_psnr"])

            base = oracle.loc[key]
            uncorrected = float(base["uncorrected_psnr"])
            oracle_psnr = float(base["oracle_psnr"])
            clean_psnr = float(base["clean_psnr"])

            records.append({
                "run": run,
                "model": model_name,
                "regime": regime,
                "seed": seed,
                "image": image,
                "dy": dy,
                "dx": dx,
                "distance": abs(dy)+abs(dx),
                "estimated_dy": estimated_dy,
                "estimated_dx": estimated_dx,
                "epe": float(est["epe"]),
                "exact": int(est["exact"]),
                "uncorrected_psnr": uncorrected,
                "step10_psnr": old_psnr,
                "step11_psnr": new_psnr,
                "oracle_psnr": oracle_psnr,
                "clean_psnr": clean_psnr,
                "gain_vs_uncorrected": new_psnr-uncorrected,
                "gain_vs_step10": new_psnr-old_psnr,
                "gap_to_oracle": new_psnr-oracle_psnr
            })

            completed.add(key)

        save_fusion()

        if index % 5 == 0 or index == len(TEST):
            count = sum(r["run"] == run for r in records)
            print(
                f"  Patch {index:02d}/40 | "
                f"Completed {count:03d}/520",
                flush=True
            )

    del model, checkpoint
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 11H — FINAL SUMMARY
# -------------------------------------------------------------------------

results = save_fusion()

assert len(results) == 2080
assert not results.duplicated(
    ["run", "image", "dy", "dx"]
).any()
assert np.isfinite(
    results[
        [
            "uncorrected_psnr",
            "step10_psnr",
            "step11_psnr",
            "oracle_psnr",
            "clean_psnr"
        ]
    ].to_numpy(float)
).all()

fusion_summary = (
    results.groupby(["model", "regime", "distance"])
    .agg(
        n=("image", "size"),
        uncorrected=("uncorrected_psnr", "mean"),
        step10=("step10_psnr", "mean"),
        step11=("step11_psnr", "mean"),
        oracle=("oracle_psnr", "mean"),
        clean=("clean_psnr", "mean"),
        step11_gain=("gain_vs_uncorrected", "mean"),
        improvement_vs_step10=("gain_vs_step10", "mean"),
        gap_to_oracle=("gap_to_oracle", "mean")
    )
    .reset_index()
)

fusion_summary.to_csv(
    OUT / "Step11_Fusion_Condition_Summary.csv",
    index=False
)

print("\n" + "=" * 88)
print("STEP 11 — FUSION RESULTS")
print("=" * 88)
print(fusion_summary.round(5).to_string(index=False))

report = {
    "selected_radius": SELECTED_RADIUS,
    "calibrated_bias": [BIAS_DY, BIAS_DX],
    "validation_exact_accuracy": float(
        winner["validation_exact_accuracy"]
    ),
    "validation_mean_epe": float(
        winner["validation_mean_epe"]
    ),
    "test_exact_accuracy": float(test_df["exact"].mean()),
    "test_mean_epe": float(test_df["epe"].mean()),
    "test_4px_accuracy": float(
        test_df[test_df["distance"] == 4]["exact"].mean()
    ),
    "test_aligned_false_correction_rate": float(
        1-test_df[test_df["distance"] == 0]["exact"].mean()
    ),
    "fusion_records": len(results),
    "ground_truth_test_shifts_used_for_model_selection": False,
    "reference_hrhsi_used_for_registration": False,
    "experiment_status": "exploratory_method_development",
    "output_directory": str(OUT)
}

with open(OUT / "Step11_Final_Report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 11 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))
print("\nSTEP 11 COMPLETE")

In [ ]:

# =============================================================================
# STEP 12 — SUBPIXEL REGISTRATION ROBUSTNESS
#
# DATA:
#   133 train patches: frozen Step 9 spectral mapping
#    32 validation patches: choose integer or subpixel estimator
#    40 test patches: final descriptive evaluation
#
# METHODS:
#   A. Uncorrected displaced MSI
#   B. Validation-selected image-estimated correction
#   C. Oracle correction with known fractional shift
#   D. Clean, unshifted MSI
#
# Fractional shifts use bilinear sampling and border replication.
# The same geometric operator is used for all correction methods.
#
# No training, no modification of original data or checkpoints.
# =============================================================================

from google.colab import drive
from pathlib import Path
from scipy.ndimage import gaussian_filter, sobel
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile
import json
import hashlib
import sys
import os
import time
import tempfile

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
DATA = REPO / "datasets" / "mdas_new"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
STEP9 = PROJECT / "Step09_Estimated_Registration"
STEP11 = PROJECT / "Step11_Search_Radius_Audit"
OUT = PROJECT / "Step12_Subpixel_Robustness"

OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

MODEL_CLASSES = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

# 1 aligned + 12 cardinal + 4 diagonal conditions = 17.
OFFSETS = [(0.0, 0.0)]

for d in [0.5, 1.5, 3.5]:
    OFFSETS += [
        (d, 0.0), (-d, 0.0),
        (0.0, d), (0.0, -d)
    ]

for dy in [-1.5, 1.5]:
    for dx in [-1.5, 1.5]:
        OFFSETS.append((dy, dx))

assert len(OFFSETS) == 17
assert len(set(OFFSETS)) == 17

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"

NORMALIZER = float(
    CONFIG["mdasn_dataset"]["max_value"]
)

assert NORMALIZER == 15000.0

with open(
    STEP11 / "Step11_Frozen_Method.json",
    encoding="utf-8"
) as f:
    FROZEN = json.load(f)

RADIUS = int(FROZEN["selected_radius"])
BIAS_DY = int(FROZEN["bias_dy"])
BIAS_DX = int(FROZEN["bias_dx"])

assert RADIUS == 5
assert (BIAS_DY, BIAS_DX) == (-1, -1)

CROP_MARGIN = 8
BLUR_SIGMA = 1.2
INTENSITY_WEIGHT = 0.7
GRADIENT_WEIGHT = 0.3

print("=" * 88, flush=True)
print("STEP 12 — SUBPIXEL REGISTRATION ROBUSTNESS", flush=True)
print("=" * 88, flush=True)
print("Device:", DEVICE)
print("Search radius:", RADIUS)
print("Frozen calibration:", (BIAS_DY, BIAS_DX))
print("Fractional test conditions:", len(OFFSETS))
print("Output:", OUT)

# -------------------------------------------------------------------------
# 12A — VERIFY PARTITIONS AND LOAD SPECTRAL PROJECTION
# -------------------------------------------------------------------------

def read_split(split):
    names = [
        line.strip().split()[0]
        for line in (DATA / f"{split}.txt").read_text().splitlines()
        if line.strip()
    ]
    assert len(names) == len(set(names))
    return names

TRAIN = read_split("train")
VAL = read_split("val")
TEST = read_split("test")

assert len(TRAIN) == 133
assert len(VAL) == 32
assert len(TEST) == 40

assert not (set(TRAIN) & set(VAL))
assert not (set(TRAIN) & set(TEST))
assert not (set(VAL) & set(TEST))

projection_file = STEP9 / "Step09_Spectral_Projection.npz"

projection_sha = hashlib.sha256(
    projection_file.read_bytes()
).hexdigest()

assert projection_sha == FROZEN["projection_sha256"]

with np.load(projection_file, allow_pickle=False) as z:
    scaler_mean = z["scaler_mean"].copy()
    scaler_scale = z["scaler_scale"].copy()
    ridge_coef = z["ridge_coefficients"].copy()
    ridge_intercept = z["ridge_intercept"].copy()

assert scaler_mean.shape == (242,)
assert scaler_scale.shape == (242,)
assert ridge_coef.shape == (4, 242)
assert ridge_intercept.shape == (4,)

print("\n12A — DATA AND PROJECTION VERIFIED")
print("Training:", len(TRAIN))
print("Validation:", len(VAL))
print("Test:", len(TEST))
print("Projection SHA256:", projection_sha)

def read_array(modality, image):
    path = DATA / modality / image

    array = tifffile.imread(
        path
    ).astype(np.float32) / NORMALIZER

    if not np.isfinite(array).all():
        raise ValueError(f"Non-finite image: {path}")

    return array

def to_tensor(array):
    return torch.from_numpy(
        np.ascontiguousarray(
            array.transpose(2, 0, 1)
        )
    ).unsqueeze(0).to(DEVICE)

def project_hsi(lrhsi):
    with torch.inference_mode():
        up = F.interpolate(
            to_tensor(lrhsi),
            size=(72, 72),
            mode="bicubic",
            align_corners=False
        )

    up = (
        up[0].permute(1, 2, 0)
        .cpu().numpy()
    )

    flat = up.reshape(-1, 242).astype(np.float64)

    standardized = (
        flat - scaler_mean
    ) / scaler_scale

    projected = (
        standardized @ ridge_coef.T
        + ridge_intercept
    )

    return projected.reshape(72, 72, 4).astype(np.float32)

# -------------------------------------------------------------------------
# 12B — FRACTIONAL-PIXEL GEOMETRIC TRANSLATION
# -------------------------------------------------------------------------

def shift_tensor(image, dy, dx):
    """
    Translation convention:
      dy > 0: move image content down
      dx > 0: move image content right

    grid_sample uses inverse sampling coordinates.
    The boundary is replicated. For fractional coordinates,
    bilinear interpolation is used.

    This is an image-domain resampling operation;
    applying its inverse does not perfectly recover the
    original radiometry.
    """
    dy = float(dy)
    dx = float(dx)

    if dy == 0.0 and dx == 0.0:
        return image

    n, c, h, w = image.shape

    yy, xx = torch.meshgrid(
        torch.arange(h, device=image.device),
        torch.arange(w, device=image.device),
        indexing="ij"
    )

    source_y = yy.to(torch.float32) - dy
    source_x = xx.to(torch.float32) - dx

    # align_corners=True maps pixel-centre coordinates
    # exactly onto the grid extrema.
    grid_y = 2.0 * source_y / (h - 1) - 1.0
    grid_x = 2.0 * source_x / (w - 1) - 1.0

    grid = torch.stack(
        [grid_x, grid_y], dim=-1
    ).unsqueeze(0).expand(n, -1, -1, -1)

    return F.grid_sample(
        image,
        grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True
    ).contiguous()

def shift_numpy(array, dy, dx):
    tensor = to_tensor(array)

    with torch.inference_mode():
        shifted = shift_tensor(
            tensor, dy, dx
        )

    return (
        shifted[0]
        .permute(1, 2, 0)
        .cpu()
        .numpy()
        .astype(np.float32)
    )

# Verify sign convention and identity.
toy = torch.arange(
    100, device=DEVICE, dtype=torch.float32
).reshape(1, 1, 10, 10)

assert torch.equal(
    shift_tensor(toy, 0, 0), toy
)

assert torch.allclose(
    shift_tensor(toy, 1, 0)[:, :, 1:, :],
    toy[:, :, :-1, :],
    atol=1e-5
)

assert torch.allclose(
    shift_tensor(toy, 0, 1)[:, :, :, 1:],
    toy[:, :, :, :-1],
    atol=1e-5
)

print("\n12B — FRACTIONAL SHIFT CONVENTION VERIFIED")

# -------------------------------------------------------------------------
# 12C — REPRODUCE FROZEN STEP 11 REGISTRATION FEATURES
# -------------------------------------------------------------------------

def registration_features(array):
    blurred = gaussian_filter(
        array.astype(np.float32),
        sigma=(BLUR_SIGMA, BLUR_SIGMA, 0)
    )

    gx = sobel(
        blurred, axis=1, mode="nearest"
    )

    gy = sobel(
        blurred, axis=0, mode="nearest"
    )

    gradient = np.sqrt(gx*gx + gy*gy)

    return blurred, gradient

def integer_shift_np(array, dy, dx):
    dy, dx = int(dy), int(dx)
    h, w = array.shape[:2]

    padded = np.pad(
        array,
        (
            (max(dy, 0), max(-dy, 0)),
            (max(dx, 0), max(-dx, 0)),
            (0, 0)
        ),
        mode="edge"
    )

    top = max(-dy, 0)
    left = max(-dx, 0)

    return np.ascontiguousarray(
        padded[top:top+h, left:left+w, :]
    )

def ncc(a, b):
    a = a[
        CROP_MARGIN:-CROP_MARGIN,
        CROP_MARGIN:-CROP_MARGIN, :
    ].reshape(-1, 4).astype(np.float64).copy()

    b = b[
        CROP_MARGIN:-CROP_MARGIN,
        CROP_MARGIN:-CROP_MARGIN, :
    ].reshape(-1, 4).astype(np.float64).copy()

    a -= a.mean(axis=0, keepdims=True)
    b -= b.mean(axis=0, keepdims=True)

    numerator = np.sum(a*b, axis=0)

    denominator = np.sqrt(
        np.sum(a*a, axis=0)
        * np.sum(b*b, axis=0)
    )

    valid = denominator > 1e-12

    if not np.any(valid):
        return -1.0

    return float(
        np.mean(
            numerator[valid] / denominator[valid]
        )
    )

def score_surface(target_features, observed_features):
    ti, tg = target_features
    oi, og = observed_features

    values = np.zeros(
        (2*RADIUS+1, 2*RADIUS+1),
        dtype=np.float64
    )

    for cy in range(-RADIUS, RADIUS+1):
        for cx in range(-RADIUS, RADIUS+1):
            si = integer_shift_np(oi, cy, cx)
            sg = integer_shift_np(og, cy, cx)

            score = (
                INTENSITY_WEIGHT * ncc(ti, si)
                + GRADIENT_WEIGHT * ncc(tg, sg)
            )

            values[
                cy+RADIUS, cx+RADIUS
            ] = score

    return values

def quadratic_vertex(left, centre, right):
    """
    Fit a parabola through scores at -1, 0 and +1.
    Return a bounded fractional displacement relative
    to the central integer candidate.

    Refinement is disabled when local curvature does not
    indicate a valid concave maximum.
    """
    denominator = (
        left - 2*centre + right
    )

    if (
        not np.isfinite(denominator)
        or denominator >= -1e-12
    ):
        return 0.0

    delta = 0.5 * (
        left - right
    ) / denominator

    return float(
        np.clip(delta, -0.5, 0.5)
    )

def estimate_both(target_features, observed_msi):
    observed_features = registration_features(
        observed_msi
    )

    scores = score_surface(
        target_features,
        observed_features
    )

    flat = int(np.argmax(scores))
    row, col = np.unravel_index(
        flat, scores.shape
    )

    correction_y = row - RADIUS
    correction_x = col - RADIUS

    delta_y = 0.0
    delta_x = 0.0

    if 0 < row < 2*RADIUS:
        delta_y = quadratic_vertex(
            scores[row-1, col],
            scores[row, col],
            scores[row+1, col]
        )

    if 0 < col < 2*RADIUS:
        delta_x = quadratic_vertex(
            scores[row, col-1],
            scores[row, col],
            scores[row, col+1]
        )

    # The raw displacement is the negative of the
    # correction applied to the observed MSI.
    integer_dy = -correction_y - BIAS_DY
    integer_dx = -correction_x - BIAS_DX

    refined_dy = (
        -(correction_y + delta_y) - BIAS_DY
    )

    refined_dx = (
        -(correction_x + delta_x) - BIAS_DX
    )

    return {
        "integer_dy": float(integer_dy),
        "integer_dx": float(integer_dx),
        "refined_dy": float(refined_dy),
        "refined_dx": float(refined_dx),
        "score": float(scores[row, col]),
        "boundary": int(
            abs(correction_y) == RADIUS
            or abs(correction_x) == RADIUS
        )
    }

# -------------------------------------------------------------------------
# 12D — VALIDATION COMPARISON, BEFORE TEST EVALUATION
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("12D — VALIDATION OF INTEGER AND SUBPIXEL ESTIMATORS")
print("=" * 88, flush=True)

validation_rows = []
val_start = time.monotonic()

for index, image in enumerate(VAL, 1):
    lr = read_array("lrhsi", image)
    clean = read_array("hrmsi", image)

    target_features = registration_features(
        project_hsi(lr)
    )

    for dy, dx in OFFSETS:
        shifted = shift_numpy(
            clean, dy, dx
        )

        estimates = estimate_both(
            target_features, shifted
        )

        integer_epe = float(
            np.hypot(
                estimates["integer_dy"] - dy,
                estimates["integer_dx"] - dx
            )
        )

        refined_epe = float(
            np.hypot(
                estimates["refined_dy"] - dy,
                estimates["refined_dx"] - dx
            )
        )

        validation_rows.append({
            "image": image,
            "true_dy": dy,
            "true_dx": dx,
            "integer_dy": estimates["integer_dy"],
            "integer_dx": estimates["integer_dx"],
            "refined_dy": estimates["refined_dy"],
            "refined_dx": estimates["refined_dx"],
            "integer_epe": integer_epe,
            "refined_epe": refined_epe,
            "boundary": estimates["boundary"]
        })

    if index % 8 == 0:
        print(
            f"  Validation patches {index:02d}/32 | "
            f"Elapsed {time.monotonic()-val_start:.1f}s",
            flush=True
        )

validation = pd.DataFrame(validation_rows)
assert len(validation) == 32*len(OFFSETS)

validation.to_csv(
    OUT / "Step12_Validation_Shifts.csv",
    index=False
)

val_integer_mae = float(
    validation["integer_epe"].mean()
)

val_refined_mae = float(
    validation["refined_epe"].mean()
)

# Validation-only model selection.
# In a tie, prefer the simpler integer estimator.
SELECTED_METHOD = (
    "quadratic_subpixel"
    if val_refined_mae < val_integer_mae - 1e-10
    else "integer"
)

print("\nVALIDATION REGISTRATION RESULTS")
print("Integer mean EPE:", round(val_integer_mae, 6))
print("Refined mean EPE:", round(val_refined_mae, 6))
print("Selected method:", SELECTED_METHOD)

selection = {
    "method": SELECTED_METHOD,
    "integer_validation_epe": val_integer_mae,
    "refined_validation_epe": val_refined_mae,
    "search_radius": RADIUS,
    "bias": [BIAS_DY, BIAS_DX],
    "validation_cases": len(validation),
    "test_labels_used_for_selection": False,
    "evaluation_status": (
        "exploratory_post_test_development"
    )
}

with open(
    OUT / "Step12_Frozen_Selection.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(selection, f, indent=2)

# -------------------------------------------------------------------------
# 12E — TEST REGISTRATION WITH FROZEN SELECTION
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("12E — HELD-OUT TEST REGISTRATION")
print("=" * 88, flush=True)

test_rows = []
test_start = time.monotonic()

for index, image in enumerate(TEST, 1):
    lr = read_array("lrhsi", image)
    clean = read_array("hrmsi", image)

    target_features = registration_features(
        project_hsi(lr)
    )

    for dy, dx in OFFSETS:
        shifted = shift_numpy(
            clean, dy, dx
        )

        estimates = estimate_both(
            target_features, shifted
        )

        if SELECTED_METHOD == "quadratic_subpixel":
            estimated_dy = estimates["refined_dy"]
            estimated_dx = estimates["refined_dx"]
        else:
            estimated_dy = estimates["integer_dy"]
            estimated_dx = estimates["integer_dx"]

        epe = float(np.hypot(
            estimated_dy - dy,
            estimated_dx - dx
        ))

        test_rows.append({
            "image": image,
            "true_dy": dy,
            "true_dx": dx,
            "shift_magnitude": float(np.hypot(dy, dx)),
            "estimated_dy": estimated_dy,
            "estimated_dx": estimated_dx,
            "epe": epe,
            "integer_epe": float(np.hypot(
                estimates["integer_dy"] - dy,
                estimates["integer_dx"] - dx
            )),
            "refined_epe": float(np.hypot(
                estimates["refined_dy"] - dy,
                estimates["refined_dx"] - dx
            )),
            "boundary": estimates["boundary"],
            "is_aligned": int(dy == 0 and dx == 0)
        })

    if index % 5 == 0:
        current = pd.DataFrame(test_rows)
        print(
            f"  Test patches {index:02d}/40 | "
            f"Mean EPE {current['epe'].mean():.4f}px | "
            f"Elapsed {time.monotonic()-test_start:.1f}s",
            flush=True
        )

test = pd.DataFrame(test_rows)

assert len(test) == len(TEST)*len(OFFSETS)

assert not test.duplicated(
    ["image", "true_dy", "true_dx"]
).any()

test.to_csv(
    OUT / "Step12_Test_Registration.csv",
    index=False
)

registration_summary = (
    test.groupby("shift_magnitude")
    .agg(
        n=("image", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max"),
        integer_mean_epe=("integer_epe", "mean"),
        refined_mean_epe=("refined_epe", "mean"),
        boundary_rate=("boundary", "mean")
    )
    .reset_index()
)

registration_summary.to_csv(
    OUT / "Step12_Registration_Summary.csv",
    index=False
)

print("\nTEST REGISTRATION SUMMARY")
print(registration_summary.round(5).to_string(index=False))

# -------------------------------------------------------------------------
# 12F — FUSION PREDICTION AND ORIGINAL PSNR
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("12F — FUSION UNDER SUBPIXEL DISPLACEMENTS")
print("=" * 88, flush=True)

test_index = test.set_index(
    ["image", "true_dy", "true_dx"]
)
assert test_index.index.is_unique

def predict_psnr(model, lr, msi, reference):
    with torch.inference_mode():
        output = model(lr, msi)

        if isinstance(output, dict):
            output = output["pred"]

        if tuple(output.shape) != (1, 242, 72, 72):
            raise ValueError(
                f"Unexpected model output: {tuple(output.shape)}"
            )

        score = PSNR(
            output[:, :, 4:-4, 4:-4].contiguous(),
            reference[:, :, 4:-4, 4:-4].contiguous()
        )

    return float(
        score.detach().cpu().item()
    )

OUTPUT_CSV = OUT / "Step12_Fusion_Patch_Results.csv"

COLS = [
    "run", "model", "regime", "image",
    "true_dy", "true_dx", "shift_magnitude",
    "estimated_dy", "estimated_dx", "epe",
    "uncorrected_psnr", "estimated_psnr",
    "oracle_psnr", "clean_psnr",
    "estimated_gain_db", "oracle_gain_db",
    "estimated_oracle_gap_db"
]

# Resume only if the previous estimates and schema agree.
if OUTPUT_CSV.exists():
    prior = pd.read_csv(OUTPUT_CSV)

    if not set(COLS).issubset(prior.columns):
        raise ValueError("Incompatible saved Step 12 results.")

    prior = prior[COLS].copy()

    if prior.duplicated(
        ["run", "image", "true_dy", "true_dx"]
    ).any():
        raise ValueError("Duplicate saved results.")

    for row in prior.itertuples(index=False):
        est = test_index.loc[
            (row.image, row.true_dy, row.true_dx)
        ]

        if (
            abs(row.estimated_dy - est["estimated_dy"]) > 1e-7
            or abs(row.estimated_dx - est["estimated_dx"]) > 1e-7
        ):
            raise ValueError(
                "Saved results use another registration method."
            )

    records = prior.to_dict("records")
    print("Resuming records:", len(records), flush=True)

else:
    records = []

completed = {
    (
        r["run"], r["image"],
        float(r["true_dy"]), float(r["true_dx"])
    )
    for r in records
}

def save_results():
    result = pd.DataFrame(
        records, columns=COLS
    )

    result = result.sort_values(
        ["run", "image", "true_dy", "true_dx"]
    ).reset_index(drop=True)

    fd, temporary = tempfile.mkstemp(
        dir=str(OUT),
        prefix="step12_",
        suffix=".csv"
    )

    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            result.to_csv(f, index=False)

        os.replace(temporary, OUTPUT_CSV)

    finally:
        if os.path.exists(temporary):
            os.remove(temporary)

    return result

for run_number, run in enumerate(RUNS, 1):
    model_name = run.split("_")[0]

    regime = (
        run.replace(model_name + "_", "")
        .rsplit("_s", 1)[0]
    )

    pending = sum(
        (run, image, float(dy), float(dx)) not in completed
        for image in TEST
        for dy, dx in OFFSETS
    )

    print(
        f"\nRUN {run_number}/4: {run} | Pending {pending}",
        flush=True
    )

    if pending == 0:
        continue

    checkpoint = torch.load(
        EXPS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    model = MODEL_CLASSES[model_name](config=CONFIG)

    model.load_state_dict(
        checkpoint["model"],
        strict=True
    )

    model = model.to(DEVICE).eval()

    print("Strict checkpoint loading: PASSED", flush=True)

    for image_index, image in enumerate(TEST, 1):
        missing = [
            (dy, dx)
            for dy, dx in OFFSETS
            if (run, image, float(dy), float(dx)) not in completed
        ]

        if not missing:
            continue

        lr = to_tensor(
            read_array("lrhsi", image)
        )

        clean = to_tensor(
            read_array("hrmsi", image)
        )

        ref = to_tensor(
            read_array("hrhsi", image)
        )

        clean_psnr = predict_psnr(
            model, lr, clean, ref
        )

        for dy, dx in missing:
            est = test_index.loc[
                (image, dy, dx)
            ]

            edy = float(est["estimated_dy"])
            edx = float(est["estimated_dx"])

            distorted = shift_tensor(
                clean, dy, dx
            )

            estimated_corrected = shift_tensor(
                distorted, -edy, -edx
            )

            oracle_corrected = shift_tensor(
                distorted, -dy, -dx
            )

            uncorrected_psnr = predict_psnr(
                model, lr, distorted, ref
            )

            estimated_psnr = predict_psnr(
                model, lr, estimated_corrected, ref
            )

            oracle_psnr = predict_psnr(
                model, lr, oracle_corrected, ref
            )

            records.append({
                "run": run,
                "model": model_name,
                "regime": regime,
                "image": image,
                "true_dy": float(dy),
                "true_dx": float(dx),
                "shift_magnitude": float(np.hypot(dy, dx)),
                "estimated_dy": edy,
                "estimated_dx": edx,
                "epe": float(est["epe"]),
                "uncorrected_psnr": uncorrected_psnr,
                "estimated_psnr": estimated_psnr,
                "oracle_psnr": oracle_psnr,
                "clean_psnr": clean_psnr,
                "estimated_gain_db":
                    estimated_psnr - uncorrected_psnr,
                "oracle_gain_db":
                    oracle_psnr - uncorrected_psnr,
                "estimated_oracle_gap_db":
                    estimated_psnr - oracle_psnr
            })

            completed.add(
                (run, image, float(dy), float(dx))
            )

        save_results()

        if (
            image_index % 5 == 0
            or image_index == len(TEST)
        ):
            count = sum(
                r["run"] == run for r in records
            )

            print(
                f"  Patch {image_index:02d}/40 | "
                f"Model records {count:03d}/{40*len(OFFSETS)}",
                flush=True
            )

    del model, checkpoint

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 12G — FINAL ANALYSIS AND QUALITY CONTROL
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("12G — FINAL SUBPIXEL PERFORMANCE SUMMARY")
print("=" * 88)

fusion = save_results()

EXPECTED_FUSION = len(RUNS)*len(TEST)*len(OFFSETS)

assert len(fusion) == EXPECTED_FUSION

assert not fusion.duplicated(
    ["run", "image", "true_dy", "true_dx"]
).any()

numeric_cols = [
    "uncorrected_psnr",
    "estimated_psnr",
    "oracle_psnr",
    "clean_psnr",
    "epe"
]

assert np.isfinite(
    fusion[numeric_cols].to_numpy(float)
).all()

fusion_summary = (
    fusion.groupby(
        ["model", "regime", "shift_magnitude"]
    )
    .agg(
        n=("image", "size"),
        uncorrected=("uncorrected_psnr", "mean"),
        estimated=("estimated_psnr", "mean"),
        oracle=("oracle_psnr", "mean"),
        clean=("clean_psnr", "mean"),
        estimated_gain=("estimated_gain_db", "mean"),
        oracle_gain=("oracle_gain_db", "mean"),
        gap_to_oracle=("estimated_oracle_gap_db", "mean"),
        mean_epe=("epe", "mean")
    )
    .reset_index()
)

fusion_summary.to_csv(
    OUT / "Step12_Fusion_Condition_Summary.csv",
    index=False
)

print(fusion_summary.round(5).to_string(index=False))

aligned_test = test[test["is_aligned"] == 1]

report = {
    "experiment": "synthetic subpixel translation robustness",
    "validation_patches": len(VAL),
    "test_patches": len(TEST),
    "offsets_per_patch": len(OFFSETS),
    "registration_cases": len(test),
    "fusion_cases": len(fusion),
    "selected_method": SELECTED_METHOD,
    "validation_integer_epe": val_integer_mae,
    "validation_refined_epe": val_refined_mae,
    "test_mean_epe": float(test["epe"].mean()),
    "test_median_epe": float(test["epe"].median()),
    "test_aligned_mean_epe": float(
        aligned_test["epe"].mean()
    ),
    "test_aligned_exact_rate_at_1e_6_px": float(
        (aligned_test["epe"] <= 1e-6).mean()
    ),
    "normalizer": NORMALIZER,
    "frozen_projection_sha256": projection_sha,
    "registration_uses_reference_hrhsi": False,
    "test_ground_truth_used_for_method_selection": False,
    "caveat": (
        "Exploratory MDAS study; integer and fractional "
        "perturbations do not establish natural-scene "
        "registration accuracy. Double resampling changes "
        "radiometry even under oracle geometric correction."
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step12_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 12 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 12 COMPLETE")
print("Original source, imagery and checkpoints unchanged.")


In [ ]:

# =============================================================================
# STEP 13 — IDENTITY-PRESERVING SUBPIXEL REGISTRATION GATE
#
# Purpose:
#   Stop unnecessary subpixel correction of already aligned images.
#
# Uses:
#   Step 12 validation and test registration estimates
#   Step 12 original/estimated/oracle fusion scores
#   Original MDAS images and four seed-7 checkpoints
#
# Calibration:
#   32 validation patches ONLY.
#
# Evaluation:
#   40 test patches × 17 displacements = 680 registration cases
#   4 models × 680 cases = 2,720 fusion comparisons
#
# No retraining.
# No changes to original checkpoints, images or earlier results.
#
# Scientific status:
#   Exploratory development following earlier MDAS test inspection.
# =============================================================================

from google.colab import drive
from pathlib import Path
import os
import sys
import json
import time
import tempfile
import hashlib

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile

# -------------------------------------------------------------------------
# 1. DRIVE AND PROJECT CONFIGURATION
# -------------------------------------------------------------------------

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")

PROJECT = (
    ROOT / "MDAS_Registration_Aware_Revision_2026"
)

REPO = (
    PROJECT / "Step05_Source_Recovery" /
    "SCSRNet_Recorded_Commit"
)

DATA = REPO / "datasets" / "mdas_new"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
STEP12 = PROJECT / "Step12_Subpixel_Robustness"

OUT = PROJECT / "Step13_Identity_Preserving_Gate"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"

NORMALIZER = float(
    CONFIG["mdasn_dataset"]["max_value"]
)

assert NORMALIZER == 15000.0

RUNS = [
    "SCSRNet_p00_none_s7",
    "SCSRNet_p50_one_two_s7",
    "MSDCNN_p00_none_s7",
    "MSDCNN_p50_one_two_s7"
]

MODEL_CLASSES = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

OFFSETS = [(0.0, 0.0)]

for d in [0.5, 1.5, 3.5]:
    OFFSETS.extend([
        (d, 0.0),
        (-d, 0.0),
        (0.0, d),
        (0.0, -d)
    ])

for dy in [-1.5, 1.5]:
    for dx in [-1.5, 1.5]:
        OFFSETS.append((dy, dx))

assert len(OFFSETS) == 17

# Candidate thresholds are fixed before validation selection.
THRESHOLDS = np.round(
    np.arange(0.0, 0.4001, 0.025),
    6
)

print("=" * 88, flush=True)
print("STEP 13 — IDENTITY-PRESERVING REGISTRATION GATE", flush=True)
print("=" * 88, flush=True)
print("Device:", DEVICE, flush=True)
print("Threshold candidates:", THRESHOLDS.tolist())
print("Outputs:", OUT, flush=True)

# -------------------------------------------------------------------------
# 2. LOAD PREVIOUS VALIDATION AND TEST ESTIMATES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("13A — VERIFYING STEP 12 ESTIMATES")
print("=" * 88)

with open(
    STEP12 / "Step12_Frozen_Selection.json",
    encoding="utf-8"
) as f:
    step12_selection = json.load(f)

if step12_selection["method"] != "quadratic_subpixel":
    raise RuntimeError(
        "Step 13 is configured for the Step 12 "
        "quadratic_subpixel method."
    )

VAL = pd.read_csv(
    STEP12 / "Step12_Validation_Shifts.csv"
)

TEST = pd.read_csv(
    STEP12 / "Step12_Test_Registration.csv"
)

PRIOR_FUSION = pd.read_csv(
    STEP12 / "Step12_Fusion_Patch_Results.csv"
)

assert len(VAL) == 32 * 17
assert len(TEST) == 40 * 17
assert len(PRIOR_FUSION) == 4 * 40 * 17

required_val = {
    "image", "true_dy", "true_dx",
    "refined_dy", "refined_dx"
}

required_test = {
    "image", "true_dy", "true_dx",
    "estimated_dy", "estimated_dx"
}

assert required_val.issubset(VAL.columns)
assert required_test.issubset(TEST.columns)

KEYS = ["image", "true_dy", "true_dx"]

assert not VAL.duplicated(KEYS).any()
assert not TEST.duplicated(KEYS).any()

assert not PRIOR_FUSION.duplicated(
    ["run"] + KEYS
).any()

def read_split(name):
    lines = (
        DATA / f"{name}.txt"
    ).read_text().splitlines()

    names = [
        line.split()[0]
        for line in lines
        if line.strip()
    ]

    assert len(names) == len(set(names))

    return names

TRAIN_NAMES = read_split("train")
VAL_NAMES = read_split("val")
TEST_NAMES = read_split("test")

assert len(TRAIN_NAMES) == 133
assert len(VAL_NAMES) == 32
assert len(TEST_NAMES) == 40

assert set(VAL["image"]) == set(VAL_NAMES)
assert set(TEST["image"]) == set(TEST_NAMES)

assert not (set(TRAIN_NAMES) & set(VAL_NAMES))
assert not (set(TRAIN_NAMES) & set(TEST_NAMES))
assert not (set(VAL_NAMES) & set(TEST_NAMES))

expected_keys = {
    (image, float(dy), float(dx))
    for image in TEST_NAMES
    for dy, dx in OFFSETS
}

actual_keys = set(
    zip(
        TEST["image"],
        TEST["true_dy"].astype(float),
        TEST["true_dx"].astype(float)
    )
)

assert actual_keys == expected_keys

print("Validation registration cases:", len(VAL))
print("Test registration cases:", len(TEST))
print("Previous fusion cases:", len(PRIOR_FUSION))
print("Data partitions: VERIFIED")
print("Step 12 method: quadratic_subpixel")

# -------------------------------------------------------------------------
# 3. DEFINE IDENTITY-PRESERVING GATE
# -------------------------------------------------------------------------

def gated_displacement(dy, dx, threshold):
    """
    If estimated displacement magnitude is <= threshold,
    preserve the input without any geometric resampling.
    Otherwise retain the original subpixel estimate.
    """
    dy = float(dy)
    dx = float(dx)

    magnitude = np.hypot(dy, dx)

    if magnitude <= float(threshold) + 1e-12:
        return 0.0, 0.0, True

    return dy, dx, False

# -------------------------------------------------------------------------
# 4. SELECT THRESHOLD ON VALIDATION DATA ONLY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("13B — VALIDATION-ONLY GATE SELECTION")
print("=" * 88)

selection_rows = []

for threshold in THRESHOLDS:

    rows = []

    for r in VAL.itertuples(index=False):

        gy, gx, snapped = gated_displacement(
            r.refined_dy,
            r.refined_dx,
            threshold
        )

        true_dy = float(r.true_dy)
        true_dx = float(r.true_dx)

        rows.append({
            "image": r.image,
            "true_dy": true_dy,
            "true_dx": true_dx,
            "gated_dy": gy,
            "gated_dx": gx,
            "snapped_to_zero": int(snapped),
            "epe": float(
                np.hypot(
                    gy - true_dy,
                    gx - true_dx
                )
            ),
            "is_aligned": int(
                true_dy == 0.0 and true_dx == 0.0
            ),
            "is_nonzero": int(
                true_dy != 0.0 or true_dx != 0.0
            )
        })

    candidate = pd.DataFrame(rows)

    aligned = candidate[
        candidate["is_aligned"] == 1
    ]

    nonzero = candidate[
        candidate["is_nonzero"] == 1
    ]

    aligned_preserved = float(
        (
            (aligned["gated_dy"] == 0.0)
            & (aligned["gated_dx"] == 0.0)
        ).mean()
    )

    nonzero_snapped_rate = float(
        nonzero["snapped_to_zero"].mean()
    )

    selection_rows.append({
        "threshold": float(threshold),
        "aligned_preserved_rate": aligned_preserved,
        "aligned_false_correction_rate":
            1.0 - aligned_preserved,
        "validation_mean_epe": float(
            candidate["epe"].mean()
        ),
        "validation_nonzero_mean_epe": float(
            nonzero["epe"].mean()
        ),
        "nonzero_snapped_rate": nonzero_snapped_rate,
        "overall_snapped_rate": float(
            candidate["snapped_to_zero"].mean()
        )
    })

SELECTION = pd.DataFrame(selection_rows)

# Primary: minimize false correction on aligned validation images.
# Secondary: minimize mean EPE across the complete validation protocol.
# Tertiary: prefer smaller threshold.
SELECTION = SELECTION.sort_values(
    [
        "aligned_false_correction_rate",
        "validation_mean_epe",
        "threshold"
    ],
    ascending=[True, True, True]
).reset_index(drop=True)

SELECTION.to_csv(
    OUT / "Step13_Validation_Threshold_Selection.csv",
    index=False
)

print(
    SELECTION.round(6).to_string(index=False)
)

WINNER = SELECTION.iloc[0]
TAU = float(WINNER["threshold"])

print("\nSELECTED GATE")
print("Threshold:", TAU, "pixels")
print(
    "Validation aligned preservation:",
    f"{100*WINNER['aligned_preserved_rate']:.2f}%"
)
print(
    "Validation mean EPE:",
    f"{WINNER['validation_mean_epe']:.6f} px"
)
print(
    "Nonzero shifts incorrectly snapped:",
    f"{100*WINNER['nonzero_snapped_rate']:.2f}%"
)

frozen = {
    "method": "identity_preserving_displacement_gate",
    "threshold_pixels": TAU,
    "selection_source": "32 MDAS validation patches",
    "selection_priority": [
        "minimum aligned false-correction rate",
        "minimum overall endpoint error",
        "smallest threshold"
    ],
    "test_labels_used_for_selection": False,
    "base_estimator": "Step12 quadratic_subpixel",
    "evaluation_status": (
        "exploratory refinement after earlier MDAS "
        "test results were inspected"
    )
}

FROZEN_PATH = OUT / "Step13_Frozen_Gate.json"

with open(
    FROZEN_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(frozen, f, indent=2)

GATE_SHA = hashlib.sha256(
    FROZEN_PATH.read_bytes()
).hexdigest()

# -------------------------------------------------------------------------
# 5. EVALUATE THE FROZEN GATE ON TEST ESTIMATES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("13C — TEST REGISTRATION EVALUATION")
print("=" * 88)

test_rows = []

for r in TEST.itertuples(index=False):

    dy = float(r.true_dy)
    dx = float(r.true_dx)

    original_dy = float(r.estimated_dy)
    original_dx = float(r.estimated_dx)

    gated_dy, gated_dx, snapped = gated_displacement(
        original_dy,
        original_dx,
        TAU
    )

    prior_epe = float(
        np.hypot(original_dy - dy, original_dx - dx)
    )

    gated_epe = float(
        np.hypot(gated_dy - dy, gated_dx - dx)
    )

    test_rows.append({
        "image": r.image,
        "true_dy": dy,
        "true_dx": dx,
        "shift_magnitude": float(
            np.hypot(dy, dx)
        ),
        "original_dy": original_dy,
        "original_dx": original_dx,
        "gated_dy": gated_dy,
        "gated_dx": gated_dx,
        "snapped_to_zero": int(snapped),
        "prior_epe": prior_epe,
        "gated_epe": gated_epe,
        "epe_improvement": prior_epe - gated_epe,
        "is_aligned": int(
            dy == 0.0 and dx == 0.0
        )
    })

TEST_GATED = pd.DataFrame(test_rows)

assert len(TEST_GATED) == 680

TEST_GATED.to_csv(
    OUT / "Step13_Gated_Test_Registration.csv",
    index=False
)

REGISTRATION_SUMMARY = (
    TEST_GATED.groupby("shift_magnitude")
    .agg(
        n=("image", "size"),
        prior_mean_epe=("prior_epe", "mean"),
        gated_mean_epe=("gated_epe", "mean"),
        median_gated_epe=("gated_epe", "median"),
        max_gated_epe=("gated_epe", "max"),
        snapped_rate=("snapped_to_zero", "mean")
    )
    .reset_index()
)

REGISTRATION_SUMMARY.to_csv(
    OUT / "Step13_Registration_Summary.csv",
    index=False
)

print(
    REGISTRATION_SUMMARY.round(6).to_string(
        index=False
    )
)

aligned_test = TEST_GATED[
    TEST_GATED["is_aligned"] == 1
]

aligned_preservation = float(
    (
        (aligned_test["gated_dy"] == 0.0)
        & (aligned_test["gated_dx"] == 0.0)
    ).mean()
)

print("\nOVERALL TEST REGISTRATION")
print(
    "Before gate:",
    f"{TEST_GATED['prior_epe'].mean():.6f} px"
)
print(
    "After gate:",
    f"{TEST_GATED['gated_epe'].mean():.6f} px"
)
print(
    "Aligned images preserved:",
    f"{100*aligned_preservation:.2f}%"
)

# -------------------------------------------------------------------------
# 6. SET UP FUSION INFERENCE
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("13D — PREPARING FUSION COMPARISONS")
print("=" * 88)

TEST_INDEX = TEST_GATED.set_index(KEYS)

PRIOR_INDEX = PRIOR_FUSION.set_index(
    ["run"] + KEYS
)

assert TEST_INDEX.index.is_unique
assert PRIOR_INDEX.index.is_unique

def load_modality(modality, name):
    array = tifffile.imread(
        DATA / modality / name
    ).astype(np.float32)

    array /= NORMALIZER

    if not np.isfinite(array).all():
        raise ValueError(
            f"Invalid image: {modality}/{name}"
        )

    return torch.from_numpy(
        np.ascontiguousarray(
            array.transpose(2, 0, 1)
        )
    ).unsqueeze(0).to(DEVICE)

def shift_tensor(image, dy, dx):
    """
    Match the Step 12 fractional translation exactly:
    grid_sample, bilinear interpolation, border replication.
    """
    dy, dx = float(dy), float(dx)

    if dy == 0.0 and dx == 0.0:
        return image

    n, c, h, w = image.shape

    yy, xx = torch.meshgrid(
        torch.arange(h, device=image.device),
        torch.arange(w, device=image.device),
        indexing="ij"
    )

    source_y = yy.float() - dy
    source_x = xx.float() - dx

    grid_y = 2.0 * source_y / (h - 1) - 1.0
    grid_x = 2.0 * source_x / (w - 1) - 1.0

    grid = torch.stack(
        (grid_x, grid_y),
        dim=-1
    ).unsqueeze(0).expand(n, -1, -1, -1)

    return F.grid_sample(
        image,
        grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True
    ).contiguous()

def compute_psnr(model, lr, msi, ref):

    with torch.inference_mode():

        output = model(lr, msi)

        if isinstance(output, dict):
            output = output["pred"]

        if tuple(output.shape) != (1, 242, 72, 72):
            raise ValueError(
                f"Incorrect prediction shape: {output.shape}"
            )

        score = PSNR(
            output[:, :, 4:-4, 4:-4].contiguous(),
            ref[:, :, 4:-4, 4:-4].contiguous()
        )

    return float(score.detach().cpu().item())

# -------------------------------------------------------------------------
# 7. RESUMABLE RESULTS
# -------------------------------------------------------------------------

OUTPUT_CSV = OUT / "Step13_Fusion_Patch_Results.csv"

COLUMNS = [
    "run", "model", "regime", "seed",
    "image", "true_dy", "true_dx",
    "shift_magnitude", "snapped_to_zero",
    "original_estimated_dy", "original_estimated_dx",
    "gated_dy", "gated_dx",
    "prior_epe", "gated_epe",
    "uncorrected_psnr", "prior_estimated_psnr",
    "gated_psnr", "oracle_psnr", "clean_psnr",
    "gain_vs_uncorrected",
    "gain_vs_step12",
    "gap_to_oracle"
]

if OUTPUT_CSV.exists():

    prior = pd.read_csv(OUTPUT_CSV)

    if not set(COLUMNS).issubset(prior.columns):
        raise ValueError(
            "Existing Step 13 CSV has an incompatible schema."
        )

    prior = prior[COLUMNS].copy()

    if prior.duplicated(
        ["run"] + KEYS
    ).any():
        raise ValueError(
            "Duplicate records in Step 13 resume file."
        )

    for r in prior.itertuples(index=False):

        test_row = TEST_INDEX.loc[
            (r.image, r.true_dy, r.true_dx)
        ]

        if (
            abs(r.gated_dy - test_row["gated_dy"]) > 1e-8
            or abs(r.gated_dx - test_row["gated_dx"]) > 1e-8
        ):
            raise ValueError(
                "Resume file uses a different gate. "
                "Preserve the old file before rerunning."
            )

    records = prior.to_dict("records")

    print(
        "Resuming existing records:",
        len(records)
    )

else:
    records = []

completed = {
    (
        r["run"],
        r["image"],
        float(r["true_dy"]),
        float(r["true_dx"])
    )
    for r in records
}

def save_progress():

    frame = pd.DataFrame(
        records,
        columns=COLUMNS
    )

    frame = frame.sort_values(
        ["run", "image", "true_dy", "true_dx"]
    ).reset_index(drop=True)

    fd, temp_path = tempfile.mkstemp(
        prefix="step13_",
        suffix=".csv",
        dir=str(OUT)
    )

    try:
        with os.fdopen(
            fd, "w", encoding="utf-8"
        ) as f:
            frame.to_csv(f, index=False)

        os.replace(
            temp_path,
            OUTPUT_CSV
        )

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    return frame

# -------------------------------------------------------------------------
# 8. FUSION EVALUATION WITH THE FROZEN GATE
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("13E — GATED FUSION EVALUATION", flush=True)
print("=" * 88, flush=True)

start = time.monotonic()

for run_index, run in enumerate(RUNS, 1):

    model_name = run.split("_")[0]
    regime = (
        run.replace(model_name + "_", "")
        .rsplit("_s", 1)[0]
    )
    seed = int(run.rsplit("_s", 1)[1])

    pending = sum(
        (
            run, image,
            float(dy), float(dx)
        ) not in completed
        for image in TEST_NAMES
        for dy, dx in OFFSETS
    )

    print(
        f"\nRUN {run_index}/4: {run} | "
        f"Pending {pending}",
        flush=True
    )

    if pending == 0:
        continue

    # Instantiate only the verified architecture.
    checkpoint = torch.load(
        EXPS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    model = MODEL_CLASSES[model_name](
        config=CONFIG
    )

    model.load_state_dict(
        checkpoint["model"],
        strict=True
    )

    model = model.to(DEVICE).eval()

    print(
        "Strict checkpoint loading: PASSED",
        flush=True
    )

    for image_index, image in enumerate(
        TEST_NAMES, 1
    ):

        missing = [
            (float(dy), float(dx))
            for dy, dx in OFFSETS
            if (
                run, image,
                float(dy), float(dx)
            ) not in completed
        ]

        if not missing:
            continue

        lr = load_modality("lrhsi", image)
        msi = load_modality("hrmsi", image)
        ref = load_modality("hrhsi", image)

        for dy, dx in missing:

            key = (image, dy, dx)

            estimate = TEST_INDEX.loc[key]
            baseline = PRIOR_INDEX.loc[
                (run, image, dy, dx)
            ]

            gated_dy = float(
                estimate["gated_dy"]
            )

            gated_dx = float(
                estimate["gated_dx"]
            )

            original_dy = float(
                estimate["original_dy"]
            )

            original_dx = float(
                estimate["original_dx"]
            )

            changed = (
                abs(gated_dy - original_dy) > 1e-12
                or abs(gated_dx - original_dx) > 1e-12
            )

            # Reuse validated Step 12 inference where the gate
            # does not change the original estimate.
            if not changed:

                gated_psnr = float(
                    baseline["estimated_psnr"]
                )

            else:

                with torch.inference_mode():

                    displaced = shift_tensor(
                        msi, dy, dx
                    )

                    corrected = shift_tensor(
                        displaced,
                        -gated_dy,
                        -gated_dx
                    )

                    gated_psnr = compute_psnr(
                        model,
                        lr,
                        corrected,
                        ref
                    )

            uncorrected = float(
                baseline["uncorrected_psnr"]
            )

            prior_estimated = float(
                baseline["estimated_psnr"]
            )

            oracle = float(
                baseline["oracle_psnr"]
            )

            clean = float(
                baseline["clean_psnr"]
            )

            records.append({
                "run": run,
                "model": model_name,
                "regime": regime,
                "seed": seed,
                "image": image,
                "true_dy": dy,
                "true_dx": dx,
                "shift_magnitude": float(
                    np.hypot(dy, dx)
                ),
                "snapped_to_zero": int(
                    estimate["snapped_to_zero"]
                ),
                "original_estimated_dy": original_dy,
                "original_estimated_dx": original_dx,
                "gated_dy": gated_dy,
                "gated_dx": gated_dx,
                "prior_epe": float(
                    estimate["prior_epe"]
                ),
                "gated_epe": float(
                    estimate["gated_epe"]
                ),
                "uncorrected_psnr": uncorrected,
                "prior_estimated_psnr":
                    prior_estimated,
                "gated_psnr": gated_psnr,
                "oracle_psnr": oracle,
                "clean_psnr": clean,
                "gain_vs_uncorrected":
                    gated_psnr - uncorrected,
                "gain_vs_step12":
                    gated_psnr - prior_estimated,
                "gap_to_oracle":
                    gated_psnr - oracle
            })

            completed.add(
                (run, image, dy, dx)
            )

        # Preserve completed patches in Drive.
        save_progress()

        if (
            image_index % 5 == 0
            or image_index == len(TEST_NAMES)
        ):

            run_count = sum(
                r["run"] == run
                for r in records
            )

            print(
                f"  Patch {image_index:02d}/40 | "
                f"Records {run_count:03d}/680",
                flush=True
            )

        del lr, msi, ref

    del model, checkpoint

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 9. FINAL FUSION SUMMARIES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("13F — FUSION PERFORMANCE SUMMARY")
print("=" * 88)

RESULTS = save_progress()

EXPECTED = len(RUNS) * len(TEST_NAMES) * len(OFFSETS)

assert len(RESULTS) == EXPECTED

assert not RESULTS.duplicated(
    ["run"] + KEYS
).any()

numeric_columns = [
    "uncorrected_psnr",
    "prior_estimated_psnr",
    "gated_psnr",
    "oracle_psnr",
    "clean_psnr",
    "prior_epe",
    "gated_epe"
]

assert np.isfinite(
    RESULTS[numeric_columns].to_numpy(float)
).all()

SUMMARY = (
    RESULTS.groupby(
        ["model", "regime", "shift_magnitude"]
    )
    .agg(
        n=("image", "size"),
        uncorrected=("uncorrected_psnr", "mean"),
        step12=("prior_estimated_psnr", "mean"),
        gated=("gated_psnr", "mean"),
        oracle=("oracle_psnr", "mean"),
        clean=("clean_psnr", "mean"),
        gain_vs_uncorrected=(
            "gain_vs_uncorrected", "mean"
        ),
        gain_vs_step12=(
            "gain_vs_step12", "mean"
        ),
        gap_to_oracle=("gap_to_oracle", "mean"),
        mean_gated_epe=("gated_epe", "mean")
    )
    .reset_index()
)

SUMMARY.to_csv(
    OUT / "Step13_Fusion_Condition_Summary.csv",
    index=False
)

print(
    SUMMARY.round(5).to_string(index=False)
)

# Separate aligned-input diagnostic.
ALIGNED = RESULTS[
    (RESULTS["true_dy"] == 0.0)
    & (RESULTS["true_dx"] == 0.0)
]

ALIGNED_SUMMARY = (
    ALIGNED.groupby(["model", "regime"])
    .agg(
        n=("image", "size"),
        clean=("clean_psnr", "mean"),
        step12=("prior_estimated_psnr", "mean"),
        gated=("gated_psnr", "mean"),
        recovered_db=("gain_vs_step12", "mean")
    )
    .reset_index()
)

ALIGNED_SUMMARY.to_csv(
    OUT / "Step13_Aligned_Input_Safety.csv",
    index=False
)

print("\nALIGNED INPUT SAFETY")
print(
    ALIGNED_SUMMARY.round(6).to_string(index=False)
)

# -------------------------------------------------------------------------
# 10. SAVE FINAL AUDIT
# -------------------------------------------------------------------------

REPORT = {
    "method": "identity-preserving subpixel correction gate",
    "selected_threshold_pixels": TAU,
    "gate_configuration_sha256": GATE_SHA,
    "validation_patches": len(VAL_NAMES),
    "test_patches": len(TEST_NAMES),
    "registration_cases": len(TEST_GATED),
    "fusion_cases": len(RESULTS),
    "validation_aligned_preservation": float(
        WINNER["aligned_preserved_rate"]
    ),
    "test_aligned_preservation": aligned_preservation,
    "test_registration_epe_before_gate": float(
        TEST_GATED["prior_epe"].mean()
    ),
    "test_registration_epe_after_gate": float(
        TEST_GATED["gated_epe"].mean()
    ),
    "test_gated_fraction": float(
        TEST_GATED["snapped_to_zero"].mean()
    ),
    "reference_hrhsi_used_for_offset_estimation": False,
    "test_labels_used_for_gate_selection": False,
    "source_models_modified": False,
    "evaluation_status": (
        "exploratory post-test methodological refinement"
    ),
    "caveat": (
        "The gate has been selected within an iterative MDAS "
        "development workflow. Independent scene-level "
        "evaluation is still required."
    ),
    "elapsed_seconds": round(
        time.monotonic() - start, 2
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step13_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(REPORT, f, indent=2)

print("\n" + "=" * 88)
print("STEP 13 — FINAL REPORT")
print("=" * 88)

print(
    json.dumps(REPORT, indent=2)
)

print("\nSTEP 13 COMPLETE")
print(
    "Original datasets, source code and checkpoints "
    "remain unchanged."
)


In [ ]:

# =============================================================================
# STEP 14 — MULTI-SEED FUSION REPLICATION AND UNCERTAINTY
#
# Models: SCSRNet, MSDCNN
# Regimes: p00_none, p50_one_two
# Seeds: 7, 17, 37
# Test patches: 40
# Conditions: 17 fractional/intact translations
#
# Total: 12 × 40 × 17 = 8,160 patch-condition records
#
# For every record:
#   A. Uncorrected displaced MSI
#   B. Step 13 validation-frozen gated correction
#   C. Known-offset oracle correction
#   D. Clean aligned input
#
# Reproduces seed-7 Step 13 results.
# Reports paired patch-level and hierarchical bootstrap uncertainty.
#
# No new training. No modification of original research files.
# Exploratory MDAS evaluation, not independent geographic validation.
# =============================================================================

from google.colab import drive
from pathlib import Path
import os, sys, json, hashlib, tempfile, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
DATA = REPO / "datasets" / "mdas_new"
EXPS = ROOT / "MDAS_JSTARS_Expanded"
STEP13 = PROJECT / "Step13_Identity_Preserving_Gate"
OUT = PROJECT / "Step14_Multiseed_Uncertainty"

OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODELS = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

REGIMES = ["p00_none", "p50_one_two"]
SEEDS = [7, 17, 37]

RUNS = [
    (model, regime, seed, f"{model}_{regime}_s{seed}")
    for model in MODELS
    for regime in REGIMES
    for seed in SEEDS
]

OFFSETS = [(0.0, 0.0)]

for value in (0.5, 1.5, 3.5):
    OFFSETS.extend([
        (value, 0.0),
        (-value, 0.0),
        (0.0, value),
        (0.0, -value)
    ])

for dy in (-1.5, 1.5):
    for dx in (-1.5, 1.5):
        OFFSETS.append((dy, dx))

assert len(OFFSETS) == 17

EXPECTED_RECORDS = len(RUNS) * 40 * len(OFFSETS)

print("=" * 88, flush=True)
print("STEP 14 — MULTI-SEED FUSION REPLICATION", flush=True)
print("=" * 88, flush=True)
print("Device:", DEVICE)
print("Checkpoints:", len(RUNS))
print("Expected comparisons:", EXPECTED_RECORDS)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 14A — VERIFY ORIGINAL DATA AND FROZEN REGISTRATION GATE
# -------------------------------------------------------------------------

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"
NORMALIZER = float(CONFIG["mdasn_dataset"]["max_value"])
assert NORMALIZER == 15000.0

def read_split(name):
    result = [
        line.split()[0]
        for line in (DATA / f"{name}.txt").read_text().splitlines()
        if line.strip()
    ]
    assert len(result) == len(set(result))
    return result

TRAIN = read_split("train")
VAL = read_split("val")
TEST = read_split("test")

assert (len(TRAIN), len(VAL), len(TEST)) == (133, 32, 40)
assert not set(TRAIN).intersection(VAL)
assert not set(TRAIN).intersection(TEST)
assert not set(VAL).intersection(TEST)

gate_path = STEP13 / "Step13_Frozen_Gate.json"

with open(gate_path, encoding="utf-8") as f:
    GATE = json.load(f)

TAU = float(GATE["threshold_pixels"])

assert abs(TAU - 0.225) < 1e-12

estimates_path = (
    STEP13 / "Step13_Gated_Test_Registration.csv"
)

ESTIMATES = pd.read_csv(estimates_path)

assert len(ESTIMATES) == 680

ESTIMATES = ESTIMATES.set_index([
    "image", "true_dy", "true_dx"
])

assert ESTIMATES.index.is_unique

expected_estimate_keys = {
    (image, float(dy), float(dx))
    for image in TEST
    for dy, dx in OFFSETS
}

assert set(ESTIMATES.index.tolist()) == expected_estimate_keys

# Ensure all 12 original checkpoints exist.
for model, regime, seed, run in RUNS:
    assert (EXPS / run / "best.pt").is_file(), run

gate_sha = hashlib.sha256(gate_path.read_bytes()).hexdigest()
estimate_sha = hashlib.sha256(
    estimates_path.read_bytes()
).hexdigest()

print("\n14A — INPUTS VERIFIED")
print("Training patches:", len(TRAIN))
print("Validation patches:", len(VAL))
print("Test patches:", len(TEST))
print("Gate threshold:", TAU)
print("Gate SHA256:", gate_sha)
print("Estimate table SHA256:", estimate_sha)

# -------------------------------------------------------------------------
# 14B — EXACT FRACTIONAL TRANSLATION FROM STEP 12/13
# -------------------------------------------------------------------------

def load_tensor(modality, image):
    path = DATA / modality / image

    array = (
        tifffile.imread(path).astype(np.float32)
        / NORMALIZER
    )

    if not np.isfinite(array).all():
        raise ValueError(f"Non-finite image: {path}")

    return torch.from_numpy(
        np.ascontiguousarray(array.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)

def shift_image(image, dy, dx):
    dy, dx = float(dy), float(dx)

    if dy == 0.0 and dx == 0.0:
        return image

    n, c, h, w = image.shape

    yy, xx = torch.meshgrid(
        torch.arange(h, device=image.device),
        torch.arange(w, device=image.device),
        indexing="ij"
    )

    source_y = yy.float() - dy
    source_x = xx.float() - dx

    grid_y = 2.0 * source_y / (h - 1) - 1.0
    grid_x = 2.0 * source_x / (w - 1) - 1.0

    grid = torch.stack(
        (grid_x, grid_y), dim=-1
    ).unsqueeze(0).expand(n, -1, -1, -1)

    return F.grid_sample(
        image,
        grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True
    ).contiguous()

def fusion_psnr(model, lr, msi, ref):
    with torch.inference_mode():
        output = model(lr, msi)

        if isinstance(output, dict):
            output = output["pred"]

        if tuple(output.shape) != (1, 242, 72, 72):
            raise ValueError(
                f"Unexpected prediction: {tuple(output.shape)}"
            )

        if not torch.isfinite(output).all():
            raise ValueError("Non-finite fusion prediction.")

        score = PSNR(
            output[:, :, 4:-4, 4:-4].contiguous(),
            ref[:, :, 4:-4, 4:-4].contiguous()
        )

    return float(score.detach().cpu().item())

# -------------------------------------------------------------------------
# 14C — RESUMABLE OUTPUT WITH PROVENANCE VERIFICATION
# -------------------------------------------------------------------------

RESULT_CSV = OUT / "Step14_Fusion_Patch_Results.csv"
PROVENANCE_PATH = OUT / "Step14_Provenance.json"

provenance = {
    "gate_sha256": gate_sha,
    "estimate_sha256": estimate_sha,
    "normalizer": NORMALIZER,
    "models": list(MODELS),
    "regimes": REGIMES,
    "seeds": SEEDS,
    "conditions": OFFSETS,
    "source": "original recovered repository",
    "evaluation": "original 64x64-crop repository PSNR"
}

if PROVENANCE_PATH.exists():
    saved = json.loads(PROVENANCE_PATH.read_text())

    if saved != provenance:
        raise ValueError(
            "Existing Step 14 outputs use different settings. "
            "Preserve the previous output directory."
        )
else:
    with open(PROVENANCE_PATH, "w") as f:
        json.dump(provenance, f, indent=2)

COLUMNS = [
    "run", "model", "regime", "seed", "image",
    "true_dy", "true_dx", "shift_magnitude",
    "estimated_dy", "estimated_dx", "epe",
    "snapped_to_zero",
    "uncorrected_psnr", "gated_psnr",
    "oracle_psnr", "clean_psnr",
    "gain_db", "oracle_gain_db",
    "gap_to_oracle_db"
]

if RESULT_CSV.exists():
    existing = pd.read_csv(RESULT_CSV)

    if not set(COLUMNS).issubset(existing.columns):
        raise ValueError("Incompatible existing Step 14 CSV.")

    existing = existing[COLUMNS]

    if existing.duplicated([
        "run", "image", "true_dy", "true_dx"
    ]).any():
        raise ValueError("Duplicate resume records.")

    allowed_runs = {run for _, _, _, run in RUNS}

    assert set(existing["run"]).issubset(allowed_runs)

    for row in existing.itertuples(index=False):
        est = ESTIMATES.loc[
            (row.image, row.true_dy, row.true_dx)
        ]

        if (
            abs(row.estimated_dy - est["gated_dy"]) > 1e-8
            or abs(row.estimated_dx - est["gated_dx"]) > 1e-8
        ):
            raise ValueError(
                "Stored results use another gate estimate."
            )

    records = existing.to_dict("records")

    print("\nResuming records:", len(records))

else:
    records = []

completed = {
    (
        r["run"], r["image"],
        float(r["true_dy"]),
        float(r["true_dx"])
    )
    for r in records
}

def atomic_save():
    frame = pd.DataFrame(records, columns=COLUMNS)
    frame = frame.sort_values([
        "run", "image", "true_dy", "true_dx"
    ]).reset_index(drop=True)

    fd, temporary = tempfile.mkstemp(
        prefix="step14_",
        suffix=".csv",
        dir=str(OUT)
    )

    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            frame.to_csv(f, index=False)

        os.replace(temporary, RESULT_CSV)

    finally:
        if os.path.exists(temporary):
            os.remove(temporary)

    return frame

# -------------------------------------------------------------------------
# 14D — INFERENCE ACROSS ALL THREE TRAINING SEEDS
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("14D — TWELVE-CHECKPOINT FUSION EVALUATION", flush=True)
print("=" * 88, flush=True)

start = time.monotonic()

for run_index, (model_name, regime, seed, run) in enumerate(
    RUNS, 1
):
    pending = sum(
        (run, image, float(dy), float(dx)) not in completed
        for image in TEST
        for dy, dx in OFFSETS
    )

    print(
        f"\nRUN {run_index:02d}/{len(RUNS)}: {run} | "
        f"Pending {pending}",
        flush=True
    )

    if pending == 0:
        continue

    checkpoint = torch.load(
        EXPS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    model = MODELS[model_name](config=CONFIG)
    model.load_state_dict(
        checkpoint["model"], strict=True
    )

    model = model.to(DEVICE).eval()

    print("Strict checkpoint loading: PASS", flush=True)

    for image_index, image in enumerate(TEST, 1):
        missing = [
            (float(dy), float(dx))
            for dy, dx in OFFSETS
            if (run, image, float(dy), float(dx))
            not in completed
        ]

        if not missing:
            continue

        lr = load_tensor("lrhsi", image)
        msi = load_tensor("hrmsi", image)
        ref = load_tensor("hrhsi", image)

        clean = fusion_psnr(model, lr, msi, ref)

        for dy, dx in missing:

            est = ESTIMATES.loc[(image, dy, dx)]

            estimated_dy = float(est["gated_dy"])
            estimated_dx = float(est["gated_dx"])
            snapped = int(est["snapped_to_zero"])

            displaced = shift_image(msi, dy, dx)

            uncorrected = fusion_psnr(
                model, lr, displaced, ref
            )

            # Gate preserves the observed MSI exactly
            # if the estimated displacement is zero.
            if (
                estimated_dy == 0.0
                and estimated_dx == 0.0
            ):
                corrected = displaced
            else:
                corrected = shift_image(
                    displaced,
                    -estimated_dy,
                    -estimated_dx
                )

            gated = fusion_psnr(
                model, lr, corrected, ref
            )

            oracle_corrected = shift_image(
                displaced, -dy, -dx
            )

            oracle = fusion_psnr(
                model, lr, oracle_corrected, ref
            )

            records.append({
                "run": run,
                "model": model_name,
                "regime": regime,
                "seed": seed,
                "image": image,
                "true_dy": dy,
                "true_dx": dx,
                "shift_magnitude": float(
                    np.hypot(dy, dx)
                ),
                "estimated_dy": estimated_dy,
                "estimated_dx": estimated_dx,
                "epe": float(est["gated_epe"]),
                "snapped_to_zero": snapped,
                "uncorrected_psnr": uncorrected,
                "gated_psnr": gated,
                "oracle_psnr": oracle,
                "clean_psnr": clean,
                "gain_db": gated-uncorrected,
                "oracle_gain_db": oracle-uncorrected,
                "gap_to_oracle_db": gated-oracle
            })

            completed.add((run, image, dy, dx))

        atomic_save()

        if (
            image_index % 5 == 0
            or image_index == len(TEST)
        ):
            count = sum(
                r["run"] == run for r in records
            )

            print(
                f"  Patch {image_index:02d}/40 | "
                f"Records {count:03d}/680 | "
                f"Elapsed {time.monotonic()-start:.1f}s",
                flush=True
            )

        del lr, msi, ref

    del model, checkpoint

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 14E — FULL INTEGRITY CHECK
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("14E — REPLICATION INTEGRITY")
print("=" * 88)

RESULTS = atomic_save()

assert len(RESULTS) == EXPECTED_RECORDS

assert not RESULTS.duplicated([
    "run", "image", "true_dy", "true_dx"
]).any()

assert np.isfinite(
    RESULTS[
        [
            "uncorrected_psnr",
            "gated_psnr",
            "oracle_psnr",
            "clean_psnr",
            "gain_db"
        ]
    ].to_numpy(float)
).all()

assert RESULTS.groupby("run").size().eq(680).all()

print("Completed comparisons:", len(RESULTS))
print("All checkpoint-condition combinations: VERIFIED")

# -------------------------------------------------------------------------
# 14F — REPRODUCE STEP 13 SEED-7 RESULTS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("14F — INDEPENDENT STEP 13 CONSISTENCY CHECK")
print("=" * 88)

old_path = STEP13 / "Step13_Fusion_Patch_Results.csv"
OLD = pd.read_csv(old_path)

old_seed7 = OLD[[
    "run", "image", "true_dy", "true_dx",
    "uncorrected_psnr", "gated_psnr",
    "oracle_psnr", "clean_psnr"
]]

new_seed7 = RESULTS[
    RESULTS["seed"] == 7
][[
    "run", "image", "true_dy", "true_dx",
    "uncorrected_psnr", "gated_psnr",
    "oracle_psnr", "clean_psnr"
]]

merged = new_seed7.merge(
    old_seed7,
    on=["run", "image", "true_dy", "true_dx"],
    how="outer",
    suffixes=("_new", "_old"),
    indicator=True,
    validate="one_to_one"
)

assert len(merged) == 2720
assert (merged["_merge"] == "both").all()

metric_differences = {}

for metric in [
    "uncorrected_psnr", "gated_psnr",
    "oracle_psnr", "clean_psnr"
]:
    difference = (
        merged[f"{metric}_new"]
        - merged[f"{metric}_old"]
    ).abs()

    metric_differences[metric] = float(
        difference.max()
    )

max_reproduction_error = max(
    metric_differences.values()
)

print("Maximum differences against Step 13:")
print(json.dumps(metric_differences, indent=2))

assert max_reproduction_error <= 0.01, (
    "Seed-7 reproduction differs from Step 13. "
    "Investigate before interpreting multi-seed results."
)

print("Seed-7 independent reproduction: PASS")

# -------------------------------------------------------------------------
# 14G — CONDITION AND SEED SUMMARIES
# -------------------------------------------------------------------------

SEED_SUMMARY = (
    RESULTS.groupby([
        "model", "regime", "seed", "shift_magnitude"
    ])
    .agg(
        n=("image", "size"),
        uncorrected=("uncorrected_psnr", "mean"),
        gated=("gated_psnr", "mean"),
        oracle=("oracle_psnr", "mean"),
        clean=("clean_psnr", "mean"),
        gain_db=("gain_db", "mean"),
        gap_to_oracle=("gap_to_oracle_db", "mean")
    )
    .reset_index()
)

SEED_SUMMARY.to_csv(
    OUT / "Step14_Per_Seed_Condition_Summary.csv",
    index=False
)

SUMMARY = (
    SEED_SUMMARY.groupby([
        "model", "regime", "shift_magnitude"
    ])
    .agg(
        seeds=("seed", "nunique"),
        uncorrected=("uncorrected", "mean"),
        gated=("gated", "mean"),
        oracle=("oracle", "mean"),
        clean=("clean", "mean"),
        gain_db=("gain_db", "mean"),
        gain_seed_sd=("gain_db", "std"),
        gap_to_oracle=("gap_to_oracle", "mean")
    )
    .reset_index()
)

SUMMARY.to_csv(
    OUT / "Step14_Three_Seed_Condition_Summary.csv",
    index=False
)

print("\n" + "=" * 88)
print("14G — THREE-SEED FUSION RESULTS")
print("=" * 88)
print(SUMMARY.round(5).to_string(index=False))

# -------------------------------------------------------------------------
# 14H — PAIRED HIERARCHICAL BOOTSTRAP
#
# Resample seeds and test patches within each model/regime,
# retaining the original paired displacement observations.
# All outputs are descriptive uncertainty intervals: only 3
# model seeds and 40 potentially spatially correlated patches.
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("14H — PAIRED BOOTSTRAP UNCERTAINTY")
print("=" * 88)

N_BOOT = 2000
RNG = np.random.default_rng(20261113)

BOOT_ROWS = []

for (model_name, regime, magnitude), subset in RESULTS.groupby(
    ["model", "regime", "shift_magnitude"]
):
    # Average over shift directions within each patch and seed.
    patch_seed = (
        subset.groupby(["seed", "image"])
        .agg(
            gain=("gain_db", "mean"),
            gated=("gated_psnr", "mean"),
            uncorrected=("uncorrected_psnr", "mean"),
            oracle_gap=("gap_to_oracle_db", "mean")
        )
        .reset_index()
    )

    seed_values = sorted(patch_seed["seed"].unique())
    image_values = sorted(patch_seed["image"].unique())

    assert seed_values == SEEDS
    assert len(image_values) == 40

    metrics = [
        "gain", "gated", "uncorrected", "oracle_gap"
    ]

    # shape: (3 seeds, 40 patches, 4 metrics)
    cube = np.empty(
        (len(seed_values), len(image_values), len(metrics)),
        dtype=np.float64
    )

    lookup = patch_seed.set_index(["seed", "image"])

    for i, seed in enumerate(seed_values):
        for j, image in enumerate(image_values):
            cube[i, j, :] = lookup.loc[
                (seed, image), metrics
            ].to_numpy(dtype=np.float64)

    observed = cube.mean(axis=(0, 1))

    bootstrap = np.empty(
        (N_BOOT, len(metrics)),
        dtype=np.float64
    )

    for b in range(N_BOOT):

        seed_ids = RNG.integers(
            0, len(seed_values), len(seed_values)
        )

        patch_ids = RNG.integers(
            0, len(image_values), len(image_values)
        )

        # Same sampled patches are used across all sampled seeds,
        # preserving pairing between methods and architectures.
        resampled = cube[
            seed_ids[:, None],
            patch_ids[None, :],
            :
        ]

        bootstrap[b] = resampled.mean(axis=(0, 1))

    lower = np.percentile(bootstrap, 2.5, axis=0)
    upper = np.percentile(bootstrap, 97.5, axis=0)

    row = {
        "model": model_name,
        "regime": regime,
        "shift_magnitude": magnitude,
        "n_patches": 40,
        "n_seeds": 3
    }

    for k, metric in enumerate(metrics):
        row[f"{metric}_mean"] = float(observed[k])
        row[f"{metric}_ci_low"] = float(lower[k])
        row[f"{metric}_ci_high"] = float(upper[k])

    BOOT_ROWS.append(row)

BOOTSTRAP = pd.DataFrame(BOOT_ROWS)

BOOTSTRAP.to_csv(
    OUT / "Step14_Bootstrap_95CI.csv",
    index=False
)

print(
    BOOTSTRAP[
        [
            "model", "regime", "shift_magnitude",
            "gain_mean", "gain_ci_low", "gain_ci_high"
        ]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 14I — PAIRED COMPARISON OF TRAINING REGIMES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("14I — AUGMENTATION VS NO AUGMENTATION")
print("=" * 88)

paired_rows = []

for model_name in MODELS:

    baseline = RESULTS[
        (RESULTS["model"] == model_name)
        & (RESULTS["regime"] == "p00_none")
    ]

    augmented = RESULTS[
        (RESULTS["model"] == model_name)
        & (RESULTS["regime"] == "p50_one_two")
    ]

    keys = [
        "seed", "image", "true_dy", "true_dx"
    ]

    comparison = baseline.merge(
        augmented,
        on=keys,
        suffixes=("_baseline", "_augmented"),
        validate="one_to_one"
    )

    assert len(comparison) == 3*40*17

    comparison["shift_magnitude"] = np.hypot(
        comparison["true_dy"],
        comparison["true_dx"]
    )

    comparison["delta_gated"] = (
        comparison["gated_psnr_augmented"]
        - comparison["gated_psnr_baseline"]
    )

    comparison["delta_uncorrected"] = (
        comparison["uncorrected_psnr_augmented"]
        - comparison["uncorrected_psnr_baseline"]
    )

    for magnitude, group in comparison.groupby(
        "shift_magnitude"
    ):
        paired_rows.append({
            "model": model_name,
            "shift_magnitude": magnitude,
            "n": len(group),
            "augmentation_delta_gated_db": float(
                group["delta_gated"].mean()
            ),
            "augmentation_delta_uncorrected_db": float(
                group["delta_uncorrected"].mean()
            )
        })

PAIRED = pd.DataFrame(paired_rows)

PAIRED.to_csv(
    OUT / "Step14_Paired_Augmentation_Comparison.csv",
    index=False
)

print(PAIRED.round(5).to_string(index=False))

# -------------------------------------------------------------------------
# 14J — FINAL MACHINE-READABLE REPORT
# -------------------------------------------------------------------------

REPORT = {
    "experiment": "multi-seed subpixel registration-aware fusion",
    "architectures": list(MODELS),
    "training_regimes": REGIMES,
    "seeds": SEEDS,
    "checkpoints": len(RUNS),
    "test_patches": len(TEST),
    "conditions_per_patch": len(OFFSETS),
    "expected_records": EXPECTED_RECORDS,
    "completed_records": len(RESULTS),
    "seed7_reproduction_max_difference_db":
        max_reproduction_error,
    "registration_gate_threshold_px": TAU,
    "gate_sha256": gate_sha,
    "estimate_table_sha256": estimate_sha,
    "bootstrap_replicates": N_BOOT,
    "bootstrap_unit": (
        "test patch and model initialization seed; "
        "directional shifts remain paired"
    ),
    "reference_hrhsi_used_for_registration": False,
    "test_shifts_used_for_registration_selection": False,
    "all_checkpoints_loaded_strictly": True,
    "source_modified": False,
    "limitations": [
        "synthetic fractional translations",
        "three training seeds",
        "possible spatial correlation between patches",
        "iterative methodological development on MDAS",
        "oracle and estimated correction involve repeated resampling",
        "not an independent scene-level confirmation"
    ],
    "elapsed_seconds": round(
        time.monotonic()-start, 2
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step14_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(REPORT, f, indent=2)

print("\n" + "=" * 88)
print("STEP 14 — FINAL REPORT")
print("=" * 88)

print(json.dumps(REPORT, indent=2))

print("\nSTEP 14 COMPLETE")
print("Original source, imagery and checkpoints remain unchanged.")


In [ ]:

# =============================================================================
# STEP 15 — MDAS SPATIAL INDEPENDENCE AND DATASET PROVENANCE AUDIT
#
# Checks:
# 1. Train/validation/test patch naming and source mosaic availability.
# 2. Patch-to-mosaic exact pixel consistency.
# 3. Cross-split duplicate HR-HSI patches (SHA256).
# 4. Georeferencing metadata and geographic footprint overlap,
#    only when valid CRS/transform metadata are available.
# 5. Within-scene patch adjacency / spatial non-independence.
# 6. Explicit PASS / UNRESOLVED / WARNING conclusions.
#
# No modification of source datasets.
# =============================================================================

from google.colab import drive
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import re
import json
import hashlib
import numpy as np
import pandas as pd
import tifffile

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
DATA = REPO / "datasets" / "mdas_new"
OUT = PROJECT / "Step15_Spatial_Independence_Audit"

OUT.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]
PATCH_SHAPE = (72, 72, 242)

print("=" * 88)
print("STEP 15 — MDAS SPATIAL INDEPENDENCE AUDIT")
print("=" * 88)
print("Dataset:", DATA)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 15A — PATCH INVENTORY
# -------------------------------------------------------------------------

def read_split(split):
    path = DATA / f"{split}.txt"
    names = [
        x.strip().split()[0]
        for x in path.read_text().splitlines()
        if x.strip()
    ]
    if len(names) != len(set(names)):
        raise ValueError(f"Duplicate filenames in {split}")
    return names

patch_rows = []
split_files = {}

pattern = re.compile(
    r"^(?P<prefix>.+)_(?P<row>\d+)_(?P<col>\d+)\.tif$"
)

for split in SPLITS:
    names = read_split(split)
    split_files[split] = names

    for name in names:
        m = pattern.match(Path(name).name)

        record = {
            "split": split,
            "image": name,
            "prefix": m.group("prefix") if m else None,
            "row": int(m.group("row")) if m else None,
            "col": int(m.group("col")) if m else None,
            "naming_parsed": bool(m),
            "patch_exists": (DATA / "hrhsi" / name).is_file()
        }

        patch_rows.append(record)

patch_df = pd.DataFrame(patch_rows)

assert [len(split_files[s]) for s in SPLITS] == [133, 32, 40]
assert patch_df["patch_exists"].all()

all_names = [n for names in split_files.values() for n in names]
assert len(all_names) == len(set(all_names))

print("\n15A — PARTITION INVENTORY")
print(
    patch_df.groupby("split")
    .agg(
        patches=("image", "size"),
        parsed_names=("naming_parsed", "sum"),
        row_min=("row", "min"),
        row_max=("row", "max"),
        col_min=("col", "min"),
        col_max=("col", "max")
    ).to_string()
)

patch_df.to_csv(
    OUT / "Step15_Patch_Inventory.csv",
    index=False
)

# -------------------------------------------------------------------------
# 15B — SOURCE MOSAIC INVENTORY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("15B — SOURCE MOSAICS")
print("=" * 88)

area_folders = [
    "area_ref",
    "area_hrmsi",
    "area_lrhsi"
]

mosaic_rows = []

for folder in area_folders:
    directory = DATA / folder

    if not directory.exists():
        print("Missing:", directory)
        continue

    for path in sorted(directory.glob("*.tif")):
        with tifffile.TiffFile(path) as tif:
            page = tif.pages[0]

            geotags = [
                tag for tag in [
                    "ModelPixelScaleTag",
                    "ModelTiepointTag",
                    "ModelTransformationTag",
                    "GeoKeyDirectoryTag",
                    "GeoAsciiParamsTag"
                ]
                if tag in page.tags
            ]

            mosaic_rows.append({
                "folder": folder,
                "filename": path.name,
                "path": str(path),
                "shape": str(tuple(tif.series[0].shape)),
                "dtype": str(tif.series[0].dtype),
                "geotiff_tags": ",".join(geotags),
                "has_geotiff_tags": len(geotags) > 0
            })

mosaics = pd.DataFrame(mosaic_rows)

mosaics.to_csv(
    OUT / "Step15_Source_Mosaic_Inventory.csv",
    index=False
)

if not mosaics.empty:
    print(
        mosaics[
            ["folder", "filename", "shape",
             "dtype", "geotiff_tags"]
        ].to_string(index=False)
    )

# -------------------------------------------------------------------------
# 15C — PATCH-TO-MOSAIC PIXEL CONSISTENCY
#
# Filename row/col are treated as hypotheses, not proof
# of real-world geographic coordinates.
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("15C — PATCH-TO-MOSAIC CONSISTENCY")
print("=" * 88)

mosaic_match_rows = []

for split in SPLITS:
    mosaic_path = DATA / "area_ref" / f"{split}.tif"

    if not mosaic_path.exists():
        print(f"{split}: No area_ref/{split}.tif")
        continue

    mosaic = tifffile.imread(mosaic_path)

    if mosaic.ndim != 3 or mosaic.shape[-1] != 242:
        print(
            f"{split}: Unsupported reference mosaic shape",
            mosaic.shape
        )
        continue

    H, W, _ = mosaic.shape
    subset = patch_df[patch_df["split"] == split]

    matches = 0
    checked = 0

    for row in subset.itertuples(index=False):
        patch = tifffile.imread(
            DATA / "hrhsi" / row.image
        )

        if tuple(patch.shape) != PATCH_SHAPE:
            raise ValueError(
                f"Unexpected patch shape: {row.image}"
            )

        if not row.naming_parsed:
            mosaic_match_rows.append({
                "split": split,
                "image": row.image,
                "match": False,
                "reason": "unparsed_filename"
            })
            continue

        top = int(row.row) * 72
        left = int(row.col) * 72

        inside = (
            top >= 0 and left >= 0
            and top + 72 <= H
            and left + 72 <= W
        )

        exact = (
            bool(np.array_equal(
                patch,
                mosaic[top:top+72, left:left+72, :]
            ))
            if inside else False
        )

        checked += 1
        matches += int(exact)

        mosaic_match_rows.append({
            "split": split,
            "image": row.image,
            "match": exact,
            "reason": (
                "exact_grid_match" if exact
                else "grid_outside_mosaic" if not inside
                else "not_equal_at_grid_location"
            ),
            "top": top,
            "left": left,
            "bottom": top + 72,
            "right": left + 72
        })

    print(
        f"{split}: exact grid matches {matches}/{checked} "
        f"| mosaic {mosaic.shape}"
    )

    del mosaic

mosaic_matches = pd.DataFrame(mosaic_match_rows)

mosaic_matches.to_csv(
    OUT / "Step15_Patch_Mosaic_Verification.csv",
    index=False
)

# -------------------------------------------------------------------------
# 15D — CROSS-SPLIT EXACT DUPLICATES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("15D — CROSS-SPLIT EXACT IMAGE DUPLICATES")
print("=" * 88)

hash_rows = []

for i, row in enumerate(patch_df.itertuples(index=False), 1):
    image = tifffile.imread(
        DATA / "hrhsi" / row.image
    )

    digest = hashlib.sha256(
        np.ascontiguousarray(image).tobytes()
    ).hexdigest()

    hash_rows.append({
        "split": row.split,
        "image": row.image,
        "sha256_pixels": digest
    })

    if i % 50 == 0:
        print(f"  Hashed {i}/{len(patch_df)} patches")

hash_df = pd.DataFrame(hash_rows)

hash_df.to_csv(
    OUT / "Step15_Patch_Pixel_Hashes.csv",
    index=False
)

duplicate_groups = (
    hash_df.groupby("sha256_pixels")
    .agg(
        n=("image", "size"),
        n_splits=("split", "nunique"),
        splits=("split", lambda x: ",".join(sorted(set(x)))),
        images=("image", lambda x: ",".join(x))
    )
    .reset_index()
)

cross_split_duplicates = duplicate_groups[
    duplicate_groups["n_splits"] > 1
]

cross_split_duplicates.to_csv(
    OUT / "Step15_Cross_Split_Duplicates.csv",
    index=False
)

print(
    "Exact cross-split duplicate groups:",
    len(cross_split_duplicates)
)

# -------------------------------------------------------------------------
# 15E — PATCH GRID ADJACENCY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("15E — PATCH GRID ADJACENCY")
print("=" * 88)

adjacency_rows = []

parsed = patch_df.dropna(
    subset=["prefix", "row", "col"]
).copy()

for (split, prefix), group in parsed.groupby(
    ["split", "prefix"]
):
    coords = set(
        zip(group["row"].astype(int),
            group["col"].astype(int))
    )

    adjacent = 0
    diagonal = 0

    for r, c in coords:
        if (r + 1, c) in coords:
            adjacent += 1
        if (r, c + 1) in coords:
            adjacent += 1

        if (r + 1, c + 1) in coords:
            diagonal += 1
        if (r + 1, c - 1) in coords:
            diagonal += 1

    adjacency_rows.append({
        "split": split,
        "prefix": prefix,
        "n_patches": len(group),
        "edge_adjacent_pairs": adjacent,
        "diagonally_adjacent_pairs": diagonal
    })

adjacency = pd.DataFrame(adjacency_rows)

adjacency.to_csv(
    OUT / "Step15_Within_Split_Adjacency.csv",
    index=False
)

print(adjacency.to_string(index=False))

print(
    "\nAdjacent patches are potentially correlated."
    "\nThey should not automatically be treated as "
    "independent geographic samples."
)

# -------------------------------------------------------------------------
# 15F — GEOREFERENCING INSPECTION
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("15F — GEOREFERENCING INSPECTION")
print("=" * 88)

geo_rows = []

try:
    import rasterio
    from rasterio.warp import transform_bounds

    rasterio_available = True

except ImportError:
    rasterio_available = False

if rasterio_available:
    for item in mosaic_rows:
        path = Path(item["path"])

        try:
            with rasterio.open(path) as src:
                has_crs = src.crs is not None
                has_transform = not src.transform.is_identity

                record = {
                    "folder": item["folder"],
                    "filename": path.name,
                    "crs": str(src.crs) if has_crs else "",
                    "has_crs": has_crs,
                    "has_nonidentity_transform": has_transform,
                    "bounds": str(tuple(src.bounds)),
                    "valid_georeference": bool(
                        has_crs and has_transform
                    )
                }

                if record["valid_georeference"]:
                    b = transform_bounds(
                        src.crs, "EPSG:4326",
                        *src.bounds,
                        densify_pts=21
                    )

                    record.update({
                        "west": b[0],
                        "south": b[1],
                        "east": b[2],
                        "north": b[3]
                    })

                geo_rows.append(record)

        except Exception as exc:
            geo_rows.append({
                "folder": item["folder"],
                "filename": path.name,
                "valid_georeference": False,
                "error": repr(exc)
            })

else:
    print(
        "Rasterio unavailable; geographic overlap "
        "cannot be checked automatically."
    )

geo = pd.DataFrame(geo_rows)

geo.to_csv(
    OUT / "Step15_Georeferencing_Metadata.csv",
    index=False
)

if not geo.empty:
    print(
        geo[
            ["folder", "filename", "crs",
             "valid_georeference"]
        ].to_string(index=False)
    )

# -------------------------------------------------------------------------
# 15G — CROSS-SPLIT GEOGRAPHIC FOOTPRINT OVERLAP
# -------------------------------------------------------------------------

geo_overlap_rows = []

ref_geo = [
    row for row in geo_rows
    if row.get("folder") == "area_ref"
    and row.get("valid_georeference", False)
    and all(
        key in row
        for key in ["west", "south", "east", "north"]
    )
]

def bbox_intersection(a, b):
    west = max(a["west"], b["west"])
    south = max(a["south"], b["south"])
    east = min(a["east"], b["east"])
    north = min(a["north"], b["north"])

    return (
        west < east and south < north
    )

for a, b in combinations(ref_geo, 2):
    geo_overlap_rows.append({
        "mosaic_a": a["filename"],
        "mosaic_b": b["filename"],
        "bbox_overlap": bbox_intersection(a, b),
        "note": (
            "Bounding-box screening only; not polygon "
            "or patch-level geographic overlap"
        )
    })

overlaps = pd.DataFrame(
    geo_overlap_rows,
    columns=[
        "mosaic_a", "mosaic_b",
        "bbox_overlap", "note"
    ]
)

overlaps.to_csv(
    OUT / "Step15_Geographic_Bbox_Screening.csv",
    index=False
)

if overlaps.empty:
    print(
        "\nNo cross-split geographic bounding-box "
        "comparisons could be established."
    )
else:
    print(overlaps.to_string(index=False))

# -------------------------------------------------------------------------
# 15H — FINAL REPORT
# -------------------------------------------------------------------------

n_exact_duplicate_groups = len(
    cross_split_duplicates
)

mosaic_verified = (
    not mosaic_matches.empty
    and mosaic_matches.groupby("split")["match"]
    .all()
    .reindex(SPLITS)
    .fillna(False)
    .all()
)

has_all_georeferences = (
    len(ref_geo) >= 3
    and all(
        (DATA / "area_ref" / f"{s}.tif").name
        in [x["filename"] for x in ref_geo]
        for s in SPLITS
    )
)

any_bbox_overlap = (
    bool(overlaps["bbox_overlap"].any())
    if len(overlaps) else None
)

if n_exact_duplicate_groups:
    status = "WARNING_CROSS_SPLIT_DUPLICATES"
elif not has_all_georeferences:
    status = "GEOGRAPHIC_INDEPENDENCE_UNRESOLVED"
elif any_bbox_overlap:
    status = "WARNING_SOURCE_FOOTPRINT_OVERLAP"
else:
    status = "NO_BBOX_OVERLAP_DETECTED_NOT_FINAL_INDEPENDENCE_PROOF"

report = {
    "total_patches": len(patch_df),
    "train_patches": len(TRAIN_NAMES) if "TRAIN_NAMES" in globals() else len(split_files["train"]),
    "validation_patches": len(split_files["val"]),
    "test_patches": len(split_files["test"]),
    "cross_split_duplicate_groups": n_exact_duplicate_groups,
    "patch_to_mosaic_matches_all": bool(mosaic_verified),
    "reference_mosaics_with_valid_georeferencing": len(ref_geo),
    "all_three_reference_mosaics_georeferenced":
        bool(has_all_georeferences),
    "any_geographic_bbox_overlap": any_bbox_overlap,
    "status": status,
    "interpretation": (
        "Filename disjointness and duplicate checks alone "
        "do not establish geographic independence. "
        "The source scene geometry and spatial sampling "
        "must also be considered."
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step15_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 15 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))
print("\nSTEP 15 COMPLETE")


#**One correction to the scientific interpretation is essential:** these results support a strong *controlled robustness analysis* and a potentially useful registration-aware processing strategy. They do not yet demonstrate an entirely new image-fusion architecture or independent geographic generalization.

#Send the Step 15 mosaic verification, adjacency, georeferencing and final report. Those results will determine whether we can use the existing MDAS partitions defensibly or need an independent spatial test before developing the final publication figures.

In [ ]:

# =============================================================================
# STEP 16 — MDAS SOURCE LINEAGE AND CROSS-SPLIT SPATIAL DIAGNOSTICS
#
# 1. Verify that the three partitions form complete patch grids.
# 2. Search recovered source files for dataset construction details.
# 3. Quantify within-partition boundary continuity.
# 4. Screen cross-partition patch boundaries for possible adjacency.
# 5. Report what is proven and what remains unresolved.
#
# Does not alter imagery, repository code, or model checkpoints.
# Boundary similarity is NOT proof of geographic adjacency.
# =============================================================================

from google.colab import drive
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import tifffile
import hashlib
import json
import re

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
DATA = REPO / "datasets" / "mdas_new"
OUT = PROJECT / "Step16_Source_Lineage_Audit"
OUT.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
BANDS = [0, 20, 40, 60, 80, 100, 120, 140,
         160, 180, 200, 220, 241]

print("=" * 88)
print("STEP 16 — MDAS SOURCE LINEAGE AND CROSS-SPLIT AUDIT")
print("=" * 88)
print("Source:", REPO)
print("Outputs:", OUT)

# -------------------------------------------------------------------------
# 16A — READ THE ORIGINAL PARTITIONS
# -------------------------------------------------------------------------

def read_split(split):
    path = DATA / f"{split}.txt"
    names = [
        x.split()[0]
        for x in path.read_text().splitlines()
        if x.strip()
    ]
    assert len(names) == len(set(names))
    return names

split_names = {s: read_split(s) for s in SPLITS}

assert [len(split_names[s]) for s in SPLITS] == [133, 32, 40]

name_pattern = re.compile(
    r"^(.*?)_(\d+)_(\d+)\.tif$"
)

inventory = []

for split in SPLITS:
    for name in split_names[split]:
        match = name_pattern.fullmatch(name)

        if match is None:
            raise ValueError(f"Unexpected patch name: {name}")

        prefix, row, col = match.groups()

        inventory.append({
            "split": split,
            "image": name,
            "prefix": prefix,
            "row": int(row),
            "col": int(col)
        })

inventory = pd.DataFrame(inventory)

assert not inventory.duplicated(
    ["split", "row", "col"]
).any()

print("\n16A — PARTITION GRID STRUCTURE")

grid_rows = []

for split in SPLITS:
    subset = inventory[inventory.split == split]

    nrows = int(subset.row.max()) + 1
    ncols = int(subset.col.max()) + 1

    grid = set(zip(subset.row, subset.col))

    complete = grid == {
        (r, c)
        for r in range(nrows)
        for c in range(ncols)
    }

    grid_rows.append({
        "split": split,
        "patches": len(subset),
        "grid_rows": nrows,
        "grid_columns": ncols,
        "complete_grid": complete,
        "reconstructed_height_px": nrows * 72,
        "reconstructed_width_px": ncols * 72
    })

grid_df = pd.DataFrame(grid_rows)

print(grid_df.to_string(index=False))

grid_df.to_csv(
    OUT / "Step16_Partition_Grid_Structure.csv",
    index=False
)

# -------------------------------------------------------------------------
# 16B — SEARCH THE RECOVERED REPOSITORY FOR DATASET PROVENANCE
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("16B — SOURCE CODE AND DOCUMENTATION AUDIT")
print("=" * 88)

keywords = [
    "mdasn_dataset",
    "mdas_new",
    "augsburg",
    "split",
    "crop",
    "patch",
    "train.txt",
    "test.txt",
    "area_ref",
    "georefer",
    "coordinates"
]

extensions = {".py", ".md", ".rst", ".txt", ".json", ".yaml", ".yml"}

source_hits = []
files_checked = 0

for path in REPO.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    # Exclude large test/train filename listings from prose search.
    if path.parent == DATA and path.name in {
        "train.txt", "val.txt", "test.txt"
    }:
        continue

    if path.stat().st_size > 2_000_000:
        continue

    try:
        content = path.read_text(
            encoding="utf-8",
            errors="replace"
        )
    except Exception:
        continue

    files_checked += 1

    for lineno, line in enumerate(content.splitlines(), 1):

        found = [
            k for k in keywords
            if k in line.lower()
        ]

        if found:
            source_hits.append({
                "file": str(path.relative_to(REPO)),
                "line": lineno,
                "matched_terms": ", ".join(found),
                "text": line.strip()[:500]
            })

source_df = pd.DataFrame(
    source_hits,
    columns=["file", "line", "matched_terms", "text"]
)

source_df.to_csv(
    OUT / "Step16_Source_Provenance_Matches.csv",
    index=False
)

print("Source files inspected:", files_checked)
print("Relevant source lines:", len(source_df))

if not source_df.empty:

    # Prefer documentation and actual dataset references.
    priority = source_df[
        source_df["matched_terms"].str.contains(
            "mdasn_dataset|mdas_new|augsburg|area_ref|coordinates",
            regex=True
        )
    ]

    print("\nHIGH-PRIORITY SOURCE REFERENCES")
    print(
        priority.head(40).to_string(
            index=False, max_colwidth=110
        )
    )

# -------------------------------------------------------------------------
# 16C — LOAD REFERENCE PATCHES AND COMPARE BOUNDARIES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("16C — PATCH BOUNDARY DIAGNOSTICS")
print("=" * 88)

patch_data = {}
internal_variation = []

for row in inventory.itertuples(index=False):

    array = tifffile.imread(
        DATA / "hrhsi" / row.image
    )

    if array.shape != (72, 72, 242):
        raise ValueError(
            f"Unexpected dimensions for {row.image}: {array.shape}"
        )

    # Spectral sampling makes the screening efficient.
    reduced = array[:, :, BANDS].astype(np.float32)

    patch_data[row.image] = reduced

    # Local pixel-to-pixel difference establishes the
    # within-patch reference scale for each image.
    v = np.mean(
        np.abs(reduced[1:] - reduced[:-1])
    )

    h = np.mean(
        np.abs(reduced[:, 1:] - reduced[:, :-1])
    )

    internal_variation.append({
        "split": row.split,
        "image": row.image,
        "vertical_local_difference": float(v),
        "horizontal_local_difference": float(h)
    })

variation_df = pd.DataFrame(internal_variation)

variation_df.to_csv(
    OUT / "Step16_Local_Spatial_Variation.csv",
    index=False
)

# Each boundary comparison asks:
# "How different are the two adjoining pixel lines?"
#
# A good match has a boundary discontinuity similar to
# ordinary neighboring pixels inside a source image.

def edge_difference(a, b, orientation):
    if orientation == "a_left_of_b":
        edge_a = a[:, -1, :]
        edge_b = b[:, 0, :]

    elif orientation == "a_right_of_b":
        edge_a = a[:, 0, :]
        edge_b = b[:, -1, :]

    elif orientation == "a_above_b":
        edge_a = a[-1, :, :]
        edge_b = b[0, :, :]

    elif orientation == "a_below_b":
        edge_a = a[0, :, :]
        edge_b = b[-1, :, :]

    else:
        raise ValueError(orientation)

    return float(np.mean(np.abs(edge_a-edge_b)))


ORIENTATIONS = [
    "a_left_of_b",
    "a_right_of_b",
    "a_above_b",
    "a_below_b"
]

# -------------------------------------------------------------------------
# 16D — WITHIN-SPLIT ADJACENCY AS POSITIVE CONTROLS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("16D — VERIFIED WITHIN-MOSAIC BOUNDARIES")
print("=" * 88)

within_rows = []

for split in SPLITS:

    subset = inventory[inventory.split == split]

    coordinates = {
        (int(row.row), int(row.col)): row.image
        for row in subset.itertuples(index=False)
    }

    for (r, c), image_a in coordinates.items():

        for neighbor, direction in [
            ((r, c+1), "a_left_of_b"),
            ((r+1, c), "a_above_b")
        ]:

            if neighbor not in coordinates:
                continue

            image_b = coordinates[neighbor]

            difference = edge_difference(
                patch_data[image_a],
                patch_data[image_b],
                direction
            )

            within_rows.append({
                "split": split,
                "image_a": image_a,
                "image_b": image_b,
                "orientation": direction,
                "boundary_mae": difference
            })

within_df = pd.DataFrame(within_rows)

within_df.to_csv(
    OUT / "Step16_Within_Split_Boundaries.csv",
    index=False
)

within_stats = (
    within_df.groupby("split")["boundary_mae"]
    .agg(["count", "median", "mean", "min", "max"])
)

print(within_stats.round(3).to_string())

# -------------------------------------------------------------------------
# 16E — CROSS-SPLIT BOUNDARY SEARCH
#
# Only compare patch edges on OUTER boundaries of each
# partition's reconstructed rectangular grid.
#
# This screens whether two split mosaics could be adjacent.
# It does not prove a shared source coordinate system.
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("16E — CROSS-SPLIT OUTER BOUNDARY SCREENING")
print("=" * 88)

outer = {}

for split in SPLITS:

    subset = inventory[inventory.split == split]

    rmax = int(subset.row.max())
    cmax = int(subset.col.max())

    outer[split] = {
        "top": subset[subset.row == 0]["image"].tolist(),
        "bottom": subset[subset.row == rmax]["image"].tolist(),
        "left": subset[subset.col == 0]["image"].tolist(),
        "right": subset[subset.col == cmax]["image"].tolist()
    }

cross_rows = []

# A boundary match is plausible only when outer sides
# have mutually compatible orientations.
side_pairs = [
    ("right", "left", "a_left_of_b"),
    ("left", "right", "a_right_of_b"),
    ("bottom", "top", "a_above_b"),
    ("top", "bottom", "a_below_b")
]

for split_a, split_b in combinations(SPLITS, 2):

    for side_a, side_b, orientation in side_pairs:

        for image_a in outer[split_a][side_a]:
            for image_b in outer[split_b][side_b]:

                difference = edge_difference(
                    patch_data[image_a],
                    patch_data[image_b],
                    orientation
                )

                cross_rows.append({
                    "split_a": split_a,
                    "split_b": split_b,
                    "side_a": side_a,
                    "side_b": side_b,
                    "image_a": image_a,
                    "image_b": image_b,
                    "orientation": orientation,
                    "boundary_mae": difference
                })

cross_df = pd.DataFrame(cross_rows)

cross_df = cross_df.sort_values(
    "boundary_mae"
).reset_index(drop=True)

cross_df.to_csv(
    OUT / "Step16_Cross_Split_Boundary_Candidates.csv",
    index=False
)

print("\nCLOSEST CROSS-SPLIT EDGE CANDIDATES")
print(
    cross_df.head(25).round(3).to_string(index=False)
)

# Compare candidate discontinuities with internal seam
# distributions. This is descriptive, not probabilistic.
comparison_rows = []

for (split_a, split_b), group in cross_df.groupby(
    ["split_a", "split_b"]
):

    reference = within_df[
        within_df["split"].isin([split_a, split_b])
    ]["boundary_mae"]

    comparison_rows.append({
        "split_a": split_a,
        "split_b": split_b,
        "tested_boundary_combinations": len(group),
        "minimum_cross_boundary_mae":
            float(group["boundary_mae"].min()),
        "median_cross_boundary_mae":
            float(group["boundary_mae"].median()),
        "median_within_boundary_mae":
            float(reference.median()),
        "cross_min_to_within_median_ratio":
            float(
                group["boundary_mae"].min()
                / max(reference.median(), 1e-12)
            )
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.to_csv(
    OUT / "Step16_Cross_Split_Boundary_Summary.csv",
    index=False
)

print("\nCROSS-SPLIT BOUNDARY SUMMARY")
print(comparison_df.round(4).to_string(index=False))

# -------------------------------------------------------------------------
# 16F — FINAL REPORT
# -------------------------------------------------------------------------

report = {
    "experiment": "MDAS source lineage and split adjacency screening",
    "train_patches": len(split_names["train"]),
    "validation_patches": len(split_names["val"]),
    "test_patches": len(split_names["test"]),
    "all_split_grids_complete": bool(
        grid_df["complete_grid"].all()
    ),
    "source_files_inspected": files_checked,
    "source_lines_flagged": len(source_df),
    "verified_within_split_boundary_pairs": len(within_df),
    "cross_split_outer_edge_pairs_screened": len(cross_df),
    "geographic_coordinates_verified": False,
    "cross_split_adjacency_proven": False,
    "geographic_independence_proven": False,
    "status": "SOURCE_LINEAGE_REQUIRES_EXTERNAL_METADATA",
    "interpretation": (
        "The test and validation mosaics were verified at "
        "the pixel-grid level in Step 15. This audit can "
        "identify potential cross-split spatial continuity "
        "but cannot determine geographic overlap without "
        "source coordinates, scene boundaries or authoritative "
        "dataset construction documentation."
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step16_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 16 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))
print("\nSTEP 16 COMPLETE")
print("No original files modified.")


In [ ]:

# =============================================================================
# STEP 17 — SPATIAL-BLOCK BOOTSTRAP FOR MDAS REGISTRATION-AWARE FUSION
#
# Input: Step 14 verified 8,160 paired test results.
#
# Test layout: 4 rows × 10 columns of 72×72-pixel patches.
#
# Block designs:
#   1×2 : 20 spatial clusters (sensitivity)
#   2×2 : 10 spatial clusters (primary)
#   2×5 :  4 spatial clusters (conservative sensitivity)
#   4×5 :  2 spatial clusters (diagnostic only, highly unstable CI)
#
# Resampling units: spatial blocks and training seeds.
# Preserve pairing of correction/baseline and augmented/unaugmented.
# Preserve shift directions within patch.
#
# No reference HR-HSI images are needed.
# No model retraining.
# =============================================================================

from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd
import json
import re
import hashlib
import time

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/MDAS_Registration_Aware_Revision_2026"
)

SOURCE = (
    PROJECT / "Step14_Multiseed_Uncertainty" /
    "Step14_Fusion_Patch_Results.csv"
)

OUT = PROJECT / "Step17_Spatial_Block_Uncertainty"
OUT.mkdir(parents=True, exist_ok=True)

SEEDS = [7, 17, 37]
BLOCKS = [(1, 2), (2, 2), (2, 5), (4, 5)]
BOOTSTRAPS = 4000
BOOT_SEED = 20261113

print("=" * 88)
print("STEP 17 — SPATIAL-BLOCK BOOTSTRAP")
print("=" * 88)
print("Source:", SOURCE)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 17A — VERIFY STEP 14 RESULTS
# -------------------------------------------------------------------------

df = pd.read_csv(SOURCE)

required = {
    "run", "model", "regime", "seed", "image",
    "true_dy", "true_dx",
    "uncorrected_psnr", "gated_psnr",
    "oracle_psnr", "clean_psnr"
}

assert required.issubset(df.columns)
assert len(df) == 8160

keys = ["run", "image", "true_dy", "true_dx"]
assert not df.duplicated(keys).any()

assert set(df["seed"]) == set(SEEDS)
assert set(df["model"]) == {"SCSRNet", "MSDCNN"}
assert set(df["regime"]) == {"p00_none", "p50_one_two"}

numeric = [
    "uncorrected_psnr",
    "gated_psnr",
    "oracle_psnr",
    "clean_psnr"
]

assert np.isfinite(df[numeric].to_numpy(float)).all()

df["gain_db"] = (
    df["gated_psnr"] - df["uncorrected_psnr"]
)

df["oracle_gap_db"] = (
    df["gated_psnr"] - df["oracle_psnr"]
)

# All experimental magnitudes are stable to five decimal places.
df["shift"] = np.round(
    np.hypot(df["true_dy"], df["true_dx"]), 5
)

magnitudes = sorted(df["shift"].unique())

assert len(magnitudes) == 5, magnitudes

# -------------------------------------------------------------------------
# 17B — RECONSTRUCT THE TEST PATCH GRID
# -------------------------------------------------------------------------

pattern = re.compile(r"^test_(\d+)_(\d+)\.tif$")

images = sorted(df["image"].unique())

assert len(images) == 40

coordinates = {}

for name in images:
    match = pattern.fullmatch(name)
    if not match:
        raise ValueError(f"Invalid test patch identifier: {name}")

    row, col = map(int, match.groups())
    coordinates[name] = (row, col)

assert set(coordinates.values()) == {
    (r, c) for r in range(4) for c in range(10)
}

images = sorted(
    images,
    key=lambda n: coordinates[n]
)

image_indices = {name: i for i, name in enumerate(images)}

df["grid_row"] = df["image"].map(
    lambda x: coordinates[x][0]
)

df["grid_col"] = df["image"].map(
    lambda x: coordinates[x][1]
)

print("\n17A — INPUT INTEGRITY")
print("Records:", len(df))
print("Seeds:", sorted(df["seed"].unique()))
print("Test images:", len(images))
print("Shift magnitudes:", magnitudes)
print("Spatial layout: 4 × 10 COMPLETE")

# -------------------------------------------------------------------------
# 17C — CONSTRUCT NONOVERLAPPING SPATIAL BLOCKS
# -------------------------------------------------------------------------

def build_blocks(block_h, block_w):
    if 4 % block_h != 0 or 10 % block_w != 0:
        raise ValueError("Block does not tile 4×10 test grid.")

    blocks = []

    for top in range(0, 4, block_h):
        for left in range(0, 10, block_w):

            members = [
                image_indices[
                    next(
                        n for n, rc in coordinates.items()
                        if rc == (r, c)
                    )
                ]
                for r in range(top, top + block_h)
                for c in range(left, left + block_w)
            ]

            blocks.append(np.asarray(members, dtype=int))

    assert sum(len(x) for x in blocks) == 40
    assert sorted(np.concatenate(blocks).tolist()) == list(range(40))

    return blocks

block_structures = {
    f"{h}x{w}": build_blocks(h, w)
    for h, w in BLOCKS
}

print("\n17B — BLOCK CONFIGURATIONS")

for label, blocks in block_structures.items():
    print(
        f"{label}: {len(blocks)} blocks, "
        f"{len(blocks[0])} patches per block"
    )

# -------------------------------------------------------------------------
# 17D — MATRIX CONSTRUCTION
#
# Aggregate shift directions within each patch/seed before
# resampling; preserve every original paired comparison.
# -------------------------------------------------------------------------

def metric_cube(subset, metric):
    patch_seed = (
        subset.groupby(["seed", "image"])[metric]
        .mean()
        .reset_index()
    )

    lookup = patch_seed.set_index(["seed", "image"])

    cube = np.empty((3, 40), dtype=float)

    for i, seed in enumerate(SEEDS):
        for j, image in enumerate(images):
            cube[i, j] = float(
                lookup.loc[(seed, image), metric]
            )

    assert np.isfinite(cube).all()
    return cube

# -------------------------------------------------------------------------
# 17E — HIERARCHICAL BLOCK BOOTSTRAP
# -------------------------------------------------------------------------

def spatial_bootstrap(cube, blocks, rng):
    observed = float(cube.mean())
    samples = np.empty(BOOTSTRAPS, dtype=np.float64)

    n_blocks = len(blocks)

    for b in range(BOOTSTRAPS):

        # Sample three model initializations with replacement.
        sampled_seed = rng.integers(0, 3, 3)

        # Resample spatial blocks; all patches in a sampled
        # block are retained together.
        sampled_blocks = rng.integers(
            0, n_blocks, n_blocks
        )

        sampled_patches = np.concatenate([
            blocks[i] for i in sampled_blocks
        ])

        # Cross product of resampled seeds and spatial patches.
        selected = cube[
            np.ix_(sampled_seed, sampled_patches)
        ]

        samples[b] = selected.mean()

    low, high = np.percentile(samples, [2.5, 97.5])

    return {
        "mean": observed,
        "ci_low": float(low),
        "ci_high": float(high),
        "ci_width": float(high - low),
        "bootstrap_sd": float(samples.std(ddof=1))
    }

# -------------------------------------------------------------------------
# 17F — CORRECTION GAINS AND ORACLE GAPS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("17C — SPATIAL-BLOCK FUSION UNCERTAINTY")
print("=" * 88)

rng = np.random.default_rng(BOOT_SEED)
result_rows = []
start = time.monotonic()

for (model, regime, shift), subset in df.groupby(
    ["model", "regime", "shift"]
):

    for metric in ["gain_db", "oracle_gap_db"]:

        cube = metric_cube(subset, metric)

        for block_label, blocks in block_structures.items():

            outcome = spatial_bootstrap(
                cube, blocks, rng
            )

            result_rows.append({
                "model": model,
                "regime": regime,
                "shift": shift,
                "metric": metric,
                "block_size": block_label,
                "n_blocks": len(blocks),
                "n_seeds": 3,
                "n_patches": 40,
                **outcome
            })

block_results = pd.DataFrame(result_rows)

block_results.to_csv(
    OUT / "Step17_Spatial_Bootstrap_Results.csv",
    index=False
)

primary = block_results[
    (block_results["block_size"] == "2x2")
    & (block_results["metric"] == "gain_db")
].copy()

print("\nPRIMARY 2×2-BLOCK CONFIDENCE INTERVALS")

print(
    primary[
        [
            "model", "regime", "shift",
            "mean", "ci_low", "ci_high"
        ]
    ].round(5).to_string(index=False)
)

print("\nElapsed:", round(time.monotonic()-start, 2), "seconds")

# -------------------------------------------------------------------------
# 17G — PAIRED AUGMENTATION COMPARISON
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("17D — AUGMENTATION EFFECT AFTER CORRECTION")
print("=" * 88)

compare_keys = [
    "model", "seed", "image",
    "true_dy", "true_dx", "shift"
]

baseline = df[df["regime"] == "p00_none"]
augmented = df[df["regime"] == "p50_one_two"]

paired = baseline.merge(
    augmented,
    on=compare_keys,
    how="inner",
    validate="one_to_one",
    suffixes=("_baseline", "_augmented")
)

assert len(paired) == 4080

paired["delta_gated_db"] = (
    paired["gated_psnr_augmented"]
    - paired["gated_psnr_baseline"]
)

paired["delta_uncorrected_db"] = (
    paired["uncorrected_psnr_augmented"]
    - paired["uncorrected_psnr_baseline"]
)

augmentation_results = []

for (model, shift), group in paired.groupby(
    ["model", "shift"]
):

    for metric in [
        "delta_gated_db",
        "delta_uncorrected_db"
    ]:

        cube = metric_cube(group, metric)

        for block_label, blocks in block_structures.items():

            outcome = spatial_bootstrap(
                cube, blocks, rng
            )

            augmentation_results.append({
                "model": model,
                "shift": shift,
                "metric": metric,
                "block_size": block_label,
                "n_blocks": len(blocks),
                **outcome
            })

augmentation_df = pd.DataFrame(augmentation_results)

augmentation_df.to_csv(
    OUT / "Step17_Paired_Augmentation_Block_CI.csv",
    index=False
)

print("\nPRIMARY 2×2-BLOCK AUGMENTATION COMPARISONS")

selected_aug = augmentation_df[
    (augmentation_df["block_size"] == "2x2")
    & (augmentation_df["metric"] == "delta_gated_db")
]

print(
    selected_aug[
        ["model", "shift", "mean", "ci_low", "ci_high"]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 17H — BLOCK-SIZE SENSITIVITY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("17E — BLOCK-SIZE SENSITIVITY")
print("=" * 88)

sensitivity = block_results[
    block_results["metric"] == "gain_db"
][
    [
        "model", "regime", "shift",
        "block_size", "n_blocks",
        "mean", "ci_width", "bootstrap_sd"
    ]
].copy()

sensitivity.to_csv(
    OUT / "Step17_Block_Size_Sensitivity.csv",
    index=False
)

print(
    sensitivity[
        sensitivity["shift"] == 3.5
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 17I — REPORT
# -------------------------------------------------------------------------

source_sha = hashlib.sha256(
    SOURCE.read_bytes()
).hexdigest()

report = {
    "experiment": "MDAS spatial-block bootstrap sensitivity",
    "source_step": 14,
    "source_records": len(df),
    "source_sha256": source_sha,
    "test_patches": 40,
    "test_grid": [4, 10],
    "training_seeds": SEEDS,
    "bootstrap_replicates": BOOTSTRAPS,
    "primary_block": "2x2",
    "primary_n_blocks": 10,
    "block_sensitivity": {
        label: len(blocks)
        for label, blocks in block_structures.items()
    },
    "paired_methods": True,
    "shift_directions_aggregated_within_patch": True,
    "test_images_used_for_method_tuning": (
        "Earlier Steps 9–14 involved exploratory "
        "development on the MDAS test partition."
    ),
    "important_limitations": [
        "Blocks are pixel-grid clusters, not proven geographic units.",
        "Only one contiguous test mosaic is available.",
        "No independent geographic scenes are introduced.",
        "Only three training seeds are available.",
        "The 4x5 case has only two blocks and its "
        "bootstrap intervals are not reliable.",
        "Confidence intervals measure sensitivity to the "
        "chosen resampling design; they do not demonstrate "
        "independent scene-level generalization."
    ],
    "output_directory": str(OUT)
}

with open(
    OUT / "Step17_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 17 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 17 COMPLETE")
print("All previous research data remain unchanged.")


In [ ]:

# =============================================================================
# STEP 18 — SPATIAL-BLOCK DELETION AND TRAINING-SEED STABILITY
#
# Purpose:
#   Diagnose dependence of estimated fusion gains on specific
#   spatial regions of the MDAS test mosaic and model seeds.
#
# Input:
#   Step14_Fusion_Patch_Results.csv (8160 records)
#
# Outputs:
#   Full-model effect summaries
#   Leave-one-spatial-block-out estimates
#   Leave-one-seed-out estimates
#   Worst-case deletion sensitivity
#   Augmentation-vs-baseline paired comparisons
#   Final JSON report
#
# No retraining. No changes to prior experiment files.
#
# Note: deletion sensitivity is NOT an independent
# geographically representative confidence interval.
# =============================================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
import re

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/MDAS_Registration_Aware_Revision_2026"
)

SOURCE = (
    PROJECT /
    "Step14_Multiseed_Uncertainty" /
    "Step14_Fusion_Patch_Results.csv"
)

OUT = PROJECT / "Step18_Spatial_Seed_Stability_Audit"
OUT.mkdir(parents=True, exist_ok=True)

SEEDS = [7, 17, 37]
BLOCK_SIZES = [(1, 2), (2, 2), (2, 5), (4, 5)]

print("=" * 88)
print("STEP 18 — SPATIAL AND TRAINING-SEED STABILITY AUDIT")
print("=" * 88)
print("Input:", SOURCE)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 18A — LOAD AND VERIFY DATA
# -------------------------------------------------------------------------

assert SOURCE.is_file(), f"Missing file: {SOURCE}"

df = pd.read_csv(SOURCE)

required = {
    "model", "regime", "seed", "image",
    "true_dy", "true_dx",
    "uncorrected_psnr", "gated_psnr",
    "oracle_psnr", "clean_psnr"
}

assert required.issubset(df.columns)
assert len(df) == 8160

KEY = [
    "model", "regime", "seed", "image",
    "true_dy", "true_dx"
]

assert not df.duplicated(KEY).any()
assert set(df["seed"]) == set(SEEDS)
assert set(df["model"]) == {"SCSRNet", "MSDCNN"}
assert set(df["regime"]) == {"p00_none", "p50_one_two"}

numeric = [
    "uncorrected_psnr",
    "gated_psnr",
    "oracle_psnr",
    "clean_psnr"
]

assert np.isfinite(df[numeric].to_numpy(float)).all()

df["shift"] = np.round(
    np.hypot(df["true_dy"], df["true_dx"]), 5
)

df["gain_db"] = (
    df["gated_psnr"] - df["uncorrected_psnr"]
)

df["oracle_gap_db"] = (
    df["gated_psnr"] - df["oracle_psnr"]
)

shifts = sorted(df["shift"].unique())
assert len(shifts) == 5

pattern = re.compile(r"^test_(\d+)_(\d+)\.tif$")

def parse_position(name):
    m = pattern.fullmatch(str(name))
    if m is None:
        raise ValueError(f"Unexpected image identifier: {name}")
    return int(m.group(1)), int(m.group(2))

positions = {
    name: parse_position(name)
    for name in df["image"].unique()
}

assert len(positions) == 40
assert set(positions.values()) == {
    (r, c)
    for r in range(4)
    for c in range(10)
}

df["grid_row"] = df["image"].map(
    lambda n: positions[n][0]
)

df["grid_col"] = df["image"].map(
    lambda n: positions[n][1]
)

# Verify all conditions are present for every seed and patch.
expected_per_group = {
    0.0: 1,
    0.5: 4,
    1.5: 4,
    2.12132: 4,
    3.5: 4
}

for (model, regime, seed, image), group in df.groupby(
    ["model", "regime", "seed", "image"]
):
    actual = group.groupby("shift").size().to_dict()
    assert actual == expected_per_group, (
        f"Missing or duplicate shifts: "
        f"{model}, {regime}, {seed}, {image}"
    )

source_sha = hashlib.sha256(
    SOURCE.read_bytes()
).hexdigest()

print("\n18A — INPUT VERIFICATION PASSED")
print("Records:", len(df))
print("Unique patches:", len(positions))
print("Training seeds:", SEEDS)
print("Displacement magnitudes:", shifts)
print("Source SHA256:", source_sha)

# -------------------------------------------------------------------------
# 18B — AGGREGATE DIRECTIONS WITHIN PATCH AND SEED
# -------------------------------------------------------------------------

patch_seed = (
    df.groupby([
        "model", "regime", "seed",
        "image", "grid_row", "grid_col", "shift"
    ])
    .agg(
        gain_db=("gain_db", "mean"),
        gated_psnr=("gated_psnr", "mean"),
        uncorrected_psnr=("uncorrected_psnr", "mean"),
        oracle_gap_db=("oracle_gap_db", "mean")
    )
    .reset_index()
)

assert len(patch_seed) == 2 * 2 * 3 * 40 * 5

# -------------------------------------------------------------------------
# 18C — REFERENCE EFFECTS
# -------------------------------------------------------------------------

reference = (
    patch_seed.groupby(["model", "regime", "shift"])
    .agg(
        gain_db=("gain_db", "mean"),
        gated_psnr=("gated_psnr", "mean"),
        uncorrected_psnr=("uncorrected_psnr", "mean"),
        oracle_gap_db=("oracle_gap_db", "mean"),
        patches=("image", "nunique"),
        seeds=("seed", "nunique")
    )
    .reset_index()
)

reference.to_csv(
    OUT / "Step18_Full_Sample_Effects.csv",
    index=False
)

print("\n18B — FULL-SAMPLE GAINS")
print(
    reference[
        [
            "model", "regime", "shift",
            "gain_db", "oracle_gap_db"
        ]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 18D — SPATIAL-BLOCK CONSTRUCTION
# -------------------------------------------------------------------------

def construct_blocks(block_height, block_width):
    assert 4 % block_height == 0
    assert 10 % block_width == 0

    blocks = {}

    counter = 0

    for row_start in range(0, 4, block_height):
        for col_start in range(0, 10, block_width):

            members = {
                name
                for name, (row, col) in positions.items()
                if row_start <= row < row_start + block_height
                and col_start <= col < col_start + block_width
            }

            assert len(members) == block_height * block_width

            block_id = f"B{counter:02d}"
            blocks[block_id] = {
                "members": members,
                "row_start": row_start,
                "col_start": col_start
            }
            counter += 1

    combined = set().union(
        *[item["members"] for item in blocks.values()]
    )

    assert combined == set(positions)

    return blocks

block_designs = {
    f"{h}x{w}": construct_blocks(h, w)
    for h, w in BLOCK_SIZES
}

block_inventory = []

for size, blocks in block_designs.items():
    for block_id, block in blocks.items():
        block_inventory.append({
            "block_size": size,
            "block_id": block_id,
            "row_start": block["row_start"],
            "col_start": block["col_start"],
            "n_patches": len(block["members"]),
            "patches": ",".join(sorted(block["members"]))
        })

pd.DataFrame(block_inventory).to_csv(
    OUT / "Step18_Spatial_Block_Inventory.csv",
    index=False
)

# -------------------------------------------------------------------------
# 18E — LEAVE-ONE-BLOCK-OUT EFFECTS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("18C — LEAVE-ONE-SPATIAL-BLOCK-OUT ANALYSIS")
print("=" * 88)

block_deletions = []

for size, blocks in block_designs.items():

    print(
        f"  Block design {size}: {len(blocks)} blocks",
        flush=True
    )

    for block_id, block in blocks.items():

        remaining = patch_seed[
            ~patch_seed["image"].isin(block["members"])
        ]

        assert remaining["image"].nunique() == (
            40 - len(block["members"])
        )

        grouped = (
            remaining.groupby(["model", "regime", "shift"])
            .agg(
                gain_after_deletion=("gain_db", "mean"),
                remaining_patches=("image", "nunique")
            )
            .reset_index()
        )

        grouped["block_size"] = size
        grouped["deleted_block"] = block_id
        grouped["deleted_patches"] = len(block["members"])

        block_deletions.append(grouped)

block_deletions = pd.concat(
    block_deletions,
    ignore_index=True
)

block_deletions = block_deletions.merge(
    reference[
        ["model", "regime", "shift", "gain_db"]
    ],
    on=["model", "regime", "shift"],
    validate="many_to_one"
)

block_deletions["change_db"] = (
    block_deletions["gain_after_deletion"]
    - block_deletions["gain_db"]
)

block_deletions.to_csv(
    OUT / "Step18_Leave_One_Block_Out.csv",
    index=False
)

block_stability = (
    block_deletions.groupby(
        ["model", "regime", "shift", "block_size"]
    )
    .agg(
        deletion_cases=("deleted_block", "size"),
        minimum_gain=("gain_after_deletion", "min"),
        maximum_gain=("gain_after_deletion", "max"),
        maximum_absolute_change=(
            "change_db",
            lambda x: float(np.max(np.abs(x)))
        )
    )
    .reset_index()
)

block_stability.to_csv(
    OUT / "Step18_Block_Deletion_Summary.csv",
    index=False
)

print("\nPRIMARY 2×2-BLOCK DELETION RESULTS")
print(
    block_stability[
        block_stability["block_size"] == "2x2"
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 18F — LEAVE-ONE-TRAINING-SEED-OUT EFFECTS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("18D — TRAINING-SEED DELETION")
print("=" * 88)

seed_deletions = []

for omitted_seed in SEEDS:

    remaining = patch_seed[
        patch_seed["seed"] != omitted_seed
    ]

    grouped = (
        remaining.groupby(["model", "regime", "shift"])
        ["gain_db"]
        .mean()
        .reset_index(
            name="gain_after_seed_deletion"
        )
    )

    grouped["omitted_seed"] = omitted_seed

    seed_deletions.append(grouped)

seed_deletions = pd.concat(
    seed_deletions,
    ignore_index=True
)

seed_deletions = seed_deletions.merge(
    reference[
        ["model", "regime", "shift", "gain_db"]
    ],
    on=["model", "regime", "shift"],
    validate="many_to_one"
)

seed_deletions["change_db"] = (
    seed_deletions["gain_after_seed_deletion"]
    - seed_deletions["gain_db"]
)

seed_deletions.to_csv(
    OUT / "Step18_Leave_One_Seed_Out.csv",
    index=False
)

seed_stability = (
    seed_deletions.groupby(
        ["model", "regime", "shift"]
    )
    .agg(
        minimum_gain=("gain_after_seed_deletion", "min"),
        maximum_gain=("gain_after_seed_deletion", "max"),
        max_absolute_change=(
            "change_db",
            lambda x: float(np.max(np.abs(x)))
        )
    )
    .reset_index()
)

seed_stability.to_csv(
    OUT / "Step18_Seed_Deletion_Summary.csv",
    index=False
)

print(
    seed_stability.round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 18G — PAIRED AUGMENTATION COMPARISON
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("18E — AUGMENTATION EFFECT STABILITY")
print("=" * 88)

comparison_keys = [
    "model", "seed", "image",
    "grid_row", "grid_col", "shift"
]

baseline = patch_seed[
    patch_seed["regime"] == "p00_none"
]

augmented = patch_seed[
    patch_seed["regime"] == "p50_one_two"
]

paired = baseline.merge(
    augmented,
    on=comparison_keys,
    suffixes=("_baseline", "_augmented"),
    validate="one_to_one"
)

assert len(paired) == 2 * 3 * 40 * 5

paired["augmentation_delta_db"] = (
    paired["gated_psnr_augmented"]
    - paired["gated_psnr_baseline"]
)

augmentation_full = (
    paired.groupby(["model", "shift"])
    ["augmentation_delta_db"]
    .mean()
    .reset_index(name="full_delta_db")
)

augmentation_deletions = []

for size, blocks in block_designs.items():
    for block_id, block in blocks.items():

        subset = paired[
            ~paired["image"].isin(block["members"])
        ]

        grouped = (
            subset.groupby(["model", "shift"])
            ["augmentation_delta_db"]
            .mean()
            .reset_index(name="delta_after_deletion")
        )

        grouped["block_size"] = size
        grouped["deleted_block"] = block_id
        augmentation_deletions.append(grouped)

augmentation_deletions = pd.concat(
    augmentation_deletions,
    ignore_index=True
)

augmentation_deletions.to_csv(
    OUT / "Step18_Augmentation_Block_Deletions.csv",
    index=False
)

aug_stability = (
    augmentation_deletions.groupby(
        ["model", "shift", "block_size"]
    )
    .agg(
        min_delta=("delta_after_deletion", "min"),
        max_delta=("delta_after_deletion", "max")
    )
    .reset_index()
)

aug_stability = aug_stability.merge(
    augmentation_full,
    on=["model", "shift"],
    validate="many_to_one"
)

aug_stability.to_csv(
    OUT / "Step18_Augmentation_Stability.csv",
    index=False
)

print("\n2×2-BLOCK AUGMENTATION STABILITY")
print(
    aug_stability[
        aug_stability["block_size"] == "2x2"
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 18H — SUMMARIZE ROBUSTNESS WITHOUT INFLATING CLAIMS
# -------------------------------------------------------------------------

primary = block_stability[
    block_stability["block_size"] == "2x2"
].copy()

# Ignore aligned input when assessing correction benefit.
nonzero = primary[primary["shift"] > 0].copy()

nonzero["positive_after_every_deletion"] = (
    nonzero["minimum_gain"] > 0
)

print("\n" + "=" * 88)
print("18F — WORST-CASE SPATIAL DELETION CHECK")
print("=" * 88)

print(
    nonzero[
        [
            "model", "regime", "shift",
            "minimum_gain", "maximum_gain",
            "positive_after_every_deletion"
        ]
    ].round(5).to_string(index=False)
)

n_positive = int(
    nonzero["positive_after_every_deletion"].sum()
)

print(
    f"\nPositive gain under every 2×2 block deletion: "
    f"{n_positive}/{len(nonzero)} comparisons"
)

# -------------------------------------------------------------------------
# 18I — FINAL REPORT
# -------------------------------------------------------------------------

report = {
    "experiment": (
        "leave-one-spatial-block-out and "
        "leave-one-seed-out robustness"
    ),
    "source_sha256": source_sha,
    "source_records": len(df),
    "models": ["SCSRNet", "MSDCNN"],
    "regimes": ["p00_none", "p50_one_two"],
    "seeds": SEEDS,
    "test_grid": [4, 10],
    "test_patches": 40,
    "block_designs": {
        label: len(blocks)
        for label, blocks in block_designs.items()
    },
    "primary_block_design": "2x2",
    "primary_nonzero_conditions": len(nonzero),
    "positive_gain_under_every_primary_block_deletion":
        n_positive,
    "no_original_files_modified": True,
    "interpretation_limitations": [
        "Deletion stability is not a confidence interval.",
        "All test patches belong to one contiguous mosaic.",
        "The number of independent scenes remains one.",
        "Only three model training seeds are available.",
        "MDAS test cases were inspected during method development.",
        "These diagnostics do not replace independent evaluation."
    ],
    "output_directory": str(OUT)
}

with open(
    OUT / "Step18_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 18 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))
print("\nSTEP 18 COMPLETE")
print("Original datasets and results remain unchanged.")


In [ ]:

# =============================================================================
# STEP 19 — CLASSICAL PHASE-CORRELATION REGISTRATION BASELINES
#
# Compare:
#   A. Phase correlation on projected MSI intensity
#   B. Phase correlation on projected MSI gradient magnitude
#   C. Step 13 validation-frozen NCC + subpixel refinement + gate
#
# Data:
#   32 validation patches × 17 offsets
#   40 test patches × 17 offsets
#
# Phase-correlation calibration and variant selection:
#   Validation only.
#
# No test-shift labels are used to choose a method.
# No fusion checkpoint evaluation in this step.
#
# This is an exploratory methodological comparison on the
# previously inspected MDAS test partition.
# =============================================================================

from google.colab import drive
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile
import cv2

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")

PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

REPO = (
    PROJECT / "Step05_Source_Recovery" /
    "SCSRNet_Recorded_Commit"
)

DATA = REPO / "datasets" / "mdas_new"

STEP9 = PROJECT / "Step09_Estimated_Registration"
STEP13 = PROJECT / "Step13_Identity_Preserving_Gate"

OUT = PROJECT / "Step19_Classical_Registration_Baselines"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

NORMALIZER = 15000.0

OFFSETS = [(0.0, 0.0)]

for d in [0.5, 1.5, 3.5]:
    OFFSETS.extend([
        (d, 0.0), (-d, 0.0),
        (0.0, d), (0.0, -d)
    ])

for dy in [-1.5, 1.5]:
    for dx in [-1.5, 1.5]:
        OFFSETS.append((dy, dx))

assert len(OFFSETS) == 17

METHODS = ["phase_intensity", "phase_gradient"]

print("=" * 88)
print("STEP 19 — CLASSICAL REGISTRATION BASELINES")
print("=" * 88)
print("Device:", DEVICE)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 19A — VERIFY PARTITIONS
# -------------------------------------------------------------------------

def names_for(split):
    names = [
        line.split()[0]
        for line in (
            DATA / f"{split}.txt"
        ).read_text().splitlines()
        if line.strip()
    ]

    assert len(names) == len(set(names))
    return names

TRAIN = names_for("train")
VAL = names_for("val")
TEST = names_for("test")

assert (len(TRAIN), len(VAL), len(TEST)) == (133, 32, 40)

assert not (set(TRAIN) & set(VAL))
assert not (set(TRAIN) & set(TEST))
assert not (set(VAL) & set(TEST))

# -------------------------------------------------------------------------
# 19B — LOAD FROZEN STEP 9 SPECTRAL PROJECTION
# -------------------------------------------------------------------------

projection_path = STEP9 / "Step09_Spectral_Projection.npz"

with np.load(projection_path, allow_pickle=False) as z:
    mean = z["scaler_mean"].copy()
    scale = z["scaler_scale"].copy()
    coefficient = z["ridge_coefficients"].copy()
    intercept = z["ridge_intercept"].copy()

assert mean.shape == (242,)
assert scale.shape == (242,)
assert coefficient.shape == (4, 242)
assert intercept.shape == (4,)
assert np.all(scale > 0)

projection_sha = hashlib.sha256(
    projection_path.read_bytes()
).hexdigest()

def read_image(modality, name):
    arr = tifffile.imread(
        DATA / modality / name
    ).astype(np.float32) / NORMALIZER

    assert np.isfinite(arr).all()
    return arr

def tensor_hwc(arr):
    return torch.from_numpy(
        np.ascontiguousarray(
            arr.transpose(2, 0, 1)
        )
    ).unsqueeze(0).to(DEVICE)

def project_hsi(lrhsi):
    with torch.inference_mode():
        up = F.interpolate(
            tensor_hwc(lrhsi),
            size=(72, 72),
            mode="bicubic",
            align_corners=False
        )

    up = (
        up[0].permute(1, 2, 0)
        .cpu().numpy()
        .reshape(-1, 242)
        .astype(np.float64)
    )

    standardized = (up - mean) / scale

    result = (
        standardized @ coefficient.T
        + intercept
    )

    return result.reshape(72, 72, 4).astype(np.float32)

# -------------------------------------------------------------------------
# 19C — MATCH STEP 12 FRACTIONAL TRANSLATIONS
# -------------------------------------------------------------------------

def shift_image(arr, dy, dx):
    if dy == 0 and dx == 0:
        return arr.copy()

    x = tensor_hwc(arr)

    _, _, h, w = x.shape

    yy, xx = torch.meshgrid(
        torch.arange(h, device=DEVICE),
        torch.arange(w, device=DEVICE),
        indexing="ij"
    )

    source_y = yy.float() - float(dy)
    source_x = xx.float() - float(dx)

    grid = torch.stack([
        2 * source_x / (w-1) - 1,
        2 * source_y / (h-1) - 1
    ], dim=-1).unsqueeze(0)

    with torch.inference_mode():
        result = F.grid_sample(
            x, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )

    return (
        result[0].permute(1, 2, 0)
        .cpu().numpy()
        .astype(np.float32)
    )

# -------------------------------------------------------------------------
# 19D — CONSTRUCT CLASSICAL FEATURE REPRESENTATIONS
# -------------------------------------------------------------------------

def normalized_gray(arr):
    # Standardize each spectral band before aggregation.
    x = arr.astype(np.float32)

    mu = x.mean(axis=(0, 1), keepdims=True)
    sd = x.std(axis=(0, 1), keepdims=True)

    x = (x - mu) / np.maximum(sd, 1e-8)

    return np.ascontiguousarray(
        x.mean(axis=2), dtype=np.float32
    )

def feature_map(arr, method):
    gray = normalized_gray(arr)

    smooth = cv2.GaussianBlur(
        gray, (0, 0), sigmaX=1.2, sigmaY=1.2
    )

    if method == "phase_intensity":
        result = smooth

    elif method == "phase_gradient":
        gx = cv2.Sobel(
            smooth, cv2.CV_32F, 1, 0, ksize=3
        )
        gy = cv2.Sobel(
            smooth, cv2.CV_32F, 0, 1, ksize=3
        )
        result = cv2.magnitude(gx, gy)

    else:
        raise ValueError(method)

    return np.ascontiguousarray(
        result, dtype=np.float32
    )

HANN = cv2.createHanningWindow(
    (72, 72), cv2.CV_32F
)

def phase_estimate(ref_feature, observed_feature):
    (dx, dy), response = cv2.phaseCorrelate(
        ref_feature, observed_feature, HANN
    )

    return float(dy), float(dx), float(response)

# Confirm phase-correlation sign with a synthetic translation.
rng = np.random.default_rng(20261113)

toy = rng.standard_normal((72, 72)).astype(np.float32)
toy = cv2.GaussianBlur(toy, (0, 0), sigmaX=2)

toy_msi = np.repeat(toy[:, :, None], 4, axis=2)
toy_shifted = shift_image(toy_msi, 2.0, -3.0)

toy_y, toy_x, _ = phase_estimate(
    toy,
    toy_shifted[:, :, 0].copy()
)

# Phase correlation must approximately recover +2, -3.
# Sign errors should never be silently corrected from test labels.
assert abs(toy_y - 2.0) < 0.5, (toy_y, toy_x)
assert abs(toy_x + 3.0) < 0.5, (toy_y, toy_x)

print("\n19A — GEOMETRIC CONVENTION VERIFIED")

# -------------------------------------------------------------------------
# 19E — VALIDATION CALIBRATION
# -------------------------------------------------------------------------

def collect_raw(names, split):
    rows = []

    for i, image in enumerate(names, 1):
        projected = project_hsi(
            read_image("lrhsi", image)
        )

        clean = read_image("hrmsi", image)

        ref_features = {
            method: feature_map(projected, method)
            for method in METHODS
        }

        for dy, dx in OFFSETS:
            observed = shift_image(clean, dy, dx)

            for method in METHODS:
                obs_feature = feature_map(
                    observed, method
                )

                raw_dy, raw_dx, response = phase_estimate(
                    ref_features[method],
                    obs_feature
                )

                rows.append({
                    "split": split,
                    "image": image,
                    "method": method,
                    "true_dy": float(dy),
                    "true_dx": float(dx),
                    "magnitude": float(np.hypot(dy, dx)),
                    "raw_dy": raw_dy,
                    "raw_dx": raw_dx,
                    "response": response
                })

        if i % 8 == 0 or i == len(names):
            print(
                f"  {split}: {i}/{len(names)} patches",
                flush=True
            )

    return pd.DataFrame(rows)

print("\n" + "=" * 88)
print("19B — VALIDATION PHASE CORRELATION")
print("=" * 88)

validation = collect_raw(VAL, "validation")

assert len(validation) == 32 * 17 * 2
assert np.isfinite(
    validation[
        ["raw_dy", "raw_dx", "response"]
    ].to_numpy(float)
).all()

validation.to_csv(
    OUT / "Step19_Validation_Raw.csv",
    index=False
)

# Calibrate each method's sensor-specific origin offset
# using ONLY aligned validation patches.
# Gate selection also uses validation only.

CANDIDATE_TAU = np.arange(
    0.0, 0.5001, 0.025
)

selection_rows = []

for method in METHODS:
    subset = validation[
        validation["method"] == method
    ].copy()

    aligned = subset[
        subset["magnitude"] == 0
    ]

    bias_dy = float(aligned["raw_dy"].median())
    bias_dx = float(aligned["raw_dx"].median())

    subset["estimated_dy"] = (
        subset["raw_dy"] - bias_dy
    )
    subset["estimated_dx"] = (
        subset["raw_dx"] - bias_dx
    )

    for tau in CANDIDATE_TAU:
        trial = subset.copy()

        estimated_norm = np.hypot(
            trial["estimated_dy"],
            trial["estimated_dx"]
        )

        snap = estimated_norm <= tau + 1e-12

        trial.loc[snap, "estimated_dy"] = 0.0
        trial.loc[snap, "estimated_dx"] = 0.0

        epe = np.hypot(
            trial["estimated_dy"] - trial["true_dy"],
            trial["estimated_dx"] - trial["true_dx"]
        )

        aligned_trial = trial[
            trial["magnitude"] == 0
        ]

        aligned_exact = (
            (aligned_trial["estimated_dy"] == 0)
            & (aligned_trial["estimated_dx"] == 0)
        ).mean()

        selection_rows.append({
            "method": method,
            "tau": float(tau),
            "bias_dy": bias_dy,
            "bias_dx": bias_dx,
            "aligned_preservation": float(aligned_exact),
            "mean_epe": float(epe.mean()),
            "median_epe": float(np.median(epe)),
            "nonzero_snapped_rate": float(
                snap[subset["magnitude"] > 0].mean()
            )
        })

selection = pd.DataFrame(selection_rows)

# Select the phase-correlation variant and its gate entirely
# from validation performance.
selection = selection.sort_values(
    [
        "aligned_preservation",
        "mean_epe",
        "tau",
        "method"
    ],
    ascending=[False, True, True, True]
).reset_index(drop=True)

selection.to_csv(
    OUT / "Step19_Validation_Selection.csv",
    index=False
)

print("\nBEST VALIDATION SETTINGS BY METHOD")

best_per_method = (
    selection.groupby("method", sort=False)
    .head(1)
)

print(
    best_per_method.round(5).to_string(index=False)
)

chosen = selection.iloc[0]

FROZEN_METHOD = str(chosen["method"])
TAU = float(chosen["tau"])
BIAS_DY = float(chosen["bias_dy"])
BIAS_DX = float(chosen["bias_dx"])

frozen = {
    "selected_method": FROZEN_METHOD,
    "gate_threshold_px": TAU,
    "bias_dy": BIAS_DY,
    "bias_dx": BIAS_DX,
    "projection_sha256": projection_sha,
    "selection_split": "validation",
    "test_offsets_used_for_selection": False,
    "scientific_status": "exploratory_existing_test_partition"
}

with open(
    OUT / "Step19_Frozen_Baseline.json",
    "w"
) as f:
    json.dump(frozen, f, indent=2)

print("\nFROZEN CLASSICAL BASELINE")
print(json.dumps(frozen, indent=2))

# -------------------------------------------------------------------------
# 19F — EVALUATE FROZEN METHOD ON TEST IMAGERY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("19C — TEST CLASSICAL REGISTRATION")
print("=" * 88)

test_raw = collect_raw(TEST, "test")

assert len(test_raw) == 40 * 17 * 2

test_raw.to_csv(
    OUT / "Step19_Test_Raw.csv",
    index=False
)

test = test_raw[
    test_raw["method"] == FROZEN_METHOD
].copy()

test["estimated_dy"] = test["raw_dy"] - BIAS_DY
test["estimated_dx"] = test["raw_dx"] - BIAS_DX

norm = np.hypot(
    test["estimated_dy"], test["estimated_dx"]
)

snap = norm <= TAU + 1e-12

test.loc[snap, "estimated_dy"] = 0.0
test.loc[snap, "estimated_dx"] = 0.0

test["snapped"] = snap.astype(int)

test["epe"] = np.hypot(
    test["estimated_dy"] - test["true_dy"],
    test["estimated_dx"] - test["true_dx"]
)

assert len(test) == 680

test.to_csv(
    OUT / "Step19_Test_Classical_Registration.csv",
    index=False
)

classical_summary = (
    test.groupby("magnitude")
    .agg(
        n=("image", "size"),
        classical_epe=("epe", "mean"),
        classical_median_epe=("epe", "median"),
        classical_max_epe=("epe", "max"),
        snapped_fraction=("snapped", "mean")
    )
    .reset_index()
)

# -------------------------------------------------------------------------
# 19G — COMPARE AGAINST FROZEN STEP 13 ESTIMATES
# -------------------------------------------------------------------------

reference = pd.read_csv(
    STEP13 / "Step13_Gated_Test_Registration.csv"
)

assert len(reference) == 680

reference["magnitude"] = np.hypot(
    reference["true_dy"],
    reference["true_dx"]
)

reference["magnitude"] = reference["magnitude"].round(5)

reference_summary = (
    reference.groupby("magnitude")
    .agg(
        step13_epe=("gated_epe", "mean"),
        step13_median_epe=("gated_epe", "median"),
        step13_max_epe=("gated_epe", "max")
    )
    .reset_index()
)

comparison = classical_summary.merge(
    reference_summary,
    on="magnitude",
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 5

comparison["step13_epe_advantage"] = (
    comparison["classical_epe"]
    - comparison["step13_epe"]
)

comparison.to_csv(
    OUT / "Step19_Registration_Comparison.csv",
    index=False
)

print("\nCLASSICAL BASELINE VS STEP 13")
print(comparison.round(5).to_string(index=False))

# Paired per-case test differences, descriptive only.
paired = test.merge(
    reference[
        [
            "image", "true_dy", "true_dx",
            "gated_epe"
        ]
    ],
    on=["image", "true_dy", "true_dx"],
    validate="one_to_one"
)

assert len(paired) == 680

paired["epe_advantage_step13"] = (
    paired["epe"] - paired["gated_epe"]
)

paired.to_csv(
    OUT / "Step19_Paired_Registration_Errors.csv",
    index=False
)

print("\nOVERALL RESULTS")
print(
    "Classical mean EPE:",
    round(float(paired["epe"].mean()), 6)
)
print(
    "Step 13 mean EPE:",
    round(float(paired["gated_epe"].mean()), 6)
)

# -------------------------------------------------------------------------
# 19H — REPORT
# -------------------------------------------------------------------------

aligned = paired[
    (paired["true_dy"] == 0)
    & (paired["true_dx"] == 0)
]

report = {
    "experiment": "classical cross-sensor phase correlation baseline",
    "validation_patches": len(VAL),
    "test_patches": len(TEST),
    "test_conditions": len(OFFSETS),
    "test_records": len(paired),
    "candidate_methods": METHODS,
    "selected_method": FROZEN_METHOD,
    "selected_gate_threshold_px": TAU,
    "selected_bias": [BIAS_DY, BIAS_DX],
    "classical_mean_epe": float(
        paired["epe"].mean()
    ),
    "step13_mean_epe": float(
        paired["gated_epe"].mean()
    ),
    "classical_aligned_mean_epe": float(
        aligned["epe"].mean()
    ),
    "step13_aligned_mean_epe": float(
        aligned["gated_epe"].mean()
    ),
    "test_ground_truth_used_for_selection": False,
    "independent_scene_evaluation": False,
    "fusion_networks_retrained": False,
    "original_files_modified": False,
    "limitation": (
        "The baseline uses the same frozen spectral projection "
        "as Step 13. Test performance remains exploratory "
        "because the MDAS test partition has already been "
        "examined during method development."
    ),
    "output_directory": str(OUT)
}

with open(
    OUT / "Step19_Final_Report.json",
    "w"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 19 — FINAL REPORT")
print("=" * 88)

print(json.dumps(report, indent=2))

print("\nSTEP 19 COMPLETE")
print("No previous results or checkpoints modified.")


In [ ]:

# =============================================================================
# STEP 19B — CORRECTED CLASSICAL REGISTRATION COMPARISON
#
# FIX:
#   Join results using image + true_dy + true_dx.
#   Round shift magnitudes only for grouping/display.
#
# RECOVER:
#   Previously completed Step 19 validation and test estimates.
#
# AUDIT:
#   1. Frozen Step 19 phase-intensity method
#   2. Alternative phase-gradient method
#   3. Step 13 gated NCC/gradient subpixel estimator
#   4. Registration errors and catastrophic failures
#   5. Validation selection-rule sensitivity
#
# No images are reprocessed.
# No GPU inference.
# Original Step 19 and Step 13 files remain unchanged.
# =============================================================================

from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd
import json
import hashlib

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

STEP19 = PROJECT / "Step19_Classical_Registration_Baselines"
STEP13 = PROJECT / "Step13_Identity_Preserving_Gate"

OUT = PROJECT / "Step19B_Corrected_Comparison"
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 88)
print("STEP 19B — CORRECTED CLASSICAL REGISTRATION AUDIT")
print("=" * 88)
print("Output directory:", OUT)

# -------------------------------------------------------------------------
# 19B-A — LOAD VERIFIED EXISTING RESULTS
# -------------------------------------------------------------------------

validation_path = STEP19 / "Step19_Validation_Raw.csv"
test_path = STEP19 / "Step19_Test_Raw.csv"
selection_path = STEP19 / "Step19_Validation_Selection.csv"
frozen_path = STEP19 / "Step19_Frozen_Baseline.json"

reference_path = (
    STEP13 / "Step13_Gated_Test_Registration.csv"
)

for path in [
    validation_path,
    test_path,
    selection_path,
    frozen_path,
    reference_path
]:
    if not path.exists():
        raise FileNotFoundError(path)

validation = pd.read_csv(validation_path)
test_raw = pd.read_csv(test_path)
selection = pd.read_csv(selection_path)
reference = pd.read_csv(reference_path)

with open(frozen_path, encoding="utf-8") as f:
    frozen = json.load(f)

METHODS = [
    "phase_intensity",
    "phase_gradient"
]

assert len(validation) == 32 * 17 * 2
assert len(test_raw) == 40 * 17 * 2
assert len(reference) == 40 * 17

assert set(validation["method"]) == set(METHODS)
assert set(test_raw["method"]) == set(METHODS)

assert (
    frozen["selected_method"] == "phase_intensity"
)

KEYS = ["image", "true_dy", "true_dx"]

assert not test_raw.duplicated(
    ["method"] + KEYS
).any()

assert not reference.duplicated(KEYS).any()

assert np.isfinite(
    test_raw[
        ["raw_dy", "raw_dx", "response"]
    ].to_numpy(float)
).all()

print("\nINPUT VERIFICATION")
print("Validation estimates:", len(validation))
print("Test estimates:", len(test_raw))
print("Step 13 reference cases:", len(reference))
print("Frozen Step 19 method:", frozen["selected_method"])

# -------------------------------------------------------------------------
# 19B-B — RECONSTRUCT VALIDATION-FROZEN SETTINGS
# -------------------------------------------------------------------------

# Choose the single best validation configuration
# for each method, exactly as archived in Step 19.

method_settings = {}

for method in METHODS:

    subset = selection[
        selection["method"] == method
    ].copy()

    assert len(subset) > 0

    subset = subset.sort_values(
        [
            "aligned_preservation",
            "mean_epe",
            "tau"
        ],
        ascending=[False, True, True]
    )

    best = subset.iloc[0]

    method_settings[method] = {
        "tau": float(best["tau"]),
        "bias_dy": float(best["bias_dy"]),
        "bias_dx": float(best["bias_dx"]),
        "validation_mean_epe": float(
            best["mean_epe"]
        ),
        "validation_aligned_preservation": float(
            best["aligned_preservation"]
        )
    }

# Verify the frozen configuration was correctly recovered.
f = method_settings[frozen["selected_method"]]

assert abs(f["tau"] - frozen["gate_threshold_px"]) < 1e-9
assert abs(f["bias_dy"] - frozen["bias_dy"]) < 1e-9
assert abs(f["bias_dx"] - frozen["bias_dx"]) < 1e-9

print("\nVALIDATION-FROZEN SETTINGS")

for method, setting in method_settings.items():
    print(method)
    print(json.dumps(setting, indent=2))

# -------------------------------------------------------------------------
# 19B-C — APPLY VALIDATION-FROZEN CALIBRATION
# -------------------------------------------------------------------------

corrected_rows = []

for method in METHODS:

    cfg = method_settings[method]

    subset = test_raw[
        test_raw["method"] == method
    ].copy()

    subset["estimated_dy"] = (
        subset["raw_dy"] - cfg["bias_dy"]
    )

    subset["estimated_dx"] = (
        subset["raw_dx"] - cfg["bias_dx"]
    )

    estimated_magnitude = np.hypot(
        subset["estimated_dy"],
        subset["estimated_dx"]
    )

    snapped = (
        estimated_magnitude <= cfg["tau"] + 1e-12
    )

    subset.loc[snapped, "estimated_dy"] = 0.0
    subset.loc[snapped, "estimated_dx"] = 0.0

    subset["snapped_to_zero"] = snapped.astype(int)

    subset["epe"] = np.hypot(
        subset["estimated_dy"] - subset["true_dy"],
        subset["estimated_dx"] - subset["true_dx"]
    )

    subset["shift_magnitude"] = np.round(
        np.hypot(
            subset["true_dy"],
            subset["true_dx"]
        ),
        5
    )

    subset["accurate_025px"] = (
        subset["epe"] <= 0.25
    ).astype(int)

    subset["accurate_050px"] = (
        subset["epe"] <= 0.50
    ).astype(int)

    subset["catastrophic_2px"] = (
        subset["epe"] > 2.0
    ).astype(int)

    corrected_rows.append(subset)

classical = pd.concat(
    corrected_rows,
    ignore_index=True
)

assert len(classical) == 1360

classical.to_csv(
    OUT / "Step19B_Both_Classical_Estimators.csv",
    index=False
)

# -------------------------------------------------------------------------
# 19B-D — RECONSTRUCT STEP 13 BASELINE
# -------------------------------------------------------------------------

reference = reference.copy()

reference["method"] = "step13_ncc_gradient"

reference["estimated_dy"] = reference["gated_dy"]
reference["estimated_dx"] = reference["gated_dx"]

reference["epe"] = reference["gated_epe"]

reference["shift_magnitude"] = np.round(
    np.hypot(
        reference["true_dy"],
        reference["true_dx"]
    ),
    5
)

reference["accurate_025px"] = (
    reference["epe"] <= 0.25
).astype(int)

reference["accurate_050px"] = (
    reference["epe"] <= 0.50
).astype(int)

reference["catastrophic_2px"] = (
    reference["epe"] > 2.0
).astype(int)

common = [
    "method",
    "image",
    "true_dy",
    "true_dx",
    "shift_magnitude",
    "estimated_dy",
    "estimated_dx",
    "epe",
    "accurate_025px",
    "accurate_050px",
    "catastrophic_2px"
]

combined = pd.concat(
    [
        classical[common],
        reference[common]
    ],
    ignore_index=True
)

assert len(combined) == 2040

assert not combined.duplicated(
    ["method"] + KEYS
).any()

# -------------------------------------------------------------------------
# 19B-E — CORRECTED FIVE-CONDITION COMPARISON
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("19B-E — REGISTRATION PERFORMANCE COMPARISON")
print("=" * 88)

summary = (
    combined.groupby(
        ["method", "shift_magnitude"]
    )
    .agg(
        n=("image", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max"),
        accuracy_025px=("accurate_025px", "mean"),
        accuracy_050px=("accurate_050px", "mean"),
        catastrophic_rate=("catastrophic_2px", "mean")
    )
    .reset_index()
)

assert len(summary) == 15
assert summary["n"].sum() == 2040

summary.to_csv(
    OUT / "Step19B_Method_Condition_Comparison.csv",
    index=False
)

print(
    summary.round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 19B-F — OVERALL SUMMARY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("19B-F — OVERALL COMPARISON")
print("=" * 88)

overall = (
    combined.groupby("method")
    .agg(
        n=("image", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max"),
        accuracy_025px=("accurate_025px", "mean"),
        accuracy_050px=("accurate_050px", "mean"),
        catastrophic_rate=("catastrophic_2px", "mean")
    )
    .reset_index()
)

assert (overall["n"] == 680).all()

overall.to_csv(
    OUT / "Step19B_Overall_Comparison.csv",
    index=False
)

print(overall.round(6).to_string(index=False))

# -------------------------------------------------------------------------
# 19B-G — ALIGNED-INPUT SAFETY
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("19B-G — ALIGNED INPUT SAFETY")
print("=" * 88)

aligned = combined[
    (combined["true_dy"] == 0)
    & (combined["true_dx"] == 0)
].copy()

aligned["preserved_exactly"] = (
    (aligned["estimated_dy"] == 0)
    & (aligned["estimated_dx"] == 0)
).astype(int)

aligned_summary = (
    aligned.groupby("method")
    .agg(
        n=("image", "size"),
        preserved_fraction=("preserved_exactly", "mean"),
        mean_epe=("epe", "mean"),
        maximum_epe=("epe", "max")
    )
    .reset_index()
)

aligned_summary.to_csv(
    OUT / "Step19B_Aligned_Input_Safety.csv",
    index=False
)

print(
    aligned_summary.round(6).to_string(index=False)
)

# -------------------------------------------------------------------------
# 19B-H — PAIRED PER-IMAGE ERROR DIFFERENCES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("19B-H — PAIRED COMPARISONS")
print("=" * 88)

ref = reference[
    KEYS + ["epe"]
].rename(
    columns={"epe": "step13_epe"}
)

paired_rows = []

for method in METHODS:

    subset = classical[
        classical["method"] == method
    ]

    paired = subset.merge(
        ref,
        on=KEYS,
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    assert len(paired) == 680
    assert (paired["_merge"] == "both").all()

    paired["step13_advantage_px"] = (
        paired["epe"] - paired["step13_epe"]
    )

    paired_rows.append(paired.drop(columns="_merge"))

paired_results = pd.concat(
    paired_rows,
    ignore_index=True
)

paired_results.to_csv(
    OUT / "Step19B_Paired_Error_Records.csv",
    index=False
)

paired_summary = (
    paired_results.groupby("method")
    .agg(
        cases=("image", "size"),
        mean_step13_advantage_px=(
            "step13_advantage_px", "mean"
        ),
        fraction_step13_better=(
            "step13_advantage_px",
            lambda s: float((s > 1e-9).mean())
        ),
        fraction_phase_better=(
            "step13_advantage_px",
            lambda s: float((s < -1e-9).mean())
        )
    )
    .reset_index()
)

paired_summary.to_csv(
    OUT / "Step19B_Paired_Comparison_Summary.csv",
    index=False
)

print(
    paired_summary.round(6).to_string(index=False)
)

# -------------------------------------------------------------------------
# 19B-I — LARGEST CLASSICAL REGISTRATION FAILURES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("19B-I — LARGEST CLASSICAL REGISTRATION FAILURES")
print("=" * 88)

worst = classical.sort_values(
    "epe", ascending=False
).head(30)

worst.to_csv(
    OUT / "Step19B_Worst_Classical_Failures.csv",
    index=False
)

print(
    worst[
        [
            "method", "image",
            "true_dy", "true_dx",
            "estimated_dy", "estimated_dx",
            "epe", "response"
        ]
    ].round(4).to_string(index=False)
)

# -------------------------------------------------------------------------
# 19B-J — SELECTION-CRITERION SENSITIVITY
# -------------------------------------------------------------------------

# Important:
# This is a diagnostic of validation criteria, not
# permission to select a new method using test results.
#
# The original Step 19 method stays frozen and is reported.
# The alternative selection shown below is based on
# validation records only.

selection_alternative = selection.sort_values(
    [
        "mean_epe",
        "aligned_preservation",
        "tau",
        "method"
    ],
    ascending=[True, False, True, True]
).iloc[0]

selection_audit = {
    "original_selected_method": frozen["selected_method"],
    "original_gate": frozen["gate_threshold_px"],
    "original_selection_priority": (
        "maximum aligned preservation, then minimum EPE"
    ),
    "epe_first_validation_method": str(
        selection_alternative["method"]
    ),
    "epe_first_validation_gate": float(
        selection_alternative["tau"]
    ),
    "epe_first_validation_epe": float(
        selection_alternative["mean_epe"]
    ),
    "epe_first_aligned_preservation": float(
        selection_alternative["aligned_preservation"]
    ),
    "warning": (
        "Method-selection priorities produce different "
        "outcomes. The original selection must not "
        "be silently replaced after test inspection."
    )
}

with open(
    OUT / "Step19B_Selection_Audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(selection_audit, f, indent=2)

print("\nVALIDATION SELECTION AUDIT")
print(json.dumps(selection_audit, indent=2))

# -------------------------------------------------------------------------
# 19B-K — FINAL REPORT
# -------------------------------------------------------------------------

input_hashes = {
    p.name: hashlib.sha256(
        p.read_bytes()
    ).hexdigest()
    for p in [
        validation_path,
        test_path,
        selection_path,
        frozen_path,
        reference_path
    ]
}

final_report = {
    "experiment": (
        "corrected classical registration baseline comparison"
    ),
    "error_fixed": (
        "floating-point shift magnitude mismatch; "
        "paired comparisons use image and exact shift coordinates"
    ),
    "validation_cases_per_method": 544,
    "test_cases_per_method": 680,
    "classical_methods": METHODS,
    "step13_reference_cases": len(reference),
    "original_step19_frozen_method": frozen["selected_method"],
    "original_selection_preserved": True,
    "test_labels_used_for_calibration": False,
    "comparisons_per_method": 680,
    "input_sha256": input_hashes,
    "limitations": [
        "synthetic translations on MDAS",
        "single test mosaic",
        "iterative testing on MDAS",
        "phase correlation may suffer large outlier errors",
        "geometry comparison only, no fusion inference",
        "validation selection priority affects chosen method"
    ],
    "output_directory": str(OUT)
}

with open(
    OUT / "Step19B_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(final_report, f, indent=2)

print("\n" + "=" * 88)
print("STEP 19B — FINAL REPORT")
print("=" * 88)
print(json.dumps(final_report, indent=2))

print("\nSTEP 19B COMPLETE")
print("Original Step 19 outputs remain unchanged.")


### What I need from the corrected run

#Please share the **overall method comparison**, **aligned-input safety**, **paired comparison**, and **largest failure cases**.

#These will establish whether classical phase correlation performs consistently or suffers from occasional large displacement errors. In particular, the high validation mean EPE suggests that the outlier distribution deserves close examination before we use phase correlation as a benchmark.

#The important methodological distinction is that we are comparing registration estimators using the same training-only spectral projection. We are not yet comparing complete, independently validated fusion systems.

In [ ]:

# =============================================================================
# STEP 20 — VALIDATION-CALIBRATED BOUNDED PHASE CORRELATION
#
# Corrects the main fairness limitation identified in Step 19B.
#
# Comparisons:
#   1. Phase correlation: intensity, bounded FFT search
#   2. Phase correlation: gradient, bounded FFT search
#   3. Step 13 NCC-gradient estimator, frozen and unchanged
#
# Validation:
#   32 patches × 17 shifts
#   Select method, search radius and identity gate.
#
# Test:
#   40 patches × 17 shifts = 680 comparisons.
#
# Frozen Step 9 spectral projection; no training.
# Original research files and results are never overwritten.
#
# STATUS: Exploratory development on previously inspected MDAS data.
# =============================================================================

from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile
import cv2
import json
import hashlib
import time

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

REPO = (
    PROJECT / "Step05_Source_Recovery" /
    "SCSRNet_Recorded_Commit"
)

DATA = REPO / "datasets" / "mdas_new"
STEP9 = PROJECT / "Step09_Estimated_Registration"
STEP13 = PROJECT / "Step13_Identity_Preserving_Gate"
STEP19B = PROJECT / "Step19B_Corrected_Comparison"

OUT = PROJECT / "Step20_Bounded_Phase_Correlation"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

NORMALIZER = 15000.0

METHODS = ["phase_intensity", "phase_gradient"]
RADII = [5, 6, 7]
TAUS = np.round(np.arange(0, 0.501, 0.025), 6)

OFFSETS = [(0.0, 0.0)]

for d in [0.5, 1.5, 3.5]:
    OFFSETS.extend([
        (d, 0.0), (-d, 0.0),
        (0.0, d), (0.0, -d)
    ])

for dy in (-1.5, 1.5):
    for dx in (-1.5, 1.5):
        OFFSETS.append((dy, dx))

assert len(OFFSETS) == 17

print("=" * 88, flush=True)
print("STEP 20 — BOUNDED PHASE-CORRELATION BASELINES", flush=True)
print("=" * 88, flush=True)
print("Device:", DEVICE)
print("Radii:", RADII)
print("Output:", OUT)

# -------------------------------------------------------------------------
# SAFE OUTPUT WRITING
# -------------------------------------------------------------------------

def save_new_csv(frame, filename):
    path = OUT / filename

    if path.exists():
        raise FileExistsError(
            f"Output already exists: {path}\n"
            "Preserve the previous experiment. "
            "Use a new output folder for a new run."
        )

    frame.to_csv(path, index=False)
    return path

def save_new_json(obj, filename):
    path = OUT / filename

    if path.exists():
        raise FileExistsError(
            f"Output already exists: {path}\n"
            "Preserve the previous experiment."
        )

    path.write_text(
        json.dumps(obj, indent=2),
        encoding="utf-8"
    )
    return path

expected_outputs = [
    "Step20_Validation_Raw.csv",
    "Step20_Validation_Selection.csv",
    "Step20_Frozen_Method.json",
    "Step20_Test_Raw.csv",
    "Step20_Test_Registration.csv",
    "Step20_Test_Condition_Summary.csv",
    "Step20_Overall_Comparison.csv",
    "Step20_Aligned_Safety.csv",
    "Step20_Paired_Comparison.csv",
    "Step20_Worst_Failures.csv",
    "Step20_Final_Report.json"
]

existing_outputs = [
    name for name in expected_outputs
    if (OUT / name).exists()
]

if existing_outputs:
    raise FileExistsError(
        "Step 20 already contains results:\n"
        + "\n".join(existing_outputs)
        + "\nUse a new output folder to avoid overwriting."
    )

# -------------------------------------------------------------------------
# 20A — VERIFY SPLITS AND FROZEN PROJECTION
# -------------------------------------------------------------------------

def read_split(name):
    rows = [
        line.split()[0]
        for line in (
            DATA / f"{name}.txt"
        ).read_text().splitlines()
        if line.strip()
    ]

    assert len(rows) == len(set(rows))
    return rows

TRAIN = read_split("train")
VAL = read_split("val")
TEST = read_split("test")

assert (len(TRAIN), len(VAL), len(TEST)) == (133, 32, 40)
assert not (set(TRAIN) & set(VAL))
assert not (set(TRAIN) & set(TEST))
assert not (set(VAL) & set(TEST))

projection_path = STEP9 / "Step09_Spectral_Projection.npz"

with np.load(projection_path, allow_pickle=False) as z:
    scaler_mean = z["scaler_mean"].copy()
    scaler_scale = z["scaler_scale"].copy()
    ridge_coef = z["ridge_coefficients"].copy()
    ridge_intercept = z["ridge_intercept"].copy()

assert scaler_mean.shape == (242,)
assert scaler_scale.shape == (242,)
assert ridge_coef.shape == (4, 242)
assert ridge_intercept.shape == (4,)

projection_sha = hashlib.sha256(
    projection_path.read_bytes()
).hexdigest()

print("\n20A — SOURCE DATA VERIFIED")
print("Train:", len(TRAIN))
print("Validation:", len(VAL))
print("Test:", len(TEST))
print("Projection SHA256:", projection_sha)

def load_image(modality, name):
    array = tifffile.imread(
        DATA / modality / name
    ).astype(np.float32) / NORMALIZER

    assert np.isfinite(array).all()
    return array

def to_tensor(array):
    return torch.from_numpy(
        np.ascontiguousarray(
            array.transpose(2, 0, 1)
        )
    ).unsqueeze(0).to(DEVICE)

def project_hsi(lrhsi):
    with torch.inference_mode():
        up = F.interpolate(
            to_tensor(lrhsi),
            size=(72, 72),
            mode="bicubic",
            align_corners=False
        )

    up = (
        up[0].permute(1, 2, 0)
        .cpu().numpy()
        .reshape(-1, 242)
        .astype(np.float64)
    )

    standardized = (
        up - scaler_mean
    ) / scaler_scale

    predicted = (
        standardized @ ridge_coef.T
        + ridge_intercept
    )

    return predicted.reshape(
        72, 72, 4
    ).astype(np.float32)

# -------------------------------------------------------------------------
# 20B — REPRODUCE STEP 12 FRACTIONAL TRANSLATION
# -------------------------------------------------------------------------

def translate_numpy(array, dy, dx):
    dy, dx = float(dy), float(dx)

    if dy == 0.0 and dx == 0.0:
        return array.copy()

    x = to_tensor(array)
    n, c, h, w = x.shape

    yy, xx = torch.meshgrid(
        torch.arange(h, device=DEVICE),
        torch.arange(w, device=DEVICE),
        indexing="ij"
    )

    src_y = yy.float() - dy
    src_x = xx.float() - dx

    grid = torch.stack([
        2.0 * src_x / (w - 1) - 1.0,
        2.0 * src_y / (h - 1) - 1.0
    ], dim=-1).unsqueeze(0)

    with torch.inference_mode():
        result = F.grid_sample(
            x,
            grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )

    return (
        result[0].permute(1, 2, 0)
        .cpu().numpy().astype(np.float32)
    )

# -------------------------------------------------------------------------
# 20C — CLASSICAL FEATURE MAPS
# -------------------------------------------------------------------------

def standardized_gray(array):
    x = array.astype(np.float32)

    means = x.mean(axis=(0, 1), keepdims=True)
    stds = x.std(axis=(0, 1), keepdims=True)

    normalized = (
        x - means
    ) / np.maximum(stds, 1e-8)

    return np.ascontiguousarray(
        normalized.mean(axis=2),
        dtype=np.float32
    )

def feature_map(array, method):
    gray = standardized_gray(array)

    smooth = cv2.GaussianBlur(
        gray,
        (0, 0),
        sigmaX=1.2,
        sigmaY=1.2
    )

    if method == "phase_intensity":
        result = smooth

    elif method == "phase_gradient":
        gx = cv2.Sobel(
            smooth, cv2.CV_32F,
            1, 0, ksize=3
        )

        gy = cv2.Sobel(
            smooth, cv2.CV_32F,
            0, 1, ksize=3
        )

        result = cv2.magnitude(gx, gy)

    else:
        raise ValueError(method)

    return np.ascontiguousarray(
        result, dtype=np.float32
    )

# -------------------------------------------------------------------------
# 20D — BOUNDED FFT PHASE CORRELATION
#
# Cross-spectrum:
#   F_observed × conjugate(F_reference)
#
# The resulting peak estimates observed-image displacement
# relative to the reference.
#
# Crucially, peak search is restricted to the declared range.
# No unrestricted centroid estimation is used.
# -------------------------------------------------------------------------

def phase_surface(reference, observed):
    assert reference.shape == observed.shape == (72, 72)

    window = np.outer(
        np.hanning(72),
        np.hanning(72)
    ).astype(np.float64)

    a = reference.astype(np.float64)
    b = observed.astype(np.float64)

    a = (a - a.mean()) * window
    b = (b - b.mean()) * window

    fa = np.fft.fft2(a)
    fb = np.fft.fft2(b)

    cross = fb * np.conj(fa)

    magnitude = np.abs(cross)

    # Stabilize low-energy frequency components.
    scale = max(float(magnitude.max()), 1e-12)

    normalized = cross / np.maximum(
        magnitude,
        scale * 1e-8
    )

    correlation = np.fft.fftshift(
        np.fft.ifft2(normalized).real
    )

    assert np.isfinite(correlation).all()
    return correlation

def vertex(a, b, c):
    denominator = a - 2*b + c

    if (
        not np.isfinite(denominator)
        or denominator >= -1e-12
    ):
        return 0.0

    displacement = 0.5 * (a - c) / denominator

    return float(
        np.clip(displacement, -0.5, 0.5)
    )

def select_bounded(surface, radius):
    h, w = surface.shape
    cy, cx = h // 2, w // 2

    region = surface[
        cy-radius:cy+radius+1,
        cx-radius:cx+radius+1
    ]

    index = int(np.argmax(region))
    iy, ix = np.unravel_index(index, region.shape)

    py = iy + cy - radius
    px = ix + cx - radius

    raw_y = float(py - cy)
    raw_x = float(px - cx)

    frac_y = vertex(
        surface[py-1, px],
        surface[py, px],
        surface[py+1, px]
    )

    frac_x = vertex(
        surface[py, px-1],
        surface[py, px],
        surface[py, px+1]
    )

    estimate_y = raw_y + frac_y
    estimate_x = raw_x + frac_x

    return {
        "raw_dy": float(estimate_y),
        "raw_dx": float(estimate_x),
        "peak_value": float(surface[py, px]),
        "at_boundary": int(
            abs(raw_y) == radius
            or abs(raw_x) == radius
        )
    }

# Known-sign check before actual imagery.
rng = np.random.default_rng(20261113)

toy = rng.standard_normal((72, 72)).astype(np.float32)

toy = cv2.GaussianBlur(
    toy,
    (0, 0),
    sigmaX=1.5,
    sigmaY=1.5
)

toy_shift = np.roll(
    toy,
    shift=(2, -3),
    axis=(0, 1)
)

toy_surface = phase_surface(
    toy, toy_shift
)

toy_estimate = select_bounded(
    toy_surface, radius=5
)

assert abs(toy_estimate["raw_dy"] - 2) < 0.5, toy_estimate
assert abs(toy_estimate["raw_dx"] + 3) < 0.5, toy_estimate

print("\n20B — FFT SHIFT SIGN CHECK: PASSED")

# -------------------------------------------------------------------------
# 20E — COMPUTE RAW ESTIMATES FOR EACH RADIUS
# -------------------------------------------------------------------------

def evaluate_partition(names, partition):
    rows = []
    start = time.monotonic()

    for i, name in enumerate(names, 1):
        projected = project_hsi(
            load_image("lrhsi", name)
        )

        clean = load_image("hrmsi", name)

        target_features = {
            method: feature_map(projected, method)
            for method in METHODS
        }

        for dy, dx in OFFSETS:
            observed = translate_numpy(
                clean, dy, dx
            )

            for method in METHODS:
                observed_feature = feature_map(
                    observed, method
                )

                surface = phase_surface(
                    target_features[method],
                    observed_feature
                )

                for radius in RADII:
                    estimate = select_bounded(
                        surface, radius
                    )

                    rows.append({
                        "partition": partition,
                        "image": name,
                        "method": method,
                        "radius": radius,
                        "true_dy": float(dy),
                        "true_dx": float(dx),
                        "shift": round(
                            float(np.hypot(dy, dx)), 5
                        ),
                        **estimate
                    })

        if i % 8 == 0 or i == len(names):
            print(
                f"  {partition}: {i}/{len(names)} | "
                f"{time.monotonic()-start:.1f}s",
                flush=True
            )

    return pd.DataFrame(rows)

# -------------------------------------------------------------------------
# 20F — VALIDATION-ONLY CALIBRATION AND SELECTION
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("20C — VALIDATION SEARCH AND CALIBRATION", flush=True)
print("=" * 88, flush=True)

validation = evaluate_partition(VAL, "validation")

assert len(validation) == 32 * 17 * 2 * len(RADII)

save_new_csv(
    validation,
    "Step20_Validation_Raw.csv"
)

selection_rows = []

for method in METHODS:
    for radius in RADII:
        subset = validation[
            (validation["method"] == method)
            & (validation["radius"] == radius)
        ].copy()

        aligned = subset[
            subset["shift"] == 0
        ]

        bias_dy = float(
            aligned["raw_dy"].median()
        )
        bias_dx = float(
            aligned["raw_dx"].median()
        )

        corrected_y = (
            subset["raw_dy"].to_numpy()
            - bias_dy
        )

        corrected_x = (
            subset["raw_dx"].to_numpy()
            - bias_dx
        )

        true_y = subset["true_dy"].to_numpy()
        true_x = subset["true_dx"].to_numpy()

        for tau in TAUS:
            ey = corrected_y.copy()
            ex = corrected_x.copy()

            snapped = np.hypot(
                ey, ex
            ) <= tau + 1e-12

            ey[snapped] = 0.0
            ex[snapped] = 0.0

            epe = np.hypot(
                ey - true_y,
                ex - true_x
            )

            is_aligned = (
                (true_y == 0)
                & (true_x == 0)
            )

            aligned_preservation = float(
                np.mean(
                    (ey[is_aligned] == 0)
                    & (ex[is_aligned] == 0)
                )
            )

            selection_rows.append({
                "method": method,
                "radius": radius,
                "tau": float(tau),
                "bias_dy": bias_dy,
                "bias_dx": bias_dx,
                "validation_mean_epe": float(epe.mean()),
                "validation_median_epe": float(
                    np.median(epe)
                ),
                "aligned_preservation":
                    aligned_preservation,
                "nonzero_snapped_rate": float(
                    snapped[~is_aligned].mean()
                )
            })

selection = pd.DataFrame(selection_rows)

save_new_csv(
    selection,
    "Step20_Validation_Selection.csv"
)

# Predeclared selection policy:
# Prefer a method with >=95% aligned preservation.
# Within eligible configurations minimize mean EPE.
# If none are eligible, fall back to highest preservation
# and then mean EPE.
eligible = selection[
    selection["aligned_preservation"] >= 0.95
].copy()

if eligible.empty:
    print(
        "WARNING: No phase configuration achieved "
        "95% aligned preservation in validation."
    )

    ranked = selection.sort_values(
        [
            "aligned_preservation",
            "validation_mean_epe",
            "radius",
            "tau",
            "method"
        ],
        ascending=[False, True, True, True, True]
    )

    selection_policy = "fallback_max_aligned_preservation"

else:
    ranked = eligible.sort_values(
        [
            "validation_mean_epe",
            "aligned_preservation",
            "radius",
            "tau",
            "method"
        ],
        ascending=[True, False, True, True, True]
    )

    selection_policy = "minimum_epe_with_95pct_alignment"

winner = ranked.iloc[0]

print("\nBEST PER-METHOD VALIDATION CONFIGURATIONS")

best_per_method = []

for method in METHODS:
    group = selection[
        selection["method"] == method
    ]

    good = group[
        group["aligned_preservation"] >= 0.95
    ]

    if good.empty:
        best = group.sort_values(
            ["aligned_preservation",
             "validation_mean_epe"],
            ascending=[False, True]
        ).iloc[0]
    else:
        best = good.sort_values(
            ["validation_mean_epe", "radius", "tau"],
            ascending=[True, True, True]
        ).iloc[0]

    best_per_method.append(best)

print(
    pd.DataFrame(best_per_method)
    .round(5).to_string(index=False)
)

METHOD = str(winner["method"])
RADIUS = int(winner["radius"])
TAU = float(winner["tau"])
BIAS_DY = float(winner["bias_dy"])
BIAS_DX = float(winner["bias_dx"])

frozen = {
    "method": METHOD,
    "radius": RADIUS,
    "tau": TAU,
    "bias_dy": BIAS_DY,
    "bias_dx": BIAS_DX,
    "validation_mean_epe": float(
        winner["validation_mean_epe"]
    ),
    "validation_aligned_preservation": float(
        winner["aligned_preservation"]
    ),
    "selection_policy": selection_policy,
    "projection_sha256": projection_sha,
    "selection_data": "validation_only",
    "test_labels_used_for_selection": False,
    "status": "exploratory_benchmark"
}

save_new_json(
    frozen,
    "Step20_Frozen_Method.json"
)

print("\nFROZEN BOUNDED PHASE METHOD")
print(json.dumps(frozen, indent=2))

# -------------------------------------------------------------------------
# 20G — TEST EVALUATION AFTER FREEZING
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("20D — TEST REGISTRATION", flush=True)
print("=" * 88, flush=True)

test_raw = evaluate_partition(
    TEST, "test"
)

assert len(test_raw) == 40 * 17 * 2 * len(RADII)

save_new_csv(
    test_raw,
    "Step20_Test_Raw.csv"
)

test = test_raw[
    (test_raw["method"] == METHOD)
    & (test_raw["radius"] == RADIUS)
].copy()

assert len(test) == 680

test["estimated_dy"] = (
    test["raw_dy"] - BIAS_DY
)

test["estimated_dx"] = (
    test["raw_dx"] - BIAS_DX
)

estimated_norm = np.hypot(
    test["estimated_dy"],
    test["estimated_dx"]
)

snapped = estimated_norm <= TAU + 1e-12

test.loc[snapped, "estimated_dy"] = 0.0
test.loc[snapped, "estimated_dx"] = 0.0

test["snapped"] = snapped.astype(int)

test["epe"] = np.hypot(
    test["estimated_dy"] - test["true_dy"],
    test["estimated_dx"] - test["true_dx"]
)

test["accuracy_025px"] = (
    test["epe"] <= 0.25
).astype(int)

test["accuracy_050px"] = (
    test["epe"] <= 0.50
).astype(int)

test["failure_over_2px"] = (
    test["epe"] > 2.0
).astype(int)

assert test["epe"].ge(0).all()
assert np.isfinite(test["epe"]).all()

save_new_csv(
    test,
    "Step20_Test_Registration.csv"
)

print("\nBOUNDED PHASE PERFORMANCE BY SHIFT")

phase_summary = (
    test.groupby("shift")
    .agg(
        n=("image", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max"),
        within_025px=("accuracy_025px", "mean"),
        within_050px=("accuracy_050px", "mean"),
        failure_over_2px=("failure_over_2px", "mean")
    )
    .reset_index()
)

print(phase_summary.round(5).to_string(index=False))

save_new_csv(
    phase_summary,
    "Step20_Test_Condition_Summary.csv"
)

# -------------------------------------------------------------------------
# 20H — PAIRED COMPARISON WITH STEP 13
# -------------------------------------------------------------------------

reference = pd.read_csv(
    STEP13 / "Step13_Gated_Test_Registration.csv"
)

assert len(reference) == 680

KEYS = ["image", "true_dy", "true_dx"]

paired = test.merge(
    reference[
        KEYS + ["gated_epe"]
    ],
    on=KEYS,
    how="outer",
    indicator=True,
    validate="one_to_one"
)

assert len(paired) == 680
assert paired["_merge"].eq("both").all()

paired = paired.drop(columns=["_merge"])

paired["step13_advantage_px"] = (
    paired["epe"] - paired["gated_epe"]
)

save_new_csv(
    paired,
    "Step20_Paired_Comparison.csv"
)

# Include Step 19B's original unconstrained baseline
# for an explicitly labelled context comparison.
unbounded_path = (
    STEP19B / "Step19B_Overall_Comparison.csv"
)

unbounded = pd.read_csv(unbounded_path)

comparisons = [
    {
        "method": "step20_bounded_phase",
        "mean_epe": float(test["epe"].mean()),
        "median_epe": float(test["epe"].median()),
        "max_epe": float(test["epe"].max()),
        "within_025px": float(
            test["accuracy_025px"].mean()
        ),
        "within_050px": float(
            test["accuracy_050px"].mean()
        ),
        "catastrophic_rate": float(
            test["failure_over_2px"].mean()
        )
    },
    {
        "method": "step13_ncc_gradient",
        "mean_epe": float(
            reference["gated_epe"].mean()
        ),
        "median_epe": float(
            reference["gated_epe"].median()
        ),
        "max_epe": float(
            reference["gated_epe"].max()
        ),
        "within_025px": float(
            (reference["gated_epe"] <= 0.25).mean()
        ),
        "within_050px": float(
            (reference["gated_epe"] <= 0.50).mean()
        ),
        "catastrophic_rate": float(
            (reference["gated_epe"] > 2).mean()
        )
    }
]

for method in METHODS:
    prior = unbounded[
        unbounded["method"] == method
    ]

    assert len(prior) == 1
    p = prior.iloc[0]

    comparisons.append({
        "method": f"step19_unbounded_{method}",
        "mean_epe": float(p["mean_epe"]),
        "median_epe": float(p["median_epe"]),
        "max_epe": float(p["max_epe"]),
        "within_025px": float(p["accuracy_025px"]),
        "within_050px": float(p["accuracy_050px"]),
        "catastrophic_rate": float(p["catastrophic_rate"])
    })

overall = pd.DataFrame(comparisons)

save_new_csv(
    overall,
    "Step20_Overall_Comparison.csv"
)

print("\n" + "=" * 88)
print("20E — OVERALL METHOD COMPARISON")
print("=" * 88)
print(overall.round(5).to_string(index=False))

# -------------------------------------------------------------------------
# 20I — ALIGNED INPUTS
# -------------------------------------------------------------------------

aligned_phase = test[
    (test["true_dy"] == 0)
    & (test["true_dx"] == 0)
]

aligned_step13 = reference[
    (reference["true_dy"] == 0)
    & (reference["true_dx"] == 0)
]

safety = pd.DataFrame([
    {
        "method": "bounded_phase",
        "n": len(aligned_phase),
        "preservation": float(
            (
                (aligned_phase["estimated_dy"] == 0)
                & (aligned_phase["estimated_dx"] == 0)
            ).mean()
        ),
        "mean_epe": float(aligned_phase["epe"].mean())
    },
    {
        "method": "step13_ncc_gradient",
        "n": len(aligned_step13),
        "preservation": float(
            (
                (aligned_step13["gated_dy"] == 0)
                & (aligned_step13["gated_dx"] == 0)
            ).mean()
        ),
        "mean_epe": float(
            aligned_step13["gated_epe"].mean()
        )
    }
])

save_new_csv(
    safety,
    "Step20_Aligned_Safety.csv"
)

print("\nALIGNED INPUT SAFETY")
print(safety.round(6).to_string(index=False))

# -------------------------------------------------------------------------
# 20J — WORST FAILURES
# -------------------------------------------------------------------------

worst = test.sort_values(
    "epe", ascending=False
).head(30)

save_new_csv(
    worst,
    "Step20_Worst_Failures.csv"
)

print("\n30 WORST BOUNDED PHASE ESTIMATES")

print(
    worst[
        [
            "image",
            "true_dy", "true_dx",
            "estimated_dy", "estimated_dx",
            "epe", "peak_value", "at_boundary"
        ]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 20K — FINAL REPORT
# -------------------------------------------------------------------------

report = {
    "experiment": (
        "bounded phase correlation versus Step 13 "
        "NCC-gradient registration"
    ),
    "validation_patches": 32,
    "test_patches": 40,
    "test_conditions": 17,
    "test_cases": 680,
    "candidate_methods": METHODS,
    "candidate_radii": RADII,
    "selected_method": METHOD,
    "selected_radius": RADIUS,
    "selected_tau": TAU,
    "selected_bias": [BIAS_DY, BIAS_DX],
    "validation_epe": float(
        winner["validation_mean_epe"]
    ),
    "test_bounded_phase_epe": float(
        test["epe"].mean()
    ),
    "test_step13_epe": float(
        reference["gated_epe"].mean()
    ),
    "test_bounded_phase_catastrophic_fraction": float(
        (test["epe"] > 2).mean()
    ),
    "test_step13_catastrophic_fraction": float(
        (reference["gated_epe"] > 2).mean()
    ),
    "step13_better_fraction": float(
        (paired["step13_advantage_px"] > 1e-9).mean()
    ),
    "projection_sha256": projection_sha,
    "test_labels_used_for_selection": False,
    "fusion_models_retrained": False,
    "original_files_modified": False,
    "caveats": [
        "Phase correlation and NCC use different registration objectives.",
        "The methods share the frozen training-only spectral projection.",
        "Search radius is selected on validation images.",
        "All shifts are synthetic.",
        "Test images were inspected in earlier exploratory steps.",
        "Only one contiguous MDAS test mosaic is evaluated.",
        "Geometric comparison does not establish fusion performance."
    ],
    "output_directory": str(OUT)
}

save_new_json(
    report,
    "Step20_Final_Report.json"
)

print("\n" + "=" * 88)
print("STEP 20 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 20 COMPLETE")
print("Original Step 9–19 files remain unchanged.")


In [ ]:

# =============================================================================
# STEP 21 — FUSION-LEVEL COMPARISON WITH BOUNDED PHASE CORRELATION
#
# Purpose:
#   Compare Step 20 bounded phase-correlation correction with
#   Step 13 NCC-gradient correction using identical fusion checkpoints.
#
# Input:
#   Step 14: 8,160 verified model/patch/shift fusion records
#   Step 20: 680 frozen bounded phase-correlation estimates
#   Original 12 trained checkpoints and MDAS test patches
#
# New GPU inference:
#   12 checkpoints x 40 patches x 17 conditions = 8,160 predictions
#
# Output:
#   Patch-level paired fusion PSNR
#   Three-seed summaries
#   Aligned-input safety
#   Paired 2x2 spatial-block bootstrap
#   Worst-case reconstruction failures
#   Final reproducibility report
#
# Original imagery, checkpoints and previous results remain unchanged.
# This remains an EXPLORATORY comparison on the same MDAS test mosaic.
# =============================================================================

from google.colab import drive
from pathlib import Path
import os
import sys
import json
import time
import hashlib
import tempfile
import re

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile

drive.mount("/content/drive")

# -------------------------------------------------------------------------
# 21A — CONFIGURATION AND SOURCE VERIFICATION
# -------------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive")

PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

REPO = (
    PROJECT / "Step05_Source_Recovery" /
    "SCSRNet_Recorded_Commit"
)

DATA = REPO / "datasets" / "mdas_new"
EXPS = ROOT / "MDAS_JSTARS_Expanded"

STEP14 = PROJECT / "Step14_Multiseed_Uncertainty"
STEP20 = PROJECT / "Step20_Bounded_Phase_Correlation"

OUT = PROJECT / "Step21_Bounded_Phase_Fusion_Comparison"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE_FUSION = (
    STEP14 / "Step14_Fusion_Patch_Results.csv"
)

SOURCE_REGISTRATION = (
    STEP20 / "Step20_Test_Registration.csv"
)

SOURCE_FROZEN = (
    STEP20 / "Step20_Frozen_Method.json"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODELS = ["SCSRNet", "MSDCNN"]
REGIMES = ["p00_none", "p50_one_two"]
SEEDS = [7, 17, 37]

RUNS = [
    (model, regime, seed, f"{model}_{regime}_s{seed}")
    for model in MODELS
    for regime in REGIMES
    for seed in SEEDS
]

OFFSETS = [(0.0, 0.0)]

for value in [0.5, 1.5, 3.5]:
    OFFSETS.extend([
        (value, 0.0),
        (-value, 0.0),
        (0.0, value),
        (0.0, -value)
    ])

for dy in [-1.5, 1.5]:
    for dx in [-1.5, 1.5]:
        OFFSETS.append((dy, dx))

assert len(OFFSETS) == 17

EXPECTED = 12 * 40 * 17

sys.path.insert(0, str(REPO))

from models.SCSRNet import SCSRNet
from models.MSDCNN import MSDCNN
from utils.metrics import PSNR

MODEL_CLASSES = {
    "SCSRNet": SCSRNet,
    "MSDCNN": MSDCNN
}

with open(
    REPO / "config_lk" / "config_rgb_lk_pavia.json",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)

CONFIG["train_dataset"] = "mdasn_dataset"
NORMALIZER = float(CONFIG["mdasn_dataset"]["max_value"])

assert NORMALIZER == 15000.0

print("=" * 88, flush=True)
print("STEP 21 — BOUNDED PHASE VS NCC FUSION", flush=True)
print("=" * 88, flush=True)

print("Device:", DEVICE)
print("Checkpoints:", len(RUNS))
print("Expected paired fusion records:", EXPECTED)
print("Output:", OUT)

# -------------------------------------------------------------------------
# 21B — STRICT SOURCE CHECKS
# -------------------------------------------------------------------------

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path in [
    SOURCE_FUSION,
    SOURCE_REGISTRATION,
    SOURCE_FROZEN
]:
    if not path.is_file():
        raise FileNotFoundError(path)

with open(SOURCE_FROZEN, encoding="utf-8") as f:
    FROZEN = json.load(f)

assert FROZEN["method"] == "phase_gradient"
assert int(FROZEN["radius"]) == 5
assert abs(float(FROZEN["tau"]) - 0.25) < 1e-12

assert FROZEN["selection_data"] == "validation_only"

def read_split(split):
    names = [
        row.split()[0]
        for row in (
            DATA / f"{split}.txt"
        ).read_text().splitlines()
        if row.strip()
    ]
    assert len(names) == len(set(names))
    return names

TRAIN = read_split("train")
VAL = read_split("val")
TEST = read_split("test")

assert (len(TRAIN), len(VAL), len(TEST)) == (133, 32, 40)

assert not set(TRAIN).intersection(VAL)
assert not set(TRAIN).intersection(TEST)
assert not set(VAL).intersection(TEST)

PREVIOUS = pd.read_csv(SOURCE_FUSION)
PHASE = pd.read_csv(SOURCE_REGISTRATION)

assert len(PREVIOUS) == EXPECTED
assert len(PHASE) == 680

JOIN_KEYS = ["image", "true_dy", "true_dx"]
FULL_KEYS = ["run"] + JOIN_KEYS

assert not PREVIOUS.duplicated(FULL_KEYS).any()
assert not PHASE.duplicated(JOIN_KEYS).any()

assert set(PREVIOUS["seed"]) == set(SEEDS)
assert set(PREVIOUS["model"]) == set(MODELS)
assert set(PREVIOUS["regime"]) == set(REGIMES)

expected_keys = {
    (name, float(dy), float(dx))
    for name in TEST
    for dy, dx in OFFSETS
}

assert set(
    PHASE.set_index(JOIN_KEYS).index.tolist()
) == expected_keys

assert set(
    PREVIOUS["run"]
) == {
    run for _, _, _, run in RUNS
}

for _, _, _, run in RUNS:
    if not (EXPS / run / "best.pt").is_file():
        raise FileNotFoundError(
            EXPS / run / "best.pt"
        )

previous_index = PREVIOUS.set_index(FULL_KEYS)
phase_index = PHASE.set_index(JOIN_KEYS)

assert previous_index.index.is_unique
assert phase_index.index.is_unique

# Check that Step 20's saved endpoint errors agree
# with the recorded displacement coordinates.
recomputed_epe = np.hypot(
    PHASE["estimated_dy"] - PHASE["true_dy"],
    PHASE["estimated_dx"] - PHASE["true_dx"]
)

assert np.allclose(
    recomputed_epe,
    PHASE["epe"],
    atol=1e-8,
    rtol=0
)

print("\n21A — INPUT VERIFICATION PASSED")
print("Step 14 fusion records:", len(PREVIOUS))
print("Step 20 phase estimates:", len(PHASE))
print("Frozen classical method:", FROZEN["method"])
print("Training/test separation: VERIFIED")

# -------------------------------------------------------------------------
# 21C — PROVENANCE AND RESUME PROTECTION
# -------------------------------------------------------------------------

RESULT_PATH = OUT / "Step21_Paired_Fusion_Results.csv"
PROVENANCE_PATH = OUT / "Step21_Provenance.json"

provenance = {
    "step14_sha256": file_hash(SOURCE_FUSION),
    "step20_estimates_sha256": file_hash(SOURCE_REGISTRATION),
    "step20_frozen_sha256": file_hash(SOURCE_FROZEN),
    "models": MODELS,
    "regimes": REGIMES,
    "seeds": SEEDS,
    "offsets": OFFSETS,
    "normalizer": NORMALIZER,
    "inference": "strict_original_checkpoints",
    "interpolation": (
        "grid_sample bilinear border align_corners=True"
    )
}

if PROVENANCE_PATH.exists():
    recorded = json.loads(
        PROVENANCE_PATH.read_text(encoding="utf-8")
    )

    if recorded != provenance:
        raise ValueError(
            "Step 21 source inputs have changed. "
            "Do not overwrite an earlier experiment."
        )

else:
    if RESULT_PATH.exists():
        raise ValueError(
            "A Step 21 output CSV exists without provenance. "
            "Inspect it rather than replacing it."
        )

    with open(
        PROVENANCE_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(provenance, f, indent=2)

# -------------------------------------------------------------------------
# 21D — EXACT STEP 12/14 IMAGE TRANSLATION
# -------------------------------------------------------------------------

def load_tensor(modality, name):
    array = tifffile.imread(
        DATA / modality / name
    ).astype(np.float32)

    array /= NORMALIZER

    if not np.isfinite(array).all():
        raise ValueError(
            f"Non-finite image: {modality}/{name}"
        )

    return torch.from_numpy(
        np.ascontiguousarray(
            array.transpose(2, 0, 1)
        )
    ).unsqueeze(0).to(DEVICE)

def translate(image, dy, dx):
    """
    Positive dy: image content moves down.
    Positive dx: image content moves right.

    Identical to the Step 14 geometric convention.
    """

    dy = float(dy)
    dx = float(dx)

    if dy == 0.0 and dx == 0.0:
        return image

    n, c, h, w = image.shape

    yy, xx = torch.meshgrid(
        torch.arange(h, device=image.device),
        torch.arange(w, device=image.device),
        indexing="ij"
    )

    sy = yy.float() - dy
    sx = xx.float() - dx

    grid = torch.stack(
        [
            2.0 * sx / (w-1) - 1.0,
            2.0 * sy / (h-1) - 1.0
        ],
        dim=-1
    ).unsqueeze(0).expand(n, -1, -1, -1)

    return F.grid_sample(
        image,
        grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True
    ).contiguous()

def infer_psnr(model, lr, msi, reference):
    with torch.inference_mode():
        output = model(lr, msi)

        if isinstance(output, dict):
            output = output["pred"]

        if tuple(output.shape) != (1, 242, 72, 72):
            raise ValueError(
                f"Incorrect model output: {output.shape}"
            )

        score = PSNR(
            output[:, :, 4:-4, 4:-4].contiguous(),
            reference[:, :, 4:-4, 4:-4].contiguous()
        )

    return float(score.detach().cpu().item())

# -------------------------------------------------------------------------
# 21E — RESUMABLE PER-PATCH RESULTS
# -------------------------------------------------------------------------

COLUMNS = [
    "run", "model", "regime", "seed",
    "image", "true_dy", "true_dx",
    "shift_magnitude",
    "phase_dy", "phase_dx",
    "phase_epe", "ncc_epe",
    "uncorrected_psnr",
    "phase_psnr",
    "ncc_psnr",
    "oracle_psnr",
    "clean_psnr",
    "phase_gain_db",
    "ncc_gain_db",
    "ncc_minus_phase_db",
    "phase_oracle_gap_db",
    "ncc_oracle_gap_db"
]

if RESULT_PATH.exists():
    prior = pd.read_csv(RESULT_PATH)

    if not set(COLUMNS).issubset(prior.columns):
        raise ValueError(
            "Incompatible Step 21 resume file."
        )

    prior = prior[COLUMNS]

    if prior.duplicated(FULL_KEYS).any():
        raise ValueError(
            "Duplicate Step 21 resume records."
        )

    records = prior.to_dict("records")

    for r in records:
        key = (
            r["image"],
            float(r["true_dy"]),
            float(r["true_dx"])
        )

        phase = phase_index.loc[key]

        if (
            abs(
                r["phase_dy"]
                - float(phase["estimated_dy"])
            ) > 1e-7
            or abs(
                r["phase_dx"]
                - float(phase["estimated_dx"])
            ) > 1e-7
        ):
            raise ValueError(
                "Resume record has different "
                "frozen phase estimates."
            )

    print("\nResuming", len(records), "saved records")

else:
    records = []

completed = {
    (
        r["run"],
        r["image"],
        float(r["true_dy"]),
        float(r["true_dx"])
    )
    for r in records
}

def save_progress():
    frame = pd.DataFrame(
        records,
        columns=COLUMNS
    ).sort_values(
        FULL_KEYS
    ).reset_index(drop=True)

    fd, temporary = tempfile.mkstemp(
        prefix="step21_",
        suffix=".csv",
        dir=str(OUT)
    )

    try:
        with os.fdopen(
            fd,
            "w",
            encoding="utf-8"
        ) as f:
            frame.to_csv(f, index=False)

        os.replace(
            temporary,
            RESULT_PATH
        )

    finally:
        if os.path.exists(temporary):
            os.remove(temporary)

    return frame

# -------------------------------------------------------------------------
# 21F — COMPUTE BOUNDED PHASE-CORRECTED FUSION
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("21B — FUSION EVALUATION", flush=True)
print("=" * 88, flush=True)

start = time.monotonic()

for run_number, (
    model_name,
    regime,
    seed,
    run
) in enumerate(RUNS, 1):

    pending = sum(
        (
            run, image,
            float(dy), float(dx)
        ) not in completed
        for image in TEST
        for dy, dx in OFFSETS
    )

    print(
        f"\nRUN {run_number:02d}/12: {run} "
        f"| Pending {pending}",
        flush=True
    )

    if pending == 0:
        continue

    checkpoint = torch.load(
        EXPS / run / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    model = MODEL_CLASSES[model_name](
        config=CONFIG
    )

    model.load_state_dict(
        checkpoint["model"],
        strict=True
    )

    model = model.to(DEVICE).eval()

    print("Strict checkpoint loading: PASSED")

    for image_number, image in enumerate(TEST, 1):

        missing = [
            (float(dy), float(dx))
            for dy, dx in OFFSETS
            if (
                run, image,
                float(dy), float(dx)
            ) not in completed
        ]

        if not missing:
            continue

        lr = load_tensor("lrhsi", image)
        msi = load_tensor("hrmsi", image)
        ref = load_tensor("hrhsi", image)

        for dy, dx in missing:

            phase = phase_index.loc[
                (image, dy, dx)
            ]

            previous = previous_index.loc[
                (run, image, dy, dx)
            ]

            phase_dy = float(
                phase["estimated_dy"]
            )

            phase_dx = float(
                phase["estimated_dx"]
            )

            displaced = translate(
                msi, dy, dx
            )

            corrected = translate(
                displaced,
                -phase_dy,
                -phase_dx
            )

            phase_psnr = infer_psnr(
                model,
                lr,
                corrected,
                ref
            )

            uncorrected = float(
                previous["uncorrected_psnr"]
            )

            ncc = float(
                previous["gated_psnr"]
            )

            oracle = float(
                previous["oracle_psnr"]
            )

            clean = float(
                previous["clean_psnr"]
            )

            records.append({
                "run": run,
                "model": model_name,
                "regime": regime,
                "seed": seed,
                "image": image,
                "true_dy": dy,
                "true_dx": dx,
                "shift_magnitude": round(
                    float(np.hypot(dy, dx)), 5
                ),
                "phase_dy": phase_dy,
                "phase_dx": phase_dx,
                "phase_epe": float(
                    phase["epe"]
                ),
                "ncc_epe": float(
                    previous["epe"]
                ),
                "uncorrected_psnr": uncorrected,
                "phase_psnr": phase_psnr,
                "ncc_psnr": ncc,
                "oracle_psnr": oracle,
                "clean_psnr": clean,
                "phase_gain_db":
                    phase_psnr - uncorrected,
                "ncc_gain_db":
                    ncc - uncorrected,
                "ncc_minus_phase_db":
                    ncc - phase_psnr,
                "phase_oracle_gap_db":
                    phase_psnr - oracle,
                "ncc_oracle_gap_db":
                    ncc - oracle
            })

            completed.add(
                (run, image, dy, dx)
            )

        save_progress()

        if (
            image_number % 5 == 0
            or image_number == len(TEST)
        ):
            count = sum(
                r["run"] == run
                for r in records
            )

            print(
                f"  Patch {image_number:02d}/40 "
                f"| Records {count:03d}/680 "
                f"| Elapsed {time.monotonic()-start:.1f}s",
                flush=True
            )

        del lr, msi, ref

    del model, checkpoint

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 21G — FINAL RECORD VALIDATION
# -------------------------------------------------------------------------

RESULTS = save_progress()

assert len(RESULTS) == EXPECTED

assert not RESULTS.duplicated(FULL_KEYS).any()

assert RESULTS.groupby("run").size().eq(680).all()

metrics = [
    "uncorrected_psnr",
    "phase_psnr",
    "ncc_psnr",
    "oracle_psnr",
    "clean_psnr",
    "phase_gain_db",
    "ncc_gain_db",
    "ncc_minus_phase_db"
]

assert np.isfinite(
    RESULTS[metrics].to_numpy(float)
).all()

assert np.allclose(
    RESULTS["ncc_minus_phase_db"],
    RESULTS["ncc_gain_db"]
    - RESULTS["phase_gain_db"],
    atol=1e-10
)

# -------------------------------------------------------------------------
# 21H — MEAN PERFORMANCE ACROSS THREE TRAINING SEEDS
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("21C — THREE-SEED FUSION COMPARISON")
print("=" * 88)

SEED_SUMMARY = (
    RESULTS.groupby([
        "model", "regime", "seed",
        "shift_magnitude"
    ])
    .agg(
        n=("image", "size"),
        uncorrected=("uncorrected_psnr", "mean"),
        phase=("phase_psnr", "mean"),
        ncc=("ncc_psnr", "mean"),
        oracle=("oracle_psnr", "mean"),
        clean=("clean_psnr", "mean"),
        phase_gain=("phase_gain_db", "mean"),
        ncc_gain=("ncc_gain_db", "mean"),
        ncc_advantage=(
            "ncc_minus_phase_db", "mean"
        )
    )
    .reset_index()
)

SEED_SUMMARY.to_csv(
    OUT / "Step21_Seed_Condition_Summary.csv",
    index=False
)

SUMMARY = (
    SEED_SUMMARY.groupby([
        "model", "regime", "shift_magnitude"
    ])
    .agg(
        seeds=("seed", "nunique"),
        uncorrected=("uncorrected", "mean"),
        phase=("phase", "mean"),
        ncc=("ncc", "mean"),
        oracle=("oracle", "mean"),
        clean=("clean", "mean"),
        phase_gain=("phase_gain", "mean"),
        ncc_gain=("ncc_gain", "mean"),
        ncc_advantage=("ncc_advantage", "mean")
    )
    .reset_index()
)

SUMMARY.to_csv(
    OUT / "Step21_Three_Seed_Fusion_Comparison.csv",
    index=False
)

print(
    SUMMARY.round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 21I — ALIGNED-INPUT SAFETY COMPARISON
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("21D — ALIGNED-INPUT FUSION SAFETY")
print("=" * 88)

ALIGNED = RESULTS[
    (RESULTS["true_dy"] == 0.0)
    & (RESULTS["true_dx"] == 0.0)
]

SAFETY = (
    ALIGNED.groupby(["model", "regime"])
    .agg(
        n=("image", "size"),
        clean=("clean_psnr", "mean"),
        phase=("phase_psnr", "mean"),
        ncc=("ncc_psnr", "mean"),
        phase_loss=("phase_gain_db", "mean"),
        ncc_loss=("ncc_gain_db", "mean"),
        ncc_advantage=("ncc_minus_phase_db", "mean")
    )
    .reset_index()
)

SAFETY.to_csv(
    OUT / "Step21_Aligned_Fusion_Safety.csv",
    index=False
)

print(
    SAFETY.round(6).to_string(index=False)
)

# -------------------------------------------------------------------------
# 21J — 2x2 SPATIAL-BLOCK PAIRED BOOTSTRAP
#
# The same block/seed draws are applied to the paired
# phase-vs-NCC difference.
#
# This is exploratory uncertainty for one test mosaic,
# not independent-scene confidence.
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("21E — PAIRED SPATIAL-BLOCK UNCERTAINTY")
print("=" * 88)

pattern = re.compile(r"^test_(\d+)_(\d+)\.tif$")

positions = {}

for image in TEST:
    match = pattern.fullmatch(image)

    if match is None:
        raise ValueError(
            f"Unexpected test image name: {image}"
        )

    positions[image] = (
        int(match.group(1)),
        int(match.group(2))
    )

assert set(positions.values()) == {
    (r, c)
    for r in range(4)
    for c in range(10)
}

ORDERED_IMAGES = sorted(
    TEST,
    key=lambda x: positions[x]
)

block_indices = []

for top in range(0, 4, 2):
    for left in range(0, 10, 2):

        indices = [
            i
            for i, name in enumerate(ORDERED_IMAGES)
            if top <= positions[name][0] < top + 2
            and left <= positions[name][1] < left + 2
        ]

        assert len(indices) == 4

        block_indices.append(indices)

assert len(block_indices) == 10

BLOCK_ARRAY = np.asarray(
    block_indices,
    dtype=int
)

N_BOOT = 4000
RNG = np.random.default_rng(20261113)

CI_ROWS = []

for (model_name, regime, shift), subset in RESULTS.groupby(
    ["model", "regime", "shift_magnitude"]
):

    aggregated = (
        subset.groupby(["seed", "image"])
        ["ncc_minus_phase_db"]
        .mean()
    )

    cube = np.empty(
        (len(SEEDS), len(ORDERED_IMAGES)),
        dtype=np.float64
    )

    for i, seed in enumerate(SEEDS):
        for j, image in enumerate(ORDERED_IMAGES):
            cube[i, j] = float(
                aggregated.loc[(seed, image)]
            )

    observed = float(cube.mean())

    draws = np.empty(
        N_BOOT,
        dtype=np.float64
    )

    for b in range(N_BOOT):

        sampled_seeds = RNG.integers(
            0, len(SEEDS), len(SEEDS)
        )

        sampled_blocks = RNG.integers(
            0, len(BLOCK_ARRAY), len(BLOCK_ARRAY)
        )

        sampled_images = BLOCK_ARRAY[
            sampled_blocks
        ].reshape(-1)

        draws[b] = cube[
            np.ix_(
                sampled_seeds,
                sampled_images
            )
        ].mean()

    low, high = np.quantile(
        draws,
        [0.025, 0.975]
    )

    CI_ROWS.append({
        "model": model_name,
        "regime": regime,
        "shift_magnitude": float(shift),
        "n_seeds": len(SEEDS),
        "n_test_patches": len(TEST),
        "n_spatial_blocks": 10,
        "n_bootstrap_replicates": N_BOOT,
        "ncc_minus_phase_mean_db": observed,
        "ci_low_db": float(low),
        "ci_high_db": float(high),
        "positive_interval": bool(low > 0)
    })

CI = pd.DataFrame(CI_ROWS)

CI.to_csv(
    OUT / "Step21_Paired_Block_Bootstrap_95CI.csv",
    index=False
)

print(
    CI[
        [
            "model", "regime", "shift_magnitude",
            "ncc_minus_phase_mean_db",
            "ci_low_db", "ci_high_db"
        ]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 21K — WORST PHASE-CORRECTION FAILURES
# -------------------------------------------------------------------------

print("\n" + "=" * 88)
print("21F — PHASE CORRECTION FAILURE AUDIT")
print("=" * 88)

WORST = RESULTS.sort_values(
    "phase_gain_db",
    ascending=True
).head(40)

WORST.to_csv(
    OUT / "Step21_Worst_Phase_Fusion_Failures.csv",
    index=False
)

print(
    WORST[
        [
            "run", "image",
            "true_dy", "true_dx",
            "phase_epe",
            "phase_gain_db",
            "ncc_gain_db",
            "ncc_minus_phase_db"
        ]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 21L — FINAL REPORT
# -------------------------------------------------------------------------

REPORT = {
    "experiment": (
        "fusion-level comparison: bounded phase "
        "correlation versus NCC-gradient registration"
    ),
    "fusion_checkpoints": len(RUNS),
    "training_seeds": SEEDS,
    "test_patches": len(TEST),
    "shift_conditions": len(OFFSETS),
    "paired_fusion_records": len(RESULTS),
    "selected_classical_method": FROZEN["method"],
    "classical_radius": int(FROZEN["radius"]),
    "classical_gate_threshold": float(FROZEN["tau"]),
    "registration_settings_frozen_before_this_experiment": True,
    "same_spectral_projection_for_both_methods": True,
    "reference_hrhsi_used_for_registration_estimation": False,
    "hrhsi_used_for_fusion_evaluation": True,
    "test_labels_used_for_registration_selection": False,
    "spatial_blocks": 10,
    "spatial_block_layout": [2, 2],
    "bootstrap_replicates": N_BOOT,
    "source_hashes": {
        "step14_fusion": provenance["step14_sha256"],
        "step20_registration": (
            provenance["step20_estimates_sha256"]
        ),
        "step20_frozen": (
            provenance["step20_frozen_sha256"]
        )
    },
    "limitations": [
        "Synthetic translations on the existing MDAS test mosaic.",
        "Step 13 was developed iteratively after previous test inspection.",
        "The 40 patches are spatially correlated.",
        "Three seeds do not represent independent test scenes.",
        "This experiment does not establish real-scene generalization.",
        "The interpolation is performed twice for fractional correction.",
        "Only two registration objectives are compared."
    ],
    "elapsed_seconds": round(
        time.monotonic() - start,
        2
    ),
    "output_directory": str(OUT)
}

REPORT_PATH = OUT / "Step21_Final_Report.json"

if REPORT_PATH.exists():
    previous_report = json.loads(
        REPORT_PATH.read_text(
            encoding="utf-8"
        )
    )

    # Do not silently replace an existing completed report.
    if (
        previous_report.get("paired_fusion_records")
        != EXPECTED
        or previous_report.get("source_hashes")
        != REPORT["source_hashes"]
    ):
        raise ValueError(
            "Existing Step 21 final report is incompatible."
        )

else:
    with open(
        REPORT_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(REPORT, f, indent=2)

print("\n" + "=" * 88)
print("STEP 21 — FINAL REPORT")
print("=" * 88)

print(json.dumps(REPORT, indent=2))

print("\nSTEP 21 COMPLETE")
print("Original source, imagery and checkpoints unchanged.")


In [ ]:

# =============================================================================
# STEP 22 — CONSOLIDATED MANUSCRIPT RESULTS AND FIGURES
#
# Inputs:
#   Step 14: multiseed fusion results
#   Step 17: spatial-block uncertainty
#   Step 20: registration comparison
#   Step 21: paired phase-vs-NCC fusion comparison
#
# Outputs:
#   Figure 1 — Fusion PSNR by displacement and method
#   Figure 2 — Registration accuracy comparison
#   Figure 3 — NCC advantage over bounded phase correlation
#   Figure 4 — Aligned-input protection
#
#   Table 1 — Dataset and evaluation protocol
#   Table 2 — Three-seed fusion performance
#   Table 3 — Registration accuracy
#   Table 4 — Paired NCC-versus-phase uncertainty
#
# Figures saved as 1200-dpi PNG and vector PDF.
# Source CSV and checkpoints remain unchanged.
# =============================================================================

from google.colab import drive
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

STEP14 = PROJECT / "Step14_Multiseed_Uncertainty"
STEP17 = PROJECT / "Step17_Spatial_Block_Uncertainty"
STEP20 = PROJECT / "Step20_Bounded_Phase_Correlation"
STEP21 = PROJECT / "Step21_Bounded_Phase_Fusion_Comparison"

OUT = PROJECT / "Step22_Manuscript_Results"
FIG = OUT / "Figures"
TAB = OUT / "Tables"

FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

# Create only new outputs. Never silently overwrite an earlier run.
if any(FIG.iterdir()) or any(TAB.iterdir()):
    raise FileExistsError(
        "Step 22 figure/table outputs already exist. "
        "Preserve the previous outputs and use a new folder "
        "for a fresh experiment."
    )

paths = {
    "fusion": STEP14 / "Step14_Fusion_Patch_Results.csv",
    "spatial": STEP17 / "Step17_Spatial_Bootstrap_Results.csv",
    "registration": STEP20 / "Step20_Overall_Comparison.csv",
    "comparison": STEP21 / "Step21_Three_Seed_Fusion_Comparison.csv",
    "uncertainty": STEP21 / "Step21_Paired_Block_Bootstrap_95CI.csv",
    "safety": STEP21 / "Step21_Aligned_Fusion_Safety.csv"
}

for label, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label}: {path}")

fusion = pd.read_csv(paths["fusion"])
spatial = pd.read_csv(paths["spatial"])
registration = pd.read_csv(paths["registration"])
comparison = pd.read_csv(paths["comparison"])
uncertainty = pd.read_csv(paths["uncertainty"])
safety = pd.read_csv(paths["safety"])

assert len(fusion) == 8160
assert len(comparison) == 20
assert len(uncertainty) == 20
assert len(safety) == 4

for frame in [comparison, uncertainty]:
    assert set(frame["model"]) == {"SCSRNet", "MSDCNN"}

# -------------------------------------------------------------------------
# PUBLICATION GRAPHICS CONFIGURATION
# -------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.9,
    "lines.linewidth": 1.9,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

COLORS = {
    "uncorrected": "#9A9A9A",
    "phase": "#D18128",
    "ncc": "#146C94",
    "oracle": "#36845B"
}

def save_figure(fig, basename):
    fig.savefig(
        FIG / f"{basename}.png",
        dpi=1200,
        bbox_inches="tight",
        pad_inches=0.08
    )
    fig.savefig(
        FIG / f"{basename}.pdf",
        bbox_inches="tight",
        pad_inches=0.08
    )
    plt.close(fig)

def x_ticks(values):
    return [
        "0", "0.5", "1.5", "2.12", "3.5"
    ]

# -------------------------------------------------------------------------
# FIGURE 1 — FUSION PERFORMANCE
# -------------------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2, figsize=(10.5, 7.3),
    sharex=True
)

for i, model in enumerate(["SCSRNet", "MSDCNN"]):
    for j, regime in enumerate(["p00_none", "p50_one_two"]):
        ax = axes[i, j]

        sub = comparison[
            (comparison["model"] == model)
            & (comparison["regime"] == regime)
        ].sort_values("shift_magnitude")

        x = np.arange(len(sub))

        for column, label in [
            ("uncorrected", "Uncorrected"),
            ("phase", "Bounded phase"),
            ("ncc", "NCC–gradient"),
            ("oracle", "Oracle")
        ]:
            ax.plot(
                x,
                sub[column].to_numpy(float),
                marker="o",
                markersize=4,
                color=COLORS[column],
                label=label
            )

        ax.set_xticks(x, x_ticks(x))
        ax.grid(
            axis="y", alpha=0.2, linewidth=0.6
        )

        ax.text(
            0.02, 0.97,
            f"({chr(97+i*2+j)})",
            transform=ax.transAxes,
            va="top",
            fontweight="bold"
        )

        if i == 1:
            ax.set_xlabel(
                "Synthetic displacement magnitude (pixels)"
            )

        if j == 0:
            ax.set_ylabel("PSNR (dB)")

        ax.set_title(
            f"{model} | "
            f"{'No augmentation' if j == 0 else '50% augmentation'}",
            fontsize=10
        )

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(
    handles, labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.025),
    ncol=4,
    frameon=False
)

fig.tight_layout(rect=[0, 0.045, 1, 1])

save_figure(fig, "Figure01_Fusion_PSNR")

# -------------------------------------------------------------------------
# FIGURE 2 — GEOMETRIC REGISTRATION ACCURACY
# -------------------------------------------------------------------------

ordered_methods = [
    "step19_unbounded_phase_intensity",
    "step19_unbounded_phase_gradient",
    "step20_bounded_phase",
    "step13_ncc_gradient"
]

labels = [
    "Unbounded phase\n(intensity)",
    "Unbounded phase\n(gradient)",
    "Bounded phase\n(gradient)",
    "NCC–gradient\nwith gate"
]

reg = (
    registration.set_index("method")
    .loc[ordered_methods]
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(10.5, 3.7)
)

x = np.arange(4)

axes[0].bar(
    x,
    reg["mean_epe"].to_numpy(float),
    color=[
        "#BDBDBD", "#969696",
        "#D18128", "#146C94"
    ],
    width=0.65
)

axes[0].set_ylabel("Mean endpoint error (pixels)")
axes[0].set_xticks(x, labels)
axes[0].grid(
    axis="y", alpha=0.2
)
axes[0].text(
    0.02, 0.98, "(a)",
    transform=axes[0].transAxes,
    va="top", fontweight="bold"
)

axes[1].bar(
    x - 0.18,
    100 * reg["within_025px"].to_numpy(float),
    width=0.36,
    color="#146C94",
    label="Within 0.25 px"
)
axes[1].bar(
    x + 0.18,
    100 * reg["within_050px"].to_numpy(float),
    width=0.36,
    color="#D18128",
    label="Within 0.50 px"
)

axes[1].set_ylabel("Registration accuracy (%)")
axes[1].set_ylim(0, 105)
axes[1].set_xticks(x, labels)
axes[1].legend(
    frameon=False, loc="upper left"
)
axes[1].grid(
    axis="y", alpha=0.2
)
axes[1].text(
    0.02, 0.98, "(b)",
    transform=axes[1].transAxes,
    va="top", fontweight="bold"
)

fig.tight_layout()
save_figure(fig, "Figure02_Registration_Accuracy")

# -------------------------------------------------------------------------
# FIGURE 3 — PAIRED NCC ADVANTAGE AND UNCERTAINTY
# -------------------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2,
    figsize=(10.5, 7.1),
    sharex=True
)

for i, model in enumerate(["SCSRNet", "MSDCNN"]):
    for j, regime in enumerate(["p00_none", "p50_one_two"]):
        ax = axes[i, j]

        sub = uncertainty[
            (uncertainty["model"] == model)
            & (uncertainty["regime"] == regime)
        ].sort_values("shift_magnitude")

        x = np.arange(len(sub))

        mean = sub[
            "ncc_minus_phase_mean_db"
        ].to_numpy(float)

        lower = sub[
            "ci_low_db"
        ].to_numpy(float)

        upper = sub[
            "ci_high_db"
        ].to_numpy(float)

        ax.errorbar(
            x, mean,
            yerr=[
                mean-lower,
                upper-mean
            ],
            fmt="o-",
            color="#146C94",
            ecolor="#747474",
            capsize=3,
            markersize=4
        )

        ax.axhline(
            0, linestyle="--",
            linewidth=0.9,
            color="#444444"
        )

        ax.set_xticks(x, x_ticks(x))
        ax.grid(axis="y", alpha=0.2)

        if j == 0:
            ax.set_ylabel(
                "NCC minus bounded-phase PSNR (dB)"
            )

        if i == 1:
            ax.set_xlabel(
                "Synthetic displacement magnitude (pixels)"
            )

        ax.set_title(
            f"{model} | "
            f"{'No augmentation' if j == 0 else '50% augmentation'}",
            fontsize=10
        )

        ax.text(
            0.02, 0.97,
            f"({chr(97+i*2+j)})",
            transform=ax.transAxes,
            va="top",
            fontweight="bold"
        )

fig.tight_layout()
save_figure(fig, "Figure03_NCC_Advantage_95CI")

# -------------------------------------------------------------------------
# FIGURE 4 — ALIGNED-INPUT SAFETY
# -------------------------------------------------------------------------

safety = safety.copy()

safety["display_name"] = (
    safety["model"]
    + " / "
    + safety["regime"].map({
        "p00_none": "No aug.",
        "p50_one_two": "50% aug."
    })
)

safety = safety.sort_values(
    ["model", "regime"]
)

fig, ax = plt.subplots(
    figsize=(8.7, 4.3)
)

x = np.arange(len(safety))
width = 0.35

ax.bar(
    x - width/2,
    safety["phase_loss"].to_numpy(float),
    width=width,
    color="#D18128",
    label="Bounded phase"
)

ax.bar(
    x + width/2,
    safety["ncc_loss"].to_numpy(float),
    width=width,
    color="#146C94",
    label="NCC–gradient"
)

ax.axhline(0, color="#333333", linewidth=0.9)

ax.set_xticks(
    x,
    safety["display_name"].to_list(),
    rotation=15,
    ha="right"
)
ax.set_ylabel(
    "PSNR change on aligned images (dB)"
)
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False)
ax.text(
    0.02, 0.97, "(a)",
    transform=ax.transAxes,
    va="top",
    fontweight="bold"
)

fig.tight_layout()
save_figure(fig, "Figure04_Aligned_Input_Safety")

# -------------------------------------------------------------------------
# TABLE 1 — PROTOCOL
# -------------------------------------------------------------------------

protocol = pd.DataFrame([
    ("Training patches", "133"),
    ("Validation patches", "32"),
    ("Test patches", "40"),
    ("Test scene structure", "One 4 × 10 patch mosaic"),
    ("Hyperspectral bands", "242"),
    ("Multispectral bands", "4"),
    ("Reference patch", "72 × 72 pixels"),
    ("Low-resolution HSI patch", "24 × 24 pixels"),
    ("Training seeds", "7, 17, 37"),
    ("Synthetic conditions", "17 per test patch"),
    ("Registration cases", "680"),
    ("Fusion evaluation cases", "8,160"),
    ("Geographic coordinates", "Not recovered"),
    ("Experimental status", "Exploratory MDAS evaluation")
], columns=["component", "value"])

protocol.to_csv(
    TAB / "Table01_Experimental_Protocol.csv",
    index=False
)

# -------------------------------------------------------------------------
# TABLE 2 — MEAN FUSION PERFORMANCE
# -------------------------------------------------------------------------

comparison.to_csv(
    TAB / "Table02_Three_Seed_Fusion.csv",
    index=False
)

# -------------------------------------------------------------------------
# TABLE 3 — REGISTRATION BENCHMARK
# -------------------------------------------------------------------------

registration.to_csv(
    TAB / "Table03_Registration_Comparison.csv",
    index=False
)

# -------------------------------------------------------------------------
# TABLE 4 — UNCERTAINTY
# -------------------------------------------------------------------------

uncertainty.to_csv(
    TAB / "Table04_Paired_Fusion_95CI.csv",
    index=False
)

# -------------------------------------------------------------------------
# FINAL MANIFEST
# -------------------------------------------------------------------------

def sha256(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

manifest = {
    "experiment": "manuscript figure and table consolidation",
    "source_steps": [14, 17, 20, 21],
    "figure_count": 4,
    "table_count": 4,
    "figure_formats": ["png_1200dpi", "pdf_vector"],
    "source_hashes": {
        key: sha256(path)
        for key, path in paths.items()
    },
    "caveats": [
        "One MDAS test mosaic; patch correlation may remain.",
        "Test data were inspected during iterative method development.",
        "Only synthetic displacement conditions were evaluated.",
        "Reported intervals describe the existing experimental sample.",
        "Results do not establish geographic generalization."
    ],
    "output_directory": str(OUT)
}

with open(
    OUT / "Step22_Manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=2)

print("=" * 88)
print("STEP 22 — MANUSCRIPT EVIDENCE CONSOLIDATED")
print("=" * 88)
print(json.dumps(manifest, indent=2))

print("\nSaved figures:")
for path in sorted(FIG.iterdir()):
    print(path.name)

print("\nSaved tables:")
for path in sorted(TAB.iterdir()):
    print(path.name)

print("\nSTEP 22 COMPLETE")
print("Original research data and checkpoints unchanged.")


In [ ]:

# =============================================================================
# STEP 23 — SPATIAL HETEROGENEITY AND DIRECTIONAL ROBUSTNESS
#
# Sources:
#   Step 21: 8160 paired fusion records across 12 checkpoints.
#
# Questions:
#   1. Do registration gains vary across the 4x10 test patch grid?
#   2. Do +dy/-dy and +dx/-dx produce different PSNR?
#   3. Is the NCC-vs-phase advantage spatially concentrated?
#   4. Do any individual patches lose PSNR after NCC correction?
#
# Output:
#   4 vector PDF + 1200-dpi PNG publication figures
#   Patch-level, direction-level and model-level CSV summaries
#   Reproducibility report
#
# No retraining, no image reprocessing.
# Existing research files unchanged.
#
# CAUTION:
#   These are source-image patch-grid diagnostics,
#   NOT georeferenced maps.
# =============================================================================

from google.colab import drive
from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

SOURCE = (
    PROJECT /
    "Step21_Bounded_Phase_Fusion_Comparison" /
    "Step21_Paired_Fusion_Results.csv"
)

OUT = PROJECT / "Step23_Spatial_Directional_Audit"
FIG = OUT / "Figures"
TAB = OUT / "Tables"

FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

# Prevent accidental replacement of previous analyses.
if list(FIG.iterdir()) or list(TAB.iterdir()):
    raise FileExistsError(
        "Step 23 already contains results. "
        "Use a new output directory to preserve earlier outputs."
    )

if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)

df = pd.read_csv(SOURCE)

required = {
    "run", "model", "regime", "seed", "image",
    "true_dy", "true_dx",
    "uncorrected_psnr", "phase_psnr", "ncc_psnr",
    "oracle_psnr", "clean_psnr"
}

assert required.issubset(df.columns)
assert len(df) == 8160

assert not df.duplicated(
    ["run", "image", "true_dy", "true_dx"]
).any()

assert set(df["seed"]) == {7, 17, 37}
assert set(df["model"]) == {"SCSRNet", "MSDCNN"}
assert set(df["regime"]) == {"p00_none", "p50_one_two"}

numeric = [
    "uncorrected_psnr",
    "phase_psnr",
    "ncc_psnr",
    "oracle_psnr",
    "clean_psnr"
]

assert np.isfinite(df[numeric].to_numpy(float)).all()

df["ncc_gain_db"] = (
    df["ncc_psnr"] - df["uncorrected_psnr"]
)

df["phase_gain_db"] = (
    df["phase_psnr"] - df["uncorrected_psnr"]
)

df["ncc_advantage_db"] = (
    df["ncc_psnr"] - df["phase_psnr"]
)

df["oracle_gap_db"] = (
    df["ncc_psnr"] - df["oracle_psnr"]
)

df["magnitude"] = np.round(
    np.hypot(df["true_dy"], df["true_dx"]), 5
)

# -------------------------------------------------------------------------
# 23A — RECONSTRUCT PATCH GRID
# -------------------------------------------------------------------------

pattern = re.compile(
    r"^test_(\d+)_(\d+)\.tif$"
)

positions = {}

for name in df["image"].unique():
    match = pattern.fullmatch(name)

    if match is None:
        raise ValueError(f"Invalid image identifier: {name}")

    positions[name] = (
        int(match.group(1)),
        int(match.group(2))
    )

assert len(positions) == 40

assert set(positions.values()) == {
    (r, c)
    for r in range(4)
    for c in range(10)
}

df["grid_row"] = df["image"].map(
    lambda x: positions[x][0]
)

df["grid_col"] = df["image"].map(
    lambda x: positions[x][1]
)

def direction_name(dy, dx):
    dy, dx = float(dy), float(dx)

    if dy == 0 and dx == 0:
        return "aligned"
    if dy > 0 and dx == 0:
        return "down"
    if dy < 0 and dx == 0:
        return "up"
    if dy == 0 and dx > 0:
        return "right"
    if dy == 0 and dx < 0:
        return "left"

    if dy < 0 and dx < 0:
        return "up-left"
    if dy < 0 and dx > 0:
        return "up-right"
    if dy > 0 and dx < 0:
        return "down-left"
    return "down-right"

df["direction"] = [
    direction_name(dy, dx)
    for dy, dx in zip(df["true_dy"], df["true_dx"])
]

print("=" * 84)
print("STEP 23 — SPATIAL AND DIRECTIONAL DIAGNOSTICS")
print("=" * 84)

print("Input records:", len(df))
print("Unique test patches:", len(positions))
print("Training seeds:", sorted(df["seed"].unique()))
print("Grid: 4 rows × 10 columns")

# -------------------------------------------------------------------------
# 23B — PATCH-LEVEL STATISTICS
# -------------------------------------------------------------------------

patch_summary = (
    df.groupby([
        "model", "regime", "image",
        "grid_row", "grid_col", "magnitude"
    ])
    .agg(
        n=("run", "size"),
        ncc_gain_db=("ncc_gain_db", "mean"),
        phase_gain_db=("phase_gain_db", "mean"),
        ncc_advantage_db=("ncc_advantage_db", "mean"),
        oracle_gap_db=("oracle_gap_db", "mean"),
        ncc_psnr=("ncc_psnr", "mean"),
        uncorrected_psnr=("uncorrected_psnr", "mean")
    )
    .reset_index()
)

patch_summary.to_csv(
    TAB / "Step23_Patch_Spatial_Statistics.csv",
    index=False
)

# -------------------------------------------------------------------------
# 23C — DISPLACEMENT DIRECTION STATISTICS
# -------------------------------------------------------------------------

direction_summary = (
    df.groupby([
        "model", "regime", "magnitude", "direction"
    ])
    .agg(
        n=("image", "size"),
        uncorrected_psnr=("uncorrected_psnr", "mean"),
        phase_psnr=("phase_psnr", "mean"),
        ncc_psnr=("ncc_psnr", "mean"),
        oracle_psnr=("oracle_psnr", "mean"),
        ncc_gain_db=("ncc_gain_db", "mean"),
        ncc_advantage_db=("ncc_advantage_db", "mean")
    )
    .reset_index()
)

direction_summary.to_csv(
    TAB / "Step23_Directional_Performance.csv",
    index=False
)

# -------------------------------------------------------------------------
# 23D — NEGATIVE GAIN DIAGNOSTIC
# -------------------------------------------------------------------------

failure_summary = (
    df.groupby([
        "model", "regime", "magnitude"
    ])
    .agg(
        n=("image", "size"),
        negative_ncc_gain_rate=(
            "ncc_gain_db",
            lambda x: float((x < -1e-6).mean())
        ),
        negative_phase_gain_rate=(
            "phase_gain_db",
            lambda x: float((x < -1e-6).mean())
        ),
        minimum_ncc_gain_db=("ncc_gain_db", "min"),
        minimum_phase_gain_db=("phase_gain_db", "min"),
        mean_ncc_advantage_db=("ncc_advantage_db", "mean")
    )
    .reset_index()
)

failure_summary.to_csv(
    TAB / "Step23_Negative_Gain_Audit.csv",
    index=False
)

print("\nNEGATIVE CORRECTION GAIN AUDIT")
print(failure_summary.round(5).to_string(index=False))

# -------------------------------------------------------------------------
# 23E — FIGURE STYLING
# -------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

def savefig(fig, stem):
    fig.savefig(
        FIG / f"{stem}.png",
        dpi=1200,
        bbox_inches="tight",
        pad_inches=0.07
    )

    fig.savefig(
        FIG / f"{stem}.pdf",
        bbox_inches="tight",
        pad_inches=0.07
    )

    plt.close(fig)

def as_grid(sub, metric):
    array = np.full((4, 10), np.nan)

    for r in sub.itertuples(index=False):
        row = int(r.grid_row)
        col = int(r.grid_col)

        if np.isfinite(array[row, col]):
            raise ValueError(
                "Duplicate patch summary grid cell."
            )

        array[row, col] = float(getattr(r, metric))

    assert np.isfinite(array).all()
    return array

# -------------------------------------------------------------------------
# FIGURE 1 — SPATIAL PATTERN OF NCC CORRECTION GAIN
#
# 3.5-pixel cardinal shifts, averaged across
# directions and three training seeds.
# -------------------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 5.8),
    constrained_layout=True
)

configs = [
    ("SCSRNet", "p00_none"),
    ("SCSRNet", "p50_one_two"),
    ("MSDCNN", "p00_none"),
    ("MSDCNN", "p50_one_two")
]

maps = []

for model, regime in configs:
    sub = patch_summary[
        (patch_summary["model"] == model)
        & (patch_summary["regime"] == regime)
        & (patch_summary["magnitude"] == 3.5)
    ]

    maps.append(as_grid(sub, "ncc_gain_db"))

minval = min(m.min() for m in maps)
maxval = max(m.max() for m in maps)

for idx, ((model, regime), values) in enumerate(
    zip(configs, maps)
):
    ax = axes.flat[idx]

    im = ax.imshow(
        values,
        cmap="viridis",
        vmin=minval,
        vmax=maxval,
        aspect="equal",
        interpolation="nearest"
    )

    ax.set_xticks(np.arange(10))
    ax.set_yticks(np.arange(4))
    ax.set_xticklabels(np.arange(10))
    ax.set_yticklabels(np.arange(4))

    ax.set_xlabel("Patch column")
    ax.set_ylabel("Patch row")

    ax.set_title(
        f"{model}, "
        f"{'no augmentation' if regime == 'p00_none' else '50% augmentation'}",
        fontsize=10
    )

    ax.set_xticks(np.arange(-0.5, 10, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)

    ax.grid(
        which="minor",
        color="white",
        linewidth=0.7
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    ax.text(
        0.02, 0.97,
        f"({chr(97 + idx)})",
        transform=ax.transAxes,
        va="top",
        color="white",
        fontweight="bold"
    )

fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    label="Mean NCC correction gain (dB)",
    shrink=0.85,
    pad=0.015
)

savefig(fig, "Figure01_Spatial_Correction_Gains")

# -------------------------------------------------------------------------
# FIGURE 2 — NCC ADVANTAGE OVER BOUNDED PHASE, PATCH GRID
# -------------------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 3.2),
    constrained_layout=True
)

model = "SCSRNet"

map_data = []

for regime in ["p00_none", "p50_one_two"]:
    sub = patch_summary[
        (patch_summary["model"] == model)
        & (patch_summary["regime"] == regime)
        & (patch_summary["magnitude"] == 3.5)
    ]

    map_data.append(
        as_grid(sub, "ncc_advantage_db")
    )

vmax = max(
    float(np.abs(x).max())
    for x in map_data
)

vmax = max(vmax, 1e-6)

for idx, (ax, values) in enumerate(
    zip(axes, map_data)
):

    im = ax.imshow(
        values,
        cmap="RdBu",
        norm=TwoSlopeNorm(
            vmin=-vmax,
            vcenter=0,
            vmax=vmax
        ),
        interpolation="nearest"
    )

    ax.set_xticks(range(10))
    ax.set_yticks(range(4))
    ax.set_xlabel("Patch column")
    ax.set_ylabel("Patch row")

    ax.set_title(
        "No augmentation" if idx == 0
        else "50% augmentation"
    )

    ax.set_xticks(
        np.arange(-0.5, 10, 1),
        minor=True
    )
    ax.set_yticks(
        np.arange(-0.5, 4, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=0.7
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    ax.text(
        0.02, 0.97,
        f"({chr(97+idx)})",
        transform=ax.transAxes,
        va="top",
        fontweight="bold"
    )

fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    label="NCC minus bounded-phase PSNR (dB)",
    shrink=0.9,
    pad=0.02
)

savefig(fig, "Figure02_Spatial_NCC_Advantage")

# -------------------------------------------------------------------------
# FIGURE 3 — DIRECTIONAL ASYMMETRY AT CARDINAL SHIFTS
# -------------------------------------------------------------------------

directions = ["up", "down", "left", "right"]
magnitudes = [0.5, 1.5, 3.5]

fig, axes = plt.subplots(
    2, 2,
    figsize=(10.7, 7.2),
    sharex=True,
    constrained_layout=True
)

direction_colors = {
    "up": "#264653",
    "down": "#2A9D8F",
    "left": "#E76F51",
    "right": "#9467BD"
}

for idx, (model, regime) in enumerate(configs):
    ax = axes.flat[idx]

    subset = direction_summary[
        (direction_summary["model"] == model)
        & (direction_summary["regime"] == regime)
        & (direction_summary["magnitude"].isin(magnitudes))
    ]

    for direction in directions:
        part = subset[
            subset["direction"] == direction
        ].sort_values("magnitude")

        assert len(part) == 3

        ax.plot(
            part["magnitude"].to_numpy(float),
            part["ncc_gain_db"].to_numpy(float),
            marker="o",
            markersize=4,
            color=direction_colors[direction],
            label=direction.capitalize()
        )

    ax.axhline(
        0,
        linewidth=0.8,
        color="gray"
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

    ax.set_xticks(magnitudes)

    if idx >= 2:
        ax.set_xlabel(
            "Cardinal displacement (pixels)"
        )

    if idx % 2 == 0:
        ax.set_ylabel(
            "Registration correction gain (dB)"
        )

    ax.set_title(
        f"{model}, "
        f"{'no augmentation' if regime == 'p00_none' else '50% augmentation'}",
        fontsize=10
    )

    ax.text(
        0.02, 0.97,
        f"({chr(97+idx)})",
        transform=ax.transAxes,
        va="top",
        fontweight="bold"
    )

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    ncol=4,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.04),
    frameon=False
)

savefig(fig, "Figure03_Directional_Correction_Gains")

# -------------------------------------------------------------------------
# FIGURE 4 — DISTRIBUTION OF PATCH-LEVEL NCC ADVANTAGE
# -------------------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(10.8, 4),
    constrained_layout=True
)

for idx, model in enumerate(["SCSRNet", "MSDCNN"]):
    ax = axes[idx]

    samples = []
    labels = []

    for regime in ["p00_none", "p50_one_two"]:
        subset = patch_summary[
            (patch_summary["model"] == model)
            & (patch_summary["regime"] == regime)
            & (patch_summary["magnitude"] > 0)
        ]

        # One value per patch: average across the
        # four nonzero shift-magnitude groups.
        patch_values = (
            subset.groupby("image")[
                "ncc_advantage_db"
            ]
            .mean()
        )

        assert len(patch_values) == 40

        samples.append(
            patch_values.to_numpy(float)
        )

        labels.append(
            "No augmentation"
            if regime == "p00_none"
            else "50% augmentation"
        )

    ax.boxplot(
        samples,
        tick_labels=labels,
        patch_artist=False,
        showfliers=True
    )

    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=0.8
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

    ax.set_ylabel(
        "Mean NCC advantage per patch (dB)"
    )

    ax.set_title(model)

    ax.text(
        0.02, 0.97,
        f"({chr(97+idx)})",
        transform=ax.transAxes,
        va="top",
        fontweight="bold"
    )

savefig(fig, "Figure04_Patchwise_NCC_Advantage")

# -------------------------------------------------------------------------
# 23F — DIRECTIONAL ASYMMETRY AUDIT
# -------------------------------------------------------------------------

direction_pairs = [
    ("up", "down"),
    ("left", "right")
]

direction_comparison_rows = []

for (model, regime, magnitude), group in (
    direction_summary.groupby(
        ["model", "regime", "magnitude"]
    )
):

    for first, second in direction_pairs:

        a = group[
            group["direction"] == first
        ]

        b = group[
            group["direction"] == second
        ]

        if len(a) != 1 or len(b) != 1:
            continue

        direction_comparison_rows.append({
            "model": model,
            "regime": regime,
            "magnitude": magnitude,
            "direction_a": first,
            "direction_b": second,
            "gain_a_db": float(a["ncc_gain_db"].iloc[0]),
            "gain_b_db": float(b["ncc_gain_db"].iloc[0]),
            "difference_db": float(
                a["ncc_gain_db"].iloc[0]
                - b["ncc_gain_db"].iloc[0]
            )
        })

direction_comparison = pd.DataFrame(
    direction_comparison_rows
)

direction_comparison.to_csv(
    TAB / "Step23_Opposite_Direction_Comparison.csv",
    index=False
)

print("\nOPPOSITE-DIRECTION COMPARISON")
print(
    direction_comparison.round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 23G — FINAL AUDIT REPORT
# -------------------------------------------------------------------------

report = {
    "experiment": (
        "spatial heterogeneity and directional robustness"
    ),
    "source_step": 21,
    "input_records": len(df),
    "source_sha256": hashlib.sha256(
        SOURCE.read_bytes()
    ).hexdigest(),
    "patch_grid": [4, 10],
    "test_patches": 40,
    "training_seeds": [7, 17, 37],
    "figure_count": 4,
    "figure_formats": [
        "png_1200dpi",
        "vector_pdf"
    ],
    "tables": sorted(
        p.name
        for p in TAB.iterdir()
    ),
    "figures": sorted(
        p.name
        for p in FIG.iterdir()
    ),
    "caveats": [
        "Patch-grid images are not geographic maps.",
        "All evaluations use the existing MDAS test mosaic.",
        "Neighboring patches may be spatially correlated.",
        "Directions are synthetic perturbation directions.",
        "No independent-scene testing is performed.",
        "This audit is descriptive; no new model is trained."
    ],
    "output_directory": str(OUT)
}

with open(
    OUT / "Step23_Final_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2
    )

print("\n" + "=" * 84)
print("STEP 23 — FINAL REPORT")
print("=" * 84)

print(json.dumps(report, indent=2))

print("\nSTEP 23 COMPLETE")
print("Original research files remain unchanged.")


In [ ]:

# =============================================================================
# STEP 24 — INDEPENDENT SCENE VALIDATION READINESS AUDIT
#
# Required external scene organization:
#
# /MyDrive/MDAS_External_Validation/
#     scene_manifest.csv
#     <scene files specified in manifest>
#
# Manifest columns:
#   scene_id, lrhsi_path, hrmsi_path, hrhsi_path,
#   source_dataset, source_acquisition, independence_evidence
#
# Paths are relative to MDAS_External_Validation.
#
# This audit checks:
#   - Provenance completeness
#   - Unique scene identities
#   - File existence
#   - Spectral channel compatibility
#   - Consistent 3x LRHSI-to-HRMSI spatial scaling
#   - Reference dimensions
#   - Whether independent validation can proceed
#
# It does not train or evaluate any models.
# It does not modify source imagery.
# =============================================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import tifffile
import json
import hashlib
import re

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

EXTERNAL = ROOT / "MDAS_External_Validation"
MANIFEST = EXTERNAL / "scene_manifest.csv"

OUT = PROJECT / "Step24_Independent_Validation_Readiness"
OUT.mkdir(parents=True, exist_ok=True)

REQUIRED = [
    "scene_id",
    "lrhsi_path",
    "hrmsi_path",
    "hrhsi_path",
    "source_dataset",
    "source_acquisition",
    "independence_evidence"
]

print("=" * 88)
print("STEP 24 — INDEPENDENT VALIDATION READINESS")
print("=" * 88)
print("External dataset location:", EXTERNAL)

def safe_relative_file(value):
    path = Path(str(value))

    if path.is_absolute() or ".." in path.parts:
        raise ValueError(
            "Manifest paths must be relative and stay "
            "inside the external dataset directory."
        )

    candidate = (EXTERNAL / path).resolve()
    root = EXTERNAL.resolve()

    if candidate != root and root not in candidate.parents:
        raise ValueError(
            f"Path escapes external dataset directory: {value}"
        )

    return candidate

def image_info(path):
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        return {
            "shape": tuple(int(v) for v in series.shape),
            "dtype": str(series.dtype)
        }

rows = []
findings = []

if not MANIFEST.is_file():
    status = "EXTERNAL_SCENE_MANIFEST_NOT_FOUND"

    print("\nNo external scene manifest found.")
    print("No independent-scene performance claim is possible.")

    template = pd.DataFrame(
        columns=REQUIRED
    )

    template_path = OUT / "Step24_External_Scene_Manifest_Template.csv"

    if not template_path.exists():
        template.to_csv(template_path, index=False)

    print("Blank manifest template:", template_path)

else:
    manifest = pd.read_csv(
        MANIFEST,
        dtype=str,
        keep_default_na=False
    )

    missing = [
        field for field in REQUIRED
        if field not in manifest.columns
    ]

    if missing:
        raise ValueError(
            "Manifest missing columns: "
            + ", ".join(missing)
        )

    manifest = manifest[REQUIRED].copy()

    if manifest["scene_id"].duplicated().any():
        raise ValueError(
            "Duplicate scene identifiers in external manifest."
        )

    for row in manifest.itertuples(index=False):

        record = dict(zip(REQUIRED, row))

        result = {
            "scene_id": record["scene_id"],
            "source_dataset": record["source_dataset"],
            "source_acquisition": record["source_acquisition"],
            "independence_evidence": record["independence_evidence"],
            "files_exist": False,
            "dimensions_compatible": False,
            "provenance_complete": False,
            "ready_for_review": False,
            "problem": ""
        }

        try:
            provenance_complete = all(
                str(record[k]).strip()
                for k in [
                    "scene_id",
                    "source_dataset",
                    "source_acquisition",
                    "independence_evidence"
                ]
            )

            paths = {
                modality: safe_relative_file(
                    record[f"{modality}_path"]
                )
                for modality in ["lrhsi", "hrmsi", "hrhsi"]
            }

            if not all(p.is_file() for p in paths.values()):
                raise FileNotFoundError(
                    "One or more declared scene files are absent"
                )

            result["files_exist"] = True

            shapes = {
                modality: image_info(path)
                for modality, path in paths.items()
            }

            lr = shapes["lrhsi"]["shape"]
            ms = shapes["hrmsi"]["shape"]
            hr = shapes["hrhsi"]["shape"]

            result["lrhsi_shape"] = str(lr)
            result["hrmsi_shape"] = str(ms)
            result["hrhsi_shape"] = str(hr)

            compatible = (
                len(lr) == len(ms) == len(hr) == 3
                and lr[2] == hr[2] == 242
                and ms[2] == 4
                and ms[:2] == hr[:2]
                and ms[0] == 3 * lr[0]
                and ms[1] == 3 * lr[1]
            )

            result["dimensions_compatible"] = bool(compatible)
            result["provenance_complete"] = bool(
                provenance_complete
            )

            result["ready_for_review"] = bool(
                compatible and provenance_complete
            )

            if not compatible:
                result["problem"] = (
                    "Input band counts or spatial scales "
                    "do not match the saved fusion models."
                )

            elif not provenance_complete:
                result["problem"] = (
                    "Scene provenance is incomplete."
                )

        except Exception as exc:
            result["problem"] = str(exc)

        rows.append(result)

    audit = pd.DataFrame(rows)

    audit.to_csv(
        OUT / "Step24_External_Scene_Audit.csv",
        index=False
    )

    ready = (
        int(audit["ready_for_review"].sum())
        if len(audit) else 0
    )

    # Compatibility and manifest completeness are preliminary
    # checks only. Scene independence still requires review
    # against the original MDAS source footprints.
    status = (
        "CANDIDATE_SCENES_REQUIRE_PROVENANCE_REVIEW"
        if ready > 0
        else "NO_COMPATIBLE_EXTERNAL_SCENES_VERIFIED"
    )

    print("\nEXTERNAL SCENE INVENTORY")
    print(audit.to_string(index=False))

    print("\nCandidates passing preliminary checks:", ready)

report = {
    "experiment": "independent scene readiness audit",
    "external_root": str(EXTERNAL),
    "manifest_found": MANIFEST.is_file(),
    "candidate_scenes_checked": len(rows),
    "preliminary_compatible_candidates": sum(
        bool(r["ready_for_review"])
        for r in rows
    ),
    "geographic_independence_verified": False,
    "independent_fusion_evaluation_completed": False,
    "status": status,
    "next_requirement": (
        "Obtain and verify external source scenes with "
        "documented provenance, compatible sensor channels, "
        "and demonstrable separation from model training "
        "and calibration imagery."
    ),
    "limitations": [
        "File compatibility does not prove scene independence.",
        "Source acquisition and geographic footprint require verification.",
        "Models may not transfer across different sensor spectral responses.",
        "No external performance metrics are calculated in this audit."
    ],
    "output_directory": str(OUT)
}

report_path = OUT / "Step24_Final_Report.json"

if report_path.exists():
    raise FileExistsError(
        "Step 24 report already exists. Preserve it and "
        "use a new output folder for a subsequent audit."
    )

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 24 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 24 COMPLETE")
print("No source imagery or checkpoints modified.")


In [ ]:

from google.colab import drive
from pathlib import Path
import os
import time

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")

KEYWORDS = (
    "hyperspectral", "multispectral", "fusion",
    "pavia", "chikusei", "botswana", "cuprite",
    "houston", "remote", "dataset", "hsi", "msi"
)

print("=" * 75, flush=True)
print("STEP 25B — FAST EXTERNAL DATASET FOLDER DISCOVERY", flush=True)
print("=" * 75, flush=True)

start = time.time()

try:
    folders = sorted(
        [
            item for item in ROOT.iterdir()
            if item.is_dir()
        ],
        key=lambda p: p.name.lower()
    )

    print("\nTOTAL TOP-LEVEL FOLDERS:", len(folders), flush=True)

    print("\nPOTENTIAL DATASET FOLDERS:\n", flush=True)

    matches = [
        folder for folder in folders
        if any(
            keyword in folder.name.lower()
            for keyword in KEYWORDS
        )
        and folder.name != "MDAS_Registration_Aware_Revision_2026"
    ]

    if matches:
        for folder in matches:
            print(folder.name, flush=True)
    else:
        print(
            "No top-level folder names matched the search terms.",
            flush=True
        )

    print("\nOTHER TOP-LEVEL FOLDERS:\n", flush=True)

    for folder in folders:
        if folder not in matches:
            print(folder.name, flush=True)

except Exception as exc:
    print("Drive listing error:", repr(exc), flush=True)

print(
    "\nElapsed seconds:",
    round(time.time() - start, 2),
    flush=True
)

print("\nSTEP 25B COMPLETE", flush=True)


In [ ]:

# =============================================================================
# STEP 25C — TARGETED EXTERNAL DATASET INVENTORY
#
# Inspect five selected folders only.
# Maximum depth: 2 directories below each selected folder.
# Maximum entries inspected: 300 per folder.
# Maximum TIFF headers inspected: 25 overall.
#
# No raster pixel arrays loaded.
# No changes to any source files.
# =============================================================================

from google.colab import drive
from pathlib import Path
import os
import time
import json
import pandas as pd
import tifffile

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")

PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
OUT = PROJECT / "Step25C_Targeted_Dataset_Inventory"
OUT.mkdir(parents=True, exist_ok=True)

FOLDERS = [
    "fusion_mvp",
    "HydroFuseNet",
    "HydroFuseNet_Global",
    "Africa_Foundation_Model_Benchmark",
    "test"
]

DATA_EXT = {
    ".tif", ".tiff", ".mat", ".h5",
    ".hdf5", ".npy", ".npz", ".nc",
    ".img", ".hdr"
}

DOC_EXT = {
    ".md", ".txt", ".json", ".csv", ".pdf"
}

MAX_DEPTH = 2
MAX_ENTRIES_PER_FOLDER = 300
MAX_TIFF_HEADERS = 25

records = []
folder_stats = []
tiff_headers_read = 0

start = time.monotonic()

print("=" * 80, flush=True)
print("STEP 25C — TARGETED EXTERNAL DATASET DISCOVERY", flush=True)
print("=" * 80, flush=True)

for folder_name in FOLDERS:

    folder = ROOT / folder_name

    print(f"\nSCANNING: {folder_name}", flush=True)

    if not folder.is_dir():
        print("  Folder unavailable", flush=True)
        folder_stats.append({
            "folder": folder_name,
            "status": "not_found",
            "entries_inspected": 0
        })
        continue

    stack = [(folder, 0)]
    visited = 0
    relevant = 0
    errors = 0

    while stack and visited < MAX_ENTRIES_PER_FOLDER:

        current, depth = stack.pop()

        try:
            with os.scandir(current) as entries:
                children = list(entries)

        except (OSError, PermissionError) as exc:
            errors += 1
            print(
                "  Cannot list:",
                current.name,
                str(exc)[:120],
                flush=True
            )
            continue

        for entry in children:

            if visited >= MAX_ENTRIES_PER_FOLDER:
                break

            visited += 1

            try:
                if entry.is_dir(follow_symlinks=False):

                    if depth < MAX_DEPTH:
                        stack.append(
                            (Path(entry.path), depth + 1)
                        )

                    continue

                if not entry.is_file(follow_symlinks=False):
                    continue

                path = Path(entry.path)
                extension = path.suffix.lower()

                if extension not in DATA_EXT | DOC_EXT:
                    continue

                relevant += 1

                record = {
                    "source_folder": folder_name,
                    "relative_path": str(
                        path.relative_to(ROOT)
                    ),
                    "extension": extension,
                    "file_size_mb": None,
                    "shape": "",
                    "dtype": "",
                    "candidate_class": (
                        "scientific_data"
                        if extension in DATA_EXT
                        else "documentation"
                    )
                }

                try:
                    record["file_size_mb"] = round(
                        entry.stat().st_size / 1048576,
                        3
                    )
                except OSError:
                    pass

                if (
                    extension in {".tif", ".tiff"}
                    and tiff_headers_read < MAX_TIFF_HEADERS
                ):

                    tiff_headers_read += 1

                    try:
                        with tifffile.TiffFile(path) as tif:
                            series = tif.series[0]
                            shape = tuple(series.shape)

                            record["shape"] = str(shape)
                            record["dtype"] = str(series.dtype)

                            if (
                                len(shape) == 3
                                and 242 in (shape[0], shape[-1])
                            ):
                                record["candidate_class"] = (
                                    "possible_242_band_HSI"
                                )

                            elif (
                                len(shape) == 3
                                and 4 in (shape[0], shape[-1])
                            ):
                                record["candidate_class"] = (
                                    "possible_4_band_MSI"
                                )

                    except Exception as exc:
                        record["candidate_class"] = (
                            "tiff_header_unreadable"
                        )
                        record["inspection_error"] = str(exc)[:150]

                records.append(record)

            except (OSError, PermissionError):
                errors += 1

        if visited and visited % 100 == 0:
            print(
                f"  Entries inspected: {visited}",
                flush=True
            )

    status = (
        "entry_limit_reached"
        if visited >= MAX_ENTRIES_PER_FOLDER
        else "bounded_scan_complete"
    )

    folder_stats.append({
        "folder": folder_name,
        "status": status,
        "entries_inspected": visited,
        "relevant_files": relevant,
        "listing_errors": errors
    })

    print(
        f"  Inspected: {visited} | "
        f"Relevant: {relevant} | "
        f"Status: {status}",
        flush=True
    )

columns = [
    "source_folder",
    "relative_path",
    "extension",
    "file_size_mb",
    "shape",
    "dtype",
    "candidate_class"
]

inventory = pd.DataFrame(records)

if inventory.empty:
    inventory = pd.DataFrame(columns=columns)

stats = pd.DataFrame(folder_stats)

inventory.to_csv(
    OUT / "Step25C_Targeted_File_Inventory.csv",
    index=False
)

stats.to_csv(
    OUT / "Step25C_Folder_Summary.csv",
    index=False
)

print("\n" + "=" * 80)
print("FOLDER SUMMARY")
print("=" * 80)
print(stats.to_string(index=False))

print("\nPOTENTIAL SCIENTIFIC DATA FILES")

scientific = inventory[
    inventory["extension"].isin(DATA_EXT)
].copy()

print(
    scientific.head(60).to_string(
        index=False,
        max_colwidth=95
    )
    if not scientific.empty
    else "No scientific data files identified within scan limits."
)

print("\nPOTENTIAL HSI/MSI TIFFS")

spectral = inventory[
    inventory["candidate_class"].isin([
        "possible_242_band_HSI",
        "possible_4_band_MSI"
    ])
]

print(
    spectral.to_string(index=False)
    if not spectral.empty
    else "None verified within inspected TIFF headers."
)

report = {
    "experiment": "bounded targeted dataset discovery",
    "folders_checked": FOLDERS,
    "maximum_depth": MAX_DEPTH,
    "maximum_entries_per_folder": MAX_ENTRIES_PER_FOLDER,
    "maximum_tiff_headers": MAX_TIFF_HEADERS,
    "total_relevant_files_found": len(inventory),
    "scientific_files_found": len(scientific),
    "spectrally_compatible_tiffs_found": len(spectral),
    "scan_is_exhaustive": False,
    "external_scene_independence_verified": False,
    "external_fusion_evaluation_completed": False,
    "elapsed_seconds": round(
        time.monotonic() - start, 2
    ),
    "output_directory": str(OUT)
}

report_path = OUT / "Step25C_Final_Report.json"

# Preserve any previously completed report.
if report_path.exists():
    print(
        "\nAn earlier Step 25C report exists; "
        "it has not been overwritten.",
        flush=True
    )
else:
    report_path.write_text(
        json.dumps(report, indent=2),
        encoding="utf-8"
    )

print("\n" + "=" * 80)
print("STEP 25C — REPORT")
print("=" * 80)
print(json.dumps(report, indent=2))

print("\nSTEP 25C COMPLETE")


In [ ]:

# =============================================================================
# STEP 26 — HYSPECNET-11K EXTERNAL BENCHMARK ACQUISITION AUDIT
#
# Goals:
#   1. Verify official dataset metadata and download endpoint.
#   2. Inspect the published hard-split archive.
#   3. Download the split archive ONLY if <= 32 MiB.
#   4. Verify its published SHA-256 hash.
#   5. Audit train/val/test source-tile disjointness.
#   6. Save a small, reproducible external benchmark plan.
#
# Safety:
#   - NEVER downloads full image archives.
#   - Never scans all of Google Drive.
#   - Downloads have a strict 32 MiB cap.
#   - No model training or fusion inference.
#   - Original MDAS files remain unchanged.
#
# Published TorchGeo archive source and SHA-256:
# https://github.com/torchgeo/torchgeo/blob/main/torchgeo/datasets/hyspecnet.py
# =============================================================================

from google.colab import drive
from pathlib import Path
from urllib.parse import quote
import requests
import hashlib
import json
import tarfile
import io
import re
import time
import pandas as pd

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

OUT = PROJECT / "Step26_External_Benchmark_Acquisition"
OUT.mkdir(parents=True, exist_ok=True)

CACHE = OUT / "Metadata"
CACHE.mkdir(exist_ok=True)

# Pinned, published TorchGeo mirror.
REVISION = "13e110422a6925cbac0f11edff610219b9399227"

BASE = (
    "https://huggingface.co/datasets/torchgeo/hyspecnet"
    f"/resolve/{REVISION}"
)

ARCHIVE_NAME = "hyspecnet-11k-splits.tar.gz"
ARCHIVE_URL = f"{BASE}/{ARCHIVE_NAME}"

EXPECTED_SHA256 = (
    "12d809d1e13ae4b2cf76b9eca2855e0bbab143372ebe728440eab383364cfb8b"
)

MAX_DOWNLOAD_BYTES = 32 * 1024 * 1024

DATASET_URL = "https://hyspecnet.rsim.berlin/"
DESCRIPTION_URL = (
    "https://hyspecnet.rsim.berlin/hyspecnet-11k-description/"
)

DRYAD_URL = (
    "https://datadryad.org/dataset/doi:10.5061/dryad.fttdz08zh"
)

HEAD_TIMEOUT = (8, 15)
DOWNLOAD_TIMEOUT = (8, 20)

print("=" * 88, flush=True)
print("STEP 26 — EXTERNAL HYPERSPECTRAL BENCHMARK ACQUISITION", flush=True)
print("=" * 88, flush=True)
print("Dataset: HySpecNet-11k")
print("Archive: SPLITS ONLY")
print("Maximum archive size:", MAX_DOWNLOAD_BYTES // (1024**2), "MiB")
print("Output:", OUT, flush=True)

# -------------------------------------------------------------------------
# 26A — DATASET PROVENANCE
# -------------------------------------------------------------------------

provenance = {
    "dataset": "HySpecNet-11k",
    "sensor": "EnMAP",
    "original_bands": 224,
    "patch_shape": [128, 128],
    "ground_sample_distance_m": 30,
    "number_of_patches_reported": 11483,
    "source_tiles_reported": 250,
    "hard_split_definition": (
        "All patches belonging to any source tile "
        "are assigned to only one split."
    ),
    "official_site": DATASET_URL,
    "description": DESCRIPTION_URL,
    "archive_repository": DRYAD_URL,
    "mirror": "TorchGeo Hugging Face",
    "mirror_revision": REVISION,
    "archive_name": ARCHIVE_NAME,
    "expected_sha256": EXPECTED_SHA256,
    "separate_from_mdas_dataset": True,
    "direct_mdas_checkpoint_compatible": False,
    "license_review_required": True,
    "intended_task": (
        "External synthetic cross-sensor registration "
        "benchmark, not pretrained MDAS fusion transfer."
    )
}

provenance_path = OUT / "Step26_Dataset_Provenance.json"

if provenance_path.exists():
    old = json.loads(
        provenance_path.read_text(encoding="utf-8")
    )
    if old != provenance:
        raise ValueError(
            "Existing provenance differs from this script. "
            "Use a new experiment directory."
        )
else:
    provenance_path.write_text(
        json.dumps(provenance, indent=2),
        encoding="utf-8"
    )

print("\n26A — DATASET PROVENANCE")
print("Published source:", DATASET_URL)
print("Mirror revision:", REVISION)
print("Required external split: hard")
print("Original hyperspectral bands:", 224)
print("MDAS fusion checkpoint compatibility: NO")

# -------------------------------------------------------------------------
# 26B — QUERY THE OFFICIAL MIRROR FOR ARCHIVE SIZE
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("26B — REMOTE ARCHIVE METADATA", flush=True)
print("=" * 88, flush=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "MDAS-External-Benchmark-Research-Audit/1.0"
})

remote_size = None
remote_url_reachable = False
head_error = None

try:
    response = session.head(
        ARCHIVE_URL,
        allow_redirects=True,
        timeout=HEAD_TIMEOUT
    )

    print("HTTP status:", response.status_code)

    response.raise_for_status()

    remote_url_reachable = True

    size_header = response.headers.get("Content-Length")

    if size_header is not None:
        remote_size = int(size_header)

        print(
            "Reported archive size:",
            round(remote_size / (1024**2), 3),
            "MiB"
        )

    else:
        print(
            "Remote server did not publish a reliable Content-Length."
        )

except Exception as exc:
    head_error = str(exc)[:250]
    print("Archive metadata unavailable:", head_error)

# -------------------------------------------------------------------------
# 26C — BOUNDED DOWNLOAD OF SPLIT ARCHIVE ONLY
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("26C — SAFE MANIFEST ACQUISITION", flush=True)
print("=" * 88, flush=True)

archive_path = CACHE / ARCHIVE_NAME

download_status = "not_attempted"
verified_sha = None
download_bytes = 0

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

if archive_path.is_file():

    verified_sha = sha256_file(archive_path)

    if verified_sha == EXPECTED_SHA256:
        download_status = "verified_existing_archive"
        download_bytes = archive_path.stat().st_size

        print("Verified previously acquired split archive.")

    else:
        download_status = "existing_archive_hash_mismatch"

        print(
            "WARNING: Existing archive checksum differs "
            "from the published checksum."
        )
        print("Existing file was not deleted or overwritten.")

elif not remote_url_reachable:

    download_status = "remote_metadata_unavailable"
    print("Download skipped: remote endpoint not verified.")

elif remote_size is None:

    download_status = "remote_size_unknown"
    print(
        "Download skipped: archive size unknown. "
        "The script will not risk downloading a large file."
    )

elif remote_size > MAX_DOWNLOAD_BYTES:

    download_status = "archive_exceeds_size_cap"

    print(
        "Download skipped safely: archive exceeds "
        "the 32 MiB limit."
    )

else:

    print("Archive is within the download cap.")

    temporary = CACHE / (ARCHIVE_NAME + ".partial")

    # Stale partial download files are not reused silently.
    if temporary.exists():
        raise FileExistsError(
            f"Existing partial file: {temporary}\n"
            "Inspect it before attempting another download."
        )

    try:
        digest = hashlib.sha256()

        with session.get(
            ARCHIVE_URL,
            stream=True,
            timeout=DOWNLOAD_TIMEOUT,
            allow_redirects=True
        ) as response:

            response.raise_for_status()

            with open(temporary, "xb") as file:

                for chunk in response.iter_content(
                    chunk_size=1024 * 256
                ):
                    if not chunk:
                        continue

                    download_bytes += len(chunk)

                    if download_bytes > MAX_DOWNLOAD_BYTES:
                        raise ValueError(
                            "Transfer exceeded the 32 MiB safety cap."
                        )

                    file.write(chunk)
                    digest.update(chunk)

        verified_sha = digest.hexdigest()

        if verified_sha != EXPECTED_SHA256:
            download_status = "download_hash_mismatch"

            print("Downloaded archive failed checksum verification.")
            print("Unverified partial file retained for inspection.")

        else:
            temporary.rename(archive_path)

            download_status = "downloaded_and_verified"

            print(
                "Split archive acquired and verified:",
                archive_path
            )

    except Exception as exc:
        download_status = "download_failed"

        print(
            "Bounded download failed:",
            str(exc)[:300]
        )

print("\nManifest acquisition status:", download_status)

# -------------------------------------------------------------------------
# 26D — INSPECT HARD SPLIT FILES, WITHOUT EXTRACTING TAR PATHS
# -------------------------------------------------------------------------

print("\n" + "=" * 88, flush=True)
print("26D — HARD SPLIT INSPECTION", flush=True)
print("=" * 88, flush=True)

split_paths = {}

if download_status in {
    "verified_existing_archive",
    "downloaded_and_verified"
}:

    with tarfile.open(
        archive_path,
        mode="r:gz"
    ) as tar:

        for member in tar.getmembers():

            if not member.isfile():
                continue

            normalized = member.name.replace("\\", "/")

            for split in ("train", "val", "test"):

                if normalized.endswith(
                    f"/splits/hard/{split}.csv"
                ):

                    # Only read modestly sized CSV metadata.
                    if member.size > 10 * 1024 * 1024:
                        raise ValueError(
                            f"Split manifest unusually large: "
                            f"{member.name}"
                        )

                    extracted = tar.extractfile(member)

                    if extracted is None:
                        raise ValueError(
                            f"Unable to inspect: {member.name}"
                        )

                    contents = extracted.read()

                    if split in split_paths:
                        raise ValueError(
                            f"Multiple {split} hard-split manifests."
                        )

                    output = CACHE / f"hard_{split}.csv"

                    if output.exists():

                        if output.read_bytes() != contents:
                            raise ValueError(
                                "Existing extracted split CSV differs. "
                                "Preserve and investigate."
                            )

                    else:
                        output.write_bytes(contents)

                    split_paths[split] = output

    print("Hard-split files recovered:", sorted(split_paths))

else:
    print(
        "Hard split not available locally. "
        "No tile-disjointness claims will be made."
    )

# -------------------------------------------------------------------------
# 26E — PARSE PUBLISHED SPLIT LISTS
# -------------------------------------------------------------------------

def read_split_entries(path):
    """
    Official TorchGeo loader treats CSV files as
    newline-separated relative data paths.
    """

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    entries = []

    for line in lines:
        value = line.strip().strip('"').strip("'")

        if not value:
            continue

        if value.lower() in {"path", "filename", "file"}:
            continue

        entries.append(value.replace("\\", "/"))

    return entries

def tile_from_path(path):
    """
    Expected path:
      tile_identifier/patch_identifier/...-DATA.npy

    This function checks path structure rather than assuming
    that the first number is a valid geographic identifier.
    """

    parts = path.strip("/").split("/")

    if len(parts) < 2:
        raise ValueError(
            f"Cannot recover source tile from: {path}"
        )

    return parts[0]

split_counts = {}
tile_counts = {}
split_overlap = None
tile_overlap = None
hard_split_verified = False
pilot_manifest = pd.DataFrame()

if set(split_paths) == {"train", "val", "test"}:

    entries = {
        split: read_split_entries(path)
        for split, path in split_paths.items()
    }

    split_counts = {
        split: len(items)
        for split, items in entries.items()
    }

    tile_sets = {
        split: set(
            tile_from_path(path)
            for path in items
        )
        for split, items in entries.items()
    }

    tile_counts = {
        split: len(tiles)
        for split, tiles in tile_sets.items()
    }

    splits = ("train", "val", "test")

    split_overlap = {}
    tile_overlap = {}

    for i, a in enumerate(splits):
        for b in splits[i+1:]:

            split_overlap[f"{a}_vs_{b}"] = len(
                set(entries[a]).intersection(entries[b])
            )

            tile_overlap[f"{a}_vs_{b}"] = len(
                tile_sets[a].intersection(tile_sets[b])
            )

    unique_within = all(
        len(items) == len(set(items))
        for items in entries.values()
    )

    hard_split_verified = (
        unique_within
        and all(v == 0 for v in split_overlap.values())
        and all(v == 0 for v in tile_overlap.values())
    )

    print("\nPATCH COUNTS:", split_counts)
    print("TILE COUNTS:", tile_counts)
    print("Cross-split duplicate paths:", split_overlap)
    print("Cross-split shared tiles:", tile_overlap)
    print("Hard split disjointness verified:", hard_split_verified)

    if hard_split_verified:

        # Select a small reproducible subset of published
        # TEST-SPLIT patches without using image contents.
        #
        # This is a pilot manifest only, not downloaded imagery.
        # Select at most two patches from each of five
        # separate source tiles.

        pilot_rows = []

        seen_per_tile = {}

        for path in sorted(entries["test"]):

            tile = tile_from_path(path)

            if (
                tile not in seen_per_tile
                and len(seen_per_tile) >= 5
            ):
                continue

            count = seen_per_tile.get(tile, 0)

            if count >= 2:
                continue

            seen_per_tile[tile] = count + 1

            pilot_rows.append({
                "published_split": "hard/test",
                "source_tile": tile,
                "relative_image_path": path.replace(
                    "-DATA.npy",
                    "-SPECTRAL_IMAGE.TIF"
                ),
                "image_downloaded": False,
                "sensor": "EnMAP",
                "independent_of_mdas_training_source": (
                    "Different source dataset; scene metadata "
                    "still requires confirmation"
                )
            })

        pilot_manifest = pd.DataFrame(pilot_rows)

        pilot_manifest.to_csv(
            OUT / "Step26_External_Pilot_Manifest.csv",
            index=False
        )

        print("\nPROPOSED EXTERNAL PILOT")
        print(
            pilot_manifest.to_string(
                index=False,
                max_colwidth=80
            )
        )

else:
    print(
        "Hard split verification not possible with "
        "the available local metadata."
    )

# -------------------------------------------------------------------------
# 26F — SCIENTIFIC READINESS ASSESSMENT
# -------------------------------------------------------------------------

if hard_split_verified:

    status = "HARD_SPLIT_VERIFIED_IMAGES_NOT_ACQUIRED"

elif download_status == "archive_exceeds_size_cap":

    status = "MANIFEST_ARCHIVE_TOO_LARGE_FOR_SAFE_AUTO_DOWNLOAD"

elif download_status in {
    "remote_size_unknown",
    "remote_metadata_unavailable"
}:

    status = "MANIFEST_DOWNLOAD_METADATA_UNAVAILABLE"

else:

    status = "EXTERNAL_SPLIT_NOT_YET_VERIFIED"

report = {
    "experiment": "HySpecNet-11k metadata and hard-split audit",
    "dataset": "HySpecNet-11k",
    "official_dataset_site": DATASET_URL,
    "repository": DRYAD_URL,
    "archive_mirror": ARCHIVE_URL,
    "archive_download_status": download_status,
    "archive_size_bytes_reported": remote_size,
    "maximum_download_bytes": MAX_DOWNLOAD_BYTES,
    "published_sha256": EXPECTED_SHA256,
    "actual_sha256": verified_sha,
    "hard_split_files_found": sorted(split_paths.keys()),
    "hard_split_patch_counts": split_counts,
    "hard_split_tile_counts": tile_counts,
    "cross_split_duplicate_paths": split_overlap,
    "cross_split_shared_source_tiles": tile_overlap,
    "hard_split_tile_disjointness_verified": hard_split_verified,
    "pilot_patches_selected": int(len(pilot_manifest)),
    "hyperspectral_images_downloaded": 0,
    "independent_registration_evaluation_completed": False,
    "pretrained_mdas_fusion_generalization_verified": False,
    "status": status,
    "next_requirement": (
        "Acquire a small number of genuine EnMAP "
        "test-tile hyperspectral patches, verify source "
        "metadata and radiometry, then establish a "
        "predeclared external registration protocol."
    ),
    "limitations": [
        "Different sensor bands prevent direct use of MDAS fusion checkpoints.",
        "A hard split verifies tile-disjoint partitioning within HySpecNet.",
        "Tile-disjoint splitting does not prove geographic distance.",
        "A synthetic MSI counterpart is not a real multispectral acquisition.",
        "No registration or fusion performance is measured in this step.",
        "Published dataset licensing must be respected."
    ],
    "output_directory": str(OUT)
}

report_path = OUT / "Step26_Final_Report.json"

if report_path.exists():
    existing = json.loads(
        report_path.read_text(encoding="utf-8")
    )

    # Preserve any completed report.
    if existing != report:
        print(
            "\nAn earlier Step 26 report exists. "
            "It has not been overwritten."
        )

else:
    report_path.write_text(
        json.dumps(report, indent=2),
        encoding="utf-8"
    )

print("\n" + "=" * 88)
print("STEP 26 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 26 COMPLETE")
print("No large image archives downloaded.")
print("Original MDAS data and checkpoints remain unchanged.")


In [ ]:

# =============================================================================
# STEP 27 — EXTERNAL ENMAP POTSDAM SCENE ACQUISITION
#
# Source: Official EnMAP-Box example-data release.
# https://github.com/EnMAP-Box/enmap-box-exampledata
#
# Objectives:
#  1. Check official example-data archive availability.
#  2. Enforce a strict 512 MiB download limit.
#  3. Extract ONLY the hyperspectral Potsdam example.
#  4. Verify image dimensions, bands, radiometry and georeferencing.
#  5. Record provenance and SHA-256 checksums.
#
# No training. No synthetic imagery generated yet.
# Does not modify existing MDAS experiments.
#
# If the archive is too large or unavailable, report that status.
# Never substitute another image silently.
# =============================================================================

from google.colab import drive
from pathlib import Path
import requests
import hashlib
import zipfile
import tempfile
import os
import shutil
import json
import time
import numpy as np

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")

PROJECT = (
    ROOT / "MDAS_Registration_Aware_Revision_2026"
)

OUT = PROJECT / "Step27_External_EnMAP_Pilot"
DATA_OUT = OUT / "Imagery"
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Official example-data release URL, used in EnMAP-Box documentation.
URL = (
    "https://github.com/EnMAP-Box/"
    "enmap-box-exampledata/releases/download/"
    "v1.1/exampledata.zip"
)

MAX_DOWNLOAD = 512 * 1024 * 1024
MAX_EXTRACTED_IMAGE = 512 * 1024 * 1024

# Keep the complete archive in local Colab temporary storage,
# not permanently in Google Drive.
CACHE = Path("/content/step27_enmap_download")
CACHE.mkdir(parents=True, exist_ok=True)

ARCHIVE = CACHE / "exampledata.zip"

TARGET = DATA_OUT / "enmap_potsdam.tif"

REPORT_PATH = OUT / "Step27_Final_Report.json"

print("=" * 88, flush=True)
print("STEP 27 — EXTERNAL ENMAP PILOT ACQUISITION", flush=True)
print("=" * 88, flush=True)
print("Source:", URL, flush=True)
print("Maximum archive:", MAX_DOWNLOAD // (1024**2), "MiB")
print("Output:", OUT, flush=True)

if REPORT_PATH.exists():
    raise FileExistsError(
        "A Step 27 report already exists. "
        "Preserve earlier results and use a new output directory."
    )

start = time.monotonic()

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

report = {
    "experiment": "external EnMAP Potsdam imagery acquisition",
    "dataset_source": "EnMAP-Box official example-data release",
    "source_url": URL,
    "target_scene": "Potsdam, Germany",
    "sensor": "EnMAP",
    "published_acquisition_date": "2022-07-24",
    "archive_available": False,
    "archive_size_bytes": None,
    "download_status": "not_started",
    "archive_sha256": None,
    "image_found": False,
    "image_sha256": None,
    "shape": None,
    "band_count": None,
    "crs": None,
    "has_georeference": False,
    "independent_from_mdas_city": True,
    "registration_evaluation_completed": False,
    "fusion_checkpoint_transfer_completed": False,
    "external_registration_pilot_ready": False
}

# -------------------------------------------------------------------------
# 27A — CHECK REMOTE ARCHIVE SIZE
# -------------------------------------------------------------------------

session = requests.Session()

session.headers["User-Agent"] = (
    "Academic-Remote-Sensing-Research/1.0"
)

remote_size = None

print("\n27A — ARCHIVE AVAILABILITY", flush=True)

try:
    response = session.head(
        URL,
        allow_redirects=True,
        timeout=(10, 20)
    )

    response.raise_for_status()

    report["archive_available"] = True

    size_header = response.headers.get(
        "Content-Length"
    )

    if size_header:
        remote_size = int(size_header)

    report["archive_size_bytes"] = remote_size

    print("HTTP status:", response.status_code)
    print(
        "Reported size:",
        (
            round(remote_size / (1024**2), 2)
            if remote_size is not None
            else "unknown"
        ),
        "MiB",
        flush=True
    )

except Exception as exc:
    report["download_status"] = (
        "archive_metadata_unavailable"
    )
    report["error"] = str(exc)[:500]

    print("Cannot verify archive:", exc, flush=True)

# -------------------------------------------------------------------------
# 27B — SAFETY-GATED DOWNLOAD
# -------------------------------------------------------------------------

if not report["archive_available"]:
    print("\nArchive not available; download skipped.")

elif remote_size is None:
    report["download_status"] = "unknown_archive_size"
    print("\nArchive size unknown; download skipped safely.")

elif remote_size > MAX_DOWNLOAD:
    report["download_status"] = "archive_too_large"
    print(
        "\nArchive exceeds 512 MiB. "
        "Download skipped safely."
    )

else:
    print("\n27B — BOUNDED DOWNLOAD", flush=True)

    if ARCHIVE.exists():
        if (
            ARCHIVE.stat().st_size > MAX_DOWNLOAD
            or not zipfile.is_zipfile(ARCHIVE)
        ):
            raise ValueError(
                "Existing cache archive failed basic validation."
            )

        print("Using existing local archive cache.")

    else:
        partial = ARCHIVE.with_suffix(".partial")

        if partial.exists():
            raise FileExistsError(
                "A partial archive already exists. "
                "Inspect it before retrying."
            )

        bytes_downloaded = 0

        try:
            with session.get(
                URL,
                stream=True,
                allow_redirects=True,
                timeout=(10, 30)
            ) as response:

                response.raise_for_status()

                with open(partial, "xb") as f:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 256
                    ):

                        if not chunk:
                            continue

                        bytes_downloaded += len(chunk)

                        if bytes_downloaded > MAX_DOWNLOAD:
                            raise ValueError(
                                "Download exceeded the 512 MiB cap."
                            )

                        f.write(chunk)

                        if (
                            bytes_downloaded // (25 * 1024 * 1024)
                        ) != (
                            (bytes_downloaded-len(chunk))
                            // (25 * 1024 * 1024)
                        ):
                            print(
                                "Downloaded:",
                                round(
                                    bytes_downloaded / (1024**2), 1
                                ),
                                "MiB",
                                flush=True
                            )

            if not zipfile.is_zipfile(partial):
                raise ValueError(
                    "Downloaded file is not a valid ZIP archive."
                )

            os.replace(partial, ARCHIVE)

        except Exception as exc:
            report["download_status"] = "download_failed"
            report["download_error"] = str(exc)[:500]

            print("Download failure:", exc, flush=True)

    if ARCHIVE.exists() and zipfile.is_zipfile(ARCHIVE):
        report["download_status"] = "archive_acquired"
        report["archive_sha256"] = sha256_file(ARCHIVE)

        print(
            "Archive acquired:",
            round(ARCHIVE.stat().st_size/(1024**2), 2),
            "MiB",
            flush=True
        )

# -------------------------------------------------------------------------
# 27C — EXTRACT ONLY THE HYPERSPECTRAL EXAMPLE
# -------------------------------------------------------------------------

if report["download_status"] == "archive_acquired":

    print("\n27C — TARGETED HSI EXTRACTION", flush=True)

    with zipfile.ZipFile(ARCHIVE, "r") as z:

        matches = [
            member
            for member in z.infolist()
            if not member.is_dir()
            and Path(member.filename).name.lower()
            == "enmap_potsdam.tif"
        ]

        if len(matches) == 1:
            member = matches[0]

            if member.file_size > MAX_EXTRACTED_IMAGE:
                report["download_status"] = (
                    "hyperspectral_image_exceeds_cap"
                )

                print(
                    "HSI image is larger than the extraction limit."
                )

            elif TARGET.exists():
                report["image_found"] = True
                print("Existing target image retained:", TARGET)

            else:
                temporary = TARGET.with_suffix(".partial")

                if temporary.exists():
                    raise FileExistsError(
                        "A partial extracted image exists."
                    )

                with z.open(member) as source:
                    with open(temporary, "xb") as destination:

                        copied = 0

                        while True:
                            chunk = source.read(1024*1024)

                            if not chunk:
                                break

                            copied += len(chunk)

                            if copied > MAX_EXTRACTED_IMAGE:
                                raise ValueError(
                                    "Image extraction exceeded "
                                    "the declared safety cap."
                                )

                            destination.write(chunk)

                os.replace(temporary, TARGET)

                report["image_found"] = True

                print("HSI extracted:", TARGET)

        elif len(matches) == 0:
            report["download_status"] = (
                "hyperspectral_image_not_found_in_archive"
            )
            print(
                "The expected enmap_potsdam.tif "
                "was not found in the archive."
            )

            # Save file names for accurate follow-up.
            names = [
                m.filename
                for m in z.infolist()
                if not m.is_dir()
            ]

            (OUT / "Step27_Archive_Contents.txt").write_text(
                "\n".join(names[:2000]),
                encoding="utf-8"
            )

        else:
            report["download_status"] = (
                "multiple_matching_hyperspectral_images"
            )

            print(
                "Multiple matching HSI files found; "
                "automatic extraction stopped."
            )

# -------------------------------------------------------------------------
# 27D — IMAGE METADATA AND DATA INTEGRITY
# -------------------------------------------------------------------------

if report["image_found"] and TARGET.is_file():

    print("\n27D — EXTERNAL HSI METADATA", flush=True)

    import rasterio

    with rasterio.open(TARGET) as src:

        shape = [
            int(src.height),
            int(src.width),
            int(src.count)
        ]

        bounds = [
            float(src.bounds.left),
            float(src.bounds.bottom),
            float(src.bounds.right),
            float(src.bounds.top)
        ]

        report["shape"] = shape
        report["band_count"] = int(src.count)
        report["dtype"] = list(src.dtypes)
        report["crs"] = str(src.crs) if src.crs else None
        report["bounds"] = bounds
        report["nodata"] = (
            float(src.nodata)
            if src.nodata is not None
            and np.isfinite(src.nodata)
            else None
        )

        has_georef = (
            src.crs is not None
            and not src.transform.is_identity
        )

        report["has_georeference"] = bool(has_georef)

        # Read a small sample window, not the entire HSI cube.
        height = min(32, src.height)
        width = min(32, src.width)

        from rasterio.windows import Window

        sample = src.read(
            indexes=list(range(1, min(8, src.count) + 1)),
            window=Window(0, 0, width, height)
        )

        finite = sample[np.isfinite(sample)]

        report["sample_finite_fraction"] = float(
            np.isfinite(sample).mean()
        )

        if finite.size:
            report["sample_min"] = float(finite.min())
            report["sample_max"] = float(finite.max())

        report["external_registration_pilot_ready"] = bool(
            src.count >= 200
            and src.height >= 96
            and src.width >= 96
            and has_georef
            and finite.size > 0
        )

    report["image_sha256"] = sha256_file(TARGET)

    print("Dimensions (H, W, bands):", report["shape"])
    print("CRS:", report["crs"])
    print("Georeferencing:", report["has_georeference"])
    print("Pilot image readiness:", report["external_registration_pilot_ready"])

# -------------------------------------------------------------------------
# 27E — FINAL REPORT
# -------------------------------------------------------------------------

report["elapsed_seconds"] = round(
    time.monotonic() - start,
    2
)

report["methodological_limits"] = [
    "External EnMAP imagery is not directly compatible "
    "with the saved 242-band MDAS fusion checkpoints.",
    "A synthetic MSI counterpart must be clearly labeled.",
    "The current step does not create synthetic observations.",
    "One independent scene is useful for a pilot "
    "but is insufficient for broad geographic generalization.",
    "The archive checksum is recorded but cannot be "
    "compared with an independently published checksum "
    "unless one is supplied."
]

report["output_directory"] = str(OUT)

REPORT_PATH.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 27 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 27 COMPLETE")
print("Original MDAS imagery and checkpoints unchanged.")


In [ ]:

# =============================================================================
# STEP 28 — INDEPENDENT ENMAP RADIOMETRIC AND VALIDITY AUDIT
#
# Input:
#   Step27_External_EnMAP_Pilot/Imagery/enmap_potsdam.tif
#
# Outputs:
#   Step28_Scene_Metadata.json
#   Step28_Band_Validity_Statistics.csv
#   Step28_Candidate_Patch_Windows.csv
#   Step28_Band_Metadata.csv
#   Step28_Final_Report.json
#
# Checks:
#   - All 224 channels and spatial dimensions
#   - Declared nodata (-32768) and non-finite data
#   - Per-band validity and radiometric distributions
#   - Wavelength / scaling metadata availability
#   - 96x96 candidate windows and coverage
#
# No synthetic registration yet.
# No MDAS weights loaded or altered.
# =============================================================================

from google.colab import drive
from pathlib import Path
import rasterio
from rasterio.windows import Window
import numpy as np
import pandas as pd
import hashlib
import json
import re

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

IMAGE = (
    PROJECT / "Step27_External_EnMAP_Pilot"
    / "Imagery" / "enmap_potsdam.tif"
)

PREVIOUS_REPORT = (
    PROJECT / "Step27_External_EnMAP_Pilot"
    / "Step27_Final_Report.json"
)

OUT = PROJECT / "Step28_EnMAP_Radiometric_Audit"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = (
    "4ba14fd27ac80c9169f3d733010dbdaa986584b3a062dc497578c3e114733329"
)

PATCH = 96
MIN_PIXEL_COVERAGE = 0.98

print("=" * 86, flush=True)
print("STEP 28 — ENMAP RADIOMETRIC VALIDITY AUDIT", flush=True)
print("=" * 86, flush=True)

assert IMAGE.is_file(), f"Missing source image: {IMAGE}"
assert PREVIOUS_REPORT.is_file()

with open(PREVIOUS_REPORT) as f:
    step27 = json.load(f)

assert step27["image_sha256"] == EXPECTED_SHA

# Preserve previous audit outputs.
output_names = [
    "Step28_Scene_Metadata.json",
    "Step28_Band_Validity_Statistics.csv",
    "Step28_Candidate_Patch_Windows.csv",
    "Step28_Band_Metadata.csv",
    "Step28_Final_Report.json"
]

existing = [name for name in output_names if (OUT/name).exists()]
if existing:
    raise FileExistsError(
        "Step 28 outputs already exist: "
        + ", ".join(existing)
        + ". Preserve the prior run."
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

actual_sha = sha256_file(IMAGE)

assert actual_sha == EXPECTED_SHA, (
    "Input imagery differs from verified Step 27 source."
)

print("\n28A — SOURCE IMAGE VERIFICATION")
print("Source:", IMAGE)
print("SHA256:", actual_sha)

with rasterio.open(IMAGE) as src:

    assert src.count == 224
    assert src.height == 200
    assert src.width == 330
    assert str(src.crs) == "EPSG:32633"
    assert not src.transform.is_identity

    nodata = src.nodata
    transform = src.transform

    image_info = {
        "height": src.height,
        "width": src.width,
        "bands": src.count,
        "crs": str(src.crs),
        "transform": list(transform)[:6],
        "bounds": [
            src.bounds.left,
            src.bounds.bottom,
            src.bounds.right,
            src.bounds.top
        ],
        "nodata": nodata,
        "dtype": src.dtypes[0],
        "dataset_tags": src.tags(),
        "descriptions": [
            str(v) if v is not None else None
            for v in src.descriptions
        ],
        "units": [
            str(v) if v is not None else None
            for v in src.units
        ],
        "scales": [float(v) for v in src.scales],
        "offsets": [float(v) for v in src.offsets],
        "input_sha256": actual_sha
    }

    # This image is modest in size. Reading ~59 MB of float32
    # data is practical and avoids thousands of Drive requests.
    cube = src.read().astype(np.float32)

    # Shape is (bands, height, width).
    assert cube.shape == (224, 200, 330)

    # Rasterio's mask catches nodata and other masked pixels.
    dataset_mask = src.read_masks() > 0

    valid = dataset_mask & np.isfinite(cube)

    if nodata is not None and np.isfinite(nodata):
        valid &= cube != nodata

    # Only use the dataset's native metadata here.
    # Do NOT assume the correct reflectance scaling
    # or that values must be between 0 and 1.

    band_records = []
    band_metadata = []

    print("\n28B — PER-BAND VALIDITY AND RADIOMETRY")

    for band_index in range(224):

        values = cube[band_index]
        mask = valid[band_index]

        selected = values[mask]

        if len(selected):
            q = np.percentile(
                selected,
                [0, 1, 5, 50, 95, 99, 100]
            )

            minimum, p01, p05, median, p95, p99, maximum = [
                float(v) for v in q
            ]

            mean = float(selected.mean(dtype=np.float64))
            std = float(selected.std(dtype=np.float64))

        else:
            minimum = p01 = p05 = median = None
            p95 = p99 = maximum = None
            mean = std = None

        band_no = band_index + 1

        tags = src.tags(band_no)

        band_records.append({
            "band": band_no,
            "valid_pixels": int(mask.sum()),
            "total_pixels": int(mask.size),
            "valid_fraction": float(mask.mean()),
            "nodata_or_masked_fraction": float(
                (~mask).mean()
            ),
            "minimum": minimum,
            "p01": p01,
            "p05": p05,
            "median": median,
            "mean": mean,
            "std": std,
            "p95": p95,
            "p99": p99,
            "maximum": maximum
        })

        band_metadata.append({
            "band": band_no,
            "description": (
                src.descriptions[band_index] or ""
            ),
            "unit": src.units[band_index] or "",
            "scale": float(src.scales[band_index]),
            "offset": float(src.offsets[band_index]),
            "tags_json": json.dumps(
                tags, sort_keys=True
            )
        })

band_df = pd.DataFrame(band_records)
metadata_df = pd.DataFrame(band_metadata)

band_df.to_csv(
    OUT / "Step28_Band_Validity_Statistics.csv",
    index=False
)

metadata_df.to_csv(
    OUT / "Step28_Band_Metadata.csv",
    index=False
)

print("\nPer-band valid fraction:")
print(
    band_df["valid_fraction"]
    .describe()
    .round(5)
    .to_string()
)

print("\nBands with lowest coverage:")
print(
    band_df.nsmallest(
        12, "valid_fraction"
    )[
        ["band", "valid_fraction", "median", "p99"]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 28C — IDENTIFY PIXELS VALID ACROSS ALL 224 BANDS
# -------------------------------------------------------------------------

print("\n28C — SPATIAL VALIDITY")

strict_pixel_valid = valid.all(axis=0)

# Separate diagnostic: a pixel with at least 95% valid
# bands might be usable after justified band filtering.
band_fraction_per_pixel = valid.mean(axis=0)
mostly_valid = band_fraction_per_pixel >= 0.95

strict_fraction = float(strict_pixel_valid.mean())
mostly_fraction = float(mostly_valid.mean())

print(
    "Pixels valid in ALL 224 bands:",
    f"{strict_fraction * 100:.3f}%"
)

print(
    "Pixels valid in >=95% of bands:",
    f"{mostly_fraction * 100:.3f}%"
)

# -------------------------------------------------------------------------
# 28D — SCREEN CANDIDATE 96x96 PATCHES
#
# Non-overlapping default positions are preferred.
# Additional candidate windows are scored but not
# treated as independent observations.
# -------------------------------------------------------------------------

print("\n28D — CANDIDATE 96x96 WINDOWS")

height = cube.shape[1]
width = cube.shape[2]

row_positions = sorted(set([
    0,
    max(0, (height-PATCH)//2),
    height-PATCH
]))

col_positions = sorted(set([
    0,
    max(0, (width-PATCH)//3),
    max(0, 2*(width-PATCH)//3),
    width-PATCH
]))

window_rows = []

for row in row_positions:
    for col in col_positions:

        sub_strict = strict_pixel_valid[
            row:row+PATCH,
            col:col+PATCH
        ]

        sub_mostly = mostly_valid[
            row:row+PATCH,
            col:col+PATCH
        ]

        # Require no nodata in any of 224 channels for
        # the strict, unfiltered experimental option.
        strict_coverage = float(sub_strict.mean())
        mostly_coverage = float(sub_mostly.mean())

        bounds = rasterio.windows.bounds(
            Window(col, row, PATCH, PATCH),
            transform
        )

        window_rows.append({
            "patch_id": f"potsdam_r{row}_c{col}",
            "row": row,
            "col": col,
            "height": PATCH,
            "width": PATCH,
            "strict_all_band_valid_fraction": strict_coverage,
            "at_least_95pct_bands_valid_fraction": mostly_coverage,
            "strict_eligible": bool(
                strict_coverage >= MIN_PIXEL_COVERAGE
            ),
            "left": float(bounds[0]),
            "bottom": float(bounds[1]),
            "right": float(bounds[2]),
            "top": float(bounds[3]),
            "crs": "EPSG:32633"
        })

windows_df = pd.DataFrame(window_rows)

windows_df.to_csv(
    OUT / "Step28_Candidate_Patch_Windows.csv",
    index=False
)

print(
    windows_df[
        [
            "patch_id",
            "strict_all_band_valid_fraction",
            "at_least_95pct_bands_valid_fraction",
            "strict_eligible"
        ]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 28E — RECORD RADIOMETRIC METADATA STATUS
# -------------------------------------------------------------------------

possible_spectral_metadata = []

for key, value in image_info["dataset_tags"].items():

    if any(
        word in key.lower()
        for word in [
            "wavelength", "reflectance", "scale",
            "unit", "fwhm", "radiance"
        ]
    ):
        possible_spectral_metadata.append({
            "scope": "dataset",
            "key": key,
            "value": str(value)[:300]
        })

for entry in band_metadata:

    tag_text = entry["tags_json"]

    if any(
        token in tag_text.lower()
        for token in ["wavelength", "fwhm", "reflectance"]
    ):
        possible_spectral_metadata.append({
            "scope": f"band_{entry['band']}",
            "key": "band_tags",
            "value": tag_text[:300]
        })

# No guessed wavelength values.
image_info["spectral_metadata_candidates"] = (
    possible_spectral_metadata[:30]
)

(OUT / "Step28_Scene_Metadata.json").write_text(
    json.dumps(image_info, indent=2, default=str),
    encoding="utf-8"
)

# -------------------------------------------------------------------------
# 28F — FINAL REPORT
# -------------------------------------------------------------------------

eligible_count = int(
    windows_df["strict_eligible"].sum()
)

report = {
    "experiment": "external EnMAP valid-pixel and radiometric audit",
    "input_sha256": actual_sha,
    "sensor": "EnMAP",
    "study_area": "Potsdam, Germany",
    "band_count": 224,
    "image_shape": [200, 330, 224],
    "crs": "EPSG:32633",
    "declared_nodata": nodata,
    "strict_valid_fraction_all_224_bands": strict_fraction,
    "fraction_pixels_with_at_least_95pct_valid_bands":
        mostly_fraction,
    "minimum_band_valid_fraction": float(
        band_df["valid_fraction"].min()
    ),
    "median_band_valid_fraction": float(
        band_df["valid_fraction"].median()
    ),
    "candidate_windows": len(windows_df),
    "strict_eligible_windows": eligible_count,
    "candidate_window_shape": [PATCH, PATCH],
    "wavelength_metadata_candidates_found": len(
        possible_spectral_metadata
    ),
    "radiometric_scaling_verified": False,
    "synthetic_msi_created": False,
    "external_registration_evaluated": False,
    "mdas_checkpoint_transfer_evaluated": False,
    "status": (
        "RADIOMETRIC_REVIEW_AND_PROTOCOL_REQUIRED"
        if eligible_count > 0
        else "NO_STRICTLY_VALID_96PX_WINDOWS"
    ),
    "next_requirement": (
        "Verify wavelength metadata and radiometric scale; "
        "then predefine a synthetic multispectral observation "
        "operator and fixed registration hyperparameters "
        "before examining external registration errors."
    ),
    "limitations": [
        "The numeric nodata sentinel is excluded explicitly.",
        "Finite values are not automatically valid observations.",
        "Overlapping candidate windows are not independent scenes.",
        "Synthetic bands do not reproduce a real multispectral sensor.",
        "The pretrained MDAS fusion model is not compatible with 224 bands."
    ],
    "output_directory": str(OUT)
}

(OUT / "Step28_Final_Report.json").write_text(
    json.dumps(report, indent=2, default=str),
    encoding="utf-8"
)

print("\n" + "=" * 86)
print("STEP 28 — FINAL REPORT")
print("=" * 86)
print(json.dumps(report, indent=2, default=str))
print("\nSTEP 28 COMPLETE")
print("Source EnMAP imagery and all MDAS files unchanged.")


In [ ]:

# =============================================================================
# STEP 29 — CORRECTED ENMAP BAND AND PATCH VALIDITY AUDIT
#
# Source: Verified Potsdam EnMAP scene acquired in Step 27
#
# Purpose:
#   1. Identify unusable spectral bands without guessing their indices.
#   2. Exclude bands with <95% valid coverage over the full scene.
#   3. Verify strict pixel validity across retained bands.
#   4. Find fully valid, non-overlapping 96x96 patches.
#   5. Inspect wavelength and radiometric metadata.
#   6. Freeze the external pilot preprocessing protocol.
#
# DOES NOT:
#   - Modify the source TIFF
#   - Estimate registration displacement
#   - Tune algorithms using the external image
#   - Test MDAS checkpoints on incompatible bands
#   - Claim independent geographic replication
# =============================================================================

from google.colab import drive
from pathlib import Path
from rasterio.windows import Window
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hashlib
import json
import re

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

IMAGE = (
    PROJECT
    / "Step27_External_EnMAP_Pilot"
    / "Imagery"
    / "enmap_potsdam.tif"
)

PREVIOUS_REPORT = (
    PROJECT
    / "Step28_EnMAP_Radiometric_Audit"
    / "Step28_Final_Report.json"
)

OUT = PROJECT / "Step29_Corrected_EnMAP_Screening"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA256 = (
    "4ba14fd27ac80c9169f3d733010dbdaa986584b3a062dc497578c3e114733329"
)

# Defined before reading data.
MIN_BAND_VALID_FRACTION = 0.95
MIN_PATCH_VALID_FRACTION = 1.00
PATCH_SIZE = 96

OUTPUT_FILES = [
    "Step29_Band_Screening.csv",
    "Step29_Nonoverlapping_Patch_Audit.csv",
    "Step29_Original_Candidate_Window_Audit.csv",
    "Step29_Selected_Pilot_Patches.csv",
    "Step29_Radiometric_Metadata.json",
    "Step29_Frozen_External_Protocol.json",
    "Step29_Final_Report.json",
    "Figure01_EnMAP_Band_Validity.png",
    "Figure01_EnMAP_Band_Validity.pdf",
    "Figure02_EnMAP_Patch_Validity.png",
    "Figure02_EnMAP_Patch_Validity.pdf"
]

existing = [
    name for name in OUTPUT_FILES
    if (OUT / name).exists()
]

if existing:
    raise FileExistsError(
        "Step 29 outputs already exist. "
        "To protect reproducibility, use a new "
        "output directory for a fresh run.\n"
        + "\n".join(existing)
    )

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

print("=" * 88)
print("STEP 29 — CORRECTED ENMAP VALIDITY SCREENING")
print("=" * 88)

assert IMAGE.is_file(), IMAGE
assert PREVIOUS_REPORT.is_file(), PREVIOUS_REPORT

source_sha = sha256(IMAGE)
assert source_sha == EXPECTED_SHA256

previous = json.loads(
    PREVIOUS_REPORT.read_text(encoding="utf-8")
)

assert previous["input_sha256"] == source_sha

print("\n29A — SOURCE CHECK")
print("Image:", IMAGE)
print("Verified SHA256:", source_sha)

# -------------------------------------------------------------------------
# 29B — READ SOURCE CUBE AND VALIDITY MASK
# -------------------------------------------------------------------------

with rasterio.open(IMAGE) as src:

    assert (src.count, src.height, src.width) == (
        224, 200, 330
    )

    assert str(src.crs) == "EPSG:32633"
    assert not src.transform.is_identity

    nodata = src.nodata
    transform = src.transform

    cube = src.read().astype(np.float32)

    mask = src.read_masks() > 0

    valid = mask & np.isfinite(cube)

    if nodata is not None and np.isfinite(nodata):
        valid &= cube != nodata

    tags = src.tags()

    descriptions = list(src.descriptions)
    units = list(src.units)
    scales = list(src.scales)
    offsets = list(src.offsets)

    all_band_tags = [
        src.tags(i)
        for i in range(1, src.count + 1)
    ]

assert cube.shape == (224, 200, 330)
assert valid.shape == cube.shape

# -------------------------------------------------------------------------
# 29C — SPECTRAL BAND SCREENING
# -------------------------------------------------------------------------

print("\n29B — BAND SCREENING")

band_valid_fraction = valid.mean(axis=(1, 2))

retained_zero_based = np.where(
    band_valid_fraction >= MIN_BAND_VALID_FRACTION
)[0]

removed_zero_based = np.where(
    band_valid_fraction < MIN_BAND_VALID_FRACTION
)[0]

retained_bands = (
    retained_zero_based + 1
).astype(int).tolist()

removed_bands = (
    removed_zero_based + 1
).astype(int).tolist()

band_table = pd.DataFrame({
    "band_1_based": np.arange(1, 225),
    "valid_fraction": band_valid_fraction,
    "retained": (
        band_valid_fraction >= MIN_BAND_VALID_FRACTION
    )
})

band_table.to_csv(
    OUT / "Step29_Band_Screening.csv",
    index=False
)

print("Original bands:", cube.shape[0])
print("Retained bands:", len(retained_bands))
print("Removed bands:", len(removed_bands))
print("Removed band IDs:", removed_bands)
print(
    "Retained band coverage range:",
    round(float(
        band_valid_fraction[retained_zero_based].min()
    ), 6),
    "to",
    round(float(
        band_valid_fraction[retained_zero_based].max()
    ), 6)
)

if len(retained_bands) < 100:
    raise RuntimeError(
        "Too few usable spectral bands. "
        "External preprocessing should not proceed."
    )

# Make no assumption that only six bands are unusable.
# All other decisions follow the explicit 95% rule.
retained_valid = valid[retained_zero_based]

strict_pixel_valid = retained_valid.all(axis=0)

strict_scene_fraction = float(
    strict_pixel_valid.mean()
)

print(
    "\nPixels valid across all retained bands:",
    f"{100 * strict_scene_fraction:.4f}%"
)

# -------------------------------------------------------------------------
# 29D — NON-OVERLAPPING PATCH SCREENING
#
# Fixed grid, with no overlap between patch interiors.
# All selected patches remain part of one geographic scene.
# -------------------------------------------------------------------------

print("\n29C — NON-OVERLAPPING 96x96 PATCH SCREENING")

height, width = strict_pixel_valid.shape

rows = list(range(
    0, height - PATCH_SIZE + 1, PATCH_SIZE
))

cols = list(range(
    0, width - PATCH_SIZE + 1, PATCH_SIZE
))

patch_rows = []

for row in rows:
    for col in cols:

        patch_valid = strict_pixel_valid[
            row:row + PATCH_SIZE,
            col:col + PATCH_SIZE
        ]

        fraction = float(patch_valid.mean())

        bounds = rasterio.windows.bounds(
            Window(
                col,
                row,
                PATCH_SIZE,
                PATCH_SIZE
            ),
            transform
        )

        patch_rows.append({
            "patch_id": f"potsdam_r{row}_c{col}",
            "row": int(row),
            "col": int(col),
            "height": PATCH_SIZE,
            "width": PATCH_SIZE,
            "strict_valid_fraction": fraction,
            "eligible": bool(
                fraction >= MIN_PATCH_VALID_FRACTION
            ),
            "left": float(bounds[0]),
            "bottom": float(bounds[1]),
            "right": float(bounds[2]),
            "top": float(bounds[3]),
            "crs": "EPSG:32633"
        })

patches = pd.DataFrame(patch_rows)

patches.to_csv(
    OUT / "Step29_Nonoverlapping_Patch_Audit.csv",
    index=False
)

selected = patches[
    patches["eligible"]
].copy().reset_index(drop=True)

selected.to_csv(
    OUT / "Step29_Selected_Pilot_Patches.csv",
    index=False
)

print(patches.round(5).to_string(index=False))
print(
    "\nFully valid non-overlapping patches:",
    len(selected)
)

# -------------------------------------------------------------------------
# 29E — RE-AUDIT THE 12 PREVIOUS CANDIDATE WINDOWS
#
# These overlap and are diagnostic only.
# They cannot be counted as independent test scenes.
# -------------------------------------------------------------------------

original_positions = [
    (0, 0), (0, 78), (0, 156), (0, 234),
    (52, 0), (52, 78), (52, 156), (52, 234),
    (104, 0), (104, 78), (104, 156), (104, 234)
]

original_rows = []

for row, col in original_positions:

    mask_patch = strict_pixel_valid[
        row:row + PATCH_SIZE,
        col:col + PATCH_SIZE
    ]

    fraction = float(mask_patch.mean())

    original_rows.append({
        "patch_id": f"potsdam_r{row}_c{col}",
        "row": row,
        "col": col,
        "strict_valid_fraction_after_band_screening":
            fraction,
        "fully_valid": bool(fraction == 1.0),
        "overlaps_other_candidates": True
    })

original_audit = pd.DataFrame(original_rows)

original_audit.to_csv(
    OUT / "Step29_Original_Candidate_Window_Audit.csv",
    index=False
)

print("\n29D — CORRECTED ORIGINAL CANDIDATE AUDIT")

print(
    original_audit.round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 29F — RADIOMETRIC METADATA AUDIT
#
# Record native metadata. Do not guess physical units.
# -------------------------------------------------------------------------

print("\n29E — RADIOMETRIC METADATA AUDIT")

metadata = {
    "source_sha256": source_sha,
    "source_crs": "EPSG:32633",
    "original_band_count": 224,
    "retained_band_count": len(retained_bands),
    "removed_band_ids_1_based": removed_bands,
    "retained_band_ids_1_based": retained_bands,
    "nodata": (
        float(nodata)
        if nodata is not None else None
    ),
    "dataset_tags": tags,
    "retained_band_metadata": []
}

for idx in retained_zero_based:

    record = {
        "band_1_based": int(idx + 1),
        "description": descriptions[idx],
        "unit": units[idx],
        "scale": float(scales[idx]),
        "offset": float(offsets[idx]),
        "tags": all_band_tags[idx]
    }

    metadata["retained_band_metadata"].append(record)

# Print representative tags without claiming the
# availability of actual wavelength values.
for original_band in [1, 10, 50, 100, 129, 136, 180, 224]:

    idx = original_band - 1

    print(f"\nBand {original_band}:")
    print("  Description:", descriptions[idx])
    print("  Unit:", units[idx])
    print("  Scale:", scales[idx])
    print("  Offset:", offsets[idx])
    print(
        "  Tags:",
        json.dumps(
            all_band_tags[idx],
            ensure_ascii=False
        )[:500]
    )

(OUT / "Step29_Radiometric_Metadata.json").write_text(
    json.dumps(metadata, indent=2, default=str),
    encoding="utf-8"
)

# -------------------------------------------------------------------------
# 29G — QUALITY-CONTROL FIGURES
# -------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white"
})

fig, ax = plt.subplots(figsize=(10, 3.7))

ax.plot(
    np.arange(1, 225),
    band_valid_fraction,
    color="#146C94",
    linewidth=1.6
)

ax.axhline(
    MIN_BAND_VALID_FRACTION,
    color="#BA4833",
    linestyle="--",
    linewidth=1.0,
    label="Band retention threshold"
)

ax.set_xlabel("EnMAP spectral band")
ax.set_ylabel("Valid-pixel fraction")
ax.set_xlim(1, 224)
ax.set_ylim(-0.04, 1.05)
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False)

fig.tight_layout()

fig.savefig(
    OUT / "Figure01_EnMAP_Band_Validity.png",
    dpi=1200,
    bbox_inches="tight"
)

fig.savefig(
    OUT / "Figure01_EnMAP_Band_Validity.pdf",
    bbox_inches="tight"
)

plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))

im = ax.imshow(
    strict_pixel_valid.astype(np.uint8),
    cmap="gray",
    vmin=0,
    vmax=1,
    interpolation="nearest"
)

for record in patch_rows:

    from matplotlib.patches import Rectangle

    color = (
        "#2A9D8F"
        if record["eligible"]
        else "#E76F51"
    )

    ax.add_patch(
        Rectangle(
            (record["col"] - 0.5,
             record["row"] - 0.5),
            PATCH_SIZE,
            PATCH_SIZE,
            fill=False,
            edgecolor=color,
            linewidth=1.6
        )
    )

    ax.text(
        record["col"] + 4,
        record["row"] + 13,
        f"{record['strict_valid_fraction']:.3f}",
        fontsize=8,
        color=color,
        weight="bold",
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.82,
            "pad": 1
        }
    )

ax.set_xlabel("Image column")
ax.set_ylabel("Image row")
ax.set_xlim(-0.5, width - 0.5)
ax.set_ylim(height - 0.5, -0.5)

fig.tight_layout()

fig.savefig(
    OUT / "Figure02_EnMAP_Patch_Validity.png",
    dpi=1200,
    bbox_inches="tight"
)

fig.savefig(
    OUT / "Figure02_EnMAP_Patch_Validity.pdf",
    bbox_inches="tight"
)

plt.close(fig)

# -------------------------------------------------------------------------
# 29H — EXTERNAL PILOT PROTOCOL
#
# Freeze preprocessing and test design before
# observing registration errors.
# -------------------------------------------------------------------------

protocol = {
    "experiment": "independent Potsdam EnMAP registration pilot",
    "protocol_version": "1.0-pre-evaluation",
    "source_image_sha256": source_sha,
    "source_dataset": "EnMAP-Box Potsdam example",
    "source_scene_count": 1,
    "source_crs": "EPSG:32633",
    "native_hsi_bands": 224,
    "spectral_retention_rule": (
        "Retain bands with >=0.95 valid-pixel fraction "
        "over the complete source image."
    ),
    "retained_band_ids_1_based": retained_bands,
    "removed_band_ids_1_based": removed_bands,
    "patch_selection_rule": (
        "Fixed non-overlapping 96x96 grid; "
        "select patches with validity fraction 1.0 "
        "across all retained bands."
    ),
    "selected_patch_ids": (
        selected["patch_id"].to_list()
    ),
    "registration_shift_conditions_pixels": [
        [0.0, 0.0],
        [0.0, -0.5], [0.0, 0.5],
        [-0.5, 0.0], [0.5, 0.0],
        [0.0, -1.5], [0.0, 1.5],
        [-1.5, 0.0], [1.5, 0.0],
        [0.0, -3.5], [0.0, 3.5],
        [-3.5, 0.0], [3.5, 0.0],
        [-1.5, -1.5], [-1.5, 1.5],
        [1.5, -1.5], [1.5, 1.5]
    ],
    "shift_convention": (
        "Positive dy shifts the image downward; "
        "positive dx shifts the image rightward."
    ),
    "registration_settings_rule": (
        "Use a predetermined registration configuration. "
        "Do not optimize thresholds, bias correction, "
        "or displacement search using this scene."
    ),
    "synthetic_msi_operator": (
        "NOT YET DEFINED: requires spectral metadata "
        "and an explicitly justified forward model."
    ),
    "radiometric_scale": (
        "NOT YET VERIFIED: native units retained "
        "for validity screening only."
    ),
    "external_scene_independence_note": (
        "Potsdam is geographically separate from Augsburg, "
        "but all selected patches share one source acquisition."
    ),
    "evaluation_status": "not_started",
    "pretrained_mdas_fusion_model_applicable": False
}

(OUT / "Step29_Frozen_External_Protocol.json").write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

# -------------------------------------------------------------------------
# 29I — FINAL REPORT
# -------------------------------------------------------------------------

if len(selected) > 0:

    status = "VALID_EXTERNAL_PATCHES_IDENTIFIED"

else:

    status = "NO_FULLY_VALID_NONOVERLAPPING_PATCHES"

report = {
    "experiment": "corrected EnMAP band and patch screening",
    "source_sha256": source_sha,
    "original_bands": 224,
    "retained_bands": len(retained_bands),
    "removed_bands_1_based": removed_bands,
    "retained_band_minimum_valid_fraction":
        float(
            band_valid_fraction[retained_zero_based].min()
        ),
    "strict_scene_valid_fraction_after_screening":
        strict_scene_fraction,
    "candidate_nonoverlapping_windows": len(patches),
    "fully_valid_nonoverlapping_windows": len(selected),
    "selected_patch_ids": selected["patch_id"].to_list(),
    "original_candidate_windows_reaudited":
        len(original_audit),
    "original_candidate_windows_fully_valid":
        int(original_audit["fully_valid"].sum()),
    "radiometric_units_verified": False,
    "synthetic_msi_generated": False,
    "external_registration_evaluated": False,
    "mdas_fusion_checkpoint_transfer_evaluated": False,
    "status": status,
    "next_requirement": (
        "Review the printed spectral and radiometric metadata, "
        "define a justified fixed synthetic MSI operator, "
        "and evaluate pretrained/frozen registration settings "
        "without tuning on the Potsdam scene."
    ),
    "limitations": [
        "Spectral band filtering is explicitly documented.",
        "Selected patches share a single geographic scene.",
        "Candidate patches must not be counted as "
        "independent source acquisitions.",
        "No registration accuracy is reported in this audit.",
        "EnMAP bands are not compatible with the "
        "original 242-band MDAS checkpoints."
    ],
    "output_directory": str(OUT)
}

(OUT / "Step29_Final_Report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 29 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 29 COMPLETE")
print("Original source imagery and MDAS checkpoints unchanged.")


In [ ]:

# =============================================================================
# STEP 30 — FROZEN EXTERNAL POTSDAM HSI–MSI BENCHMARK GENERATION
#
# SOURCE:
#   Verified EnMAP Potsdam image from Step 27
#   Selected non-overlapping valid patches from Step 29
#
# OUTPUT:
#   Five paired benchmark NPZ files
#   Seventeen synthetic MSI shifts per patch
#   Spectral response matrix
#   Patch and shift manifests
#   Frozen forward-model protocol
#   Integrity checks and final report
#
# IMPORTANT:
#   - This is a synthetic paired-observation benchmark.
#   - It is NOT a real co-acquired HSI–MSI dataset.
#   - The source acquisition count is ONE.
#   - The MDAS pretrained fusion checkpoints are incompatible.
#   - No registration algorithm is run or tuned here.
# =============================================================================

from google.colab import drive
from pathlib import Path
from scipy.ndimage import gaussian_filter, shift as ndi_shift
import rasterio
import numpy as np
import pandas as pd
import hashlib
import json
import time

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

SOURCE = (
    PROJECT / "Step27_External_EnMAP_Pilot"
    / "Imagery" / "enmap_potsdam.tif"
)

STEP29 = PROJECT / "Step29_Corrected_EnMAP_Screening"

SELECTION = STEP29 / "Step29_Selected_Pilot_Patches.csv"
SCREENING = STEP29 / "Step29_Band_Screening.csv"
PREVIOUS = STEP29 / "Step29_Final_Report.json"

OUT = PROJECT / "Step30_Frozen_Potsdam_Benchmark"
DATA_OUT = OUT / "Paired_Observations"

DATA_OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = (
    "4ba14fd27ac80c9169f3d733010dbdaa986584b3a062dc497578c3e114733329"
)

# -------------------------------------------------------------------------
# FROZEN OBSERVATION MODEL — SET BEFORE READING SCENE CONTENT
# -------------------------------------------------------------------------

PATCH_SIZE = 96
SPATIAL_FACTOR = 3

# Spatial degradation in native EnMAP pixel coordinates.
BLUR_SIGMA_PX = 1.0

# Gaussian spectral-response approximation.
# Values are modelling assumptions, not calibrated sensor SRFs.
MSI_CENTERS_NM = [490.0, 560.0, 665.0, 842.0]
MSI_FWHM_NM = [60.0, 45.0, 35.0, 115.0]

# Same shift set as the original MDAS robustness design.
OFFSETS = [
    (0.0, 0.0),
    (0.0, -0.5), (0.0, 0.5),
    (-0.5, 0.0), (0.5, 0.0),
    (0.0, -1.5), (0.0, 1.5),
    (-1.5, 0.0), (1.5, 0.0),
    (0.0, -3.5), (0.0, 3.5),
    (-3.5, 0.0), (3.5, 0.0),
    (-1.5, -1.5), (-1.5, 1.5),
    (1.5, -1.5), (1.5, 1.5)
]

# Native image is scaled using its TIFF metadata.
# Do not clip valid negative values silently.
REQUIRE_FULL_PATCH_VALIDITY = True

print("=" * 88, flush=True)
print("STEP 30 — FROZEN EXTERNAL REGISTRATION BENCHMARK", flush=True)
print("=" * 88, flush=True)

started = time.monotonic()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

def write_json_once(path, obj):
    """
    Preserve previous outputs, including partial runs.
    A matching existing file is accepted.
    """
    encoded = json.dumps(
        obj, indent=2, sort_keys=True
    ).encode("utf-8")

    if path.exists():
        if path.read_bytes() != encoded:
            raise FileExistsError(
                f"Existing file differs: {path}"
            )
    else:
        path.write_bytes(encoded)

def write_csv_once(path, frame):
    content = frame.to_csv(index=False)

    if path.exists():
        if path.read_text(encoding="utf-8") != content:
            raise FileExistsError(
                f"Existing CSV differs: {path}"
            )
    else:
        path.write_text(
            content,
            encoding="utf-8"
        )

# -------------------------------------------------------------------------
# 30A — VERIFY PREVIOUS DATA AND SELECTION
# -------------------------------------------------------------------------

for path in [SOURCE, SELECTION, SCREENING, PREVIOUS]:
    if not path.is_file():
        raise FileNotFoundError(path)

source_sha = sha256(SOURCE)

assert source_sha == EXPECTED_SHA

previous = json.loads(
    PREVIOUS.read_text(encoding="utf-8")
)

assert previous["source_sha256"] == source_sha
assert previous["external_registration_evaluated"] is False

selection = pd.read_csv(SELECTION)
screening = pd.read_csv(SCREENING)

assert len(selection) == 5
assert selection["patch_id"].is_unique
assert selection["eligible"].astype(bool).all()

retained = screening.loc[
    screening["retained"].astype(str).str.lower() == "true",
    "band_1_based"
].to_numpy(dtype=int)

removed = screening.loc[
    screening["retained"].astype(str).str.lower() != "true",
    "band_1_based"
].to_numpy(dtype=int)

assert len(retained) == 218
assert removed.tolist() == [130, 131, 132, 133, 134, 135]

print("\n30A — SOURCE AND PATCH CHECK")
print("SHA256:", source_sha)
print("Retained bands:", len(retained))
print("Selected patches:", len(selection))
print("Synthetic shift conditions:", len(OFFSETS))

# -------------------------------------------------------------------------
# 30B — READ WAVELENGTH METADATA AND BUILD FIXED RESPONSES
# -------------------------------------------------------------------------

with rasterio.open(SOURCE) as src:

    assert src.count == 224
    assert (src.height, src.width) == (200, 330)
    assert str(src.crs) == "EPSG:32633"

    raw_wavelengths = []

    for band_id in retained:

        tags = src.tags(int(band_id))

        if "wavelength" not in tags:
            raise RuntimeError(
                f"Missing wavelength for band {band_id}"
            )

        unit = tags.get(
            "wavelength_units", ""
        ).lower()

        if "nanometer" not in unit:
            raise RuntimeError(
                f"Unexpected wavelength unit: {unit}"
            )

        raw_wavelengths.append(
            float(tags["wavelength"])
        )

    wavelengths = np.asarray(
        raw_wavelengths, dtype=np.float64
    )

    scales = np.asarray(
        src.scales,
        dtype=np.float64
    )[retained - 1]

    offsets = np.asarray(
        src.offsets,
        dtype=np.float64
    )[retained - 1]

    assert np.all(np.isfinite(wavelengths))
    assert np.all(np.diff(wavelengths) > 0)
    assert np.all(np.isfinite(scales))
    assert np.all(scales > 0)
    assert np.all(np.isfinite(offsets))

    cube = src.read(
        indexes=retained.tolist()
    ).astype(np.float32)

    valid_mask = (
        src.read_masks(
            indexes=retained.tolist()
        ) > 0
    )

    valid_mask &= np.isfinite(cube)

    if src.nodata is not None:
        valid_mask &= cube != src.nodata

    # Per-band TIFF scaling.
    cube = (
        cube.astype(np.float64)
        * scales[:, None, None]
        + offsets[:, None, None]
    ).astype(np.float32)

    transform = src.transform

assert cube.shape == (218, 200, 330)
assert np.isfinite(cube[valid_mask]).all()

# Four normalized Gaussian spectral response functions.
response_rows = []

for center, fwhm in zip(
    MSI_CENTERS_NM,
    MSI_FWHM_NM
):

    sigma_nm = fwhm / (
        2.0 * np.sqrt(2.0 * np.log(2.0))
    )

    response = np.exp(
        -0.5 * (
            (wavelengths - center) / sigma_nm
        ) ** 2
    )

    if response.max() < 1e-6:
        raise RuntimeError(
            f"No meaningful coverage around {center} nm."
        )

    response /= response.sum()

    response_rows.append(response)

# Shape: 4 MSI bands × 218 retained HSI bands.
RESPONSE = np.stack(
    response_rows,
    axis=0
).astype(np.float32)

assert RESPONSE.shape == (4, 218)
assert np.allclose(
    RESPONSE.sum(axis=1),
    1.0,
    atol=1e-6
)

response_table = pd.DataFrame({
    "original_hsi_band": retained,
    "wavelength_nm": wavelengths,
    "synthetic_blue_weight": RESPONSE[0],
    "synthetic_green_weight": RESPONSE[1],
    "synthetic_red_weight": RESPONSE[2],
    "synthetic_nir_weight": RESPONSE[3]
})

write_csv_once(
    OUT / "Step30_Spectral_Response_Matrix.csv",
    response_table
)

# -------------------------------------------------------------------------
# 30C — FREEZE FORWARD MODEL BEFORE GENERATING DATA
# -------------------------------------------------------------------------

protocol = {
    "experiment": "Potsdam synthetic HSI-MSI registration benchmark",
    "protocol_version": "1.0",
    "source_image_sha256": source_sha,
    "source_scene_count": 1,
    "native_sensor": "EnMAP",
    "source_crs": "EPSG:32633",
    "selected_patch_ids": sorted(
        selection["patch_id"].tolist()
    ),
    "patch_shape": [96, 96],
    "retained_hsi_bands": retained.tolist(),
    "excluded_hsi_bands": removed.tolist(),
    "tiff_band_scales_applied": True,
    "radiometric_value_clipping": False,
    "synthetic_msi_band_centers_nm": MSI_CENTERS_NM,
    "synthetic_msi_band_fwhm_nm": MSI_FWHM_NM,
    "spectral_response_kind": "normalized Gaussian surrogate",
    "spatial_degradation": {
        "method": "Gaussian blur, then decimation",
        "gaussian_sigma_native_pixels": BLUR_SIGMA_PX,
        "boundary_mode": "reflect",
        "downsample_factor": SPATIAL_FACTOR,
        "sample_origin_native_pixels": [1, 1]
    },
    "synthetic_shift_operator": {
        "method": "scipy.ndimage.shift",
        "interpolation_order": 1,
        "boundary_mode": "nearest",
        "prefilter": False,
        "positive_dy": "down",
        "positive_dx": "right"
    },
    "synthetic_offsets_dy_dx": [
        [float(dy), float(dx)]
        for dy, dx in OFFSETS
    ],
    "registration_algorithms_evaluated": False,
    "registration_hyperparameters_tuned": False,
    "reference_hsi_available_for_diagnostics": True,
    "pretrained_mdas_fusion_checkpoints_compatible": False,
    "notes": [
        "MSI is derived mathematically from EnMAP HSI.",
        "Synthetic MSI is not an observed Sentinel-2 image.",
        "Patches belong to one source acquisition.",
        "Synthetic subpixel translations use interpolation.",
        "A fixed cross-sensor registration protocol is "
        "required before measuring errors."
    ]
}

write_json_once(
    OUT / "Step30_Frozen_Forward_Model.json",
    protocol
)

print("\n30B — FORWARD MODEL FROZEN")
print("HSI wavelengths:", len(wavelengths))
print("MSI channels:", RESPONSE.shape[0])
print("Downsampling factor:", SPATIAL_FACTOR)
print("Synthetic displacement conditions:", len(OFFSETS))

# -------------------------------------------------------------------------
# 30D — GENERATE PAIRED OBSERVATIONS
# -------------------------------------------------------------------------

manifest_rows = []
shift_rows = []

print("\n30C — GENERATING PAIRED OBSERVATIONS", flush=True)

for record in selection.sort_values(
    "patch_id"
).itertuples(index=False):

    patch_id = str(record.patch_id)
    row = int(record.row)
    col = int(record.col)

    hsi = cube[
        :,
        row:row + PATCH_SIZE,
        col:col + PATCH_SIZE
    ].copy()

    patch_valid = valid_mask[
        :,
        row:row + PATCH_SIZE,
        col:col + PATCH_SIZE
    ]

    assert hsi.shape == (218, 96, 96)
    assert patch_valid.shape == hsi.shape

    if REQUIRE_FULL_PATCH_VALIDITY:
        assert patch_valid.all(), (
            f"Unexpected nodata in selected patch {patch_id}"
        )

    assert np.isfinite(hsi).all()

    # LRHSI: spatial Gaussian blur and 3x decimation.
    # No spectral degradation is applied to LRHSI.
    blurred = gaussian_filter(
        hsi,
        sigma=(0.0, BLUR_SIGMA_PX, BLUR_SIGMA_PX),
        mode="reflect"
    )

    lrhsi = blurred[
        :,
        1::SPATIAL_FACTOR,
        1::SPATIAL_FACTOR
    ].astype(np.float32)

    assert lrhsi.shape == (218, 32, 32)

    # HRMSI: fixed linear spectral integration.
    hrmsi = np.einsum(
        "bc,chw->bhw",
        RESPONSE,
        hsi,
        optimize=True
    ).astype(np.float32)

    assert hrmsi.shape == (4, 96, 96)

    # Shift MSI observations only.
    shifted_list = []

    for condition_id, (dy, dx) in enumerate(OFFSETS):

        shifted = ndi_shift(
            hrmsi,
            shift=(0.0, float(dy), float(dx)),
            order=1,
            mode="nearest",
            prefilter=False
        ).astype(np.float32)

        assert shifted.shape == hrmsi.shape
        assert np.isfinite(shifted).all()

        if condition_id == 0:
            assert np.array_equal(
                shifted, hrmsi
            ), "Aligned condition was modified."

        shifted_list.append(shifted)

        shift_rows.append({
            "patch_id": patch_id,
            "condition_id": condition_id,
            "true_dy": float(dy),
            "true_dx": float(dx),
            "shift_magnitude": float(
                np.hypot(dy, dx)
            ),
            "is_aligned": bool(
                dy == 0 and dx == 0
            )
        })

    shifted_msi = np.stack(
        shifted_list,
        axis=0
    )

    assert shifted_msi.shape == (
        len(OFFSETS), 4, 96, 96
    )

    output_path = DATA_OUT / f"{patch_id}.npz"

    if output_path.exists():

        # Resume only if structural and numerical checks pass.
        with np.load(
            output_path,
            allow_pickle=False
        ) as existing:

            assert np.array_equal(
                existing["reference_hsi"], hsi
            )
            assert np.array_equal(
                existing["lrhsi"], lrhsi
            )
            assert np.array_equal(
                existing["hrmsi"], hrmsi
            )
            assert np.array_equal(
                existing["shifted_msi"], shifted_msi
            )

        print("Verified existing:", patch_id, flush=True)

    else:

        np.savez_compressed(
            output_path,
            reference_hsi=hsi,
            lrhsi=lrhsi,
            hrmsi=hrmsi,
            shifted_msi=shifted_msi,
            retained_band_ids=retained.astype(np.int16),
            wavelengths_nm=wavelengths.astype(np.float32),
            shift_offsets_dy_dx=np.asarray(
                OFFSETS,
                dtype=np.float32
            )
        )

        print("Saved:", patch_id, flush=True)

    manifest_rows.append({
        "patch_id": patch_id,
        "row": row,
        "col": col,
        "source_scene": "EnMAP Potsdam",
        "source_image_sha256": source_sha,
        "npz_path": str(output_path),
        "npz_sha256": sha256(output_path),
        "reference_shape": "(218, 96, 96)",
        "lrhsi_shape": "(218, 32, 32)",
        "hrmsi_shape": "(4, 96, 96)",
        "shifted_msi_shape": str(
            (len(OFFSETS), 4, 96, 96)
        ),
        "conditions": len(OFFSETS),
        "source_acquisition_count": 1
    })

manifest = pd.DataFrame(manifest_rows)
shift_manifest = pd.DataFrame(shift_rows)

assert len(manifest) == 5
assert len(shift_manifest) == 85

assert not shift_manifest.duplicated(
    ["patch_id", "condition_id"]
).any()

write_csv_once(
    OUT / "Step30_Paired_Observation_Manifest.csv",
    manifest
)

write_csv_once(
    OUT / "Step30_Shift_Conditions.csv",
    shift_manifest
)

# -------------------------------------------------------------------------
# 30E — QUALITY AND VALIDATION SUMMARY
# -------------------------------------------------------------------------

negative_fraction = float(
    (cube[valid_mask] < 0).mean()
)

finite_fraction = float(
    np.isfinite(cube[valid_mask]).mean()
)

report = {
    "experiment": "Potsdam synthetic paired-observation generation",
    "source_image_sha256": source_sha,
    "external_source_acquisitions": 1,
    "patches_generated": len(manifest),
    "conditions_per_patch": len(OFFSETS),
    "total_shift_cases": len(shift_manifest),
    "original_enmap_bands": 224,
    "retained_enmap_bands": 218,
    "simulated_msi_bands": 4,
    "reference_shape": [218, 96, 96],
    "lrhsi_shape": [218, 32, 32],
    "hrmsi_shape": [4, 96, 96],
    "input_image_hash_verified": True,
    "native_radiometric_scales_applied": True,
    "fraction_negative_valid_scaled_values":
        negative_fraction,
    "fraction_finite_valid_scaled_values":
        finite_fraction,
    "all_selected_patches_fully_valid": True,
    "aligned_msi_identity_check": True,
    "forward_model_frozen_before_registration": True,
    "registration_evaluated": False,
    "mdas_fusion_checkpoints_evaluated": False,
    "status": "EXTERNAL_SYNTHETIC_BENCHMARK_PREPARED",
    "limitations": [
        "Only one independent Potsdam acquisition.",
        "Observations use simulated spatial degradation.",
        "The MSI observation is synthetic, not co-acquired.",
        "Known synthetic translations are used.",
        "Patch results will be correlated.",
        "No direct transfer of incompatible MDAS checkpoints."
    ],
    "output_directory": str(OUT),
    "elapsed_seconds": round(
        time.monotonic() - started, 2
    )
}

write_json_once(
    OUT / "Step30_Final_Report.json",
    report
)

print("\n" + "=" * 88)
print("STEP 30 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))
print("\nSTEP 30 COMPLETE")
print("Original EnMAP image and MDAS checkpoints unchanged.")


In [ ]:

# =============================================================================
# STEP 30 — CORRECTED EXTERNAL ENMAP HSI–MSI BENCHMARK
# =============================================================================
#
# RESEARCH PROJECT:
# MDAS Registration-Aware Hyperspectral–Multispectral Fusion
#
# PURPOSE:
#   1. Verify the original EnMAP Potsdam source image.
#   2. Read the 218 spectral bands retained by Step 29.
#   3. Preserve native band ordering, including VNIR/SWIR transitions.
#   4. Apply the documented TIFF radiometric scale and offset.
#   5. Generate four synthetic MSI bands using fixed Gaussian
#      spectral-response functions.
#   6. Generate lower-resolution HSI observations using spatial
#      Gaussian degradation followed by 3x downsampling.
#   7. Generate 17 predefined shifted MSI observations.
#   8. Save five paired, non-overlapping patch datasets.
#   9. Verify all arrays, source hashes and output dimensions.
#  10. Save an auditable, frozen forward model and final report.
#
# THIS SCRIPT DOES NOT:
#   - Modify the original EnMAP image.
#   - Train or evaluate any registration or fusion model.
#   - Tune hyperparameters using the Potsdam scene.
#   - Claim that simulated MSI is a real multispectral acquisition.
#   - Claim direct generalization of 242-band MDAS checkpoints.
#
# IMPORTANT SCIENTIFIC DISTINCTION:
#   Potsdam is a separate external EnMAP acquisition, but all five
#   selected patches originate from ONE scene.
#
# FIX FROM PREVIOUS VERSION:
#   EnMAP wavelengths are not required to increase monotonically.
#   All spectral weights are constructed in ORIGINAL band order.
# =============================================================================

from google.colab import drive
from pathlib import Path

import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import rasterio

from scipy.ndimage import gaussian_filter, shift as ndi_shift


# =============================================================================
# 30A — MOUNT DRIVE AND DEFINE PATHS
# =============================================================================

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")

PROJECT = (
    ROOT
    / "MDAS_Registration_Aware_Revision_2026"
)

STEP27 = PROJECT / "Step27_External_EnMAP_Pilot"

STEP29 = PROJECT / "Step29_Corrected_EnMAP_Screening"

OUT = PROJECT / "Step30_Frozen_Potsdam_Benchmark"

DATA_OUT = OUT / "Paired_Observations"

DATA_OUT.mkdir(parents=True, exist_ok=True)

SOURCE = (
    STEP27
    / "Imagery"
    / "enmap_potsdam.tif"
)

SELECTION_FILE = (
    STEP29
    / "Step29_Selected_Pilot_Patches.csv"
)

SCREENING_FILE = (
    STEP29
    / "Step29_Band_Screening.csv"
)

STEP29_REPORT_FILE = (
    STEP29
    / "Step29_Final_Report.json"
)

EXPECTED_SOURCE_SHA256 = (
    "4ba14fd27ac80c9169f3d733010dbdaa986584b3a062dc497578c3e114733329"
)


# =============================================================================
# 30B — FROZEN EXPERIMENT CONFIGURATION
# =============================================================================

PATCH_SIZE = 96

SPATIAL_FACTOR = 3

SPATIAL_BLUR_SIGMA = 1.0

# Deliberately simplified Gaussian spectral responses.
# These are NOT official Sentinel-2 or EnMAP response functions.
#
# Center wavelengths are in nanometers.

MSI_CENTERS_NM = [
    490.0,
    560.0,
    665.0,
    842.0
]

MSI_FWHM_NM = [
    60.0,
    45.0,
    35.0,
    115.0
]

MSI_NAMES = [
    "synthetic_blue",
    "synthetic_green",
    "synthetic_red",
    "synthetic_nir"
]

# Same synthetic displacement design used in MDAS.
# Positive dy = downward image movement.
# Positive dx = rightward image movement.

OFFSETS = [
    (0.0, 0.0),

    (0.0, -0.5),
    (0.0, 0.5),

    (-0.5, 0.0),
    (0.5, 0.0),

    (0.0, -1.5),
    (0.0, 1.5),

    (-1.5, 0.0),
    (1.5, 0.0),

    (0.0, -3.5),
    (0.0, 3.5),

    (-3.5, 0.0),
    (3.5, 0.0),

    (-1.5, -1.5),
    (-1.5, 1.5),

    (1.5, -1.5),
    (1.5, 1.5)
]

EXPECTED_PATCHES = 5

EXPECTED_HSI_BANDS = 218

EXPECTED_SHIFT_CASES = (
    EXPECTED_PATCHES * len(OFFSETS)
)

# The TIFF supplies its own per-band scale factors.
# No additional clipping or normalization is imposed.
#
# The method is intentionally frozen before evaluating
# any registration results.

START = time.monotonic()

print("=" * 90, flush=True)
print(
    "STEP 30 — CORRECTED EXTERNAL ENMAP HSI–MSI BENCHMARK",
    flush=True
)
print("=" * 90, flush=True)

print("\nDevice: CPU (GPU not required)")
print("Project:", PROJECT)
print("Source image:", SOURCE)
print("Output directory:", OUT)
print("Expected patches:", EXPECTED_PATCHES)
print("Expected displacement conditions:", len(OFFSETS))
print("Expected paired cases:", EXPECTED_SHIFT_CASES)


# =============================================================================
# 30C — REPRODUCIBILITY HELPERS
# =============================================================================

def sha256_file(path):
    """
    Compute SHA-256 without loading the entire file into RAM.
    """

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


def save_json_verified(path, obj):
    """
    Write a JSON file only if it does not exist.

    If it exists, verify logical content equality.
    Do not overwrite previous experimental records.
    """

    path = Path(path)

    if path.exists():

        existing = json.loads(
            path.read_text(encoding="utf-8")
        )

        if existing != obj:

            raise FileExistsError(
                f"Existing JSON differs from this run:\n{path}\n"
                "Preserve the previous output or choose a new directory."
            )

        return "verified_existing"

    temporary = path.with_suffix(
        path.suffix + ".partial"
    )

    if temporary.exists():

        raise FileExistsError(
            f"Partial JSON file already exists: {temporary}"
        )

    encoded = json.dumps(
        obj,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False
    )

    with open(
        temporary,
        "x",
        encoding="utf-8"
    ) as handle:

        handle.write(encoded)

    os.replace(temporary, path)

    return "created"


def save_csv_verified(path, dataframe):
    """
    Preserve existing CSV files.

    Compare parsed values when rerunning so harmless
    differences in float formatting do not cause failure.
    """

    path = Path(path)

    if path.exists():

        previous = pd.read_csv(path)

        pd.testing.assert_frame_equal(
            previous.reset_index(drop=True),
            dataframe.reset_index(drop=True),
            check_dtype=False,
            check_exact=False,
            rtol=1e-10,
            atol=1e-12
        )

        return "verified_existing"

    temporary = path.with_suffix(
        path.suffix + ".partial"
    )

    if temporary.exists():

        raise FileExistsError(
            f"Partial CSV already exists: {temporary}"
        )

    dataframe.to_csv(
        temporary,
        index=False,
        float_format="%.12g"
    )

    os.replace(temporary, path)

    return "created"


def save_npz_verified(path, arrays):
    """
    Save one benchmark patch atomically.

    If a patch file already exists, compare all arrays.
    Do not regenerate a different benchmark silently.
    """

    path = Path(path)

    if path.exists():

        with np.load(
            path,
            allow_pickle=False
        ) as previous:

            existing_keys = set(previous.files)

            expected_keys = set(arrays.keys())

            if existing_keys != expected_keys:

                raise ValueError(
                    f"Saved NPZ key mismatch: {path}\n"
                    f"Existing: {sorted(existing_keys)}\n"
                    f"Expected: {sorted(expected_keys)}"
                )

            for key, current in arrays.items():

                old = previous[key]

                if not np.array_equal(
                    old,
                    current,
                    equal_nan=True
                ):

                    raise ValueError(
                        f"Saved NPZ data mismatch for '{key}': "
                        f"{path}"
                    )

        return "verified_existing"

    temporary = Path(
        str(path) + ".partial"
    )

    if temporary.exists():

        raise FileExistsError(
            "A partial benchmark file already exists:\n"
            f"{temporary}"
        )

    # Pass an open file handle to avoid NumPy appending
    # an extra .npz extension to the temporary filename.

    with open(temporary, "xb") as handle:

        np.savez_compressed(
            handle,
            **arrays
        )

    # Validate the written file before committing it.
    with np.load(
        temporary,
        allow_pickle=False
    ) as check:

        assert set(check.files) == set(arrays)

        for key, current in arrays.items():

            assert np.array_equal(
                check[key],
                current,
                equal_nan=True
            )

    os.replace(temporary, path)

    return "created"


# =============================================================================
# 30D — VERIFY ORIGINAL FILES AND STEP 29 PROVENANCE
# =============================================================================

print("\n" + "=" * 90)
print("30A — SOURCE AND PROVENANCE VERIFICATION")
print("=" * 90)

required_files = [
    SOURCE,
    SELECTION_FILE,
    SCREENING_FILE,
    STEP29_REPORT_FILE
]

for path in required_files:

    if not path.is_file():

        raise FileNotFoundError(
            f"Required research input missing:\n{path}"
        )

source_sha = sha256_file(SOURCE)

if source_sha != EXPECTED_SOURCE_SHA256:

    raise RuntimeError(
        "The source EnMAP image differs from the "
        "previously verified file.\n"
        f"Expected: {EXPECTED_SOURCE_SHA256}\n"
        f"Actual:   {source_sha}"
    )

step29_report = json.loads(
    STEP29_REPORT_FILE.read_text(
        encoding="utf-8"
    )
)

assert (
    step29_report["source_sha256"]
    == source_sha
)

assert (
    step29_report["retained_bands"]
    == EXPECTED_HSI_BANDS
)

assert (
    step29_report["fully_valid_nonoverlapping_windows"]
    == EXPECTED_PATCHES
)

selection = pd.read_csv(
    SELECTION_FILE
)

screening = pd.read_csv(
    SCREENING_FILE
)

assert len(selection) == EXPECTED_PATCHES
assert selection["patch_id"].is_unique

# Robust conversion, including cases where a CSV
# was read with boolean or string types.

selected_eligible = (
    selection["eligible"]
    .astype(str)
    .str.lower()
    .eq("true")
)

assert selected_eligible.all()

retained_mask = (
    screening["retained"]
    .astype(str)
    .str.lower()
    .eq("true")
)

retained_band_ids = (
    screening.loc[
        retained_mask,
        "band_1_based"
    ]
    .astype(int)
    .to_numpy()
)

removed_band_ids = (
    screening.loc[
        ~retained_mask,
        "band_1_based"
    ]
    .astype(int)
    .to_numpy()
)

assert len(retained_band_ids) == 218

assert removed_band_ids.tolist() == [
    130, 131, 132, 133, 134, 135
]

assert np.array_equal(
    retained_band_ids,
    np.sort(retained_band_ids)
)

assert (
    len(np.unique(retained_band_ids))
    == len(retained_band_ids)
)

print("Verified image SHA256:", source_sha)

print("Original spectral bands:", 224)

print(
    "Retained spectral bands:",
    len(retained_band_ids)
)

print(
    "Excluded spectral bands:",
    removed_band_ids.tolist()
)

print(
    "Eligible source patches:",
    len(selection)
)

print(
    "Source scene count: 1"
)

print("\nSelected patches:")

for name in selection["patch_id"]:

    print(" ", name)


# =============================================================================
# 30E — EXTRACT NATIVE WAVELENGTHS AND RADIOMETRIC SCALING
# =============================================================================

print("\n" + "=" * 90)
print("30B — ENMAP SPECTRAL AND RADIOMETRIC AUDIT")
print("=" * 90)

with rasterio.open(SOURCE) as src:

    assert src.count == 224

    assert (
        src.height,
        src.width
    ) == (200, 330)

    assert str(src.crs) == "EPSG:32633"

    assert not src.transform.is_identity

    nodata = src.nodata

    wavelengths_list = []

    fwhm_list = []

    bbl_list = []

    for band_id in retained_band_ids:

        band_id = int(band_id)

        band_tags = src.tags(band_id)

        if "wavelength" not in band_tags:

            raise RuntimeError(
                f"Wavelength metadata missing "
                f"for original EnMAP band {band_id}"
            )

        units = (
            band_tags.get(
                "wavelength_units",
                ""
            )
            .strip()
            .lower()
        )

        if (
            "nanometer" not in units
            and units != "nm"
        ):

            raise ValueError(
                f"Unexpected wavelength unit for "
                f"band {band_id}: {units}"
            )

        wavelength = float(
            band_tags["wavelength"]
        )

        wavelengths_list.append(
            wavelength
        )

        fwhm_value = band_tags.get(
            "fwhm"
        )

        fwhm_list.append(
            float(fwhm_value)
            if fwhm_value not in (None, "")
            else np.nan
        )

        bbl_list.append(
            band_tags.get(
                "bbl",
                ""
            )
        )

    wavelengths_nm = np.asarray(
        wavelengths_list,
        dtype=np.float64
    )

    fwhm_nm = np.asarray(
        fwhm_list,
        dtype=np.float64
    )

    all_scales = np.asarray(
        src.scales,
        dtype=np.float64
    )

    all_offsets = np.asarray(
        src.offsets,
        dtype=np.float64
    )

    scales = all_scales[
        retained_band_ids - 1
    ]

    offsets = all_offsets[
        retained_band_ids - 1
    ]

    # ---------------------------------------------------------
    # CORRECTED WAVELENGTH VERIFICATION
    #
    # VNIR/SWIR overlap may cause adjacent wavelengths
    # to decrease in the original EnMAP band sequence.
    #
    # We preserve the original channel order.
    # ---------------------------------------------------------

    assert len(wavelengths_nm) == 218

    assert np.isfinite(
        wavelengths_nm
    ).all()

    assert (
        wavelengths_nm > 0
    ).all()

    wavelength_differences = np.diff(
        wavelengths_nm
    )

    decreasing_indices = np.where(
        wavelength_differences < 0
    )[0]

    duplicate_indices = np.where(
        wavelength_differences == 0
    )[0]

    print(
        "Minimum wavelength:",
        round(float(wavelengths_nm.min()), 3),
        "nm"
    )

    print(
        "Maximum wavelength:",
        round(float(wavelengths_nm.max()), 3),
        "nm"
    )

    print(
        "Adjacent wavelength decreases:",
        len(decreasing_indices)
    )

    print(
        "Adjacent identical wavelengths:",
        len(duplicate_indices)
    )

    if len(decreasing_indices):

        print(
            "\nNative wavelength-order transitions:"
        )

        for idx in decreasing_indices[:20]:

            print(
                f"  Original band "
                f"{retained_band_ids[idx]} "
                f"({wavelengths_nm[idx]:.3f} nm)"
                f" -> "
                f"{retained_band_ids[idx + 1]} "
                f"({wavelengths_nm[idx + 1]:.3f} nm)"
            )

    # Do not sort wavelengths or reorder HSI channels.

    assert np.isfinite(
        scales
    ).all()

    assert (
        scales > 0
    ).all()

    assert np.isfinite(
        offsets
    ).all()

    print("\nRadiometric scale:")

    print(
        "Minimum:",
        float(scales.min())
    )

    print(
        "Maximum:",
        float(scales.max())
    )

    print(
        "Unique rounded scales:",
        np.unique(
            np.round(scales, 10)
        ).tolist()[:20]
    )

    print(
        "Offset range:",
        float(offsets.min()),
        "to",
        float(offsets.max())
    )

    # Read only the 218 retained bands.
    raw_cube = src.read(
        indexes=retained_band_ids.tolist()
    ).astype(np.float32)

    mask_cube = (
        src.read_masks(
            indexes=retained_band_ids.tolist()
        ) > 0
    )

    valid_cube = (
        mask_cube
        & np.isfinite(raw_cube)
    )

    if (
        nodata is not None
        and np.isfinite(nodata)
    ):

        valid_cube &= (
            raw_cube != nodata
        )

    transform = src.transform

assert raw_cube.shape == (
    218, 200, 330
)

assert valid_cube.shape == raw_cube.shape

# Apply per-band radiometric scaling.
#
# This follows the declared TIFF scale and offset.
# We do not infer additional correction factors.

scaled_cube = (
    raw_cube.astype(np.float64)
    * scales[:, None, None]
    + offsets[:, None, None]
).astype(np.float32)

valid_values = scaled_cube[
    valid_cube
]

assert valid_values.size > 0

assert np.isfinite(
    valid_values
).all()

negative_fraction = float(
    (valid_values < 0).mean()
)

strict_scene_valid = valid_cube.all(
    axis=0
)

strict_scene_fraction = float(
    strict_scene_valid.mean()
)

print("\nSpectral/radiometric validation: PASSED")

print(
    "Native HSI channel order:",
    "PRESERVED"
)

print(
    "Strict valid-pixel fraction:",
    f"{strict_scene_fraction * 100:.4f}%"
)

print(
    "Scaled valid-data minimum:",
    float(valid_values.min())
)

print(
    "Scaled valid-data median:",
    float(np.median(valid_values))
)

print(
    "Scaled valid-data maximum:",
    float(valid_values.max())
)

print(
    "Fraction of valid values below zero:",
    round(negative_fraction, 8)
)

# Avoid carrying unnecessary copies of the raw array.
del raw_cube
del mask_cube


# =============================================================================
# 30F — BUILD THE FOUR SYNTHETIC SPECTRAL RESPONSES
# =============================================================================

print("\n" + "=" * 90)
print("30C — FIXED SPECTRAL OBSERVATION OPERATOR")
print("=" * 90)

response_rows = []

for center_nm, fwhm in zip(
    MSI_CENTERS_NM,
    MSI_FWHM_NM
):

    sigma_nm = (
        float(fwhm)
        / (
            2.0
            * np.sqrt(
                2.0 * np.log(2.0)
            )
        )
    )

    response = np.exp(
        -0.5
        * (
            (
                wavelengths_nm - center_nm
            ) / sigma_nm
        ) ** 2
    )

    assert np.isfinite(
        response
    ).all()

    if response.max() < 1e-6:

        raise RuntimeError(
            f"The EnMAP wavelengths do not provide "
            f"adequate support for MSI center "
            f"{center_nm} nm."
        )

    response_sum = float(
        response.sum()
    )

    assert response_sum > 0

    response = (
        response / response_sum
    )

    response_rows.append(
        response
    )

RESPONSE = np.stack(
    response_rows,
    axis=0
).astype(np.float32)

assert RESPONSE.shape == (
    4, 218
)

assert np.allclose(
    RESPONSE.sum(axis=1),
    1.0,
    rtol=0,
    atol=1e-6
)

assert (
    RESPONSE >= 0
).all()

response_table = pd.DataFrame({
    "original_hsi_band": (
        retained_band_ids
    ),
    "wavelength_nm": (
        wavelengths_nm
    ),
    "native_fwhm_nm": (
        fwhm_nm
    ),
    "native_bbl": (
        bbl_list
    ),
    "synthetic_blue_weight": (
        RESPONSE[0]
    ),
    "synthetic_green_weight": (
        RESPONSE[1]
    ),
    "synthetic_red_weight": (
        RESPONSE[2]
    ),
    "synthetic_nir_weight": (
        RESPONSE[3]
    )
})

response_file = (
    OUT
    / "Step30_Spectral_Response_Matrix.csv"
)

response_status = save_csv_verified(
    response_file,
    response_table
)

print(
    "Spectral response matrix:",
    RESPONSE.shape
)

print(
    "Response normalization:",
    RESPONSE.sum(axis=1)
)

print(
    "Response file status:",
    response_status
)

for band_name, center, fwhm in zip(
    MSI_NAMES,
    MSI_CENTERS_NM,
    MSI_FWHM_NM
):

    print(
        f"  {band_name}: "
        f"center={center:.1f} nm, "
        f"FWHM={fwhm:.1f} nm"
    )


# =============================================================================
# 30G — FREEZE THE COMPLETE FORWARD MODEL
# =============================================================================

print("\n" + "=" * 90)
print("30D — FREEZING EXTERNAL OBSERVATION PROTOCOL")
print("=" * 90)

selected_patch_ids = sorted(
    selection["patch_id"].astype(str).tolist()
)

protocol = {
    "experiment": (
        "External Potsdam synthetic HSI-MSI "
        "registration benchmark"
    ),
    "protocol_version": "1.1-wavelength-order-correction",
    "source_dataset": (
        "EnMAP-Box example-data Potsdam"
    ),
    "source_scene_count": 1,
    "source_image_sha256": source_sha,
    "source_crs": "EPSG:32633",
    "original_band_count": 224,
    "retained_band_count": 218,
    "retained_band_ids_1_based": (
        retained_band_ids.tolist()
    ),
    "removed_band_ids_1_based": (
        removed_band_ids.tolist()
    ),
    "native_wavelength_order_preserved": True,
    "wavelength_sorting_applied": False,
    "adjacent_wavelength_decreases": (
        int(len(decreasing_indices))
    ),
    "adjacent_identical_wavelengths": (
        int(len(duplicate_indices))
    ),
    "native_tiff_scales_applied": True,
    "native_tiff_offsets_applied": True,
    "radiometric_clipping_applied": False,
    "radiometric_physical_units": (
        "Scaled TIFF values; reflectance "
        "interpretation requires source documentation"
    ),
    "selected_patch_ids": (
        selected_patch_ids
    ),
    "patch_selection": {
        "window_height": 96,
        "window_width": 96,
        "nonoverlapping": True,
        "validity_requirement": (
            "100% of retained HSI band pixels valid"
        ),
        "selected_patches": 5
    },
    "synthetic_msi": {
        "number_of_channels": 4,
        "channel_names": MSI_NAMES,
        "response_type": (
            "Discrete normalized Gaussian "
            "spectral-response surrogate"
        ),
        "centers_nm": MSI_CENTERS_NM,
        "fwhm_nm": MSI_FWHM_NM,
        "not_a_real_sensor_response": True
    },
    "synthetic_lrhsi": {
        "spatial_factor": SPATIAL_FACTOR,
        "gaussian_sigma_pixels": SPATIAL_BLUR_SIGMA,
        "gaussian_boundary_mode": "reflect",
        "decimation_start_row": 1,
        "decimation_start_col": 1
    },
    "synthetic_registration": {
        "shift_operator": (
            "scipy.ndimage.shift"
        ),
        "interpolation_order": 1,
        "boundary_mode": "nearest",
        "prefilter": False,
        "positive_dy": "down",
        "positive_dx": "right",
        "offsets_dy_dx": [
            [
                float(dy),
                float(dx)
            ]
            for dy, dx in OFFSETS
        ]
    },
    "evaluation_policy": {
        "registration_estimator_tuning": False,
        "external_truth_used_for_method_selection": False,
        "existing_mdas_fusion_weights_applicable": False,
        "independent_source_acquisitions": 1
    },
    "limitations": [
        "Synthetic multispectral observations.",
        "Single independent EnMAP source scene.",
        "Five non-overlapping patches are not "
        "five independent acquisitions.",
        "No evaluation of pretrained MDAS fusion models.",
        "Gaussian response functions are not "
        "calibrated real-sensor SRFs.",
        "Native sensor wavelength ordering preserved."
    ]
}

protocol_file = (
    OUT
    / "Step30_Frozen_Forward_Model.json"
)

protocol_status = save_json_verified(
    protocol_file,
    protocol
)

protocol_sha = sha256_file(
    protocol_file
)

print("Frozen protocol status:", protocol_status)
print("Protocol SHA256:", protocol_sha)
print("Wavelength sorting: NONE")
print("Registration algorithms evaluated: NO")


# =============================================================================
# 30H — GENERATE FIVE PAIRED OBSERVATION DATASETS
# =============================================================================

print("\n" + "=" * 90)
print("30E — GENERATING PAIRED EXTERNAL OBSERVATIONS")
print("=" * 90)

manifest_rows = []

shift_rows = []

offset_array = np.asarray(
    OFFSETS,
    dtype=np.float32
)

assert offset_array.shape == (
    17, 2
)

for index, row_data in enumerate(
    selection.sort_values(
        "patch_id"
    ).itertuples(index=False),
    start=1
):

    patch_id = str(
        row_data.patch_id
    )

    top = int(
        row_data.row
    )

    left = int(
        row_data.col
    )

    # ---------------------------------------------------------
    # Reference HSI
    # ---------------------------------------------------------

    reference_hsi = scaled_cube[
        :,
        top:top + PATCH_SIZE,
        left:left + PATCH_SIZE
    ].copy()

    patch_mask = valid_cube[
        :,
        top:top + PATCH_SIZE,
        left:left + PATCH_SIZE
    ]

    assert reference_hsi.shape == (
        218, 96, 96
    )

    assert patch_mask.shape == (
        218, 96, 96
    )

    assert patch_mask.all(), (
        "Selected patch contains invalid "
        f"hyperspectral pixels: {patch_id}"
    )

    assert np.isfinite(
        reference_hsi
    ).all()

    # ---------------------------------------------------------
    # Simulated LRHSI
    #
    # Gaussian blur over spatial axes only.
    # Preserve the spectral axis.
    # ---------------------------------------------------------

    spatially_blurred_hsi = gaussian_filter(
        reference_hsi,
        sigma=(
            0.0,
            SPATIAL_BLUR_SIGMA,
            SPATIAL_BLUR_SIGMA
        ),
        mode="reflect"
    )

    lrhsi = spatially_blurred_hsi[
        :,
        1::SPATIAL_FACTOR,
        1::SPATIAL_FACTOR
    ].astype(np.float32)

    assert lrhsi.shape == (
        218, 32, 32
    )

    assert np.isfinite(
        lrhsi
    ).all()

    # ---------------------------------------------------------
    # Simulated HRMSI
    #
    # Output channels:
    #   blue, green, red, near-infrared
    #
    # The spectral response is a normalized surrogate,
    # not an observed real multispectral sensor.
    # ---------------------------------------------------------

    hrmsi = np.einsum(
        "bc,chw->bhw",
        RESPONSE,
        reference_hsi,
        optimize=True
    ).astype(np.float32)

    assert hrmsi.shape == (
        4, 96, 96
    )

    assert np.isfinite(
        hrmsi
    ).all()

    # ---------------------------------------------------------
    # Generate synthetic MSI displacement conditions.
    # ---------------------------------------------------------

    shifted_observations = []

    for condition_id, (dy, dx) in enumerate(
        OFFSETS
    ):

        # Prevent any interpolation, copying artefacts or
        # numerical differences for the aligned condition.

        if dy == 0.0 and dx == 0.0:

            shifted = hrmsi.copy()

        else:

            shifted = ndi_shift(
                hrmsi,
                shift=(
                    0.0,
                    float(dy),
                    float(dx)
                ),
                order=1,
                mode="nearest",
                prefilter=False
            ).astype(np.float32)

        assert shifted.shape == (
            4, 96, 96
        )

        assert np.isfinite(
            shifted
        ).all()

        if condition_id == 0:

            assert np.array_equal(
                shifted,
                hrmsi
            )

        shifted_observations.append(
            shifted
        )

        shift_rows.append({
            "patch_id": patch_id,
            "condition_id": int(condition_id),
            "true_dy": float(dy),
            "true_dx": float(dx),
            "shift_magnitude": float(
                np.hypot(dy, dx)
            ),
            "is_aligned": bool(
                dy == 0.0 and dx == 0.0
            )
        })

    shifted_msi = np.stack(
        shifted_observations,
        axis=0
    )

    assert shifted_msi.shape == (
        17, 4, 96, 96
    )

    # ---------------------------------------------------------
    # Save benchmark patch.
    # ---------------------------------------------------------

    arrays = {
        "reference_hsi": (
            reference_hsi.astype(
                np.float32
            )
        ),
        "lrhsi": (
            lrhsi
        ),
        "hrmsi": (
            hrmsi
        ),
        "shifted_msi": (
            shifted_msi
        ),
        "retained_band_ids": (
            retained_band_ids.astype(
                np.int16
            )
        ),
        "wavelengths_nm": (
            wavelengths_nm.astype(
                np.float32
            )
        ),
        "shift_offsets_dy_dx": (
            offset_array
        )
    }

    output_path = (
        DATA_OUT
        / f"{patch_id}.npz"
    )

    file_status = save_npz_verified(
        output_path,
        arrays
    )

    file_sha = sha256_file(
        output_path
    )

    manifest_rows.append({
        "patch_id": patch_id,
        "source_scene": "EnMAP Potsdam",
        "row": top,
        "col": left,
        "reference_hsi_bands": 218,
        "reference_height": 96,
        "reference_width": 96,
        "lrhsi_height": 32,
        "lrhsi_width": 32,
        "synthetic_msi_bands": 4,
        "shift_conditions": 17,
        "source_acquisition_count": 1,
        "output_filename": output_path.name,
        "output_path": str(output_path),
        "output_sha256": file_sha
    })

    print(
        f"  Patch {index:02d}/{EXPECTED_PATCHES}: "
        f"{patch_id} | {file_status} | "
        f"{output_path.stat().st_size / 1048576:.2f} MiB",
        flush=True
    )


# =============================================================================
# 30I — CHECK COMPLETE BENCHMARK STRUCTURE
# =============================================================================

print("\n" + "=" * 90)
print("30F — BENCHMARK INTEGRITY VERIFICATION")
print("=" * 90)

manifest = pd.DataFrame(
    manifest_rows
)

shift_manifest = pd.DataFrame(
    shift_rows
)

assert len(manifest) == (
    EXPECTED_PATCHES
)

assert len(shift_manifest) == (
    EXPECTED_SHIFT_CASES
)

assert manifest["patch_id"].is_unique

assert not shift_manifest.duplicated(
    [
        "patch_id",
        "condition_id"
    ]
).any()

assert (
    shift_manifest.groupby(
        "patch_id"
    ).size() == 17
).all()

assert (
    shift_manifest.groupby(
        "patch_id"
    )["is_aligned"].sum() == 1
).all()

# Confirm that all source patch windows are non-overlapping.

spatial_boxes = []

for record in manifest.itertuples(
    index=False
):

    spatial_boxes.append({
        "patch_id": record.patch_id,
        "top": int(record.row),
        "bottom": int(record.row) + PATCH_SIZE,
        "left": int(record.col),
        "right": int(record.col) + PATCH_SIZE
    })

for i in range(
    len(spatial_boxes)
):

    for j in range(
        i + 1,
        len(spatial_boxes)
    ):

        a = spatial_boxes[i]
        b = spatial_boxes[j]

        overlap_h = max(
            0,
            min(
                a["bottom"],
                b["bottom"]
            ) - max(
                a["top"],
                b["top"]
            )
        )

        overlap_w = max(
            0,
            min(
                a["right"],
                b["right"]
            ) - max(
                a["left"],
                b["left"]
            )
        )

        assert (
            overlap_h * overlap_w
        ) == 0, (
            "The selected benchmark patches overlap."
        )

# Verify each saved file again from disk.

for record in manifest.itertuples(
    index=False
):

    path = Path(
        record.output_path
    )

    assert path.is_file()

    assert sha256_file(path) == (
        record.output_sha256
    )

    with np.load(
        path,
        allow_pickle=False
    ) as data:

        assert data["reference_hsi"].shape == (
            218, 96, 96
        )

        assert data["lrhsi"].shape == (
            218, 32, 32
        )

        assert data["hrmsi"].shape == (
            4, 96, 96
        )

        assert data["shifted_msi"].shape == (
            17, 4, 96, 96
        )

        assert np.array_equal(
            data["shifted_msi"][0],
            data["hrmsi"]
        )

        assert np.array_equal(
            data["retained_band_ids"],
            retained_band_ids.astype(
                np.int16
            )
        )

        assert np.array_equal(
            data["shift_offsets_dy_dx"],
            offset_array
        )

print("Source checksum: PASSED")
print("Patch count: PASSED")
print("Retained-band count: PASSED")
print("Reference HSI dimensions: PASSED")
print("LRHSI dimensions: PASSED")
print("HRMSI dimensions: PASSED")
print("Displacement conditions: PASSED")
print("Aligned MSI identity: PASSED")
print("Non-overlapping patch audit: PASSED")
print("Saved NPZ integrity: PASSED")


# =============================================================================
# 30J — SAVE DATA MANIFESTS
# =============================================================================

print("\n" + "=" * 90)
print("30G — SAVING BENCHMARK MANIFESTS")
print("=" * 90)

manifest_file = (
    OUT
    / "Step30_Paired_Observation_Manifest.csv"
)

shift_file = (
    OUT
    / "Step30_Shift_Conditions.csv"
)

manifest_status = save_csv_verified(
    manifest_file,
    manifest
)

shift_status = save_csv_verified(
    shift_file,
    shift_manifest
)

print(
    "Paired observation manifest:",
    manifest_status
)

print(
    "Shift condition manifest:",
    shift_status
)

print(
    "Total benchmark cases:",
    len(shift_manifest)
)


# =============================================================================
# 30K — FINAL AUDIT REPORT
# =============================================================================

print("\n" + "=" * 90)
print("30H — FINAL EXTERNAL BENCHMARK REPORT")
print("=" * 90)

report = {
    "experiment": (
        "External Potsdam synthetic HSI-MSI "
        "benchmark generation"
    ),
    "status": (
        "EXTERNAL_SYNTHETIC_BENCHMARK_PREPARED"
    ),
    "source_dataset": (
        "EnMAP-Box Potsdam example"
    ),
    "source_scene_count": 1,
    "source_image_sha256": source_sha,
    "source_crs": "EPSG:32633",
    "source_image_shape": [
        200, 330, 224
    ],
    "retained_spectral_bands": 218,
    "excluded_spectral_bands": (
        removed_band_ids.tolist()
    ),
    "native_wavelength_order_preserved": True,
    "adjacent_wavelength_decreases": (
        int(len(decreasing_indices))
    ),
    "adjacent_identical_wavelengths": (
        int(len(duplicate_indices))
    ),
    "tiff_radiometric_scales_applied": True,
    "tiff_offsets_applied": True,
    "valid_scaled_minimum": (
        float(valid_values.min())
    ),
    "valid_scaled_median": (
        float(np.median(valid_values))
    ),
    "valid_scaled_maximum": (
        float(valid_values.max())
    ),
    "fraction_negative_valid_values": (
        negative_fraction
    ),
    "strict_scene_valid_fraction": (
        strict_scene_fraction
    ),
    "patches_generated": (
        len(manifest)
    ),
    "patches_nonoverlapping": True,
    "patches_fully_valid": True,
    "shift_conditions_per_patch": (
        len(OFFSETS)
    ),
    "total_registration_cases": (
        len(shift_manifest)
    ),
    "reference_hsi_shape": [
        218, 96, 96
    ],
    "simulated_lrhsi_shape": [
        218, 32, 32
    ],
    "simulated_hrmsi_shape": [
        4, 96, 96
    ],
    "shifted_msi_shape": [
        17, 4, 96, 96
    ],
    "synthetic_msi_response": (
        "Four normalized Gaussian spectral "
        "response surrogate functions"
    ),
    "spatial_degradation": (
        "Gaussian blur sigma=1.0 pixels "
        "followed by factor-3 decimation"
    ),
    "shift_interpolation": (
        "Bilinear interpolation with nearest "
        "boundary extension"
    ),
    "frozen_protocol_sha256": (
        protocol_sha
    ),
    "paired_observation_manifest_sha256": (
        sha256_file(manifest_file)
    ),
    "shift_manifest_sha256": (
        sha256_file(shift_file)
    ),
    "response_matrix_sha256": (
        sha256_file(response_file)
    ),
    "all_saved_npz_files_verified": True,
    "external_registration_evaluated": False,
    "pretrained_mdas_fusion_checkpoints_evaluated": False,
    "registration_method_selection_using_potsdam": False,
    "limitations": [
        "Only one independent EnMAP source scene.",
        "Five spatially distinct patches are not "
        "independent source acquisitions.",
        "The MSI observation is simulated.",
        "The LRHSI observation is simulated.",
        "The synthetic MSI responses are not "
        "measured real-sensor response functions.",
        "The experimental shifts are synthetic.",
        "No direct testing of incompatible "
        "242-band MDAS fusion checkpoints.",
        "Radiometric scaling follows the TIFF "
        "metadata; physical reflectance "
        "interpretation should cite the source.",
        "Benchmark generation does not establish "
        "external registration accuracy."
    ],
    "output_directory": str(OUT)
}

final_report_path = (
    OUT
    / "Step30_Final_Report.json"
)

# Final reports contain no transient timestamps, allowing
# deterministic rerun verification.

report_status = save_json_verified(
    final_report_path,
    report
)

print(json.dumps(
    report,
    indent=2
))

print(
    "\nFinal report:",
    report_status
)

print(
    "Elapsed seconds:",
    round(
        time.monotonic() - START,
        2
    )
)

print("\n" + "=" * 90)
print("STEP 30 COMPLETE")
print("=" * 90)

print(
    "Benchmark patches:",
    len(manifest)
)

print(
    "Synthetic shift conditions:",
    len(OFFSETS)
)

print(
    "Total registration cases:",
    len(shift_manifest)
)

print(
    "External registration evaluated: NO"
)

print(
    "Original EnMAP imagery unchanged."
)

print(
    "Original MDAS imagery and checkpoints unchanged."
)


In [ ]:

# =============================================================================
# STEP 31 — EXTERNAL POTSDAM REGISTRATION EVALUATION
#
# Dataset:
#   Five independent-of-MDAS EnMAP Potsdam patches
#   17 synthetic shifts per patch = 85 paired conditions
#
# Methods:
#   1. Identity (no correction)
#   2. Fixed exhaustive NCC-gradient registration adaptation
#   3. FFT gradient phase-correlation adaptation
#
# IMPORTANT:
#   These are transparent, fixed external implementations.
#   They are NOT claimed to be bitwise/exact reproductions
#   of the original MDAS Step 13 and Step 20 estimators.
#
# No training, no tuning, no reference HSI used by estimators.
# Ground-truth shifts are read ONLY for evaluation.
# =============================================================================

from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import json
import hashlib
import time

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

STEP30 = PROJECT / "Step30_Frozen_Potsdam_Benchmark"

OUT = PROJECT / "Step31_External_Registration_Evaluation"
OUT.mkdir(parents=True, exist_ok=True)

MANIFEST = STEP30 / "Step30_Paired_Observation_Manifest.csv"
SHIFT_MANIFEST = STEP30 / "Step30_Shift_Conditions.csv"
RESPONSE_FILE = STEP30 / "Step30_Spectral_Response_Matrix.csv"
PROTOCOL_FILE = STEP30 / "Step30_Frozen_Forward_Model.json"
STEP30_REPORT = STEP30 / "Step30_Final_Report.json"

RESULT_FILE = OUT / "Step31_Paired_Registration_Results.csv"
SUMMARY_FILE = OUT / "Step31_Method_Comparison.csv"
PATCH_FILE = OUT / "Step31_Patch_Level_Comparison.csv"
REPORT_FILE = OUT / "Step31_Final_Report.json"
METHOD_FILE = OUT / "Step31_Frozen_Methods.json"

print("=" * 88, flush=True)
print("STEP 31 — EXTERNAL POTSDAM REGISTRATION TEST", flush=True)
print("=" * 88, flush=True)

# -------------------------------------------------------------------------
# 31A — CONFIGURATION FROZEN BEFORE THE EVALUATION
# -------------------------------------------------------------------------

CONFIG = {
    "ncc_method": "fixed_NCC_gradient_external_adaptation",
    "phase_method": "FFT_phase_gradient_external_adaptation",
    "identity_method": "no_correction",
    "max_translation_pixels": 5.0,
    "ncc_integer_search_range": 5,
    "ncc_first_refinement_step": 0.5,
    "ncc_second_refinement_step": 0.25,
    "ncc_identity_gate_pixels": 0.225,
    "ncc_intensity_weight": 0.5,
    "ncc_gradient_weight": 0.5,
    "phase_identity_gate_pixels": 0.25,
    "phase_radius_rejection_pixels": 5.0,
    "common_gradient_blur_sigma_pixels": 1.2,
    "interior_margin_pixels": 8,
    "upsampling_method": "cv2_INTER_LINEAR",
    "registration_uses_ground_truth": False,
    "registration_uses_reference_hrhsi": False,
    "method_identical_to_original_mdas_step13": False,
    "method_identical_to_original_mdas_step20": False,
    "external_hyperparameter_tuning": False
}

def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)
    return h.hexdigest()

def write_json_safely(path, data):
    if path.exists():
        old = json.loads(path.read_text(encoding="utf-8"))
        if old != data:
            raise FileExistsError(
                f"Existing output differs: {path}"
            )
    else:
        path.write_text(
            json.dumps(data, indent=2, sort_keys=True),
            encoding="utf-8"
        )

def save_csv_safely(path, frame):
    if path.exists():
        raise FileExistsError(
            f"Previous evaluation exists: {path}. "
            "Preserve the result and choose a new output folder."
        )
    frame.to_csv(path, index=False)

# Verify no previous evaluation exists.
for p in [RESULT_FILE, SUMMARY_FILE, PATCH_FILE, REPORT_FILE]:
    if p.exists():
        raise FileExistsError(
            f"Existing Step 31 result: {p}\n"
            "Preserve it and use a new output directory."
        )

for p in [
    MANIFEST, SHIFT_MANIFEST,
    RESPONSE_FILE, PROTOCOL_FILE, STEP30_REPORT
]:
    if not p.is_file():
        raise FileNotFoundError(p)

source_report = json.loads(
    STEP30_REPORT.read_text(encoding="utf-8")
)

assert source_report["status"] == (
    "EXTERNAL_SYNTHETIC_BENCHMARK_PREPARED"
)

assert source_report["patches_generated"] == 5
assert source_report["total_registration_cases"] == 85

frozen_forward = json.loads(
    PROTOCOL_FILE.read_text(encoding="utf-8")
)

manifest = pd.read_csv(MANIFEST)
shift_table = pd.read_csv(SHIFT_MANIFEST)
response_table = pd.read_csv(RESPONSE_FILE)

assert len(manifest) == 5
assert len(shift_table) == 85
assert len(response_table) == 218

R = response_table[
    [
        "synthetic_blue_weight",
        "synthetic_green_weight",
        "synthetic_red_weight",
        "synthetic_nir_weight"
    ]
].to_numpy(dtype=np.float32).T.copy()

assert R.shape == (4, 218)
assert np.allclose(R.sum(axis=1), 1.0, atol=1e-6)

write_json_safely(METHOD_FILE, {
    "methods": CONFIG,
    "step30_forward_protocol_sha256": digest(PROTOCOL_FILE),
    "step30_benchmark_manifest_sha256": digest(MANIFEST),
    "step30_response_sha256": digest(RESPONSE_FILE),
    "scientific_status": (
        "Fixed external algorithm adaptation; "
        "not exact MDAS implementation reproduction."
    )
})

print("\n31A — INPUTS VERIFIED")
print("Patches:", len(manifest))
print("Shift conditions:", len(shift_table))
print("Response matrix:", R.shape)
print("Method configuration:", METHOD_FILE)

# -------------------------------------------------------------------------
# 31B — FEATURE EXTRACTION
# -------------------------------------------------------------------------

MARGIN = CONFIG["interior_margin_pixels"]
MAX_SHIFT = CONFIG["max_translation_pixels"]

def normalize_feature(image):
    """
    Normalize feature channels for registration only.
    Shape: H,W,C.
    """
    image = np.asarray(image, dtype=np.float32)

    # Use whole-image channel means/stds; no reference target.
    mean = image.mean(axis=(0, 1), keepdims=True)
    std = image.std(axis=(0, 1), keepdims=True)

    return (
        (image - mean) / np.maximum(std, 1e-6)
    ).astype(np.float32)

def gradient_feature(image):
    """
    Multi-channel spatial gradient magnitude.
    """
    src = cv2.GaussianBlur(
        image,
        (0, 0),
        sigmaX=CONFIG["common_gradient_blur_sigma_pixels"],
        sigmaY=CONFIG["common_gradient_blur_sigma_pixels"]
    )

    gx = cv2.Sobel(
        src, cv2.CV_32F,
        1, 0, ksize=3
    )

    gy = cv2.Sobel(
        src, cv2.CV_32F,
        0, 1, ksize=3
    )

    magnitude = cv2.magnitude(gx, gy)

    return normalize_feature(magnitude)

def make_features(image):
    """
    Features are computed without consulting shift truth.
    """
    image = np.asarray(image, dtype=np.float32)

    return (
        normalize_feature(image),
        gradient_feature(image)
    )

def warp_to_reference(image, dy, dx):
    """
    Estimate is the displacement of the observed MSI
    relative to the reference.
    Alignment applies the inverse translation.
    """
    h, w = image.shape[:2]

    matrix = np.array([
        [1.0, 0.0, -float(dx)],
        [0.0, 1.0, -float(dy)]
    ], dtype=np.float32)

    return cv2.warpAffine(
        image,
        matrix,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE
    )

def ncc(a, b):
    """
    Normalized cross correlation over common interior.
    """
    a = a[MARGIN:-MARGIN, MARGIN:-MARGIN]
    b = b[MARGIN:-MARGIN, MARGIN:-MARGIN]

    a = a.astype(np.float64).ravel()
    b = b.astype(np.float64).ravel()

    a -= a.mean()
    b -= b.mean()

    denominator = (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

    if denominator < 1e-12:
        return 0.0

    return float(np.dot(a, b) / denominator)

def score_displacement(ref_features, obs_features, dy, dx):
    """
    Fixed 50/50 weighting of intensity and gradient NCC.
    """
    ref_intensity, ref_gradient = ref_features
    obs_intensity, obs_gradient = obs_features

    aligned_intensity = warp_to_reference(
        obs_intensity, dy, dx
    )

    aligned_gradient = warp_to_reference(
        obs_gradient, dy, dx
    )

    return (
        CONFIG["ncc_intensity_weight"]
        * ncc(ref_intensity, aligned_intensity)
        +
        CONFIG["ncc_gradient_weight"]
        * ncc(ref_gradient, aligned_gradient)
    )

# -------------------------------------------------------------------------
# 31C — FIXED NCC SEARCH
# -------------------------------------------------------------------------

def estimate_ncc(ref_features, obs_features):

    radius = CONFIG["ncc_integer_search_range"]

    best_score = -np.inf
    best_dy, best_dx = 0.0, 0.0

    # Integer-grid search.
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):

            if np.hypot(dy, dx) > MAX_SHIFT:
                continue

            s = score_displacement(
                ref_features, obs_features,
                dy, dx
            )

            if s > best_score:
                best_score = s
                best_dy, best_dx = float(dy), float(dx)

    # Two fixed subpixel refinements, no tuning.
    for step in [
        CONFIG["ncc_first_refinement_step"],
        CONFIG["ncc_second_refinement_step"]
    ]:

        anchor_y, anchor_x = best_dy, best_dx

        for offset_y in [-step, 0.0, step]:
            for offset_x in [-step, 0.0, step]:

                dy = anchor_y + offset_y
                dx = anchor_x + offset_x

                if np.hypot(dy, dx) > MAX_SHIFT:
                    continue

                s = score_displacement(
                    ref_features,
                    obs_features,
                    dy,
                    dx
                )

                if s > best_score:
                    best_score = s
                    best_dy, best_dx = dy, dx

    raw_dy, raw_dx = best_dy, best_dx

    if np.hypot(raw_dy, raw_dx) < CONFIG[
        "ncc_identity_gate_pixels"
    ]:
        best_dy, best_dx = 0.0, 0.0

    return {
        "estimated_dy": float(best_dy),
        "estimated_dx": float(best_dx),
        "raw_dy": float(raw_dy),
        "raw_dx": float(raw_dx),
        "objective": float(best_score)
    }

# -------------------------------------------------------------------------
# 31D — FIXED FFT PHASE-CORRELATION ADAPTATION
# -------------------------------------------------------------------------

def estimate_phase(ref_features, obs_features):
    """
    OpenCV phase correlation on mean gradient features.

    This is a global FFT estimator with a fixed maximum-radius
    rejection rule, not the previously developed custom bounded
    FFT method from MDAS Step 20.
    """

    ref_gradient = ref_features[1]
    obs_gradient = obs_features[1]

    ref_image = np.mean(
        ref_gradient, axis=2
    ).astype(np.float32)

    obs_image = np.mean(
        obs_gradient, axis=2
    ).astype(np.float32)

    h, w = ref_image.shape

    window = cv2.createHanningWindow(
        (w, h), cv2.CV_32F
    )

    displacement, response = cv2.phaseCorrelate(
        ref_image, obs_image, window
    )

    dx, dy = map(float, displacement)

    if not np.isfinite([dy, dx]).all():
        dy, dx = 0.0, 0.0

    raw_dy, raw_dx = dy, dx

    magnitude = float(
        np.hypot(dy, dx)
    )

    if (
        magnitude < CONFIG["phase_identity_gate_pixels"]
        or magnitude > CONFIG["phase_radius_rejection_pixels"]
    ):
        dy, dx = 0.0, 0.0

    return {
        "estimated_dy": float(dy),
        "estimated_dx": float(dx),
        "raw_dy": float(raw_dy),
        "raw_dx": float(raw_dx),
        "objective": float(response)
    }

# -------------------------------------------------------------------------
# 31E — EXTERNAL EVALUATION
# -------------------------------------------------------------------------

print("\n31B — REGISTRATION EVALUATION", flush=True)

records = []
started = time.monotonic()

for patch_index, item in enumerate(
    manifest.itertuples(index=False), 1
):

    path = STEP30 / "Paired_Observations" / item.output_filename

    if digest(path) != item.output_sha256:
        raise RuntimeError(
            f"Benchmark checksum mismatch: {path}"
        )

    with np.load(path, allow_pickle=False) as data:
        lrhsi = data["lrhsi"].copy()
        shifted_msi = data["shifted_msi"].copy()
        offsets = data["shift_offsets_dy_dx"].copy()

    assert lrhsi.shape == (218, 32, 32)
    assert shifted_msi.shape == (17, 4, 96, 96)
    assert offsets.shape == (17, 2)

    # Construct matching low-resolution MSI using the
    # frozen spectral response matrix, then upsample.
    #
    # This is a model-based proxy derived from the LRHSI
    # observation, not from the HRHSI reference image.
    low_msi = np.einsum(
        "bc,chw->bhw",
        R, lrhsi,
        optimize=True
    ).astype(np.float32)

    ref_image = cv2.resize(
        low_msi.transpose(1, 2, 0),
        (96, 96),
        interpolation=cv2.INTER_LINEAR
    )

    ref_features = make_features(ref_image)

    for condition_id in range(17):

        observed = shifted_msi[
            condition_id
        ].transpose(1, 2, 0)

        obs_features = make_features(observed)

        # Neither estimator is passed the true shift.
        ncc_result = estimate_ncc(
            ref_features,
            obs_features
        )

        phase_result = estimate_phase(
            ref_features,
            obs_features
        )

        # Ground truth enters only after both methods
        # have returned their displacement estimates.
        true_dy, true_dx = map(
            float,
            offsets[condition_id]
        )

        for method, result in [
            ("identity", {
                "estimated_dy": 0.0,
                "estimated_dx": 0.0,
                "raw_dy": 0.0,
                "raw_dx": 0.0,
                "objective": np.nan
            }),
            ("ncc_gradient_adaptation", ncc_result),
            ("fft_gradient_adaptation", phase_result)
        ]:

            est_dy = result["estimated_dy"]
            est_dx = result["estimated_dx"]

            epe = float(
                np.hypot(
                    est_dy - true_dy,
                    est_dx - true_dx
                )
            )

            records.append({
                "patch_id": item.patch_id,
                "condition_id": condition_id,
                "method": method,
                "true_dy": true_dy,
                "true_dx": true_dx,
                "shift_magnitude": float(
                    np.hypot(true_dy, true_dx)
                ),
                "estimated_dy": est_dy,
                "estimated_dx": est_dx,
                "raw_dy": result["raw_dy"],
                "raw_dx": result["raw_dx"],
                "objective": result["objective"],
                "epe": epe,
                "within_025px": bool(epe <= 0.25),
                "within_050px": bool(epe <= 0.50),
                "within_100px": bool(epe <= 1.00),
                "aligned_condition": bool(
                    true_dy == 0 and true_dx == 0
                )
            })

    print(
        f"Patch {patch_index}/5: {item.patch_id} | "
        f"cases processed: {patch_index * 17} | "
        f"elapsed: {time.monotonic()-started:.1f}s",
        flush=True
    )

results = pd.DataFrame(records)

assert len(results) == 85 * 3
assert results.groupby("method").size().eq(85).all()

save_csv_safely(RESULT_FILE, results)

# -------------------------------------------------------------------------
# 31F — SUMMARIES
# -------------------------------------------------------------------------

summary = (
    results.groupby("method")
    .agg(
        n=("epe", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max"),
        within_025=("within_025px", "mean"),
        within_050=("within_050px", "mean"),
        within_100=("within_100px", "mean")
    )
    .reset_index()
)

aligned = results[
    results["aligned_condition"]
].groupby("method").agg(
    aligned_n=("epe", "size"),
    aligned_exact=(
        "epe",
        lambda x: int(np.isclose(x, 0, atol=1e-8).sum())
    ),
    aligned_mean_epe=("epe", "mean")
).reset_index()

summary = summary.merge(
    aligned, on="method", validate="one_to_one"
)

patch_summary = (
    results.groupby(["patch_id", "method"])
    .agg(
        n=("epe", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        aligned_epe=(
            "epe",
            lambda x: float(x.iloc[0])
        )
    )
    .reset_index()
)

save_csv_safely(SUMMARY_FILE, summary)
save_csv_safely(PATCH_FILE, patch_summary)

print("\n" + "=" * 88)
print("31C — EXTERNAL REGISTRATION COMPARISON")
print("=" * 88)

print(summary.round(6).to_string(index=False))

print("\nPATCH-LEVEL SUMMARY")
print(patch_summary.round(6).to_string(index=False))

# -------------------------------------------------------------------------
# 31G — FINAL REPORT
# -------------------------------------------------------------------------

report = {
    "experiment": (
        "Potsdam external synthetic registration evaluation"
    ),
    "status": "EXTERNAL_ALGORITHM_ADAPTATION_EVALUATED",
    "source_scene_count": 1,
    "source_patch_count": 5,
    "shift_conditions": 17,
    "registration_cases_per_method": 85,
    "methods": summary.to_dict(orient="records"),
    "mdas_exact_ncc_reproduced": False,
    "mdas_exact_bounded_fft_reproduced": False,
    "spectral_projection": (
        "Fixed Step30 synthetic response applied to LRHSI"
    ),
    "external_ground_truth_used_for_tuning": False,
    "reference_hrhsi_used_by_estimators": False,
    "source_benchmark_manifest_sha256": digest(MANIFEST),
    "source_forward_protocol_sha256": digest(PROTOCOL_FILE),
    "frozen_method_config_sha256": digest(METHOD_FILE),
    "result_sha256": digest(RESULT_FILE),
    "limitations": [
        "Five patches share one source scene.",
        "HSI and MSI observations are synthetic degradations.",
        "Estimators are documented external adaptations.",
        "Results are not an exact transfer test of the "
        "original MDAS Step13 or Step20 implementation.",
        "No pretrained MDAS fusion checkpoint was evaluated.",
        "The fixed synthetic spectral response differs "
        "from the train-only MDAS spectral ridge projection.",
        "No independent test-scene replication."
    ],
    "elapsed_seconds": round(
        time.monotonic() - started, 2
    ),
    "output_directory": str(OUT)
}

write_json_safely(REPORT_FILE, report)

print("\n" + "=" * 88)
print("STEP 31 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))
print("\nSTEP 31 COMPLETE")
print("Original EnMAP imagery and MDAS files unchanged.")


In [ ]:

# =============================================================================
# STEP 32 — EXTERNAL REGISTRATION ERROR-ATTRIBUTION AUDIT
#
# Inputs:
#   Step31_Paired_Registration_Results.csv
#   Step31_Frozen_Methods.json
#   Step30_Final_Report.json
#
# Analyses:
#   1. Validate paired experiment completeness and source hashes
#   2. Examine aligned-input residual displacement vectors
#   3. Identify signed systematic biases
#   4. Compare error by shift magnitude
#   5. Compare FFT and NCC on paired cases
#   6. Check patch-level consistency
#   7. Audit NCC refinement-grid quantization
#   8. Generate publication-quality diagnostic figures
#
# Does NOT:
#   - Rerun registration estimators
#   - Adjust identity thresholds or bias corrections
#   - Select algorithms using Potsdam performance
#   - Claim independent replication across five scenes
# =============================================================================

from google.colab import drive
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

STEP30 = PROJECT / "Step30_Frozen_Potsdam_Benchmark"
STEP31 = PROJECT / "Step31_External_Registration_Evaluation"

OUT = PROJECT / "Step32_External_Registration_Error_Audit"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE = STEP31 / "Step31_Paired_Registration_Results.csv"
METHODS = STEP31 / "Step31_Frozen_Methods.json"
STEP31_REPORT = STEP31 / "Step31_Final_Report.json"

SHIFT_MANIFEST = STEP30 / "Step30_Shift_Conditions.csv"

OUTPUTS = [
    "Step32_Aligned_Error_Vectors.csv",
    "Step32_Shift_Magnitude_Summary.csv",
    "Step32_Paired_FFT_NCC_Comparison.csv",
    "Step32_Patchwise_Comparison.csv",
    "Step32_Quantization_Audit.csv",
    "Step32_Final_Report.json",
    "Figure01_Aligned_Residual_Vectors.png",
    "Figure01_Aligned_Residual_Vectors.pdf",
    "Figure02_Registration_Error_by_Magnitude.png",
    "Figure02_Registration_Error_by_Magnitude.pdf",
    "Figure03_Patchwise_FFT_NCC_Comparison.png",
    "Figure03_Patchwise_FFT_NCC_Comparison.pdf"
]

existing = [x for x in OUTPUTS if (OUT / x).exists()]
if existing:
    raise FileExistsError(
        "Step 32 outputs already exist. "
        "Preserve earlier audit results.\n"
        + "\n".join(existing)
    )

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(block)
    return digest.hexdigest()

print("=" * 86)
print("STEP 32 — EXTERNAL REGISTRATION ERROR ATTRIBUTION")
print("=" * 86)

for path in [SOURCE, METHODS, STEP31_REPORT, SHIFT_MANIFEST]:
    if not path.is_file():
        raise FileNotFoundError(path)

df = pd.read_csv(SOURCE)
shifts = pd.read_csv(SHIFT_MANIFEST)
method_config = json.loads(
    METHODS.read_text(encoding="utf-8")
)
previous_report = json.loads(
    STEP31_REPORT.read_text(encoding="utf-8")
)

assert len(df) == 255
assert len(shifts) == 85
assert df["patch_id"].nunique() == 5

METHOD_NAMES = [
    "identity",
    "ncc_gradient_adaptation",
    "fft_gradient_adaptation"
]

assert set(df["method"]) == set(METHOD_NAMES)
assert df.groupby("method").size().eq(85).all()

KEY = ["patch_id", "condition_id"]

assert not df.duplicated(KEY + ["method"]).any()
assert not shifts.duplicated(KEY).any()

# Verify all methods share the exact same conditions.
expected_keys = set(
    map(tuple, shifts[KEY].to_numpy())
)

for method, group in df.groupby("method"):
    actual_keys = set(
        map(tuple, group[KEY].to_numpy())
    )
    assert actual_keys == expected_keys, (
        f"Incomplete paired cases for {method}"
    )

# Check values against the saved Step 31 summary.
assert (
    previous_report["result_sha256"]
    == sha256(SOURCE)
), "Step 31 source checksum mismatch."

required_numeric = [
    "true_dy", "true_dx",
    "estimated_dy", "estimated_dx", "epe"
]

assert np.isfinite(
    df[required_numeric].to_numpy(float)
).all()

df["error_dy"] = (
    df["estimated_dy"] - df["true_dy"]
)

df["error_dx"] = (
    df["estimated_dx"] - df["true_dx"]
)

recomputed = np.hypot(
    df["error_dy"], df["error_dx"]
)

assert np.allclose(
    recomputed,
    df["epe"],
    atol=1e-9,
    rtol=0
)

df["magnitude_group"] = np.round(
    np.hypot(df["true_dy"], df["true_dx"]),
    5
)

print("\n32A — INPUT VERIFICATION PASSED")
print("Registration records:", len(df))
print("Paired conditions:", len(shifts))
print("Distinct source patches:", df["patch_id"].nunique())
print("Source scene acquisitions: 1")

# -------------------------------------------------------------------------
# 32B — ALIGNED-IMAGE RESIDUALS
# -------------------------------------------------------------------------

aligned = df[
    np.isclose(df["true_dy"], 0)
    & np.isclose(df["true_dx"], 0)
].copy()

assert len(aligned) == 15

aligned = aligned.sort_values(
    ["method", "patch_id"]
).reset_index(drop=True)

aligned.to_csv(
    OUT / "Step32_Aligned_Error_Vectors.csv",
    index=False
)

print("\n32B — ALIGNED RESIDUAL DISPLACEMENT VECTORS")
print(
    aligned[
        [
            "patch_id", "method",
            "estimated_dy", "estimated_dx", "epe"
        ]
    ].round(6).to_string(index=False)
)

ncc_aligned = aligned[
    aligned["method"] == "ncc_gradient_adaptation"
]

ncc_common_vector = bool(
    ncc_aligned[
        ["estimated_dy", "estimated_dx"]
    ].round(9).drop_duplicates().shape[0] == 1
)

print(
    "\nNCC aligned residual identical on all patches:",
    ncc_common_vector
)

print(
    "NCC mean aligned residual dy:",
    round(float(ncc_aligned["estimated_dy"].mean()), 6)
)

print(
    "NCC mean aligned residual dx:",
    round(float(ncc_aligned["estimated_dx"].mean()), 6)
)

# -------------------------------------------------------------------------
# 32C — SHIFT-MAGNITUDE SUMMARY
# -------------------------------------------------------------------------

magnitude_summary = (
    df.groupby(["method", "magnitude_group"])
    .agg(
        cases=("epe", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max"),
        signed_dy_bias=("error_dy", "mean"),
        signed_dx_bias=("error_dx", "mean"),
        within_025=("epe", lambda x: float((x <= 0.25).mean())),
        within_050=("epe", lambda x: float((x <= 0.50).mean()))
    )
    .reset_index()
)

magnitude_summary.to_csv(
    OUT / "Step32_Shift_Magnitude_Summary.csv",
    index=False
)

print("\n32C — ERROR BY SHIFT MAGNITUDE")
print(
    magnitude_summary.round(6).to_string(index=False)
)

# -------------------------------------------------------------------------
# 32D — PAIRED FFT VS NCC CASE COMPARISON
# -------------------------------------------------------------------------

ncc = df[
    df["method"] == "ncc_gradient_adaptation"
].copy()

fft = df[
    df["method"] == "fft_gradient_adaptation"
].copy()

paired = ncc.merge(
    fft,
    on=KEY,
    suffixes=("_ncc", "_fft"),
    validate="one_to_one"
)

assert len(paired) == 85

assert np.allclose(
    paired["true_dy_ncc"],
    paired["true_dy_fft"]
)
assert np.allclose(
    paired["true_dx_ncc"],
    paired["true_dx_fft"]
)

# Positive means FFT has lower EPE (better).
paired["fft_advantage_px"] = (
    paired["epe_ncc"] - paired["epe_fft"]
)

paired["fft_better"] = (
    paired["fft_advantage_px"] > 1e-9
)

paired["ncc_better"] = (
    paired["fft_advantage_px"] < -1e-9
)

paired["tie"] = (
    paired["fft_advantage_px"].abs() <= 1e-9
)

paired.to_csv(
    OUT / "Step32_Paired_FFT_NCC_Comparison.csv",
    index=False
)

print("\n32D — PAIRED CASE COMPARISON")
print(
    "FFT better:",
    int(paired["fft_better"].sum()),
    "/ 85"
)
print(
    "NCC better:",
    int(paired["ncc_better"].sum()),
    "/ 85"
)
print(
    "Ties:",
    int(paired["tie"].sum()),
    "/ 85"
)
print(
    "Mean FFT advantage (pixels):",
    round(float(paired["fft_advantage_px"].mean()), 6)
)

# -------------------------------------------------------------------------
# 32E — PATCH-LEVEL COMPARISON
# -------------------------------------------------------------------------

patch_summary = (
    paired.groupby("patch_id")
    .agg(
        cases=("fft_advantage_px", "size"),
        ncc_mean_epe=("epe_ncc", "mean"),
        fft_mean_epe=("epe_fft", "mean"),
        fft_advantage_px=("fft_advantage_px", "mean"),
        fft_better_cases=("fft_better", "sum"),
        ncc_better_cases=("ncc_better", "sum")
    )
    .reset_index()
)

patch_summary.to_csv(
    OUT / "Step32_Patchwise_Comparison.csv",
    index=False
)

print("\n32E — PATCH-LEVEL COMPARISON")
print(
    patch_summary.round(6).to_string(index=False)
)

# No inferential claim based on 85 independent samples:
# all observations are dependent perturbations of one scene.

# -------------------------------------------------------------------------
# 32F — SEARCH-GRID QUANTIZATION
# -------------------------------------------------------------------------

# The NCC adaptation's second refinement step is 0.25 px.
# Check whether output displacements lie on that grid.
# This is a diagnostic, not a parameter update.

grid_step = 0.25

quantization_rows = []

for method in METHOD_NAMES:

    subset = df[df["method"] == method]

    y = subset["estimated_dy"].to_numpy(float)
    x = subset["estimated_dx"].to_numpy(float)

    residual_y = np.abs(
        y / grid_step - np.round(y / grid_step)
    )

    residual_x = np.abs(
        x / grid_step - np.round(x / grid_step)
    )

    quantization_rows.append({
        "method": method,
        "grid_step_px": grid_step,
        "fraction_on_grid": float(
            np.mean(
                (residual_y < 1e-6)
                & (residual_x < 1e-6)
            )
        ),
        "unique_estimated_dy": int(
            subset["estimated_dy"].nunique()
        ),
        "unique_estimated_dx": int(
            subset["estimated_dx"].nunique()
        ),
        "mean_signed_error_dy": float(
            subset["error_dy"].mean()
        ),
        "mean_signed_error_dx": float(
            subset["error_dx"].mean()
        )
    })

quantization = pd.DataFrame(quantization_rows)

quantization.to_csv(
    OUT / "Step32_Quantization_Audit.csv",
    index=False
)

print("\n32F — QUANTIZATION AUDIT")
print(
    quantization.round(6).to_string(index=False)
)

# -------------------------------------------------------------------------
# 32G — PUBLICATION-QUALITY DIAGNOSTIC FIGURES
# -------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "savefig.facecolor": "white"
})

COLORS = {
    "identity": "#888888",
    "ncc_gradient_adaptation": "#146C94",
    "fft_gradient_adaptation": "#D18128"
}

LABELS = {
    "identity": "Identity",
    "ncc_gradient_adaptation": "NCC adaptation",
    "fft_gradient_adaptation": "FFT adaptation"
}

def save_figure(fig, name):
    fig.savefig(
        OUT / f"{name}.png",
        dpi=1200,
        bbox_inches="tight",
        pad_inches=0.08
    )
    fig.savefig(
        OUT / f"{name}.pdf",
        bbox_inches="tight",
        pad_inches=0.08
    )
    plt.close(fig)

# Figure 1: Aligned residual displacement vectors.
fig, ax = plt.subplots(figsize=(6.6, 5.3))

for method in [
    "ncc_gradient_adaptation",
    "fft_gradient_adaptation"
]:

    subset = aligned[aligned["method"] == method]

    dx = subset["estimated_dx"].to_numpy(float)
    dy = subset["estimated_dy"].to_numpy(float)

    ax.scatter(
        dx, dy,
        color=COLORS[method],
        label=LABELS[method],
        s=55,
        alpha=0.85
    )

    for xx, yy in zip(dx, dy):
        ax.plot(
            [0, xx], [0, yy],
            color=COLORS[method],
            linewidth=0.7,
            alpha=0.35
        )

ax.scatter(
    [0], [0],
    marker="+",
    color="black",
    s=120,
    label="True alignment"
)

ax.axhline(0, color="gray", linewidth=0.7)
ax.axvline(0, color="gray", linewidth=0.7)

ax.set_xlabel("Estimated horizontal displacement (px)")
ax.set_ylabel("Estimated vertical displacement (px)")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.15)
ax.legend(frameon=False)

fig.tight_layout()
save_figure(
    fig, "Figure01_Aligned_Residual_Vectors"
)

# Figure 2: EPE by shift magnitude.
fig, ax = plt.subplots(figsize=(8.5, 4.5))

for method in METHOD_NAMES:

    sub = magnitude_summary[
        magnitude_summary["method"] == method
    ].sort_values("magnitude_group")

    ax.plot(
        sub["magnitude_group"].to_numpy(float),
        sub["mean_epe"].to_numpy(float),
        marker="o",
        markersize=5,
        color=COLORS[method],
        label=LABELS[method]
    )

ax.set_xlabel("Synthetic displacement magnitude (px)")
ax.set_ylabel("Mean endpoint error (px)")
ax.set_xticks(
    sorted(df["magnitude_group"].unique())
)
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False)

fig.tight_layout()
save_figure(
    fig, "Figure02_Registration_Error_by_Magnitude"
)

# Figure 3: Paired patch-level comparison.
fig, ax = plt.subplots(figsize=(9.2, 4.5))

patch_summary = patch_summary.sort_values("patch_id")

x = np.arange(len(patch_summary))
width = 0.35

ax.bar(
    x - width/2,
    patch_summary["ncc_mean_epe"],
    width,
    color=COLORS["ncc_gradient_adaptation"],
    label="NCC adaptation"
)

ax.bar(
    x + width/2,
    patch_summary["fft_mean_epe"],
    width,
    color=COLORS["fft_gradient_adaptation"],
    label="FFT adaptation"
)

ax.set_xticks(
    x,
    patch_summary["patch_id"],
    rotation=25,
    ha="right"
)
ax.set_ylabel("Mean endpoint error (px)")
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False)

fig.tight_layout()
save_figure(
    fig, "Figure03_Patchwise_FFT_NCC_Comparison"
)

# -------------------------------------------------------------------------
# 32H — FINAL REPORT
# -------------------------------------------------------------------------

report = {
    "experiment": "External Potsdam registration error attribution",
    "status": "EXTERNAL_ERROR_AUDIT_COMPLETED",
    "source_step31_results_sha256": sha256(SOURCE),
    "source_step31_methods_sha256": sha256(METHODS),
    "source_step30_shift_manifest_sha256": sha256(SHIFT_MANIFEST),
    "source_acquisitions": 1,
    "source_patches": 5,
    "paired_conditions": 85,
    "evaluated_estimators": METHOD_NAMES,
    "ncc_aligned_common_residual_vector": ncc_common_vector,
    "ncc_aligned_mean_dy": float(
        ncc_aligned["estimated_dy"].mean()
    ),
    "ncc_aligned_mean_dx": float(
        ncc_aligned["estimated_dx"].mean()
    ),
    "fft_better_cases": int(
        paired["fft_better"].sum()
    ),
    "ncc_better_cases": int(
        paired["ncc_better"].sum()
    ),
    "tied_cases": int(
        paired["tie"].sum()
    ),
    "mean_fft_advantage_px": float(
        paired["fft_advantage_px"].mean()
    ),
    "method_parameters_changed": False,
    "new_registration_estimations_performed": False,
    "external_ground_truth_used_for_tuning": False,
    "exact_original_mdas_estimators_tested": False,
    "interpretation_limits": [
        "Observed systematic bias does not establish its cause.",
        "The algorithms are external adaptations, not exact MDAS methods.",
        "All five patches come from one Potsdam acquisition.",
        "The 85 perturbations are not independent source scenes.",
        "Any subsequent correction informed by Potsdam is exploratory.",
        "No external fusion checkpoint transfer was tested."
    ],
    "output_directory": str(OUT)
}

(OUT / "Step32_Final_Report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 86)
print("STEP 32 — FINAL REPORT")
print("=" * 86)
print(json.dumps(report, indent=2))

print("\nSTEP 32 COMPLETE")
print("No registration estimators rerun.")
print("No thresholds or biases changed.")
print("Original experimental results unchanged.")


In [ ]:

# =============================================================================
# STEP 33 — ORIGINAL MDAS REGISTRATION SOURCE RECOVERY AUDIT
#
# PURPOSE
#   Locate the actual executable source implementing:
#       Step 13 — NCC/gradient + identity-preserving gate
#       Step 20 — bounded FFT phase correlation
#
# Search scope:
#   Relevant project folders only, NOT all of Google Drive.
#
# Outputs:
#   Step33_Frozen_Configurations.json
#   Step33_Source_Inventory.csv
#   Step33_Code_Matches.csv
#   Step33_Candidate_Excerpts.txt
#   Step33_Final_Report.json
#
# No model inference, no parameter changes, no source modification.
# =============================================================================

from google.colab import drive
from pathlib import Path
import os
import json
import re
import hashlib
import pandas as pd

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

OUT = PROJECT / "Step33_Original_Method_Source_Audit"
OUT.mkdir(parents=True, exist_ok=True)

STEP13 = PROJECT / "Step13_Identity_Preserving_Gate"
STEP20 = PROJECT / "Step20_Bounded_Phase_Correlation"
STEP9 = PROJECT / "Step09_Estimated_Registration"

FROZEN_FILES = {
    "step13_gate": STEP13 / "Step13_Frozen_Gate.json",
    "step20_method": STEP20 / "Step20_Frozen_Method.json",
    "step30_forward_model": (
        PROJECT / "Step30_Frozen_Potsdam_Benchmark"
        / "Step30_Frozen_Forward_Model.json"
    ),
    "step31_external_methods": (
        PROJECT / "Step31_External_Registration_Evaluation"
        / "Step31_Frozen_Methods.json"
    )
}

# Search only targeted experimental folders.
SEARCH_ROOTS = [
    STEP9,
    STEP13,
    STEP20,
    PROJECT / "Step19B_Corrected_Comparison",
    PROJECT / "Step19_Classical_Registration",
    PROJECT / "Step05_Source_Recovery"
]

# Also inspect scripts/notebooks saved directly at project root.
MAX_DEPTH = 4
MAX_FILES = 4000
MAX_TEXT_BYTES = 4 * 1024 * 1024

SOURCE_EXTENSIONS = {
    ".py", ".ipynb", ".txt", ".md"
}

PATTERNS = {
    "ncc_objective": re.compile(
        r"normalized.cross.correlation|"
        r"\bncc\b|"
        r"cross.?correlat",
        re.IGNORECASE
    ),
    "gradient_feature": re.compile(
        r"\bgradient\b|"
        r"\bsobel\b|"
        r"\bscharr\b",
        re.IGNORECASE
    ),
    "identity_gate": re.compile(
        r"identity|"
        r"snap|"
        r"gate|"
        r"threshold",
        re.IGNORECASE
    ),
    "phase_correlation": re.compile(
        r"phaseCorrelate|"
        r"phase.correlation|"
        r"phase_corr|"
        r"\bfft2\b|"
        r"\brfft2\b",
        re.IGNORECASE
    ),
    "bounded_search": re.compile(
        r"radius|"
        r"search.window|"
        r"max.shift|"
        r"subpixel|"
        r"refin",
        re.IGNORECASE
    ),
    "spectral_projection": re.compile(
        r"ridge_coefficients|"
        r"spectral.projection|"
        r"scaler_mean|"
        r"scaler_scale",
        re.IGNORECASE
    )
}

print("=" * 88, flush=True)
print("STEP 33 — ORIGINAL REGISTRATION SOURCE AUDIT", flush=True)
print("=" * 88, flush=True)

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()

# -------------------------------------------------------------------------
# 33A — INSPECT FROZEN CONFIGURATION FILES
# -------------------------------------------------------------------------

print("\n33A — FROZEN METHOD CONFIGURATIONS", flush=True)

frozen = {}

for label, path in FROZEN_FILES.items():

    if not path.is_file():
        frozen[label] = {
            "found": False,
            "path": str(path)
        }
        print(label, "NOT FOUND", flush=True)
        continue

    data = json.loads(
        path.read_text(encoding="utf-8")
    )

    frozen[label] = {
        "found": True,
        "path": str(path),
        "sha256": sha256(path),
        "settings": data
    }

    print(
        f"\n{label}:",
        path.name,
        flush=True
    )

    print(
        json.dumps(
            data,
            indent=2
        )[:5000],
        flush=True
    )

assert frozen["step13_gate"]["found"], (
    "Original Step 13 gate configuration missing."
)

assert frozen["step20_method"]["found"], (
    "Original Step 20 frozen configuration missing."
)

# -------------------------------------------------------------------------
# 33B — READ SOURCE FILES SAFELY
# -------------------------------------------------------------------------

def source_lines(path):
    """
    Return lines from a Python/text file or notebook.

    The original files are never modified.
    """

    if path.stat().st_size > MAX_TEXT_BYTES:
        return None

    text = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    if path.suffix.lower() != ".ipynb":
        return text.splitlines()

    notebook = json.loads(text)

    lines = []

    for cell_no, cell in enumerate(
        notebook.get("cells", [])
    ):

        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])

        if isinstance(source, list):
            source = "".join(source)

        lines.append(
            f"# --- NOTEBOOK CODE CELL {cell_no} ---"
        )

        lines.extend(
            source.splitlines()
        )

    return lines

def walk_limited(root, depth_limit):
    """
    Targeted, bounded traversal.
    No scanning of the entire Google Drive.
    """

    if not root.is_dir():
        return

    stack = [(root, 0)]

    while stack:

        folder, depth = stack.pop()

        try:
            with os.scandir(folder) as iterator:
                entries = list(iterator)

        except OSError:
            continue

        for entry in entries:

            path = Path(entry.path)

            try:
                if entry.is_dir(follow_symlinks=False):

                    if (
                        depth < depth_limit
                        and not entry.name.startswith(".")
                        and entry.name != "__pycache__"
                    ):
                        stack.append(
                            (path, depth + 1)
                        )

                elif entry.is_file(follow_symlinks=False):

                    if path.suffix.lower() in SOURCE_EXTENSIONS:
                        yield path

            except OSError:
                continue

# Include relevant scripts directly at PROJECT root.
candidate_paths = []

for path in PROJECT.iterdir():

    if (
        path.is_file()
        and path.suffix.lower() in SOURCE_EXTENSIONS
    ):
        candidate_paths.append(path)

for root in SEARCH_ROOTS:

    print(
        "Searching:",
        root.name,
        flush=True
    )

    for path in walk_limited(root, MAX_DEPTH):
        candidate_paths.append(path)

candidate_paths = sorted(set(candidate_paths))

if len(candidate_paths) > MAX_FILES:

    print(
        "WARNING: Source inventory was truncated.",
        flush=True
    )

    candidate_paths = candidate_paths[:MAX_FILES]

print(
    "\nCandidate source files:",
    len(candidate_paths),
    flush=True
)

# -------------------------------------------------------------------------
# 33C — FIND ACTUAL CODE REFERENCES
# -------------------------------------------------------------------------

inventory_rows = []
match_rows = []
excerpt_blocks = []

for file_index, path in enumerate(
    candidate_paths, 1
):

    try:
        lines = source_lines(path)

        if lines is None:
            continue

        file_sha = sha256(path)

        relative = str(
            path.relative_to(PROJECT)
        )

        found_categories = set()
        match_count = 0

        for line_no, line in enumerate(
            lines, 1
        ):

            matched = [
                category
                for category, pattern in PATTERNS.items()
                if pattern.search(line)
            ]

            if not matched:
                continue

            found_categories.update(matched)
            match_count += 1

            match_rows.append({
                "source_file": relative,
                "line_number": line_no,
                "categories": ";".join(matched),
                "matched_code": line[:300]
            })

        inventory_rows.append({
            "source_file": relative,
            "sha256": file_sha,
            "size_bytes": path.stat().st_size,
            "line_count": len(lines),
            "pattern_matches": match_count,
            "categories": ";".join(
                sorted(found_categories)
            )
        })

        if match_count:

            excerpt_blocks.append(
                "\n" + "=" * 75
                + f"\nFILE: {relative}"
                + f"\nSHA256: {file_sha}"
                + "\n" + "=" * 75
            )

            # Show at most 18 source locations per file.
            matching_indices = [
                i
                for i, line in enumerate(lines)
                if any(
                    p.search(line)
                    for p in PATTERNS.values()
                )
            ][:18]

            displayed = set()

            for i in matching_indices:

                start = max(0, i - 2)
                end = min(len(lines), i + 3)

                for j in range(start, end):

                    if j in displayed:
                        continue

                    excerpt_blocks.append(
                        f"{j + 1:05d}: {lines[j][:240]}"
                    )

                    displayed.add(j)

                excerpt_blocks.append("")

    except Exception as exc:

        inventory_rows.append({
            "source_file": str(path),
            "sha256": "",
            "size_bytes": None,
            "line_count": None,
            "pattern_matches": 0,
            "categories": f"READ_ERROR: {str(exc)[:120]}"
        })

inventory = pd.DataFrame(
    inventory_rows
)

matches = pd.DataFrame(
    match_rows,
    columns=[
        "source_file",
        "line_number",
        "categories",
        "matched_code"
    ]
)

if not inventory.empty:
    inventory = inventory.sort_values(
        "pattern_matches",
        ascending=False
    )

# -------------------------------------------------------------------------
# 33D — REVIEW HIGHEST-PRIORITY SOURCE CANDIDATES
# -------------------------------------------------------------------------

print("\n33B — HIGHEST-PRIORITY SOURCE FILES", flush=True)

if inventory.empty:

    print(
        "No scripts/notebooks found in the targeted locations."
    )

else:

    print(
        inventory[
            [
                "source_file",
                "pattern_matches",
                "categories"
            ]
        ].head(25).to_string(
            index=False,
            max_colwidth=90
        ),
        flush=True
    )

# Candidate files are NOT automatically verified implementations.
#
# A genuine original-method recovery requires checking:
#
# - The exact feature normalization
# - The registration scoring function
# - Search grid and subpixel refinement
# - Bias correction and identity gate
# - Shift sign conventions
# - Spectral projection and resizing
# - Validation-selected frozen parameters
#
# Merely finding the words "NCC" or "phase" is insufficient.

ncc_candidates = 0
phase_candidates = 0

if not inventory.empty:

    ncc_candidates = int(
        inventory["categories"]
        .fillna("")
        .str.contains("ncc_objective")
        .sum()
    )

    phase_candidates = int(
        inventory["categories"]
        .fillna("")
        .str.contains("phase_correlation")
        .sum()
    )

print("\n33C — CANDIDATE COUNTS")

print(
    "Files mentioning NCC-related code:",
    ncc_candidates
)

print(
    "Files mentioning phase correlation:",
    phase_candidates
)

# -------------------------------------------------------------------------
# 33E — WRITE AUDIT OUTPUTS
# -------------------------------------------------------------------------

inventory.to_csv(
    OUT / "Step33_Source_Inventory.csv",
    index=False
)

matches.to_csv(
    OUT / "Step33_Code_Matches.csv",
    index=False
)

(OUT / "Step33_Candidate_Excerpts.txt").write_text(
    "\n".join(excerpt_blocks),
    encoding="utf-8"
)

(OUT / "Step33_Frozen_Configurations.json").write_text(
    json.dumps(
        frozen,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

report = {
    "experiment": (
        "Original MDAS registration source recovery audit"
    ),
    "searched_project": str(PROJECT),
    "search_roots": [
        str(p)
        for p in SEARCH_ROOTS
    ],
    "source_files_inspected": int(
        len(inventory)
    ),
    "matched_source_lines": int(
        len(matches)
    ),
    "ncc_candidate_files": (
        ncc_candidates
    ),
    "phase_candidate_files": (
        phase_candidates
    ),
    "step13_frozen_config_found": bool(
        frozen["step13_gate"]["found"]
    ),
    "step20_frozen_config_found": bool(
        frozen["step20_method"]["found"]
    ),
    "original_ncc_source_verified": False,
    "original_bounded_phase_source_verified": False,
    "exact_external_registration_reproduction_completed": False,
    "source_files_modified": False,
    "status": (
        "CANDIDATE_SOURCE_REQUIRES_CODE_REVIEW"
        if (
            ncc_candidates > 0
            or phase_candidates > 0
        )
        else "ORIGINAL_IMPLEMENTATIONS_NOT_LOCATED"
    ),
    "next_requirement": (
        "Review candidate source excerpts and recover "
        "the exact original Step 13 and Step 20 implementations "
        "before testing their frozen behavior on Potsdam."
    ),
    "limitations": [
        "Text matches do not establish implementation identity.",
        "The scan covers targeted project folders only.",
        "Original Colab code may not have been saved to Drive.",
        "No registration algorithm was executed.",
        "No parameters were changed using Potsdam results."
    ],
    "output_directory": str(OUT)
}

(OUT / "Step33_Final_Report.json").write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 33 — FINAL REPORT")
print("=" * 88)

print(
    json.dumps(
        report,
        indent=2
    )
)

print("\nSTEP 33 COMPLETE")
print("No source code or prior results modified.")


In [ ]:

# =============================================================================
# STEP 34 — RECOVER ORIGINAL REGISTRATION CODE FROM COLAB NOTEBOOKS
#
# Search:
#   1. MyDrive/Colab Notebooks
#   2. Python/notebook files directly inside MyDrive
#   3. Python/notebook files in the MDAS project, to depth 2
#
# Targets:
#   - Step 12 quadratic subpixel NCC estimator
#   - Step 13 identity-preserving displacement gate
#   - Step 20 bounded phase-correlation implementation
#
# Safeguards:
#   - Bounded file search
#   - No full-Drive recursive traversal
#   - Reads notebook CODE cells only, not embedded outputs
#   - Never executes recovered code
#   - Never declares source equivalence automatically
#   - Preserves existing audit results
# =============================================================================

from google.colab import drive
from pathlib import Path
import json
import hashlib
import os
import re
import time
import pandas as pd

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

OUT = PROJECT / "Step34_Colab_Source_Recovery"
OUT.mkdir(parents=True, exist_ok=True)

SEARCH_FOLDERS = [
    ROOT / "Colab Notebooks",
    PROJECT
]

MAX_DEPTH = {
    str(ROOT / "Colab Notebooks"): 4,
    str(PROJECT): 2
}

MAX_FILES_PER_FOLDER = 600
MAX_FILE_BYTES = 12 * 1024 * 1024

CODE_EXTENSIONS = {".ipynb", ".py"}

# Ignore source repositories already inspected by Step 33.
SKIP_DIRS = {
    "Step05_Source_Recovery",
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "node_modules"
}

# Strong signals of an actual registration implementation.
PATTERNS = {
    "quadratic_subpixel": re.compile(
        r"quadratic.subpixel|quadratic.*refin|"
        r"parabolic.*subpixel|subpixel.*quadratic",
        re.I
    ),
    "ncc_algorithm": re.compile(
        r"\bncc\b|normalized.cross.correlation|"
        r"normalized_cross_correlation",
        re.I
    ),
    "phase_algorithm": re.compile(
        r"phaseCorrelate|phase_correlate|"
        r"phase_correlation|phase_corr|"
        r"cross.power.spectrum",
        re.I
    ),
    "fft_implementation": re.compile(
        r"\bfft2\b|\brfft2\b|"
        r"\bifft2\b|\birfft2\b",
        re.I
    ),
    "identity_gate": re.compile(
        r"identity.preserv|identity_gate|"
        r"snap.*zero|threshold_pixels|"
        r"0\.225",
        re.I
    ),
    "bounded_phase": re.compile(
        r"bounded.phase|bounded.*correlat|"
        r"phase_gradient|"
        r"bias_dy|bias_dx",
        re.I
    ),
    "registration_shift": re.compile(
        r"estimated_dy|estimated_dx|"
        r"true_dy|true_dx|"
        r"endpoint.error|\bepe\b",
        re.I
    ),
    "spectral_projection": re.compile(
        r"ridge_coefficients|"
        r"spectral_projection|"
        r"scaler_mean|scaler_scale",
        re.I
    )
}

OUTPUT_NAMES = [
    "Step34_Notebook_Inventory.csv",
    "Step34_Code_Matches.csv",
    "Step34_High_Priority_Candidates.csv",
    "Step34_Candidate_Code_Excerpts.txt",
    "Step34_Final_Report.json"
]

if any((OUT / name).exists() for name in OUTPUT_NAMES):
    raise FileExistsError(
        "Step 34 outputs already exist. "
        "Preserve the previous source-recovery audit "
        "and choose a new output directory for a rerun."
    )

print("=" * 88, flush=True)
print("STEP 34 — GOOGLE COLAB REGISTRATION SOURCE RECOVERY", flush=True)
print("=" * 88, flush=True)

started = time.monotonic()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

def discover(folder, max_depth, limit):
    """
    Bounded traversal. Does not enter unrelated
    folders across the entire Google Drive.
    """
    if not folder.is_dir():
        return [], False, 0

    pending = [(folder, 0)]
    found = []
    inspected = 0

    while pending and inspected < limit:
        current, depth = pending.pop()

        try:
            with os.scandir(current) as handle:
                entries = list(handle)
        except OSError:
            continue

        for entry in entries:
            inspected += 1

            if inspected > limit:
                return found, True, inspected

            path = Path(entry.path)

            try:
                if entry.is_dir(follow_symlinks=False):
                    if (
                        depth < max_depth
                        and entry.name not in SKIP_DIRS
                        and not entry.name.startswith(".")
                    ):
                        pending.append((path, depth + 1))

                elif entry.is_file(follow_symlinks=False):
                    if path.suffix.lower() in CODE_EXTENSIONS:
                        found.append(path)

            except OSError:
                continue

    return found, bool(pending), inspected

def extract_code(path):
    """
    For notebooks, inspect only code cells.
    Output cells are excluded to keep parsing fast.
    """
    if path.stat().st_size > MAX_FILE_BYTES:
        return None, "oversize"

    try:
        if path.suffix.lower() == ".py":
            text = path.read_text(
                encoding="utf-8",
                errors="replace"
            )
            return [
                {
                    "cell": 0,
                    "lines": text.splitlines()
                }
            ], "ok"

        notebook = json.loads(
            path.read_text(encoding="utf-8")
        )

        cells = []

        for index, cell in enumerate(
            notebook.get("cells", [])
        ):
            if cell.get("cell_type") != "code":
                continue

            source = cell.get("source", "")

            if isinstance(source, list):
                source = "".join(source)

            cells.append({
                "cell": index,
                "lines": source.splitlines()
            })

        return cells, "ok"

    except Exception as exc:
        return None, f"read_error: {str(exc)[:120]}"

# -------------------------------------------------------------------------
# 34A — BOUNDED DISCOVERY
# -------------------------------------------------------------------------

candidate_paths = []
scope_reports = []

for folder in SEARCH_FOLDERS:
    paths, truncated, inspected = discover(
        folder,
        MAX_DEPTH[str(folder)],
        MAX_FILES_PER_FOLDER
    )

    candidate_paths.extend(paths)

    scope_reports.append({
        "folder": str(folder),
        "entries_inspected": inspected,
        "source_files_found": len(paths),
        "scan_truncated": truncated
    })

    print(
        f"\nFolder: {folder}\n"
        f"Entries inspected: {inspected}\n"
        f"Source files found: {len(paths)}\n"
        f"Truncated: {truncated}",
        flush=True
    )

# Also inspect notebook/script files directly in MyDrive.
try:
    root_files = [
        p for p in ROOT.iterdir()
        if p.is_file()
        and p.suffix.lower() in CODE_EXTENSIONS
    ]
except OSError:
    root_files = []

candidate_paths.extend(root_files)
candidate_paths = sorted(set(candidate_paths))

print("\nTotal candidate code files:", len(candidate_paths))

# -------------------------------------------------------------------------
# 34B — SOURCE CLASSIFICATION
# -------------------------------------------------------------------------

inventory = []
matches = []
excerpts = []

for file_index, path in enumerate(candidate_paths, start=1):
    cells, read_status = extract_code(path)

    if cells is None:
        inventory.append({
            "path": str(path),
            "file_sha256": "",
            "code_cells": 0,
            "matched_lines": 0,
            "score": 0,
            "matched_categories": "",
            "status": read_status
        })
        continue

    categories = set()
    file_matches = []

    for cell in cells:
        cell_id = int(cell["cell"])
        lines = cell["lines"]

        for line_index, line in enumerate(lines):
            current = [
                name for name, pattern in PATTERNS.items()
                if pattern.search(line)
            ]

            if not current:
                continue

            categories.update(current)

            match = {
                "path": str(path),
                "cell_index": cell_id,
                "line_in_cell": line_index + 1,
                "categories": ";".join(current),
                "code": line[:320]
            }

            matches.append(match)
            file_matches.append(match)

    # Score implementation evidence rather than number
    # of generic search hits.
    score = (
        7 * ("quadratic_subpixel" in categories)
        + 6 * ("phase_algorithm" in categories)
        + 4 * ("fft_implementation" in categories)
        + 4 * ("ncc_algorithm" in categories)
        + 4 * ("identity_gate" in categories)
        + 4 * ("bounded_phase" in categories)
        + 2 * ("registration_shift" in categories)
        + 2 * ("spectral_projection" in categories)
    )

    # Prefer source files that include multiple algorithm
    # elements in the same notebook.
    if (
        "ncc_algorithm" in categories
        and "quadratic_subpixel" in categories
    ):
        score += 8

    if (
        "phase_algorithm" in categories
        and "fft_implementation" in categories
    ):
        score += 8

    inventory.append({
        "path": str(path),
        "file_sha256": sha256(path),
        "code_cells": len(cells),
        "matched_lines": len(file_matches),
        "score": score,
        "matched_categories": ";".join(sorted(categories)),
        "status": "parsed"
    })

    if score >= 8:
        excerpts.append(
            "\n" + "=" * 88
            + f"\nFILE: {path}"
            + f"\nSCORE: {score}"
            + "\n" + "=" * 88
        )

        # Include local code context for a bounded number of
        # matches, without copying entire notebooks.
        selected_matches = 0

        for cell in cells:
            lines = cell["lines"]

            for i, line in enumerate(lines):
                if not any(p.search(line) for p in PATTERNS.values()):
                    continue

                selected_matches += 1

                if selected_matches > 12:
                    break

                excerpts.append(
                    f"\n--- Cell {cell['cell']}, "
                    f"line {i + 1} ---"
                )

                for k in range(
                    max(0, i - 4),
                    min(len(lines), i + 8)
                ):
                    excerpts.append(
                        f"{k + 1:04d}: {lines[k][:240]}"
                    )

            if selected_matches > 12:
                break

inventory_df = pd.DataFrame(inventory)

matches_df = pd.DataFrame(
    matches,
    columns=[
        "path",
        "cell_index",
        "line_in_cell",
        "categories",
        "code"
    ]
)

if not inventory_df.empty:
    inventory_df = inventory_df.sort_values(
        ["score", "matched_lines"],
        ascending=[False, False]
    )

high_priority = (
    inventory_df[inventory_df["score"] >= 8].copy()
    if not inventory_df.empty
    else pd.DataFrame(columns=inventory_df.columns)
)

# -------------------------------------------------------------------------
# 34C — SAVE AUDIT EVIDENCE
# -------------------------------------------------------------------------

inventory_df.to_csv(
    OUT / "Step34_Notebook_Inventory.csv",
    index=False
)

matches_df.to_csv(
    OUT / "Step34_Code_Matches.csv",
    index=False
)

high_priority.to_csv(
    OUT / "Step34_High_Priority_Candidates.csv",
    index=False
)

(OUT / "Step34_Candidate_Code_Excerpts.txt").write_text(
    "\n".join(excerpts),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("34C — HIGHEST-PRIORITY CODE CANDIDATES")
print("=" * 88)

if high_priority.empty:
    print(
        "No high-priority registration source candidates "
        "were identified within the bounded search."
    )
else:
    print(
        high_priority[
            ["path", "score", "matched_categories"]
        ].head(20).to_string(
            index=False,
            max_colwidth=110
        )
    )

ncc_high_priority = (
    int(high_priority["matched_categories"].str.contains(
        "quadratic_subpixel"
    ).sum())
    if not high_priority.empty else 0
)

phase_high_priority = (
    int(high_priority["matched_categories"].str.contains(
        "phase_algorithm|fft_implementation",
        regex=True
    ).sum())
    if not high_priority.empty else 0
)

# -------------------------------------------------------------------------
# 34D — FINAL REPORT
# -------------------------------------------------------------------------

report = {
    "experiment": "Colab notebook source-recovery audit",
    "search_scopes": scope_reports,
    "root_level_notebooks_found": len(root_files),
    "candidate_source_files": len(candidate_paths),
    "source_files_parsed": int(
        (inventory_df["status"] == "parsed").sum()
    ) if not inventory_df.empty else 0,
    "matched_code_lines": len(matches_df),
    "high_priority_source_candidates": len(high_priority),
    "quadratic_subpixel_candidate_files": ncc_high_priority,
    "phase_or_fft_candidate_files": phase_high_priority,
    "original_step13_implementation_verified": False,
    "original_step20_implementation_verified": False,
    "external_reproduction_completed": False,
    "source_files_modified": False,
    "status": (
        "SOURCE_CANDIDATES_REQUIRE_MANUAL_VERIFICATION"
        if len(high_priority)
        else "NO_HIGH_PRIORITY_SOURCE_FOUND_IN_SEARCH_SCOPE"
    ),
    "limitations": [
        "Code keywords alone cannot establish method identity.",
        "Notebook source search is intentionally bounded.",
        "Notebooks may exist outside inspected directories.",
        "A Colab runtime may contain unsaved execution history.",
        "No recovered source was executed.",
        "No registration parameters were modified."
    ],
    "next_requirement": (
        "Review any high-priority candidate functions, "
        "compare them with the saved frozen configurations "
        "and historical MDAS outputs, then establish whether "
        "exact reproduction is possible."
    ),
    "elapsed_seconds": round(
        time.monotonic() - started, 2
    ),
    "output_directory": str(OUT)
}

(OUT / "Step34_Final_Report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 34 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 34 COMPLETE")
print("Existing research data, scripts and checkpoints unchanged.")


In [ ]:

# =============================================================================
# STEP 35 — TARGETED ORIGINAL MDAS REGISTRATION CODE RECOVERY
# =============================================================================
#
# PURPOSE
#   Inspect the specific Colab notebook identified in Step 34.
#   Recover candidate source for:
#       - Quadratic subpixel NCC (Step 12)
#       - Identity-preserving gate (Step 13)
#       - Bounded gradient phase correlation (Step 20)
#
#   Compare recovered code against frozen configurations.
#   Preserve source-cell positions and SHA-256 provenance.
#
# IMPORTANT
#   - Read-only examination of the original notebook.
#   - No execution of notebook cells.
#   - No source-code modifications.
#   - No external registration re-evaluation.
#   - Candidate matches are NOT automatically exact reproductions.
#
# OUTPUT
#   Step35_Notebook_Summary.json
#   Step35_Function_Inventory.csv
#   Step35_Registration_Code_Matches.csv
#   Step35_Candidate_Functions.txt
#   Step35_Relevant_Code_Cells.txt
#   Step35_Config_Consistency_Audit.csv
#   Step35_Final_Report.json
# =============================================================================

from google.colab import drive
from pathlib import Path

import ast
import hashlib
import json
import re
import time

import numpy as np
import pandas as pd

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")

PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

NOTEBOOK = (
    ROOT
    / "Colab Notebooks"
    / "Sensitivity of Hyperspectral and Multispectral Image .ipynb"
)

STEP13_CONFIG = (
    PROJECT
    / "Step13_Identity_Preserving_Gate"
    / "Step13_Frozen_Gate.json"
)

STEP20_CONFIG = (
    PROJECT
    / "Step20_Bounded_Phase_Correlation"
    / "Step20_Frozen_Method.json"
)

OUT = PROJECT / "Step35_Original_Registration_Code_Recovery"

OUT.mkdir(parents=True, exist_ok=True)

OUTPUTS = [
    "Step35_Notebook_Summary.json",
    "Step35_Function_Inventory.csv",
    "Step35_Registration_Code_Matches.csv",
    "Step35_Candidate_Functions.txt",
    "Step35_Relevant_Code_Cells.txt",
    "Step35_Config_Consistency_Audit.csv",
    "Step35_Final_Report.json"
]

existing = [name for name in OUTPUTS if (OUT/name).exists()]

if existing:
    raise FileExistsError(
        "Step 35 outputs already exist. "
        "Preserve the previous audit before rerunning:\n"
        + "\n".join(existing)
    )

print("=" * 88, flush=True)
print("STEP 35 — ORIGINAL MDAS REGISTRATION CODE RECOVERY", flush=True)
print("=" * 88, flush=True)

started = time.monotonic()


# =============================================================================
# 35A — VERIFY SOURCE AND READ FROZEN CONFIGURATIONS
# =============================================================================

def sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


for path in [
    NOTEBOOK,
    STEP13_CONFIG,
    STEP20_CONFIG
]:
    if not path.is_file():
        raise FileNotFoundError(path)

notebook_hash = sha256(NOTEBOOK)

step13 = json.loads(
    STEP13_CONFIG.read_text(encoding="utf-8")
)

step20 = json.loads(
    STEP20_CONFIG.read_text(encoding="utf-8")
)

assert step13["threshold_pixels"] == 0.225

assert step13["base_estimator"] == (
    "Step12 quadratic_subpixel"
)

assert step20["radius"] == 5
assert step20["tau"] == 0.25

notebook = json.loads(
    NOTEBOOK.read_text(encoding="utf-8")
)

cells = notebook.get("cells", [])

print("\n35A — VERIFIED SOURCE")
print("Notebook:", NOTEBOOK)
print("Notebook SHA256:", notebook_hash)
print("Total notebook cells:", len(cells))
print("Step 13 gate:", step13["threshold_pixels"])
print("Step 20 radius:", step20["radius"])
print("Step 20 tau:", step20["tau"])
print("Step 20 bias dy:", step20["bias_dy"])
print("Step 20 bias dx:", step20["bias_dx"])


# =============================================================================
# 35B — EXTRACT CODE CELLS AND SEARCH FOR REGISTRATION COMPONENTS
# =============================================================================

PATTERNS = {
    "quadratic_subpixel": re.compile(
        r"quadratic_subpixel|quadratic.*subpixel|"
        r"subpixel.*quadratic|parabolic.*refin",
        re.I
    ),
    "ncc": re.compile(
        r"\bncc\b|normalized.cross.correlation|"
        r"normalized_cross_correlation",
        re.I
    ),
    "gradient": re.compile(
        r"\bgradient\b|\bsobel\b|\bscharr\b",
        re.I
    ),
    "identity_gate": re.compile(
        r"identity.preserv|identity.gate|"
        r"0\.225|snap.*zero|threshold_pixels",
        re.I
    ),
    "phase_correlation": re.compile(
        r"phase.correlat|phase_corr|"
        r"phaseCorrelate|cross.power.spectrum",
        re.I
    ),
    "fft": re.compile(
        r"\bfft2\b|\bifft2\b|"
        r"\brfft2\b|\birfft2\b",
        re.I
    ),
    "bounded_search": re.compile(
        r"max.shift|search.radius|"
        r"bounded|radius|"
        r"subpixel|refin",
        re.I
    ),
    "phase_bias": re.compile(
        r"bias_dy|bias_dx|"
        r"-1\.010922|-1\.004048",
        re.I
    ),
    "registration_output": re.compile(
        r"estimated_dy|estimated_dx|"
        r"true_dy|true_dx|"
        r"\bepe\b|endpoint.error",
        re.I
    )
}

code_cells = []

for cell_index, cell in enumerate(cells):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])

    if isinstance(source, list):
        source = "".join(source)

    if not source.strip():
        continue

    code_cells.append({
        "cell_index": cell_index,
        "source": source,
        "lines": source.splitlines()
    })

print("\n35B — CODE CELL INVENTORY")
print("Nonempty code cells:", len(code_cells))

match_rows = []
cell_scores = []

for cell in code_cells:

    lines = cell["lines"]

    categories = set()
    matches_in_cell = 0

    for line_number, line in enumerate(lines, 1):

        matching = [
            category
            for category, pattern in PATTERNS.items()
            if pattern.search(line)
        ]

        if not matching:
            continue

        categories.update(matching)
        matches_in_cell += 1

        match_rows.append({
            "cell_index": cell["cell_index"],
            "line_in_cell": line_number,
            "categories": ";".join(matching),
            "code": line[:400]
        })

    # Favor code containing multiple registration components.
    score = (
        8 * ("quadratic_subpixel" in categories)
        + 5 * ("ncc" in categories)
        + 3 * ("gradient" in categories)
        + 8 * ("identity_gate" in categories)
        + 7 * ("phase_correlation" in categories)
        + 5 * ("fft" in categories)
        + 3 * ("bounded_search" in categories)
        + 8 * ("phase_bias" in categories)
        + 2 * ("registration_output" in categories)
    )

    cell_scores.append({
        "cell_index": cell["cell_index"],
        "line_count": len(lines),
        "match_count": matches_in_cell,
        "score": score,
        "categories": ";".join(sorted(categories))
    })

match_df = pd.DataFrame(
    match_rows,
    columns=[
        "cell_index",
        "line_in_cell",
        "categories",
        "code"
    ]
)

score_df = pd.DataFrame(cell_scores)

score_df = score_df.sort_values(
    ["score", "match_count"],
    ascending=[False, False]
)

match_df.to_csv(
    OUT / "Step35_Registration_Code_Matches.csv",
    index=False
)

print("\nMost relevant code cells:")

print(
    score_df.head(20).to_string(index=False)
)


# =============================================================================
# 35C — IDENTIFY FUNCTION DEFINITIONS
# =============================================================================
#
# AST parsing can fail for notebook cells that contain
# Colab magics, shell commands or incomplete Python.
#
# We therefore use AST where possible and an indentation-aware
# textual fallback for function extraction.
# =============================================================================

def safe_ast_tree(source):

    try:
        return ast.parse(source)
    except (SyntaxError, ValueError):
        return None


def collect_ast_functions(tree):

    functions = []

    if tree is None:
        return functions

    for node in ast.walk(tree):

        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef)
        ):

            functions.append({
                "name": node.name,
                "start_line": node.lineno,
                "end_line": node.end_lineno
            })

    return functions


def collect_textual_functions(lines):

    found = []

    start_pattern = re.compile(
        r"^(\s*)(?:async\s+)?def\s+"
        r"([A-Za-z_][A-Za-z0-9_]*)\s*\("
    )

    for index, line in enumerate(lines):

        match = start_pattern.match(line)

        if not match:
            continue

        base_indent = len(match.group(1))
        name = match.group(2)

        end_index = len(lines)

        for j in range(index + 1, len(lines)):

            line_j = lines[j]

            if not line_j.strip():
                continue

            if line_j.lstrip().startswith("#"):
                continue

            indent_j = (
                len(line_j) - len(line_j.lstrip())
            )

            if indent_j <= base_indent:
                end_index = j
                break

        found.append({
            "name": name,
            "start_line": index + 1,
            "end_line": end_index
        })

    return found


function_rows = []
candidate_blocks = []

REGISTRATION_TERMS = re.compile(
    r"ncc|correlat|phase|fft|shift|"
    r"register|subpixel|quadratic|"
    r"gradient|gate|displacement|"
    r"estimate|align",
    re.I
)

for cell in code_cells:

    lines = cell["lines"]

    tree = safe_ast_tree(
        cell["source"]
    )

    definitions = collect_ast_functions(tree)

    extraction_method = "AST"

    if tree is None:
        definitions = collect_textual_functions(lines)
        extraction_method = "TEXT_FALLBACK"

    for definition in definitions:

        start_line = definition["start_line"]
        end_line = definition["end_line"]

        text = "\n".join(
            lines[start_line - 1:end_line]
        )

        categories = [
            name
            for name, pattern in PATTERNS.items()
            if pattern.search(text)
        ]

        relevance = (
            bool(REGISTRATION_TERMS.search(
                definition["name"]
            ))
            or any(
                category in categories
                for category in [
                    "quadratic_subpixel",
                    "ncc",
                    "identity_gate",
                    "phase_correlation",
                    "phase_bias"
                ]
            )
        )

        function_rows.append({
            "cell_index": cell["cell_index"],
            "function_name": definition["name"],
            "start_line": start_line,
            "end_line": end_line,
            "line_count": end_line - start_line + 1,
            "extraction_method": extraction_method,
            "categories": ";".join(categories),
            "candidate_registration_function": relevance,
            "source_sha256": hashlib.sha256(
                text.encode("utf-8")
            ).hexdigest()
        })

        if relevance:

            candidate_blocks.append(
                "\n" + "=" * 88
                + f"\nFUNCTION: {definition['name']}"
                + f"\nCELL: {cell['cell_index']}"
                + f"\nLINES: {start_line}–{end_line}"
                + f"\nEXTRACTION: {extraction_method}"
                + "\n" + "=" * 88
                + "\n"
                + text
                + "\n"
            )

function_df = pd.DataFrame(
    function_rows,
    columns=[
        "cell_index",
        "function_name",
        "start_line",
        "end_line",
        "line_count",
        "extraction_method",
        "categories",
        "candidate_registration_function",
        "source_sha256"
    ]
)

function_df.to_csv(
    OUT / "Step35_Function_Inventory.csv",
    index=False
)

(OUT / "Step35_Candidate_Functions.txt").write_text(
    "\n".join(candidate_blocks),
    encoding="utf-8"
)

candidate_functions = function_df[
    function_df["candidate_registration_function"]
].copy()

print("\n35C — REGISTRATION FUNCTION CANDIDATES")

if candidate_functions.empty:

    print("No candidate function definitions extracted.")

else:

    print(
        candidate_functions[
            [
                "cell_index",
                "function_name",
                "start_line",
                "end_line",
                "categories"
            ]
        ].to_string(
            index=False,
            max_colwidth=90
        )
    )


# =============================================================================
# 35D — SAVE RELEVANT COMPLETE CODE CELLS
# =============================================================================
#
# This file may contain multiple versions of an algorithm.
# It preserves the source so that implementations can be
# compared without relying on keyword summaries.
# =============================================================================

relevant_cell_indices = set(
    score_df.loc[
        score_df["score"] >= 8,
        "cell_index"
    ].astype(int)
)

relevant_cell_blocks = []

for cell in code_cells:

    if cell["cell_index"] not in relevant_cell_indices:
        continue

    numbered_lines = [
        f"{line_number:05d}: {line}"
        for line_number, line
        in enumerate(cell["lines"], 1)
    ]

    relevant_cell_blocks.append(
        "\n" + "=" * 88
        + f"\nNOTEBOOK CELL {cell['cell_index']}"
        + f"\nSOURCE LINES: {len(cell['lines'])}"
        + "\n" + "=" * 88
        + "\n"
        + "\n".join(numbered_lines)
    )

(OUT / "Step35_Relevant_Code_Cells.txt").write_text(
    "\n".join(relevant_cell_blocks),
    encoding="utf-8"
)

print("\nRelevant code cells exported:", len(relevant_cell_blocks))


# =============================================================================
# 35E — CHECK FROZEN CONFIGURATION VALUES IN SOURCE
# =============================================================================
#
# A text match is evidence of appearance only.
# It is NOT proof that a particular notebook cell
# executed during the historical experiment.
# =============================================================================

all_code = "\n".join(
    cell["source"] for cell in code_cells
)

check_specs = [
    {
        "component": "Step13 identity gate",
        "expected_setting": "0.225",
        "regex": r"(?<!\d)0\.225(?!\d)"
    },
    {
        "component": "Step12 quadratic refinement",
        "expected_setting": "quadratic subpixel estimator",
        "regex": r"quadratic_subpixel|quadratic.*subpixel"
    },
    {
        "component": "Step20 bounded phase",
        "expected_setting": "phase gradient/FFT",
        "regex": r"phase_gradient|phase_corr|phaseCorrelate"
    },
    {
        "component": "Step20 radius",
        "expected_setting": "5",
        "regex": r"radius\s*[=:]\s*5\b|"
                 r"['\"]radius['\"]\s*:\s*5\b"
    },
    {
        "component": "Step20 tau",
        "expected_setting": "0.25",
        "regex": r"\btau\s*[=:]\s*0\.25\b|"
                 r"['\"]tau['\"]\s*:\s*0\.25\b"
    },
    {
        "component": "Step20 bias dy",
        "expected_setting": str(step20["bias_dy"]),
        "regex": re.escape(str(step20["bias_dy"]))
    },
    {
        "component": "Step20 bias dx",
        "expected_setting": str(step20["bias_dx"]),
        "regex": re.escape(str(step20["bias_dx"]))
    }
]

consistency_rows = []

for spec in check_specs:

    found = bool(
        re.search(
            spec["regex"],
            all_code,
            flags=re.IGNORECASE | re.DOTALL
        )
    )

    consistency_rows.append({
        "component": spec["component"],
        "expected_setting": spec["expected_setting"],
        "source_text_match_found": found,
        "establishes_exact_implementation": False
    })

consistency_df = pd.DataFrame(
    consistency_rows
)

consistency_df.to_csv(
    OUT / "Step35_Config_Consistency_Audit.csv",
    index=False
)

print("\n35D — FROZEN CONFIGURATION CONSISTENCY")

print(
    consistency_df[
        [
            "component",
            "expected_setting",
            "source_text_match_found"
        ]
    ].to_string(index=False)
)


# =============================================================================
# 35F — NOTEBOOK PROVENANCE RECORD
# =============================================================================

notebook_summary = {
    "notebook_path": str(NOTEBOOK),
    "notebook_sha256": notebook_hash,
    "notebook_format": notebook.get("nbformat"),
    "notebook_minor_version": notebook.get("nbformat_minor"),
    "total_cells": len(cells),
    "nonempty_code_cells": len(code_cells),
    "candidate_function_count": len(candidate_functions),
    "relevant_code_cells_exported": len(relevant_cell_blocks),
    "step13_frozen_config_sha256": sha256(STEP13_CONFIG),
    "step20_frozen_config_sha256": sha256(STEP20_CONFIG),
    "original_notebook_modified": False
}

(OUT / "Step35_Notebook_Summary.json").write_text(
    json.dumps(notebook_summary, indent=2),
    encoding="utf-8"
)


# =============================================================================
# 35G — FINAL REPORT
# =============================================================================

report = {
    "experiment": (
        "Targeted original MDAS registration code recovery"
    ),
    "source_notebook_sha256": notebook_hash,
    "source_notebook": str(NOTEBOOK),
    "code_cells_examined": len(code_cells),
    "matched_code_lines": len(match_df),
    "function_definitions_found": len(function_df),
    "candidate_registration_functions": len(
        candidate_functions
    ),
    "relevant_code_cells_exported": len(
        relevant_cell_blocks
    ),
    "frozen_settings_text_matches": {
        row["component"]: bool(
            row["source_text_match_found"]
        )
        for row in consistency_rows
    },
    "step13_exact_source_verified": False,
    "step20_exact_source_verified": False,
    "historical_mdas_output_reproduced": False,
    "potsdam_exact_method_transfer_completed": False,
    "original_source_modified": False,
    "status": (
        "CANDIDATE_FUNCTIONS_RECOVERED_FOR_VERIFICATION"
        if len(candidate_functions) > 0
        else "NO_REGISTRATION_FUNCTIONS_EXTRACTED"
    ),
    "next_requirement": (
        "Inspect the recovered candidate functions; "
        "identify historical Step12/13/20 implementation "
        "versions and verify their outputs against saved "
        "MDAS registration records before external evaluation."
    ),
    "methodological_limitations": [
        "A function-name or text match does not establish "
        "historical execution identity.",
        "The notebook may contain revised versions "
        "of the same estimator.",
        "AST extraction may be incomplete for cells "
        "containing notebook magics.",
        "No recovered function was executed.",
        "Frozen settings do not fully define algorithms.",
        "External testing should occur only after "
        "historical reproduction is established."
    ],
    "elapsed_seconds": round(
        time.monotonic() - started, 2
    ),
    "output_directory": str(OUT)
}

(OUT / "Step35_Final_Report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 35 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 35 COMPLETE")
print("Original notebook and research results unchanged.")


In [ ]:

# =============================================================================
# STEP 36 — CORRECTED HISTORICAL MDAS SOURCE VERIFICATION
# =============================================================================
# FIX:
#   A changed notebook SHA256 no longer causes automatic failure.
#   Actual registration function source is compared with Step 35.
#
# SOURCE:
#   MyDrive/Colab Notebooks/
#   Sensitivity of Hyperspectral and Multispectral Image .ipynb
#
# INPUT AUDIT:
#   Step35_Function_Inventory.csv
#
# OUTPUTS:
#   Step36_Function_Provenance.csv
#   Step36_Recovered_Functions.txt
#   Step36_Notebook_Change_Audit.json
#   Step36_Historical_Result_Inventory.csv
#   Step36_Configuration_Evidence.csv
#   Step36_Final_Report.json
#
# DOES NOT EXECUTE RECOVERED SOURCE.
# DOES NOT MODIFY NOTEBOOKS OR HISTORICAL RESULTS.
# =============================================================================

from google.colab import drive
from pathlib import Path
import ast
import hashlib
import json
import re
import time
import pandas as pd

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

NOTEBOOK = (
    ROOT / "Colab Notebooks"
    / "Sensitivity of Hyperspectral and Multispectral Image .ipynb"
)

STEP35 = PROJECT / "Step35_Original_Registration_Code_Recovery"

STEP35_INVENTORY = (
    STEP35 / "Step35_Function_Inventory.csv"
)

STEP35_NOTEBOOK_SUMMARY = (
    STEP35 / "Step35_Notebook_Summary.json"
)

STEP13_CONFIG = (
    PROJECT / "Step13_Identity_Preserving_Gate"
    / "Step13_Frozen_Gate.json"
)

STEP20_CONFIG = (
    PROJECT / "Step20_Bounded_Phase_Correlation"
    / "Step20_Frozen_Method.json"
)

OUT = PROJECT / "Step36_Historical_Algorithm_Verification"
OUT.mkdir(parents=True, exist_ok=True)

TARGETS = {
    15: [
        "integer_shift_np",
        "ncc",
        "score_surface",
        "quadratic_vertex",
        "estimate_both",
    ],
    16: [
        "gated_displacement",
    ],
    22: [
        "phase_estimate",
    ],
    24: [
        "phase_surface",
    ],
}

OUTPUT_NAMES = [
    "Step36_Function_Provenance.csv",
    "Step36_Recovered_Functions.txt",
    "Step36_Notebook_Change_Audit.json",
    "Step36_Historical_Result_Inventory.csv",
    "Step36_Configuration_Evidence.csv",
    "Step36_Final_Report.json",
]

existing = [
    name for name in OUTPUT_NAMES
    if (OUT / name).exists()
]

if existing:
    raise FileExistsError(
        "Previous Step 36 outputs exist. "
        "Preserve them before rerunning:\n"
        + "\n".join(existing)
    )

print("=" * 88, flush=True)
print(
    "STEP 36 — CORRECTED HISTORICAL MDAS SOURCE VERIFICATION",
    flush=True
)
print("=" * 88, flush=True)

started = time.monotonic()


# =============================================================================
# 36A — INPUT VERIFICATION
# =============================================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)
    return h.hexdigest()


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def load_json(path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )


def save_json(path, obj):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            default=str
        ),
        encoding="utf-8"
    )


required = [
    NOTEBOOK,
    STEP35_INVENTORY,
    STEP35_NOTEBOOK_SUMMARY,
    STEP13_CONFIG,
    STEP20_CONFIG
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)

notebook = load_json(NOTEBOOK)
old_summary = load_json(STEP35_NOTEBOOK_SUMMARY)
step13 = load_json(STEP13_CONFIG)
step20 = load_json(STEP20_CONFIG)

step35_functions = pd.read_csv(STEP35_INVENTORY)

old_notebook_sha = old_summary["notebook_sha256"]
new_notebook_sha = sha256_file(NOTEBOOK)

assert len(notebook.get("cells", [])) >= 25

assert float(step13["threshold_pixels"]) == 0.225
assert int(step20["radius"]) == 5
assert float(step20["tau"]) == 0.25

print("\n36A — NOTEBOOK PROVENANCE")

print("Step 35 notebook SHA:", old_notebook_sha)
print("Current notebook SHA:", new_notebook_sha)

notebook_changed = (
    old_notebook_sha != new_notebook_sha
)

print("Whole notebook changed:", notebook_changed)

if notebook_changed:
    print(
        "\nNOTE: Notebook container changed."
        "\nThis does not establish that function source changed."
        "\nComparing function bodies next."
    )


# =============================================================================
# 36B — EXTRACT TARGET FUNCTIONS
# =============================================================================

def cell_source(cell_index):
    cell = notebook["cells"][cell_index]

    if cell.get("cell_type") != "code":
        raise ValueError(
            f"Cell {cell_index} is not code."
        )

    source = cell.get("source", "")

    if isinstance(source, list):
        source = "".join(source)

    return str(source)


def extract_function(source, function_name):
    """
    Match Step 35's extraction method where possible.

    First use AST to find a function.
    If notebook syntax prevents AST parsing,
    use the same indentation-based fallback.
    """

    lines = source.splitlines()

    try:
        tree = ast.parse(source)

        functions = [
            node
            for node in ast.walk(tree)
            if isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            )
            and node.name == function_name
        ]

        if functions:
            node = functions[0]

            text = "\n".join(
                lines[
                    node.lineno - 1:
                    node.end_lineno
                ]
            )

            return {
                "found": True,
                "source": text,
                "start_line": node.lineno,
                "end_line": node.end_lineno,
                "extraction": "AST"
            }

    except (SyntaxError, ValueError):
        pass

    # Fallback: preserve Step 35's indentation logic.
    pattern = re.compile(
        r"^(\s*)(?:async\s+)?def\s+"
        + re.escape(function_name)
        + r"\s*\("
    )

    for index, line in enumerate(lines):
        match = pattern.match(line)

        if not match:
            continue

        base_indent = len(match.group(1))
        end_index = len(lines)

        for j in range(index + 1, len(lines)):
            candidate = lines[j]

            if not candidate.strip():
                continue

            if candidate.lstrip().startswith("#"):
                continue

            indent = (
                len(candidate)
                - len(candidate.lstrip())
            )

            if indent <= base_indent:
                end_index = j
                break

        text = "\n".join(
            lines[index:end_index]
        )

        return {
            "found": True,
            "source": text,
            "start_line": index + 1,
            "end_line": end_index,
            "extraction": "TEXT_FALLBACK"
        }

    return {
        "found": False,
        "source": "",
        "start_line": None,
        "end_line": None,
        "extraction": "NOT_FOUND"
    }


# =============================================================================
# 36C — COMPARE WITH STEP 35 FUNCTION HASHES
# =============================================================================

print("\n36B — FUNCTION-LEVEL SOURCE VERIFICATION")

rows = []
source_blocks = []

for cell_index, function_names in TARGETS.items():

    original_cell_source = cell_source(
        cell_index
    )

    for function_name in function_names:

        extracted = extract_function(
            original_cell_source,
            function_name
        )

        old_rows = step35_functions[
            (step35_functions["cell_index"] == cell_index)
            &
            (
                step35_functions["function_name"]
                == function_name
            )
        ]

        if len(old_rows) == 1:
            old_hash = str(
                old_rows.iloc[0]["source_sha256"]
            )
            if old_hash.lower() == "nan":
                old_hash = None
        else:
            old_hash = None

        if extracted["found"]:
            new_hash = sha256_text(
                extracted["source"]
            )
        else:
            new_hash = None

        match = (
            old_hash is not None
            and new_hash is not None
            and old_hash == new_hash
        )

        if not extracted["found"]:
            status = "MISSING_CURRENT_FUNCTION"

        elif old_hash is None:
            status = "NO_STEP35_HASH"

        elif match:
            status = "SOURCE_UNCHANGED"

        else:
            status = "SOURCE_CHANGED"

        rows.append({
            "cell_index": cell_index,
            "function_name": function_name,
            "found_current": extracted["found"],
            "step35_source_sha256": old_hash,
            "current_source_sha256": new_hash,
            "source_hash_match": match,
            "status": status,
            "start_line": extracted["start_line"],
            "end_line": extracted["end_line"],
            "extraction_method": extracted["extraction"]
        })

        if extracted["found"]:
            source_blocks.append(
                "\n" + "=" * 88
                + f"\nCELL: {cell_index}"
                + f"\nFUNCTION: {function_name}"
                + f"\nSTATUS: {status}"
                + f"\nCURRENT SHA256: {new_hash}"
                + "\n" + "=" * 88
                + "\n"
                + extracted["source"]
                + "\n"
            )

        print(
            f"Cell {cell_index:02d} | "
            f"{function_name:22s} | {status}",
            flush=True
        )

function_df = pd.DataFrame(rows)

function_df.to_csv(
    OUT / "Step36_Function_Provenance.csv",
    index=False
)

(OUT / "Step36_Recovered_Functions.txt").write_text(
    "\n".join(source_blocks),
    encoding="utf-8"
)

total_targets = len(function_df)

matching_count = int(
    function_df["source_hash_match"].sum()
)

changed_count = int(
    (function_df["status"] == "SOURCE_CHANGED").sum()
)

missing_count = int(
    (function_df["status"] == "MISSING_CURRENT_FUNCTION").sum()
)

print("\nSource matches:", matching_count, "/", total_targets)
print("Changed function bodies:", changed_count)
print("Missing current functions:", missing_count)


# =============================================================================
# 36D — NOTEBOOK CHANGE INTERPRETATION
# =============================================================================

print("\n36C — NOTEBOOK CHANGE ATTRIBUTION")

all_target_functions_unchanged = (
    matching_count == total_targets
)

if (
    notebook_changed
    and all_target_functions_unchanged
):
    interpretation = (
        "Whole notebook changed, but all selected "
        "registration function bodies match Step 35. "
        "The differing bytes may be notebook metadata, "
        "outputs, execution counts, or other cells. "
        "The exact cause is not established."
    )

elif (
    not notebook_changed
    and all_target_functions_unchanged
):
    interpretation = (
        "Whole notebook and all selected "
        "registration functions match Step 35."
    )

else:
    interpretation = (
        "At least one targeted function differs from "
        "Step 35 or cannot be verified. "
        "Do not claim source equivalence."
    )

change_audit = {
    "step35_notebook_sha256": old_notebook_sha,
    "current_notebook_sha256": new_notebook_sha,
    "whole_notebook_changed": notebook_changed,
    "target_functions": total_targets,
    "matching_function_hashes": matching_count,
    "changed_function_hashes": changed_count,
    "missing_current_functions": missing_count,
    "all_target_functions_unchanged": (
        all_target_functions_unchanged
    ),
    "interpretation": interpretation
}

save_json(
    OUT / "Step36_Notebook_Change_Audit.json",
    change_audit
)

print(interpretation)


# =============================================================================
# 36E — HISTORICAL CONFIGURATION SOURCE EVIDENCE
# =============================================================================

print("\n36D — FROZEN PARAMETER EVIDENCE")

focus_cells = [
    12, 13, 14, 15, 16, 17,
    22, 23, 24, 25, 26
]

patterns = {
    "gate_0_225": re.compile(
        r"(?<!\d)0\.225(?!\d)"
    ),
    "quadratic": re.compile(
        r"quadratic|parabolic",
        re.I
    ),
    "ncc": re.compile(
        r"\bncc\b|normalized.*correlation",
        re.I
    ),
    "phase": re.compile(
        r"phase|cross.power.spectrum",
        re.I
    ),
    "fft": re.compile(
        r"\bfft2\b|\bifft2\b|\brfft2\b",
        re.I
    ),
    "tau": re.compile(
        r"\btau\b",
        re.I
    ),
    "bias": re.compile(
        r"bias_dy|bias_dx|"
        r"1\.010922|1\.004048",
        re.I
    )
}

evidence_rows = []

for cell_index in focus_cells:
    source = cell_source(cell_index)

    for line_no, line in enumerate(
        source.splitlines(), 1
    ):

        matched = [
            name
            for name, pattern in patterns.items()
            if pattern.search(line)
        ]

        if matched:
            evidence_rows.append({
                "cell_index": cell_index,
                "line_number": line_no,
                "categories": ";".join(matched),
                "source_line": line[:400]
            })

evidence_df = pd.DataFrame(
    evidence_rows,
    columns=[
        "cell_index",
        "line_number",
        "categories",
        "source_line"
    ]
)

evidence_df.to_csv(
    OUT / "Step36_Configuration_Evidence.csv",
    index=False
)

print(
    "Historical configuration references:",
    len(evidence_df)
)

print(
    evidence_df[
        [
            "cell_index",
            "line_number",
            "categories",
            "source_line"
        ]
    ].head(25).to_string(
        index=False,
        max_colwidth=75
    )
)


# =============================================================================
# 36F — TARGETED HISTORICAL RESULT INVENTORY
# =============================================================================

print("\n36E — HISTORICAL RESULT FILE INVENTORY")

RESULT_FOLDERS = [
    PROJECT / "Step09_Estimated_Registration",
    PROJECT / "Step10_Registration_Comparison",
    PROJECT / "Step11_Subpixel_Registration",
    PROJECT / "Step12_Quadratic_Subpixel",
    PROJECT / "Step13_Identity_Preserving_Gate",
    PROJECT / "Step19B_Corrected_Comparison",
    PROJECT / "Step20_Bounded_Phase_Correlation"
]

# Discover directly relevant Step folders if exact names differ.
for folder in PROJECT.iterdir():
    if (
        folder.is_dir()
        and re.match(
            r"Step(?:09|10|11|12|13|19|20)(?:\D|$)",
            folder.name,
            flags=re.I
        )
    ):
        RESULT_FOLDERS.append(folder)

RESULT_FOLDERS = sorted(
    set(RESULT_FOLDERS)
)

result_rows = []

for folder in RESULT_FOLDERS:

    if not folder.is_dir():
        continue

    # One folder level only. Avoid another Drive-wide scan.
    for path in sorted(folder.iterdir()):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (
            ".csv", ".json", ".npz"
        ):
            continue

        record = {
            "folder": folder.name,
            "filename": path.name,
            "path": str(path),
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
            "columns": ""
        }

        if path.suffix.lower() == ".csv":
            try:
                columns = pd.read_csv(
                    path,
                    nrows=0
                ).columns.tolist()

                record["columns"] = ";".join(columns)

            except Exception as exc:
                record["columns"] = (
                    "ERROR:" + str(exc)[:100]
                )

        result_rows.append(record)

result_df = pd.DataFrame(
    result_rows,
    columns=[
        "folder",
        "filename",
        "path",
        "size_bytes",
        "sha256",
        "columns"
    ]
)

result_df.to_csv(
    OUT / "Step36_Historical_Result_Inventory.csv",
    index=False
)

print("Historical files inventoried:", len(result_df))

if not result_df.empty:

    likely = result_df[
        result_df["filename"].str.contains(
            "registr|displac|shift|phase|"
            "gate|ncc|test",
            case=False,
            regex=True
        )
    ]

    print("\nLikely reproduction-reference files:")

    print(
        likely[
            ["folder", "filename", "columns"]
        ].head(25).to_string(
            index=False,
            max_colwidth=75
        )
    )


# =============================================================================
# 36G — FINAL REPORT
# =============================================================================

print("\n" + "=" * 88)
print("STEP 36 — FINAL REPORT")
print("=" * 88)

if all_target_functions_unchanged:
    status = "TARGET_FUNCTION_SOURCE_MATCHES_STEP35"
else:
    status = "TARGET_SOURCE_VERSION_MISMATCH_REQUIRES_REVIEW"

report = {
    "experiment": (
        "Historical MDAS registration source provenance verification"
    ),
    "status": status,
    "notebook_changed_since_step35": notebook_changed,
    "step35_notebook_sha256": old_notebook_sha,
    "current_notebook_sha256": new_notebook_sha,
    "target_functions": total_targets,
    "function_hashes_matching_step35": matching_count,
    "function_hashes_changed": changed_count,
    "missing_target_functions": missing_count,
    "all_target_functions_unchanged": (
        all_target_functions_unchanged
    ),
    "historical_result_files_inventoried": len(result_df),
    "step13_identity_gate_threshold": float(
        step13["threshold_pixels"]
    ),
    "step20_radius": int(step20["radius"]),
    "step20_tau": float(step20["tau"]),
    "step20_bias_dy": float(step20["bias_dy"]),
    "step20_bias_dx": float(step20["bias_dx"]),
    "original_step13_algorithm_numerically_reproduced": False,
    "original_step20_algorithm_numerically_reproduced": False,
    "exact_potsdam_transfer_completed": False,
    "notebook_code_executed": False,
    "historical_results_modified": False,
    "next_requirement": (
        "Review extracted functions and dependencies, "
        "identify their original historical execution context, "
        "and compare reproduced displacements with saved "
        "MDAS reference outputs before external transfer."
    ),
    "limitations": [
        "Matching function bodies does not verify dependencies.",
        "Matching source does not establish historical execution.",
        "The full notebook may change through Colab autosave.",
        "Notebook execution order may differ from cell order.",
        "The frozen phase configuration may be stored outside the notebook.",
        "No historical displacement reproduction was performed.",
        "No Potsdam registration method was rerun."
    ],
    "elapsed_seconds": round(
        time.monotonic() - started, 2
    ),
    "output_directory": str(OUT)
}

save_json(
    OUT / "Step36_Final_Report.json",
    report
)

print(json.dumps(report, indent=2))

print("\nSTEP 36 COMPLETE")
print("Original notebook and registration outputs unchanged.")


In [ ]:

# =============================================================================
# STEP 37 — MDAS HISTORICAL NUMERICAL CONSISTENCY AUDIT
#
# Objectives:
#   1. Verify the Step 36 recovered-source provenance.
#   2. Replay the frozen 0.225-pixel identity gate using
#      the saved Step 13 original displacement estimates.
#   3. Check whether Step 12 and Step 13 estimates agree
#      on matching image/translation cases.
#   4. Inspect the saved Step 20 phase-correlation columns.
#   5. Identify external dependencies of recovered functions.
#
# This script DOES NOT:
#   - Execute recovered notebook code.
#   - Rerun NCC or phase correlation.
#   - Adjust the gate threshold.
#   - Tune anything using MDAS test or Potsdam results.
#   - Overwrite historical experimental outputs.
#
# Source integrity:
#   - Uses Step 36 function-level provenance report.
#   - Uses the original historical CSVs as references.
# =============================================================================

from google.colab import drive
from pathlib import Path
import ast
import builtins
import hashlib
import json
import symtable
import time

import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

STEP12 = PROJECT / "Step12_Subpixel_Robustness"
STEP13 = PROJECT / "Step13_Identity_Preserving_Gate"
STEP20 = PROJECT / "Step20_Bounded_Phase_Correlation"
STEP36 = PROJECT / "Step36_Historical_Algorithm_Verification"

OUT = PROJECT / "Step37_Historical_Numerical_Consistency"
OUT.mkdir(parents=True, exist_ok=True)

FILES = {
    "step12": STEP12 / "Step12_Test_Registration.csv",
    "step13": STEP13 / "Step13_Gated_Test_Registration.csv",
    "step13_config": STEP13 / "Step13_Frozen_Gate.json",
    "step20": STEP20 / "Step20_Test_Registration.csv",
    "step20_config": STEP20 / "Step20_Frozen_Method.json",
    "source_provenance": STEP36 / "Step36_Function_Provenance.csv",
    "recovered_functions": STEP36 / "Step36_Recovered_Functions.txt",
}

OUTPUT_NAMES = [
    "Step37_Gate_Replay.csv",
    "Step37_Step12_Step13_Reconciliation.csv",
    "Step37_Historical_Data_Schema.json",
    "Step37_Recovered_Function_Dependencies.csv",
    "Step37_Final_Report.json"
]

print("=" * 88)
print("STEP 37 — HISTORICAL NUMERICAL CONSISTENCY")
print("=" * 88)

for name, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

existing = [
    name for name in OUTPUT_NAMES
    if (OUT / name).exists()
]

if existing:
    raise FileExistsError(
        "Previous Step 37 outputs exist. "
        "Please preserve them:\n" + "\n".join(existing)
    )

start = time.monotonic()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

def save_json(name, obj):
    (OUT / name).write_text(
        json.dumps(obj, indent=2, default=str),
        encoding="utf-8"
    )

def normalize_images(values):
    return (
        values.astype(str)
        .str.replace("\\", "/", regex=False)
        .str.split("/")
        .str[-1]
    )

# -------------------------------------------------------------------------
# 37A — VERIFY RECOVERED SOURCE PROVENANCE
# -------------------------------------------------------------------------

print("\n37A — SOURCE PROVENANCE")

source = pd.read_csv(FILES["source_provenance"])

assert len(source) == 8

assert (
    source["status"] == "SOURCE_UNCHANGED"
).all(), "Not all recovered source functions are verified."

config13 = json.loads(
    FILES["step13_config"].read_text(encoding="utf-8")
)

config20 = json.loads(
    FILES["step20_config"].read_text(encoding="utf-8")
)

GATE = float(config13["threshold_pixels"])

assert np.isclose(GATE, 0.225)
assert int(config20["radius"]) == 5
assert np.isclose(float(config20["tau"]), 0.25)

print("Verified historical functions:", len(source))
print("Identity gate:", GATE, "pixels")
print("Phase radius:", config20["radius"])
print("Phase tau:", config20["tau"])
print("Phase bias dy:", config20["bias_dy"])
print("Phase bias dx:", config20["bias_dx"])

# -------------------------------------------------------------------------
# 37B — REPLAY THE FROZEN IDENTITY GATE
# -------------------------------------------------------------------------

print("\n37B — STEP 13 GATE NUMERICAL REPLAY")

step13 = pd.read_csv(FILES["step13"])

required = [
    "image", "true_dy", "true_dx",
    "original_dy", "original_dx",
    "gated_dy", "gated_dx"
]

missing = [c for c in required if c not in step13.columns]

if missing:
    raise ValueError(
        "Unexpected Step 13 schema. Missing: "
        + ", ".join(missing)
        + "\nAvailable columns: "
        + ", ".join(step13.columns)
    )

for column in required[1:]:
    step13[column] = pd.to_numeric(
        step13[column], errors="raise"
    )

original_y = step13["original_dy"].to_numpy(float)
original_x = step13["original_dx"].to_numpy(float)

recorded_y = step13["gated_dy"].to_numpy(float)
recorded_x = step13["gated_dx"].to_numpy(float)

assert np.isfinite(
    np.column_stack([
        original_y, original_x,
        recorded_y, recorded_x
    ])
).all()

magnitude = np.hypot(original_y, original_x)

# Both boundary conventions are checked.
# This is diagnostic and is not a threshold optimization.

candidate_rules = {}

for rule_name, snap_mask in [
    ("magnitude_less_than_threshold", magnitude < GATE),
    ("magnitude_less_equal_threshold", magnitude <= GATE)
]:
    predicted_y = np.where(
        snap_mask, 0.0, original_y
    )
    predicted_x = np.where(
        snap_mask, 0.0, original_x
    )

    match = (
        np.isclose(predicted_y, recorded_y, atol=1e-8, rtol=0)
        & np.isclose(predicted_x, recorded_x, atol=1e-8, rtol=0)
    )

    candidate_rules[rule_name] = {
        "matches": int(match.sum()),
        "total": int(len(match)),
        "all_match": bool(match.all()),
        "snapped_count": int(snap_mask.sum())
    }

# Use the stated strict-less-than convention as the main
# replay. The alternative is reported, not substituted.

replay_snap = magnitude < GATE

replay_y = np.where(
    replay_snap, 0.0, original_y
)
replay_x = np.where(
    replay_snap, 0.0, original_x
)

replay_match = (
    np.isclose(replay_y, recorded_y, atol=1e-8, rtol=0)
    & np.isclose(replay_x, recorded_x, atol=1e-8, rtol=0)
)

gate_replay = step13.copy()

gate_replay["original_magnitude"] = magnitude
gate_replay["replayed_gated_dy"] = replay_y
gate_replay["replayed_gated_dx"] = replay_x
gate_replay["gate_replay_match"] = replay_match

gate_replay.to_csv(
    OUT / "Step37_Gate_Replay.csv",
    index=False
)

print("Historical cases:", len(step13))

for rule, stats in candidate_rules.items():
    print(
        f"{rule}: "
        f"{stats['matches']}/{stats['total']} matches, "
        f"{stats['snapped_count']} snapped"
    )

if not replay_match.all():
    print("\nFirst gate mismatches:")
    print(
        gate_replay.loc[
            ~gate_replay["gate_replay_match"],
            [
                "image", "true_dy", "true_dx",
                "original_dy", "original_dx",
                "gated_dy", "gated_dx",
                "replayed_gated_dy", "replayed_gated_dx"
            ]
        ].head(12).to_string(index=False)
    )

# -------------------------------------------------------------------------
# 37C — STEP 12 / STEP 13 HISTORICAL RECONCILIATION
# -------------------------------------------------------------------------

print("\n37C — STEP 12 AND STEP 13 RECONCILIATION")

step12 = pd.read_csv(FILES["step12"])

step12_columns = step12.columns.tolist()

comparison = {
    "attempted": False,
    "matched_cases": 0,
    "step13_cases": int(len(step13)),
    "mean_absolute_dy_difference": None,
    "mean_absolute_dx_difference": None,
    "exact_displacement_matches": None,
    "note": ""
}

# Step 12 and 13 should have comparable image/shift keys.
keys12 = ["image", "true_dy", "true_dx"]

if all(k in step12.columns for k in keys12):

    left = step13[
        [
            "image", "true_dy", "true_dx",
            "original_dy", "original_dx"
        ]
    ].copy()

    right = step12.copy()

    left["image_key"] = normalize_images(left["image"])
    right["image_key"] = normalize_images(right["image"])

    join_keys = ["image_key", "true_dy", "true_dx"]

    for df in [left, right]:
        for key in ["true_dy", "true_dx"]:
            df[key] = pd.to_numeric(
                df[key], errors="raise"
            ).round(6)

    # Use original Step 12 refined estimates if available.
    possible_pairs = [
        ("estimated_dy", "estimated_dx"),
        ("refined_dy", "refined_dx")
    ]

    pair = next(
        (
            p for p in possible_pairs
            if all(k in right.columns for k in p)
        ),
        None
    )

    if pair is not None:

        right = right[
            join_keys + list(pair)
        ].copy()

        right = right.rename(
            columns={
                pair[0]: "step12_dy",
                pair[1]: "step12_dx"
            }
        )

        # Avoid accidental many-to-many matching.
        left_unique = not left.duplicated(join_keys).any()
        right_unique = not right.duplicated(join_keys).any()

        if left_unique and right_unique:

            merged = left.merge(
                right,
                on=join_keys,
                how="left",
                validate="one_to_one",
                indicator=True
            )

            matched = merged[
                merged["_merge"] == "both"
            ].copy()

            if len(matched):

                difference_y = (
                    matched["original_dy"].to_numpy(float)
                    - matched["step12_dy"].to_numpy(float)
                )

                difference_x = (
                    matched["original_dx"].to_numpy(float)
                    - matched["step12_dx"].to_numpy(float)
                )

                exact = (
                    np.isclose(difference_y, 0.0, atol=1e-8, rtol=0)
                    & np.isclose(difference_x, 0.0, atol=1e-8, rtol=0)
                )

                comparison = {
                    "attempted": True,
                    "matched_cases": int(len(matched)),
                    "step13_cases": int(len(step13)),
                    "mean_absolute_dy_difference": float(
                        np.mean(np.abs(difference_y))
                    ),
                    "mean_absolute_dx_difference": float(
                        np.mean(np.abs(difference_x))
                    ),
                    "exact_displacement_matches": int(exact.sum()),
                    "note": (
                        "Joined on image basename and rounded true displacement."
                    )
                }

            merged.to_csv(
                OUT / "Step37_Step12_Step13_Reconciliation.csv",
                index=False
            )

        else:
            comparison["note"] = (
                "Duplicate join keys detected; "
                "comparison deliberately withheld."
            )

    else:
        comparison["note"] = (
            "No expected Step 12 displacement pair found."
        )

else:
    comparison["note"] = (
        "Step 12 does not contain the expected join keys."
    )

# Always create the output CSV even if comparison is unavailable.
reconciliation_path = (
    OUT / "Step37_Step12_Step13_Reconciliation.csv"
)

if not reconciliation_path.exists():
    pd.DataFrame(
        columns=[
            "image_key",
            "true_dy", "true_dx",
            "original_dy", "original_dx",
            "step12_dy", "step12_dx"
        ]
    ).to_csv(reconciliation_path, index=False)

print(json.dumps(comparison, indent=2))

# -------------------------------------------------------------------------
# 37D — STEP 20 REFERENCE SCHEMA
# -------------------------------------------------------------------------

print("\n37D — SAVED PHASE-CORRELATION RECORDS")

step20 = pd.read_csv(FILES["step20"])

print("Rows:", len(step20))
print("Columns:", step20.columns.tolist())

display_columns = [
    column for column in [
        "partition", "image", "method", "radius",
        "true_dy", "true_dx",
        "raw_dy", "raw_dx",
        "estimated_dy", "estimated_dx",
        "epe"
    ]
    if column in step20.columns
]

print(
    step20[display_columns]
    .head(8)
    .to_string(index=False)
)

# -------------------------------------------------------------------------
# 37E — DEPENDENCIES OF RECOVERED FUNCTIONS
# -------------------------------------------------------------------------

print("\n37E — RECOVERED SOURCE DEPENDENCIES")

# Extract text blocks recorded by Step 36.
# Full source is preserved as evidence; never execute it.

source_text = FILES["recovered_functions"].read_text(
    encoding="utf-8"
)

# Inspect definitions directly from the original notebook,
# using AST where possible, without executing them.
NOTEBOOK = (
    Path("/content/drive/MyDrive/Colab Notebooks")
    / "Sensitivity of Hyperspectral and Multispectral Image .ipynb"
)

notebook = json.loads(
    NOTEBOOK.read_text(encoding="utf-8")
)

TARGETS = {
    15: [
        "integer_shift_np",
        "ncc",
        "score_surface",
        "quadratic_vertex",
        "estimate_both"
    ],
    16: ["gated_displacement"],
    22: ["phase_estimate"],
    24: ["phase_surface"]
}

dependency_rows = []
builtin_names = set(dir(builtins))

for cell_index, names in TARGETS.items():

    cell = notebook["cells"][cell_index]

    code = cell.get("source", "")

    if isinstance(code, list):
        code = "".join(code)

    try:
        module = ast.parse(code)
    except SyntaxError:
        print(
            f"Cell {cell_index}: AST unavailable; "
            "dependency parsing skipped."
        )
        continue

    lines = code.splitlines()

    functions = [
        node for node in ast.walk(module)
        if isinstance(
            node, (ast.FunctionDef, ast.AsyncFunctionDef)
        )
    ]

    for name in names:

        found = [
            node for node in functions
            if node.name == name
        ]

        if not found:
            continue

        node = found[0]

        definition = "\n".join(
            lines[
                node.lineno - 1:node.end_lineno
            ]
        )

        try:
            table = symtable.symtable(
                definition,
                f"cell_{cell_index}_{name}.py",
                "exec"
            )

            children = [
                child for child in table.get_children()
                if child.get_name() == name
            ]

            if not children:
                continue

            scope = children[0]

            globals_used = sorted({
                symbol.get_name()
                for symbol in scope.get_symbols()
                if (
                    symbol.is_global()
                    and symbol.is_referenced()
                    and symbol.get_name() not in builtin_names
                )
            })

        except (SyntaxError, ValueError):
            globals_used = []

        dependency_rows.append({
            "cell_index": cell_index,
            "function_name": name,
            "source_sha256": hashlib.sha256(
                definition.encode("utf-8")
            ).hexdigest(),
            "referenced_global_names": ";".join(globals_used),
            "number_of_global_references": len(globals_used)
        })

dependencies = pd.DataFrame(
    dependency_rows,
    columns=[
        "cell_index",
        "function_name",
        "source_sha256",
        "referenced_global_names",
        "number_of_global_references"
    ]
)

dependencies.to_csv(
    OUT / "Step37_Recovered_Function_Dependencies.csv",
    index=False
)

print(
    dependencies[
        [
            "cell_index",
            "function_name",
            "referenced_global_names"
        ]
    ].to_string(
        index=False,
        max_colwidth=125
    )
)

# -------------------------------------------------------------------------
# 37F — SAVE SCHEMA AND AUDIT REPORT
# -------------------------------------------------------------------------

schema = {
    "step12": {
        "path": str(FILES["step12"]),
        "sha256": sha256(FILES["step12"]),
        "rows": int(len(step12)),
        "columns": step12_columns
    },
    "step13": {
        "path": str(FILES["step13"]),
        "sha256": sha256(FILES["step13"]),
        "rows": int(len(step13)),
        "columns": step13.columns.tolist()
    },
    "step20": {
        "path": str(FILES["step20"]),
        "sha256": sha256(FILES["step20"]),
        "rows": int(len(step20)),
        "columns": step20.columns.tolist()
    }
}

save_json(
    "Step37_Historical_Data_Schema.json",
    schema
)

report = {
    "experiment": (
        "Historical MDAS registration numerical consistency audit"
    ),
    "gate_threshold_pixels": GATE,
    "historical_step13_cases": int(len(step13)),
    "gate_rule_comparisons": candidate_rules,
    "gate_strict_less_than_replay_matches": int(
        replay_match.sum()
    ),
    "gate_strict_less_than_replay_total": int(
        len(replay_match)
    ),
    "step12_step13_comparison": comparison,
    "step20_reference_rows": int(len(step20)),
    "recovered_function_dependencies_inspected": int(
        len(dependencies)
    ),
    "historical_source_functions_verified": 8,
    "exact_ncc_estimator_reexecuted": False,
    "exact_phase_estimator_reexecuted": False,
    "external_potsdam_transfer_completed": False,
    "parameters_modified": False,
    "limitations": [
        "Gate replay applies the documented mathematical rule.",
        "Matching gate outputs does not reproduce NCC estimation.",
        "Step 12/13 reconciliation depends on matching record keys.",
        "Dependencies are statically identified, not executed.",
        "Step 20 phase results are inspected, not reproduced.",
        "Historical MDAS test results were previously "
        "examined during exploratory method development.",
        "External test-scene independence is not established "
        "by replaying historical records."
    ],
    "elapsed_seconds": round(
        time.monotonic() - start, 2
    ),
    "output_directory": str(OUT)
}

save_json("Step37_Final_Report.json", report)

print("\n" + "=" * 88)
print("STEP 37 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 37 COMPLETE")
print("Original notebooks, MDAS records and EnMAP files unchanged.")


In [ ]:

# =============================================================================
# STEP 38 — FROZEN PHASE CORRECTION REPLAY AND NCC SOURCE PREPARATION
#
# Purpose:
#   A. Replay Step 20 displacement bias correction.
#   B. Verify the 0.25-pixel identity-snap rule.
#   C. Compare against all 680 recorded phase estimates.
#   D. Check original EPE and snapped flags.
#   E. Extract the Step 12 estimator source and dependencies.
#
# No external test results used.
# No parameter optimization.
# No registration functions executed.
# No historical records overwritten.
#
# NOTE:
#   This reproduces the PHASE POSTPROCESSING, not yet the
#   original FFT phase-correlation surface itself.
# =============================================================================

from google.colab import drive
from pathlib import Path

import ast
import hashlib
import json
import re
import time

import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

STEP12 = PROJECT / "Step12_Subpixel_Robustness"
STEP20 = PROJECT / "Step20_Bounded_Phase_Correlation"

NOTEBOOK = (
    Path("/content/drive/MyDrive/Colab Notebooks")
    / "Sensitivity of Hyperspectral and Multispectral Image .ipynb"
)

SOURCE_CSV = (
    STEP20 / "Step20_Test_Registration.csv"
)

FROZEN_CONFIG = (
    STEP20 / "Step20_Frozen_Method.json"
)

STEP12_CSV = (
    STEP12 / "Step12_Test_Registration.csv"
)

OUT = PROJECT / "Step38_Phase_Numerical_Reproduction"
OUT.mkdir(parents=True, exist_ok=True)

FILES = [
    "Step38_Phase_Replay.csv",
    "Step38_Phase_Replay_Summary.csv",
    "Step38_Phase_Rule_Comparison.csv",
    "Step38_NCC_Source_Dependencies.txt",
    "Step38_Final_Report.json"
]

existing = [
    name for name in FILES
    if (OUT / name).exists()
]

if existing:
    raise FileExistsError(
        "Step 38 output files already exist. "
        "Previous results will not be overwritten:\n"
        + "\n".join(existing)
    )

print("=" * 88, flush=True)
print("STEP 38 — HISTORICAL PHASE-CORRECTION REPRODUCTION", flush=True)
print("=" * 88, flush=True)

start = time.monotonic()


# =============================================================================
# 38A — SOURCE VERIFICATION
# =============================================================================

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()


def save_json(path, obj):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
            default=str
        ),
        encoding="utf-8"
    )


for path in [
    SOURCE_CSV,
    FROZEN_CONFIG,
    STEP12_CSV,
    NOTEBOOK
]:
    if not path.is_file():
        raise FileNotFoundError(path)

config = json.loads(
    FROZEN_CONFIG.read_text(encoding="utf-8")
)

records = pd.read_csv(SOURCE_CSV)

step12_records = pd.read_csv(STEP12_CSV)

assert len(records) == 680

assert config["method"] == "phase_gradient"

RADIUS = int(config["radius"])
TAU = float(config["tau"])

BIAS_DY = float(config["bias_dy"])
BIAS_DX = float(config["bias_dx"])

assert RADIUS == 5
assert TAU == 0.25

print("\n38A — FROZEN CONFIGURATION")
print("Records:", len(records))
print("Method:", config["method"])
print("Search radius:", RADIUS)
print("Identity threshold:", TAU)
print("Bias dy:", BIAS_DY)
print("Bias dx:", BIAS_DX)
print("Phase output SHA256:", sha256(SOURCE_CSV))


# =============================================================================
# 38B — VERIFY NUMERICAL INPUT SCHEMA
# =============================================================================

required = [
    "image",
    "true_dy",
    "true_dx",
    "raw_dy",
    "raw_dx",
    "estimated_dy",
    "estimated_dx",
    "snapped",
    "epe",
    "radius",
    "method"
]

missing = [
    col for col in required
    if col not in records.columns
]

if missing:
    raise ValueError(
        "Missing columns: " + ", ".join(missing)
    )

numerical = [
    "true_dy", "true_dx",
    "raw_dy", "raw_dx",
    "estimated_dy", "estimated_dx",
    "epe", "radius"
]

for col in numerical:
    records[col] = pd.to_numeric(
        records[col],
        errors="raise"
    )

assert np.isfinite(
    records[numerical].to_numpy(float)
).all()

assert (
    records["radius"] == RADIUS
).all()

assert (
    records["method"].astype(str)
    == "phase_gradient"
).all()

# Robustly decode boolean flags.
flag_text = (
    records["snapped"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if not flag_text.isin(
    ["true", "false", "1", "0"]
).all():
    raise ValueError(
        "Unexpected snapped flag encoding."
    )

recorded_snapped = flag_text.isin(
    ["true", "1"]
).to_numpy(bool)

raw_dy = records["raw_dy"].to_numpy(float)
raw_dx = records["raw_dx"].to_numpy(float)

recorded_dy = (
    records["estimated_dy"].to_numpy(float)
)

recorded_dx = (
    records["estimated_dx"].to_numpy(float)
)

true_dy = records["true_dy"].to_numpy(float)
true_dx = records["true_dx"].to_numpy(float)


# =============================================================================
# 38C — REPLAY THE FROZEN BIAS CORRECTION
#
# Historical mathematical interpretation:
#
#   corrected_dy = raw_dy - bias_dy
#   corrected_dx = raw_dx - bias_dx
#
# The sign is not inferred from performance; it is
# explicitly stated here as the candidate correction
# to verify against saved historical records.
# =============================================================================

corrected_dy = raw_dy - BIAS_DY
corrected_dx = raw_dx - BIAS_DX

corrected_magnitude = np.hypot(
    corrected_dy,
    corrected_dx
)

print("\n38B — BIAS-CORRECTED DISPLACEMENTS")

print(
    "Raw displacement mean dy:",
    round(float(raw_dy.mean()), 6)
)

print(
    "Raw displacement mean dx:",
    round(float(raw_dx.mean()), 6)
)

print(
    "Corrected displacement mean dy:",
    round(float(corrected_dy.mean()), 6)
)

print(
    "Corrected displacement mean dx:",
    round(float(corrected_dx.mean()), 6)
)


# =============================================================================
# 38D — TEST THE TWO DOCUMENTED THRESHOLD BOUNDARIES
#
# Neither rule is selected using test performance.
# They are compared only to identify which mathematical
# convention is consistent with the historical output.
# =============================================================================

candidate_rules = {
    "magnitude_less_than_tau": (
        corrected_magnitude < TAU
    ),
    "magnitude_less_equal_tau": (
        corrected_magnitude <= TAU
    )
}

rule_rows = []
replayed_arrays = {}

ATOL = 1e-8

for name, snap_mask in candidate_rules.items():

    predicted_dy = np.where(
        snap_mask,
        0.0,
        corrected_dy
    )

    predicted_dx = np.where(
        snap_mask,
        0.0,
        corrected_dx
    )

    displacement_match = (
        np.isclose(
            predicted_dy,
            recorded_dy,
            rtol=0,
            atol=ATOL
        )
        &
        np.isclose(
            predicted_dx,
            recorded_dx,
            rtol=0,
            atol=ATOL
        )
    )

    flag_match = (
        snap_mask == recorded_snapped
    )

    predicted_epe = np.hypot(
        predicted_dy - true_dy,
        predicted_dx - true_dx
    )

    epe_match = np.isclose(
        predicted_epe,
        records["epe"].to_numpy(float),
        rtol=0,
        atol=ATOL
    )

    replayed_arrays[name] = {
        "dy": predicted_dy,
        "dx": predicted_dx,
        "snapped": snap_mask,
        "epe": predicted_epe,
        "displacement_match": displacement_match,
        "flag_match": flag_match,
        "epe_match": epe_match
    }

    rule_rows.append({
        "candidate_rule": name,
        "cases": len(records),
        "displacement_matches": int(
            displacement_match.sum()
        ),
        "snapped_flag_matches": int(
            flag_match.sum()
        ),
        "epe_matches": int(
            epe_match.sum()
        ),
        "replayed_snapped_cases": int(
            snap_mask.sum()
        ),
        "maximum_displacement_difference": float(
            np.max(
                np.hypot(
                    predicted_dy - recorded_dy,
                    predicted_dx - recorded_dx
                )
            )
        )
    })

rules_df = pd.DataFrame(rule_rows)

rules_df.to_csv(
    OUT / "Step38_Phase_Rule_Comparison.csv",
    index=False
)

print("\n38C — PHASE CORRECTION RULE REPLAY")
print(
    rules_df.to_string(index=False)
)

# The strict comparison is the primary diagnostic
# convention. If it fails, no alternative is silently
# substituted in the final report.
primary_name = "magnitude_less_than_tau"
primary = replayed_arrays[primary_name]

all_displacements_match = bool(
    primary["displacement_match"].all()
)

all_flags_match = bool(
    primary["flag_match"].all()
)

all_epe_match = bool(
    primary["epe_match"].all()
)

# Determine whether either historical record lies
# exactly on the decision boundary.
boundary_cases = int(
    np.isclose(
        corrected_magnitude,
        TAU,
        rtol=0,
        atol=1e-10
    ).sum()
)

# Full table including discrepancies.
replay = records.copy()

replay["corrected_dy_before_gate"] = corrected_dy
replay["corrected_dx_before_gate"] = corrected_dx

replay["corrected_magnitude"] = corrected_magnitude

replay["replayed_dy"] = primary["dy"]
replay["replayed_dx"] = primary["dx"]
replay["replayed_snapped"] = primary["snapped"]
replay["replayed_epe"] = primary["epe"]

replay["displacement_match"] = (
    primary["displacement_match"]
)

replay["snapped_flag_match"] = (
    primary["flag_match"]
)

replay["epe_match"] = (
    primary["epe_match"]
)

replay.to_csv(
    OUT / "Step38_Phase_Replay.csv",
    index=False
)

if not (
    all_displacements_match
    and all_flags_match
    and all_epe_match
):

    print("\nREPLAY DISCREPANCIES")

    mismatches = replay[
        ~(
            replay["displacement_match"]
            & replay["snapped_flag_match"]
            & replay["epe_match"]
        )
    ]

    print(
        mismatches[
            [
                "image",
                "true_dy",
                "true_dx",
                "raw_dy",
                "raw_dx",
                "estimated_dy",
                "estimated_dx",
                "replayed_dy",
                "replayed_dx",
                "snapped",
                "replayed_snapped"
            ]
        ].head(20).to_string(index=False)
    )


# =============================================================================
# 38E — REPRODUCED ERROR SUMMARY
# =============================================================================

summary_rows = []

for label, epe in [
    (
        "recorded_step20",
        records["epe"].to_numpy(float)
    ),
    (
        "replayed_phase_correction",
        primary["epe"]
    )
]:

    summary_rows.append({
        "evaluation": label,
        "cases": len(epe),
        "mean_epe": float(epe.mean()),
        "median_epe": float(np.median(epe)),
        "maximum_epe": float(epe.max()),
        "within_025": float(
            (epe <= 0.25).mean()
        ),
        "within_050": float(
            (epe <= 0.50).mean()
        ),
        "failure_over_2px": float(
            (epe > 2.0).mean()
        )
    })

summary = pd.DataFrame(summary_rows)

summary.to_csv(
    OUT / "Step38_Phase_Replay_Summary.csv",
    index=False
)

print("\n38D — HISTORICAL ERROR SUMMARY")
print(summary.round(6).to_string(index=False))


# =============================================================================
# 38F — EXTRACT ESSENTIAL ORIGINAL NCC SOURCE CONTEXT
#
# This section documents the source and constants required
# to rerun the historical Step 12 estimator.
#
# It does not execute extracted notebook code.
# =============================================================================

print("\n38E — NCC ESTIMATOR SOURCE PREPARATION")

notebook = json.loads(
    NOTEBOOK.read_text(encoding="utf-8")
)

def code_cell(index):
    cell = notebook["cells"][index]

    assert cell["cell_type"] == "code"

    source = cell.get("source", "")

    return (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

source15 = code_cell(15)
source16 = code_cell(16)

# Capture complete original cell 15, including imports,
# feature extraction and calibration statements.
# Text file is explicitly non-executable as saved here.

extract_text = [
    "ORIGINAL NOTEBOOK SOURCE AUDIT",
    "=" * 80,
    f"Notebook SHA256: {sha256(NOTEBOOK)}",
    "",
    "CELL 15 — STEP 12 ESTIMATOR",
    "=" * 80,
    source15,
    "",
    "CELL 16 — STEP 13 GATE",
    "=" * 80,
    source16
]

(OUT / "Step38_NCC_Source_Dependencies.txt").write_text(
    "\n".join(extract_text),
    encoding="utf-8"
)

# Report top-level assignments in the historical code.
try:
    tree = ast.parse(source15)
except SyntaxError:
    tree = None

relevant_names = {
    "RADIUS",
    "CROP_MARGIN",
    "INTENSITY_WEIGHT",
    "GRADIENT_WEIGHT",
    "BIAS_DY",
    "BIAS_DX"
}

assignment_sources = {}

if tree is not None:

    for node in tree.body:

        if isinstance(node, ast.Assign):
            assigned_names = [
                target.id
                for target in node.targets
                if isinstance(target, ast.Name)
            ]

        elif isinstance(node, ast.AnnAssign):
            assigned_names = (
                [node.target.id]
                if isinstance(node.target, ast.Name)
                else []
            )

        else:
            continue

        for name in assigned_names:

            if name in relevant_names:

                assignment_sources.setdefault(
                    name, []
                ).append(
                    ast.get_source_segment(
                        source15, node
                    )
                )

print("\nRecovered constant assignments:")

for name in sorted(relevant_names):

    values = assignment_sources.get(name, [])

    print(
        f"  {name}:",
        values if values else "NOT FOUND AS TOP-LEVEL ASSIGNMENT"
    )

print("\nStep 12 historical CSV columns:")
print(step12_records.columns.tolist())


# =============================================================================
# 38G — FINAL REPORT
# =============================================================================

if (
    all_displacements_match
    and all_flags_match
    and all_epe_match
):
    status = "PHASE_POSTPROCESSING_REPRODUCED"
else:
    status = "PHASE_POSTPROCESSING_DISCREPANCY"

report = {
    "experiment": (
        "Historical MDAS bounded-phase "
        "postprocessing numerical replay"
    ),
    "status": status,
    "source_step20_sha256": sha256(SOURCE_CSV),
    "source_step20_config_sha256": sha256(FROZEN_CONFIG),
    "source_step12_sha256": sha256(STEP12_CSV),
    "notebook_sha256_at_replay": sha256(NOTEBOOK),
    "historical_phase_cases": len(records),
    "frozen_radius": RADIUS,
    "frozen_tau": TAU,
    "frozen_bias_dy": BIAS_DY,
    "frozen_bias_dx": BIAS_DX,
    "displacement_replay_matches": int(
        primary["displacement_match"].sum()
    ),
    "snapped_flag_replay_matches": int(
        primary["flag_match"].sum()
    ),
    "epe_replay_matches": int(
        primary["epe_match"].sum()
    ),
    "all_phase_postprocessing_matches": bool(
        all_displacements_match
        and all_flags_match
        and all_epe_match
    ),
    "cases_on_tau_boundary": boundary_cases,
    "ncc_constant_assignment_sources": assignment_sources,
    "original_fft_phase_surface_reexecuted": False,
    "original_ncc_estimator_reexecuted": False,
    "external_potsdam_transfer_completed": False,
    "parameter_tuning_performed": False,
    "limitations": [
        "This replays saved phase outputs after raw FFT estimation.",
        "It does not recompute the original FFT correlation surface.",
        "Gate boundary convention may be unidentifiable "
        "without observations exactly at tau.",
        "NCC source is extracted but not executed.",
        "Potsdam data are not used for model selection.",
        "Source and historical results are preserved."
    ],
    "next_requirement": (
        "Reexecute original frozen NCC and bounded-phase "
        "estimators on original MDAS inputs; compare numerical "
        "outputs with Steps 12, 13 and 20 before cross-scene transfer."
    ),
    "elapsed_seconds": round(
        time.monotonic() - start,
        2
    ),
    "output_directory": str(OUT)
}

save_json(
    OUT / "Step38_Final_Report.json",
    report
)

print("\n" + "=" * 88)
print("STEP 38 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 38 COMPLETE")
print("No estimator executed, tuned or modified.")
print("Original research data and notebooks unchanged.")


In [ ]:

# =============================================================================
# STEP 39 — EXACT-SOURCE MDAS REGISTRATION ESTIMATOR REPLAY
#
# Tests:
#   Step 12: original quadratic-subpixel NCC estimator
#   Step 20: original bounded FFT phase-correlation estimator
#
# Procedure:
#   - Verify original notebook function source against Step 36
#   - Verify train-derived projection and frozen configurations
#   - Reconstruct the original 72x72 reference and shifted MSI
#   - Reexecute ORIGINAL functions extracted from notebook code cells
#   - Compare all 680 saved displacement cases per method
#
# Historical functions are compiled individually from AST.
# Complete notebook cells are NEVER executed.
#
# No training, model fitting, threshold tuning, or output overwrites.
# =============================================================================

from google.colab import drive
from pathlib import Path
from scipy.ndimage import gaussian_filter, sobel
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tifffile
import cv2
import ast
import json
import hashlib
import time
import os

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"
REPO = PROJECT / "Step05_Source_Recovery" / "SCSRNet_Recorded_Commit"
DATA = REPO / "datasets" / "mdas_new"

STEP9 = PROJECT / "Step09_Estimated_Registration"
STEP11 = PROJECT / "Step11_Search_Radius_Audit"
STEP12 = PROJECT / "Step12_Subpixel_Robustness"
STEP13 = PROJECT / "Step13_Identity_Preserving_Gate"
STEP20 = PROJECT / "Step20_Bounded_Phase_Correlation"
STEP36 = PROJECT / "Step36_Historical_Algorithm_Verification"

NOTEBOOK = (
    ROOT / "Colab Notebooks"
    / "Sensitivity of Hyperspectral and Multispectral Image .ipynb"
)

OUT = PROJECT / "Step39_Original_Estimator_Numerical_Replay"
OUT.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = [
    "Step39_NCC_Replay.csv",
    "Step39_Phase_Replay.csv",
    "Step39_Method_Summary.csv",
    "Step39_Function_Provenance.csv",
    "Step39_Final_Report.json"
]

if any((OUT / x).exists() for x in OUTPUT_FILES):
    raise FileExistsError(
        "Step 39 outputs already exist. "
        "Preserve the previous run and use a new output directory."
    )

# Floating-point comparisons:
# 1e-5 pixels allows small CPU/GPU numerical differences.
# This is a reproducibility tolerance, not an accuracy criterion.
ATOL = 1e-5

# Stop rather than silently proceed if the first
# 10 cases of either original method do not reproduce.
PILOT_CASES = 10

DEVICE = torch.device(
    "cpu"  # Deterministic, portable replay baseline
)

print("=" * 90, flush=True)
print("STEP 39 — ORIGINAL MDAS ESTIMATOR NUMERICAL REPLAY", flush=True)
print("=" * 90, flush=True)

start = time.monotonic()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()

def sha256_text(value):
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

# -------------------------------------------------------------------------
# 39A — SOURCE AND HISTORICAL CONFIGURATION
# -------------------------------------------------------------------------

required = {
    "notebook": NOTEBOOK,
    "projection": STEP9 / "Step09_Spectral_Projection.npz",
    "ncc_frozen": STEP11 / "Step11_Frozen_Method.json",
    "ncc_historical": STEP12 / "Step12_Test_Registration.csv",
    "phase_frozen": STEP20 / "Step20_Frozen_Method.json",
    "phase_historical": STEP20 / "Step20_Test_Registration.csv",
    "source_hashes": STEP36 / "Step36_Function_Provenance.csv",
    "test_split": DATA / "test.txt"
}

for label, path in required.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label}: {path}")

nb = load_json(NOTEBOOK)
ncc_config = load_json(required["ncc_frozen"])
phase_config = load_json(required["phase_frozen"])

source_hashes = pd.read_csv(
    required["source_hashes"]
)

with np.load(required["projection"], allow_pickle=False) as z:
    scaler_mean = z["scaler_mean"].copy()
    scaler_scale = z["scaler_scale"].copy()
    ridge_coef = z["ridge_coefficients"].copy()
    ridge_intercept = z["ridge_intercept"].copy()

assert scaler_mean.shape == (242,)
assert scaler_scale.shape == (242,)
assert ridge_coef.shape == (4, 242)
assert ridge_intercept.shape == (4,)
assert np.all(scaler_scale > 0)

projection_hash = sha256_file(
    required["projection"]
)

assert projection_hash == ncc_config["projection_sha256"]
assert projection_hash == phase_config["projection_sha256"]

RADIUS = int(ncc_config["selected_radius"])
BIAS_DY = int(ncc_config["bias_dy"])
BIAS_DX = int(ncc_config["bias_dx"])

assert RADIUS == 5
assert (BIAS_DY, BIAS_DX) == (-1, -1)

PHASE_RADIUS = int(phase_config["radius"])
PHASE_TAU = float(phase_config["tau"])
PHASE_BIAS_DY = float(phase_config["bias_dy"])
PHASE_BIAS_DX = float(phase_config["bias_dx"])

assert PHASE_RADIUS == 5
assert PHASE_TAU == 0.25

CROP_MARGIN = 8
BLUR_SIGMA = 1.2
INTENSITY_WEIGHT = 0.7
GRADIENT_WEIGHT = 0.3
NORMALIZER = 15000.0

print("\n39A — FROZEN SETTINGS")
print("Source notebook:", NOTEBOOK.name)
print("Projection SHA256:", projection_hash)
print("NCC radius:", RADIUS)
print("NCC bias:", BIAS_DY, BIAS_DX)
print("NCC intensity/gradient weights:", 0.7, 0.3)
print("Phase radius:", PHASE_RADIUS)
print("Phase tau:", PHASE_TAU)
print("Phase bias:", PHASE_BIAS_DY, PHASE_BIAS_DX)

# -------------------------------------------------------------------------
# 39B — EXTRACT HISTORICAL FUNCTIONS SAFELY
#
# Only top-level function definitions are compiled.
# No complete notebook cells or top-level notebook commands run.
# -------------------------------------------------------------------------

def get_cell(index):
    cell = nb["cells"][index]
    assert cell["cell_type"] == "code"
    s = cell["source"]
    return "".join(s) if isinstance(s, list) else str(s)

def extract_function(cell_index, name):
    text = get_cell(cell_index)
    tree = ast.parse(text)
    lines = text.splitlines()

    nodes = [
        n for n in tree.body
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
        and n.name == name
    ]

    if len(nodes) != 1:
        raise RuntimeError(
            f"Expected exactly one top-level function "
            f"{name} in cell {cell_index}; found {len(nodes)}."
        )

    node = nodes[0]

    if node.decorator_list:
        raise RuntimeError(
            f"Decorated function requires manual review: {name}"
        )

    src = "\n".join(
        lines[node.lineno-1:node.end_lineno]
    )

    return src, node

# Definitions recovered from the original historical cells.
FUNCTIONS = {
    15: [
        "registration_features",
        "integer_shift_np",
        "ncc",
        "score_surface",
        "quadratic_vertex",
        "estimate_both"
    ],
    24: [
        "standardized_gray",
        "feature_map",
        "phase_surface",
        "vertex",
        "select_bounded"
    ]
}

# First verify source hashes for the functions already
# certified in Step 36. Other functions are logged as
# newly inspected dependencies, not previously certified.
verified = {}

for row in source_hashes.itertuples(index=False):
    verified[
        (int(row.cell_index), str(row.function_name))
    ] = str(row.current_source_sha256)

functions = {}
provenance_rows = []

for cell_index, names in FUNCTIONS.items():
    for name in names:
        source, node = extract_function(cell_index, name)
        new_hash = sha256_text(source)
        prior_hash = verified.get((cell_index, name))

        if prior_hash is not None and new_hash != prior_hash:
            raise RuntimeError(
                f"Recovered function changed: cell {cell_index}, "
                f"{name}. Stop before historical replay."
            )

        functions[(cell_index, name)] = node
        provenance_rows.append({
            "cell_index": cell_index,
            "function_name": name,
            "source_sha256": new_hash,
            "matches_step36": (
                True if prior_hash is not None else None
            ),
            "previously_hashed": prior_hash is not None
        })

provenance = pd.DataFrame(provenance_rows)
print("\n39B — RECOVERED FUNCTIONS")
print(provenance.to_string(index=False))

# Compile definitions in two controlled namespaces.
# This executes only function-defining statements.
# The historical functions then operate on our
# explicitly provided, frozen inputs and constants.

def install_functions(cell_index, globals_dict):
    names = FUNCTIONS[cell_index]
    nodes = [functions[(cell_index, name)] for name in names]

    module = ast.Module(
        body=nodes,
        type_ignores=[]
    )
    ast.fix_missing_locations(module)

    code = compile(
        module,
        filename=f"historical_notebook_cell_{cell_index}",
        mode="exec"
    )

    exec(code, globals_dict)

ncc_env = {
    "np": np,
    "gaussian_filter": gaussian_filter,
    "sobel": sobel,
    "CROP_MARGIN": CROP_MARGIN,
    "BLUR_SIGMA": BLUR_SIGMA,
    "RADIUS": RADIUS,
    "BIAS_DY": BIAS_DY,
    "BIAS_DX": BIAS_DX,
    "INTENSITY_WEIGHT": INTENSITY_WEIGHT,
    "GRADIENT_WEIGHT": GRADIENT_WEIGHT
}

phase_env = {
    "np": np,
    "cv2": cv2
}

install_functions(15, ncc_env)
install_functions(24, phase_env)

for name in FUNCTIONS[15]:
    assert callable(ncc_env[name])

for name in FUNCTIONS[24]:
    assert callable(phase_env[name])

# -------------------------------------------------------------------------
# 39C — RECREATE ORIGINAL INPUT PREPROCESSING
# -------------------------------------------------------------------------

def load_image(modality, filename):
    a = tifffile.imread(
        DATA / modality / filename
    ).astype(np.float32) / NORMALIZER

    if not np.isfinite(a).all():
        raise ValueError(f"Nonfinite image: {modality}/{filename}")

    return a

def torch_tensor(array):
    return torch.from_numpy(
        np.ascontiguousarray(array.transpose(2, 0, 1))
    ).unsqueeze(0).to(DEVICE)

def project_hsi(lrhsi):
    # Exact historical interpolation settings.
    with torch.inference_mode():
        up = F.interpolate(
            torch_tensor(lrhsi),
            size=(72, 72),
            mode="bicubic",
            align_corners=False
        )

    flat = (
        up[0].permute(1, 2, 0)
        .cpu().numpy()
        .reshape(-1, 242)
        .astype(np.float64)
    )

    standardized = (
        flat - scaler_mean
    ) / scaler_scale

    predicted = (
        standardized @ ridge_coef.T
        + ridge_intercept
    )

    return predicted.reshape(
        72, 72, 4
    ).astype(np.float32)

def translate_numpy(array, dy, dx):
    # Original MDAS Step 12 / 20 shift convention.
    dy = float(dy)
    dx = float(dx)

    if dy == 0.0 and dx == 0.0:
        return array.copy()

    x = torch_tensor(array)
    n, c, h, w = x.shape

    yy, xx = torch.meshgrid(
        torch.arange(h, device=DEVICE),
        torch.arange(w, device=DEVICE),
        indexing="ij"
    )

    src_y = yy.float() - dy
    src_x = xx.float() - dx

    grid = torch.stack(
        [
            2.0 * src_x / (w - 1) - 1.0,
            2.0 * src_y / (h - 1) - 1.0
        ],
        dim=-1
    ).unsqueeze(0)

    with torch.inference_mode():
        shifted = F.grid_sample(
            x, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )

    return (
        shifted[0].permute(1, 2, 0)
        .cpu().numpy().astype(np.float32)
    )

# -------------------------------------------------------------------------
# 39D — HISTORICAL REFERENCE RECORDS
# -------------------------------------------------------------------------

ncc_ref = pd.read_csv(
    required["ncc_historical"]
)

phase_ref = pd.read_csv(
    required["phase_historical"]
)

assert len(ncc_ref) == 680
assert len(phase_ref) == 680

required_ncc = [
    "image", "true_dy", "true_dx",
    "estimated_dy", "estimated_dx"
]

required_phase = [
    "image", "true_dy", "true_dx",
    "raw_dy", "raw_dx",
    "estimated_dy", "estimated_dx"
]

assert set(required_ncc).issubset(ncc_ref.columns)
assert set(required_phase).issubset(phase_ref.columns)

test_names = [
    line.split()[0]
    for line in required["test_split"]
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]

assert len(test_names) == 40
assert len(set(test_names)) == 40

# This is the frozen Step 12 shift ordering.
OFFSETS = [(0.0, 0.0)]

for d in [0.5, 1.5, 3.5]:
    OFFSETS.extend([
        (d, 0.0), (-d, 0.0),
        (0.0, d), (0.0, -d)
    ])

for dy in (-1.5, 1.5):
    for dx in (-1.5, 1.5):
        OFFSETS.append((dy, dx))

assert len(OFFSETS) == 17

# Canonical keys avoid dependence on CSV row order.
def key_of(image, dy, dx):
    return (
        Path(str(image).replace("\\", "/")).name,
        round(float(dy), 6),
        round(float(dx), 6)
    )

ncc_lookup = {}
phase_lookup = {}

for row in ncc_ref.itertuples(index=False):
    key = key_of(row.image, row.true_dy, row.true_dx)
    if key in ncc_lookup:
        raise RuntimeError(f"Duplicate NCC key: {key}")
    ncc_lookup[key] = row

for row in phase_ref.itertuples(index=False):
    key = key_of(row.image, row.true_dy, row.true_dx)
    if key in phase_lookup:
        raise RuntimeError(f"Duplicate phase key: {key}")
    phase_lookup[key] = row

assert len(ncc_lookup) == 680
assert len(phase_lookup) == 680

# -------------------------------------------------------------------------
# 39E — REPLAY ONE ORIGINAL PAIR
# -------------------------------------------------------------------------

def replay_pair(image, dy, dx):
    lr = load_image("lrhsi", image)
    clean_msi = load_image("hrmsi", image)

    target = project_hsi(lr)
    observed = translate_numpy(clean_msi, dy, dx)

    # Original Step 12 NCC / gradient pipeline.
    target_features = ncc_env["registration_features"](target)

    ncc_output = ncc_env["estimate_both"](
        target_features, observed
    )

    # Original Step 20 bounded FFT pipeline.
    phase_target = phase_env["feature_map"](
        target, "phase_gradient"
    )

    phase_observed = phase_env["feature_map"](
        observed, "phase_gradient"
    )

    surface = phase_env["phase_surface"](
        phase_target,
        phase_observed
    )

    phase_output = phase_env["select_bounded"](
        surface, PHASE_RADIUS
    )

    return ncc_output, phase_output

def replay_and_compare(image, dy, dx):
    key = key_of(image, dy, dx)

    expected_ncc = ncc_lookup[key]
    expected_phase = phase_lookup[key]

    ncc_out, phase_out = replay_pair(image, dy, dx)

    ncc_diff = float(np.hypot(
        float(ncc_out["refined_dy"])
        - float(expected_ncc.estimated_dy),
        float(ncc_out["refined_dx"])
        - float(expected_ncc.estimated_dx)
    ))

    phase_diff = float(np.hypot(
        float(phase_out["raw_dy"])
        - float(expected_phase.raw_dy),
        float(phase_out["raw_dx"])
        - float(expected_phase.raw_dx)
    ))

    ncc_row = {
        "image": image,
        "true_dy": dy,
        "true_dx": dx,
        "recorded_dy": float(expected_ncc.estimated_dy),
        "recorded_dx": float(expected_ncc.estimated_dx),
        "replayed_dy": float(ncc_out["refined_dy"]),
        "replayed_dx": float(ncc_out["refined_dx"]),
        "difference_epe": ncc_diff,
        "match": bool(ncc_diff <= ATOL),
        "historical_source": "Step12"
    }

    phase_row = {
        "image": image,
        "true_dy": dy,
        "true_dx": dx,
        "recorded_raw_dy": float(expected_phase.raw_dy),
        "recorded_raw_dx": float(expected_phase.raw_dx),
        "replayed_raw_dy": float(phase_out["raw_dy"]),
        "replayed_raw_dx": float(phase_out["raw_dx"]),
        "difference_epe": phase_diff,
        "match": bool(phase_diff <= ATOL),
        "historical_source": "Step20"
    }

    return ncc_row, phase_row

# -------------------------------------------------------------------------
# 39F — DIAGNOSTIC REPLAY FIRST
# -------------------------------------------------------------------------

print("\n39C — FIRST-CASE REPLAY", flush=True)

# Spread diagnostic cases across more than one patch
# and across aligned / subpixel / larger displacements.
diagnostic_cases = []

for image in test_names[:5]:
    diagnostic_cases.append((image, 0.0, 0.0))
    diagnostic_cases.append((image, 1.5, 0.0))

diagnostic_cases = diagnostic_cases[:PILOT_CASES]

ncc_rows = []
phase_rows = []

for image, dy, dx in diagnostic_cases:
    nr, pr = replay_and_compare(image, dy, dx)
    ncc_rows.append(nr)
    phase_rows.append(pr)

pilot_ncc = pd.DataFrame(ncc_rows)
pilot_phase = pd.DataFrame(phase_rows)

print("\nNCC diagnostic:")
print(pilot_ncc[
    ["image", "true_dy", "true_dx", "difference_epe", "match"]
].to_string(index=False))

print("\nPhase diagnostic:")
print(pilot_phase[
    ["image", "true_dy", "true_dx", "difference_epe", "match"]
].to_string(index=False))

pilot_passed = bool(
    pilot_ncc["match"].all()
    and pilot_phase["match"].all()
)

if not pilot_passed:
    # Save diagnostic records without claiming full replay.
    pilot_ncc.to_csv(
        OUT / "Step39_NCC_Diagnostic_Mismatch.csv",
        index=False
    )
    pilot_phase.to_csv(
        OUT / "Step39_Phase_Diagnostic_Mismatch.csv",
        index=False
    )

    report = {
        "status": "HISTORICAL_ESTIMATOR_DIAGNOSTIC_MISMATCH",
        "ncc_matches": int(pilot_ncc["match"].sum()),
        "phase_matches": int(pilot_phase["match"].sum()),
        "pilot_cases": len(diagnostic_cases),
        "comparison_tolerance_pixels": ATOL,
        "full_replay_attempted": False,
        "original_parameters_modified": False,
        "note": (
            "Exact-source diagnostic mismatch detected. "
            "Review reference preprocessing, dependencies "
            "and historical numerical environment."
        )
    }

    (OUT / "Step39_Final_Report.json").write_text(
        json.dumps(report, indent=2),
        encoding="utf-8"
    )

    raise RuntimeError(
        "Step 39 diagnostic did not reproduce historical "
        "outputs. Review the saved diagnostic CSVs. "
        "The full 680-case replay has not been run."
    )

print("\nInitial NCC and FFT replay: PASSED", flush=True)

# -------------------------------------------------------------------------
# 39G — COMPLETE 680-CASE REPLAY
# -------------------------------------------------------------------------

print("\n39D — COMPLETE HISTORICAL REPLAY", flush=True)

completed_keys = set(
    key_of(r["image"], r["true_dy"], r["true_dx"])
    for r in ncc_rows
)

total_processed = len(completed_keys)

for image_index, image in enumerate(test_names, 1):

    lr = load_image("lrhsi", image)
    clean_msi = load_image("hrmsi", image)

    target = project_hsi(lr)
    target_features = ncc_env["registration_features"](target)

    phase_target = phase_env["feature_map"](
        target, "phase_gradient"
    )

    for dy, dx in OFFSETS:
        key = key_of(image, dy, dx)

        if key in completed_keys:
            continue

        observed = translate_numpy(clean_msi, dy, dx)

        ncc_out = ncc_env["estimate_both"](
            target_features, observed
        )

        phase_obs = phase_env["feature_map"](
            observed, "phase_gradient"
        )

        phase_out = phase_env["select_bounded"](
            phase_env["phase_surface"](
                phase_target, phase_obs
            ),
            PHASE_RADIUS
        )

        expected_ncc = ncc_lookup[key]
        expected_phase = phase_lookup[key]

        ndiff = float(np.hypot(
            float(ncc_out["refined_dy"])
            - float(expected_ncc.estimated_dy),
            float(ncc_out["refined_dx"])
            - float(expected_ncc.estimated_dx)
        ))

        pdiff = float(np.hypot(
            float(phase_out["raw_dy"])
            - float(expected_phase.raw_dy),
            float(phase_out["raw_dx"])
            - float(expected_phase.raw_dx)
        ))

        ncc_rows.append({
            "image": image,
            "true_dy": dy,
            "true_dx": dx,
            "recorded_dy": float(expected_ncc.estimated_dy),
            "recorded_dx": float(expected_ncc.estimated_dx),
            "replayed_dy": float(ncc_out["refined_dy"]),
            "replayed_dx": float(ncc_out["refined_dx"]),
            "difference_epe": ndiff,
            "match": bool(ndiff <= ATOL),
            "historical_source": "Step12"
        })

        phase_rows.append({
            "image": image,
            "true_dy": dy,
            "true_dx": dx,
            "recorded_raw_dy": float(expected_phase.raw_dy),
            "recorded_raw_dx": float(expected_phase.raw_dx),
            "replayed_raw_dy": float(phase_out["raw_dy"]),
            "replayed_raw_dx": float(phase_out["raw_dx"]),
            "difference_epe": pdiff,
            "match": bool(pdiff <= ATOL),
            "historical_source": "Step20"
        })

        total_processed += 1

    if image_index % 5 == 0:
        print(
            f"  Patches {image_index:02d}/40 | "
            f"Cases {len(ncc_rows):03d}/680 | "
            f"Elapsed {time.monotonic()-start:.1f}s",
            flush=True
        )

ncc_df = pd.DataFrame(ncc_rows)
phase_df = pd.DataFrame(phase_rows)

assert len(ncc_df) == 680
assert len(phase_df) == 680

assert not ncc_df.duplicated(
    ["image", "true_dy", "true_dx"]
).any()

assert not phase_df.duplicated(
    ["image", "true_dy", "true_dx"]
).any()

ncc_df.to_csv(
    OUT / "Step39_NCC_Replay.csv",
    index=False
)

phase_df.to_csv(
    OUT / "Step39_Phase_Replay.csv",
    index=False
)

provenance.to_csv(
    OUT / "Step39_Function_Provenance.csv",
    index=False
)

# -------------------------------------------------------------------------
# 39H — REPRODUCIBILITY SUMMARY
# -------------------------------------------------------------------------

summary = pd.DataFrame([
    {
        "method": "original_step12_ncc",
        "cases": len(ncc_df),
        "matches": int(ncc_df["match"].sum()),
        "max_difference_px": float(
            ncc_df["difference_epe"].max()
        ),
        "mean_difference_px": float(
            ncc_df["difference_epe"].mean()
        )
    },
    {
        "method": "original_step20_phase_raw",
        "cases": len(phase_df),
        "matches": int(phase_df["match"].sum()),
        "max_difference_px": float(
            phase_df["difference_epe"].max()
        ),
        "mean_difference_px": float(
            phase_df["difference_epe"].mean()
        )
    }
])

summary.to_csv(
    OUT / "Step39_Method_Summary.csv",
    index=False
)

print("\n" + "=" * 90)
print("STEP 39 — NUMERICAL REPRODUCTION SUMMARY")
print("=" * 90)
print(summary.to_string(index=False))

all_pass = bool(
    ncc_df["match"].all()
    and phase_df["match"].all()
)

report = {
    "experiment": (
        "Original MDAS registration estimator numerical replay"
    ),
    "status": (
        "ORIGINAL_ESTIMATORS_REPRODUCED"
        if all_pass
        else "HISTORICAL_REPLAY_DISCREPANCIES"
    ),
    "ncc_cases": len(ncc_df),
    "ncc_matches": int(ncc_df["match"].sum()),
    "phase_cases": len(phase_df),
    "phase_matches": int(phase_df["match"].sum()),
    "tolerance_pixels": ATOL,
    "original_notebook_sha256": sha256_file(NOTEBOOK),
    "original_projection_sha256": projection_hash,
    "ncc_source_cell": 15,
    "phase_source_cell": 24,
    "original_ncc_weights": [0.7, 0.3],
    "original_ncc_crop_margin": 8,
    "original_ncc_radius": RADIUS,
    "original_phase_radius": PHASE_RADIUS,
    "phase_raw_estimator_replayed": True,
    "original_parameters_modified": False,
    "external_data_used": False,
    "external_transfer_completed": False,
    "limitations": [
        "Replay is on the already-explored MDAS test mosaic.",
        "Historical reproduction is not independent validation.",
        "Bitwise identity is not required; numerical "
        "agreement uses the declared absolute tolerance.",
        "Phase postprocessing was independently checked "
        "in Step 38; this step tests the raw FFT estimator."
    ],
    "elapsed_seconds": round(
        time.monotonic()-start, 2
    ),
    "output_directory": str(OUT)
}

(OUT / "Step39_Final_Report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 90)
print("STEP 39 — FINAL REPORT")
print("=" * 90)
print(json.dumps(report, indent=2))
print("\nSTEP 39 COMPLETE")
print("Historical MDAS files and Potsdam benchmark unchanged.")
